In [ ]:
import cv2
import os
import time
import numpy as np
import tqdm
import torch
from ultralytics import YOLO

# =========================================================
# ⚙️ 하이퍼파라미터 (Hyperparameters) - Dynamic Upscale
# =========================================================
MODEL_FILTER_PATH = 'model/best_nano.pt'
MODEL_MAIN_PATH = 'model/best_small.pt'

CONF_GLOBAL = 0.3
CONF_FILTER = 0.1     
CONF_DENSE = 0.3
CONF_TETRIS = 0.3
CONF_UC = 0.3

IOU_FILTER_MATCH = 0.1
NMS_CONF_THRESH = 0.3
NMS_IOU_THRESH = 0.4    

# 💡 1차 필터가 찾은 모든 객체를 캔버스로 보냄
FILTER_MAX_SIZE = 9999 

# 💡 밀집 구역은 그대로 유지
DENSE_WINDOW_SIZE = 512
DENSE_STEP = 320      

MERGE_PAD = 16
CROP_PAD_LARGE = 80     
CROP_PAD_SMALL = 16
CROP_PAD_THRESH = 200

CANVAS_SIZE = 960
CANVAS_MARGIN = 2
CANVAS_BG_COLOR = 114

# 💡 [NEW] 업스케일 파라미터
UPSCALE_RATIO = 1.5         # 소형 객체를 몇 배 확대할 것인가?
UPSCALE_MAX_THRESH = 200    # 이 픽셀 이하의 객체만 확대함 (너무 큰 객체 확대 방지)

NUM_TEST_IMAGES = 500
HR_THRESHOLD = 1920 * 1080

# =========================================================
dataset_root = 'data/test'
img_dir, lbl_dir = os.path.join(dataset_root, 'images'), os.path.join(dataset_root, 'labels')
img_list = sorted(os.listdir(img_dir))[:NUM_TEST_IMAGES]

print(f"🚀 [DOU 최적화] Dynamic Upscale + 1.10 GCC 벤치마크 시작! (Total {len(img_list)} images)")

m1 = YOLO(MODEL_FILTER_PATH)
m2 = YOLO(MODEL_MAIN_PATH)

def calculate_iou(box1, box2):
    xi1, yi1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    xi2, yi2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])

def get_size_category(w, h):
    area = w * h
    if area < 32 ** 2: return 'small'
    elif area < 96 ** 2: return 'medium'
    else: return 'large'

def merge_clusters_dynamic(boxes, img_w, img_h, merge_pad=MERGE_PAD):
    if not len(boxes): return []
    def get_padded(b, pad): return [max(0, b[0]-pad), max(0, b[1]-pad), min(img_w, b[2]+pad), min(img_h, b[3]+pad)]
    def is_overlap(b1, b2):
        p1, p2 = get_padded(b1, merge_pad), get_padded(b2, merge_pad)
        return (min(p1[2], p2[2]) > max(p1[0], p2[0])) and (min(p1[3], p2[3]) > max(p1[1], p2[1]))
    curr = boxes.copy()
    while True:
        merged, flags = [], [False]*len(curr)
        for i in range(len(curr)):
            if flags[i]: continue
            b = curr[i]
            for j in range(i+1, len(curr)):
                if not flags[j] and is_overlap(b, curr[j]):
                    b = [min(b[0], curr[j][0]), min(b[1], curr[j][1]), max(b[2], curr[j][2]), max(b[3], curr[j][3])]
                    flags[j] = True
            merged.append(b)
        if len(merged) == len(curr): break
        curr = merged
    final_boxes = []
    for b in curr:
        bw, bh = b[2] - b[0], b[3] - b[1]
        crop_pad = CROP_PAD_LARGE if max(bw, bh) < CROP_PAD_THRESH else CROP_PAD_SMALL 
        final_boxes.append(get_padded(b, crop_pad))
    return final_boxes

def run_ultimate_benchmark(method_name):
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        
    all_gts = {}; all_preds = []
    
    stats = {
        'ALL': {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'HR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'LR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()}
    }

    pbar = tqdm.tqdm(img_list, desc=f"⏳ {method_name} 추론 중", bar_format='{l_bar}{bar:30}{r_bar}')
    for img_idx, img_name in enumerate(pbar):
        img_path, lbl_path = os.path.join(img_dir, img_name), os.path.join(lbl_dir, img_name.replace('.jpg', '.txt'))
        img = cv2.imread(img_path); h, w, _ = img.shape
        
        gts = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    c, xc, yc, bw, bh = map(float, line.split())
                    bw_pix, bh_pix = bw * w, bh * h
                    gts.append([int(c), (xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h, False, get_size_category(bw_pix, bh_pix)])
        all_gts[img_idx] = gts

        t_pipe_start = time.time()
        img_inf_time, img_inf_cnt = 0, 0
        
        # ---------------------------------------------------------
        # [비교군] UC (2x2 Uniform Crop)
        # ---------------------------------------------------------
        if method_name == "UC (2x2 Uniform Crop)":
            ch, cw = h // 2, w // 2
            # crops, offsets = [img], [(0, 0)]
            # for y in [0, ch]:
            #     for x in [0, cw]:
            #         crops.append(img[y:y+ch, x:x+cw])
            #         offsets.append((x, y))
            
            # t_inf_start = time.time()
            # results2 = m2.predict(crops, conf=CONF_UC, verbose=False, batch=5)
            # img_inf_time += (time.time() - t_inf_start)
            # img_inf_cnt += 5 
            
            # temp_boxes, temp_scores, temp_classes = [], [], []
            # for i, res in enumerate(results2):
            #     ox, oy = offsets[i]
            #     for b in res.boxes:
            #         temp_boxes.append([b.xyxy[0][0]+ox, b.xyxy[0][1]+oy, b.xyxy[0][2]+ox, b.xyxy[0][3]+oy])
            #         temp_scores.append(float(b.conf[0])); temp_classes.append(int(b.cls[0]))
                    
            # for c in set(temp_classes):
            #     c_boxes = [b for j, b in enumerate(temp_boxes) if temp_classes[j] == c]
            #     c_scores = [s for j, s in enumerate(temp_scores) if temp_classes[j] == c]
            #     cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
            #     indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
            #     if len(indices) > 0:
            #         for idx in indices.flatten():
            #             all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])
                        
        # ---------------------------------------------------------
        # ⭐️ [제안군] Ours (DOU + Hierarchical NMS + GCC + UBI)
        # ---------------------------------------------------------
        elif method_name == "Ours (Boundary-Aware Tetris Packing)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=CONF_GLOBAL, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = min(1.0, float(b.conf[0]) * 1.10) # GCC 1.10
                global_final_boxes.append([bx1, by1, bx2, by2])
                global_final_scores.append(conf)
                global_final_classes.append(int(b.cls[0]))

            t_inf_start = time.time()
            r1 = m1.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                # 💡 중복검사 삭제. 무조건 다 잡아서 캔버스로 넘김 (필터 Max Size 9999)
                roi_boxes.append([bx1, by1, bx2, by2])
            
            best_region, dense_boxes, remaining_boxes = None, [], []
            if len(roi_boxes) > 0:
                best_count = -1
                for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                        count = sum(1 for rb in roi_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                        if count > best_count: best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                
                if best_region and best_count > 0:
                    dx1, dy1, dx2, dy2 = best_region
                    for rb in roi_boxes:
                        if rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2: dense_boxes.append(rb)
                        else: remaining_boxes.append(rb)
                else: remaining_boxes = roi_boxes

            unified_infer_list = []
            dense_idx, canvas_start_idx = -1, -1
            cw_dense, ch_dense, d_dx1, d_dy1 = 0, 0, 0, 0

            if best_region and len(dense_boxes) > 0:
                d_dx1, d_dy1, dx2, dy2 = best_region
                cw_dense, ch_dense = dx2 - d_dx1, dy2 - d_dy1
                unified_infer_list.append(img[d_dy1:dy2, d_dx1:dx2])
                dense_idx = 0

            canvases, canvas_infos = [], []
            if len(remaining_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(remaining_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb)
                    cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    
                    # 💡 [NEW] Dynamic Object Upscaling (동적 돋보기)
                    # 조각이 작다면 확대해서 캔버스에 담는다!
                    scale_ratio = 1.0
                    if max(cw_org, ch_org) < UPSCALE_MAX_THRESH:
                        scale_ratio = UPSCALE_RATIO
                    
                    # 확대된 크기 계산
                    cw_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE)
                    ch_crop = min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        
                        # 확대 수행
                        if scale_ratio > 1.0:
                            crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else:
                            # 확대를 안해도 너무 큰 박스는 캔버스 사이즈에 맞게 자름
                            crop_img = crop_img[:ch_crop, :cw_crop]
                            
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: 
                        canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    
                    # 💡 스케일 정보도 같이 저장
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # 💡 SCE (Seamless Context Expansion) - 확대된 스케일에 맞춰 문맥 팽창
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            
                            # 원본 기준 박스 크기
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                # 빈 공간을 원본 스케일로 환산하여 자르고 다시 확대!
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                                    
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

                if len(canvases) > 0:
                    canvas_start_idx = len(unified_infer_list)
                    unified_infer_list.extend(canvases)

            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                if dense_idx != -1:
                    res_dense = res_all[dense_idx]
                    for b in res_dense.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                        conf = float(b.conf[0])
                        # BCP
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+d_dx1, by1+d_dy1, bx2+d_dx1, by2+d_dy1])
                        local_scores.append(conf); local_classes.append(int(b.cls[0]))
                
                if canvas_start_idx != -1:
                    res_pack = res_all[canvas_start_idx:]
                    for c_idx, res in enumerate(res_pack):
                        for b in res.boxes:
                            bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                            conf = float(b.conf[0])
                            bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                            for info in canvas_infos:
                                if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                    if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                    
                                    # 💡 [NEW] 역산 시 스케일(scale) 복원
                                    s = info['scale']
                                    orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']
                                    orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                    orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']
                                    orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                    
                                    local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2])
                                    local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                    break
                                        
            # 계층적 NMS
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]
                c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]
            combined_scores = global_final_scores + [p[1] for p in final_local_preds]
            combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]
                c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # 통계 저장
        img_total_time = time.time() - t_pipe_start
        is_hr = (w * h >= HR_THRESHOLD)
        target_keys = ['ALL', 'HR'] if is_hr else ['ALL', 'LR']
        for k in target_keys:
            stats[k]['count'] += 1
            stats[k]['inf_time'] += img_inf_time
            stats[k]['total_time'] += img_total_time
            stats[k]['inf_cnt'] += img_inf_cnt
            stats[k]['indices'].add(img_idx)

    # ---------------------------------------------------------
    # 💡 AP 연산 엔진
    # ---------------------------------------------------------
    def calc_metrics_for_subset(subset_indices):
        if not subset_indices: return {"AP50": 0, "AP50s": 0, "AP50m": 0, "AP50l": 0}
        
        sub_preds = [p for p in all_preds if p[0] in subset_indices]
        sub_preds.sort(key=lambda x: x[2], reverse=True) 
        unique_classes = set([gt[0] for idx in subset_indices for gt in all_gts[idx]])
        metrics = {'all': [], 'small': [], 'medium': [], 'large': []}
        
        for c in unique_classes:
            c_preds = [p for p in sub_preds if p[1] == c]
            for size_target in ['all', 'small', 'medium', 'large']:
                if size_target == 'all': c_gts = {idx: [list(g) for g in all_gts[idx] if g[0] == c] for idx in subset_indices}
                else: c_gts = {idx: [list(g) for g in all_gts[idx] if g[0] == c and g[6] == size_target] for idx in subset_indices}
                    
                npos = sum(len(gts) for gts in c_gts.values())
                if npos == 0: continue
                
                tp, fp = np.zeros(len(c_preds)), np.zeros(len(c_preds))
                for i, pred in enumerate(c_preds):
                    img_idx, _, _, px1, py1, px2, py2 = pred
                    pred_box = [px1, py1, px2, py2]
                    gts = c_gts[img_idx]
                    
                    pw, ph = px2 - px1, py2 - py1
                    p_size = get_size_category(pw, ph)
                    if size_target != 'all' and p_size != size_target: continue
                    
                    best_iou, best_idx = 0.5, -1
                    for j, gt in enumerate(gts):
                        if gt[5]: continue 
                        iou = calculate_iou(pred_box, gt[1:5])
                        if iou >= best_iou: best_iou, best_idx = iou, j
                            
                    if best_idx >= 0:
                        tp[i] = 1; gts[best_idx][5] = True
                    else:
                        fp[i] = 1
                        
                fp_cumsum, tp_cumsum = np.cumsum(fp), np.cumsum(tp)
                rec = tp_cumsum / npos
                prec = tp_cumsum / np.maximum(tp_cumsum + fp_cumsum, np.finfo(np.float64).eps)
                metrics[size_target].append(compute_ap(rec, prec))
                
        return {
            "AP50": np.mean(metrics['all']) if metrics['all'] else 0,
            "AP50s": np.mean(metrics['small']) if metrics['small'] else 0,
            "AP50m": np.mean(metrics['medium']) if metrics['medium'] else 0,
            "AP50l": np.mean(metrics['large']) if metrics['large'] else 0,
        }

    result_dict = {}
    for group in ['ALL', 'HR', 'LR']:
        c = stats[group]['count']
        res = calc_metrics_for_subset(stats[group]['indices'])
        res['Img_Cnt'] = c
        res['Avg_Inf_Cnt'] = stats[group]['inf_cnt'] / c if c else 0
        res['Avg_Inf_Time'] = (stats[group]['inf_time'] / c) * 1000 if c else 0
        res['Avg_Tot_Time'] = (stats[group]['total_time'] / c) * 1000 if c else 0
        result_dict[group] = res
        
    result_dict['Peak_VRAM'] = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0
    return result_dict

# =========================================================
# 실행 및 표 그리기
# =========================================================
methods = ["UC (2x2 Uniform Crop)", "Ours (Boundary-Aware Tetris Packing)"]
final_stats = {}
for m in methods: 
    final_stats[m] = run_ultimate_benchmark(m)

print("\n" + "="*140)
print(f"🏆 [최종 진화형 DOU 적용] COCO-Style Results (Total {NUM_TEST_IMAGES} Images) 🏆")
print("="*140)
print(f"{'Method':<35} | {'Type':<4} | {'Img':<4} | {'AP50':<6} | {'AP50s':<6} | {'AP50m':<6} | {'AP50l':<6} | {'Inf Cnt':<7} | {'Inf Time':<9} | {'Tot Time':<9}")
print("-" * 140)

for m, groups in final_stats.items():
    for g in ['ALL', 'HR', 'LR']:
        s = groups[g]
        if s['Img_Cnt'] == 0: continue
        print(f"{m if g == 'ALL' else '':<35} | {g:<4} | {s['Img_Cnt']:<4} | {s['AP50']:.4f} | {s['AP50s']:.4f} | {s['AP50m']:.4f} | {s['AP50l']:.4f} | {s['Avg_Inf_Cnt']:4.1f} /i | {s['Avg_Inf_Time']:5.1f} ms | {s['Avg_Tot_Time']:5.1f} ms")
    print(f"{'':<35} > Peak VRAM: {groups['Peak_VRAM']:.1f} MB")
    print("-" * 140)

🚀 [DOU 최적화] Dynamic Upscale + 1.10 GCC 벤치마크 시작! (Total 430 images)


⏳ Ours (Boundary-Aware Tetris Packing) 추론 중: 100%|██████████████████████████████| 430/430 [00:35<00:00, 12.27it/s]



🏆 [최종 진화형 DOU 적용] COCO-Style Results (Total 500 Images) 🏆
Method                              | Type | Img  | AP50   | AP50s  | AP50m  | AP50l  | Inf Cnt | Inf Time  | Tot Time 
--------------------------------------------------------------------------------------------------------------------------------------------
UC (2x2 Uniform Crop)               | ALL  | 430  | 0.0000 | 0.0000 | 0.0000 | 0.0000 |  0.0 /i |   0.0 ms |   0.0 ms
                                    | HR   | 54   | 0.0000 | 0.0000 | 0.0000 | 0.0000 |  0.0 /i |   0.0 ms |   0.0 ms
                                    | LR   | 376  | 0.0000 | 0.0000 | 0.0000 | 0.0000 |  0.0 /i |   0.0 ms |   0.0 ms
                                    > Peak VRAM: 54.2 MB
--------------------------------------------------------------------------------------------------------------------------------------------
Ours (Boundary-Aware Tetris Packing) | ALL  | 430  | 0.4211 | 0.2667 | 0.5085 | 0.5546 |  4.4 /i |  47.3 ms |  68.7 ms
         

In [ ]:
import cv2
import os
import time
import numpy as np
import tqdm
import torch
from ultralytics import YOLO

# =========================================================
# ⚙️ 하이퍼파라미터 (Hyperparameters)
# =========================================================
MODEL_FILTER_PATH = 'model/best_nano.pt'
MODEL_MAIN_PATH = 'model/best_small.pt'

CONF_GLOBAL = 0.3
CONF_FILTER = 0.1     
CONF_DENSE = 0.3
CONF_TETRIS = 0.3
CONF_UC = 0.3

IOU_FILTER_MATCH = 0.1
NMS_CONF_THRESH = 0.3
NMS_IOU_THRESH = 0.4    

DENSE_WINDOW_SIZE = 512
DENSE_STEP = 320      
DENSE_MIN_COUNT = 3         # 완전체 모델에서 DAHI가 작동할 최소 객체 수

MERGE_PAD = 16
CROP_PAD_LARGE = 80     
CROP_PAD_SMALL = 16
CROP_PAD_THRESH = 200

CANVAS_SIZE = 960
CANVAS_MARGIN = 2
CANVAS_BG_COLOR = 114

UPSCALE_RATIO = 1.5       
UPSCALE_MAX_THRESH = 200    

NUM_TEST_IMAGES = 500
HR_THRESHOLD = 1920 * 1080 

# =========================================================
dataset_root = 'data/test'
img_dir, lbl_dir = os.path.join(dataset_root, 'images'), os.path.join(dataset_root, 'labels')
img_list = sorted(os.listdir(img_dir))[:NUM_TEST_IMAGES]

print(f"🚀 [Ablation Study] 가독성 최적화 버전 벤치마크 시작! (Total {len(img_list)} images)")

m1 = YOLO(MODEL_FILTER_PATH)
m2 = YOLO(MODEL_MAIN_PATH)

def calculate_iou(box1, box2):
    xi1, yi1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    xi2, yi2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])

def get_size_category(w, h):
    area = w * h
    if area < 32 ** 2: return 'small'
    elif area < 96 ** 2: return 'medium'
    else: return 'large'

def merge_clusters_dynamic(boxes, img_w, img_h, merge_pad=MERGE_PAD):
    if not len(boxes): return []
    def get_padded(b, pad): return [max(0, b[0]-pad), max(0, b[1]-pad), min(img_w, b[2]+pad), min(img_h, b[3]+pad)]
    def is_overlap(b1, b2):
        p1, p2 = get_padded(b1, merge_pad), get_padded(b2, merge_pad)
        return (min(p1[2], p2[2]) > max(p1[0], p2[0])) and (min(p1[3], p2[3]) > max(p1[1], p2[1]))
    curr = boxes.copy()
    while True:
        merged, flags = [], [False]*len(curr)
        for i in range(len(curr)):
            if flags[i]: continue
            b = curr[i]
            for j in range(i+1, len(curr)):
                if not flags[j] and is_overlap(b, curr[j]):
                    b = [min(b[0], curr[j][0]), min(b[1], curr[j][1]), max(b[2], curr[j][2]), max(b[3], curr[j][3])]
                    flags[j] = True
            merged.append(b)
        if len(merged) == len(curr): break
        curr = merged
    final_boxes = []
    for b in curr:
        bw, bh = b[2] - b[0], b[3] - b[1]
        crop_pad = CROP_PAD_LARGE if max(bw, bh) < CROP_PAD_THRESH else CROP_PAD_SMALL 
        final_boxes.append(get_padded(b, crop_pad))
    return final_boxes

def run_ablation_benchmark(method_name):
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        
    all_gts = {}; all_preds = []
    stats = {
        'ALL': {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'HR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'LR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()}
    }

    pbar = tqdm.tqdm(img_list, desc=f"⏳ {method_name}", bar_format='{l_bar}{bar:30}{r_bar}')
    for img_idx, img_name in enumerate(pbar):
        img_path, lbl_path = os.path.join(img_dir, img_name), os.path.join(lbl_dir, img_name.replace('.jpg', '.txt'))
        img = cv2.imread(img_path); h, w, _ = img.shape
        
        gts = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    c, xc, yc, bw, bh = map(float, line.split())
                    bw_pix, bh_pix = bw * w, bh * h
                    gts.append([int(c), (xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h, False, get_size_category(bw_pix, bh_pix)])
        all_gts[img_idx] = gts

        t_pipe_start = time.time()
        img_inf_time, img_inf_cnt = 0, 0
        
        # =====================================================================
        # 1. Baseline: UC (2x2 Uniform Crop)
        # =====================================================================
        if method_name == "UC (2x2 Uniform Crop)":
            ch, cw = h // 2, w // 2
            # crops, offsets = [img], [(0, 0)]
            # for y in [0, ch]:
            #     for x in [0, cw]:
            #         crops.append(img[y:y+ch, x:x+cw])
            #         offsets.append((x, y))
            
            # t_inf_start = time.time()
            # results2 = m2.predict(crops, conf=CONF_UC, verbose=False, batch=5)
            # img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 5 
            
            # temp_boxes, temp_scores, temp_classes = [], [], []
            # for i, res in enumerate(results2):
            #     ox, oy = offsets[i]
            #     for b in res.boxes:
            #         temp_boxes.append([b.xyxy[0][0]+ox, b.xyxy[0][1]+oy, b.xyxy[0][2]+ox, b.xyxy[0][3]+oy])
            #         temp_scores.append(float(b.conf[0])); temp_classes.append(int(b.cls[0]))
                    
            # for c in set(temp_classes):
            #     c_boxes = [b for j, b in enumerate(temp_boxes) if temp_classes[j] == c]
            #     c_scores = [s for j, s in enumerate(temp_scores) if temp_classes[j] == c]
            #     cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
            #     indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
            #     if len(indices) > 0:
            #         for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 2. 제안 1: Ours (DAHI Only)
        # =====================================================================
        elif method_name == "Ours (DAHI Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            # Global
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=CONF_GLOBAL, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            # Filter
            t_inf_start = time.time()
            r1 = m1.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                is_found = False
                for gb in global_final_boxes:
                    if calculate_iou([bx1, by1, bx2, by2], gb) > IOU_FILTER_MATCH: is_found = True; break
                if not is_found: roi_boxes.append([bx1, by1, bx2, by2])
            
            # DAHI 로직 (모든 객체가 커버될 때까지 반복)
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            while len(remaining_boxes) > 0:
                best_count, best_region = -1, None
                for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                        count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                        if count > best_count: 
                            best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                if best_region and best_count >= 1:
                    dense_regions.append(best_region)
                    dx1, dy1, dx2, dy2 = best_region
                    remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                else: break
            
            # 통합 추론 (DAHI 윈도우들)
            unified_infer_list = [img[dy1:dy2, dx1:dx2] for dx1, dy1, dx2, dy2 in dense_regions]
            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for idx, (dx1, dy1, dx2, dy2) in enumerate(dense_regions):
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1
                    for b in res_all[idx].boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                        conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
            
            # 계층적 NMS
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 3. 제안 2: Ours (Tetris Only)
        # =====================================================================
        elif method_name == "Ours (Tetris Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            # Global
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=CONF_GLOBAL, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            # Filter
            t_inf_start = time.time()
            r1 = m1.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                is_found = False
                for gb in global_final_boxes:
                    if calculate_iou([bx1, by1, bx2, by2], gb) > IOU_FILTER_MATCH: is_found = True; break
                if not is_found: roi_boxes.append([bx1, by1, bx2, by2])
            
            # Tetris 로직 (모든 roi_boxes를 캔버스로)
            canvases, canvas_infos = [], []
            if len(roi_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(roi_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE (Seamless Context Expansion)
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

            # 통합 추론 (Tetris 캔버스들)
            if len(canvases) > 0:
                t_inf_start = time.time()
                res_pack = m2.predict(canvases, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(canvases)
                
                for c_idx, res in enumerate(res_pack):
                    for b in res.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                        for info in canvas_infos:
                            if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                s = info['scale']
                                orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                break
                                        
            # 계층적 NMS
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 4. 제안 3: Ours (DAHI + Tetris) 완전체 모델
        # =====================================================================
        elif method_name == "Ours (DAHI + Tetris)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            # Global
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=CONF_GLOBAL, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            # Filter
            t_inf_start = time.time()
            r1 = m1.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                is_found = False
                for gb in global_final_boxes:
                    if calculate_iou([bx1, by1, bx2, by2], gb) > IOU_FILTER_MATCH: is_found = True; break
                if not is_found: roi_boxes.append([bx1, by1, bx2, by2])
            
            # DAHI 로직 (밀집 구역 추출, 임계값: DENSE_MIN_COUNT)
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            best_count, best_region = -1, None
            for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                    if count > best_count: 
                        best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                        
            if best_region and best_count > 0: # 1개라도 있으면 추출
                dense_regions.append(best_region)
                dx1, dy1, dx2, dy2 = best_region
                remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
            
            unified_infer_list = []
            dense_idx_list = []
            
            for dx1, dy1, dx2, dy2 in dense_regions:
                unified_infer_list.append(img[dy1:dy2, dx1:dx2])
                dense_idx_list.append((len(unified_infer_list) - 1, dx1, dy1, dx2, dy2))

            # Tetris 로직 (남은 찌꺼기 팩킹)
            canvases, canvas_infos = [], []
            canvas_start_idx = -1
            if len(remaining_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(remaining_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

                if len(canvases) > 0:
                    canvas_start_idx = len(unified_infer_list)
                    unified_infer_list.extend(canvases)

            # 통합 추론 (DAHI + Tetris)
            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for d_idx, dx1, dy1, dx2, dy2 in dense_idx_list:
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1; res_dense = res_all[d_idx]
                    for b in res_dense.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                
                if canvas_start_idx != -1:
                    res_pack = res_all[canvas_start_idx:]
                    for c_idx, res in enumerate(res_pack):
                        for b in res.boxes:
                            bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                            bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                            for info in canvas_infos:
                                if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                    if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                    s = info['scale']
                                    orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                    orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                    local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                    break
                                        
            # 계층적 NMS
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # 통계 저장
        img_total_time = time.time() - t_pipe_start
        is_hr = (w * h >= HR_THRESHOLD)
        target_keys = ['ALL', 'HR'] if is_hr else ['ALL', 'LR']
        for k in target_keys:
            stats[k]['count'] += 1
            stats[k]['inf_time'] += img_inf_time
            stats[k]['total_time'] += img_total_time
            stats[k]['inf_cnt'] += img_inf_cnt
            stats[k]['indices'].add(img_idx)

    # ---------------------------------------------------------
    # 💡 AP 연산 엔진
    # ---------------------------------------------------------
    def calc_metrics_for_subset(subset_indices):
        if not subset_indices: return {"AP50": 0, "AP50s": 0, "AP50m": 0, "AP50l": 0}
        
        sub_preds = [p for p in all_preds if p[0] in subset_indices]
        sub_preds.sort(key=lambda x: x[2], reverse=True) 
        unique_classes = set([gt[0] for idx in subset_indices for gt in all_gts[idx]])
        metrics = {'all': [], 'small': [], 'medium': [], 'large': []}
        
        for c in unique_classes:
            c_preds = [p for p in sub_preds if p[1] == c]
            for size_target in ['all', 'small', 'medium', 'large']:
                if size_target == 'all': c_gts = {idx: [list(g) for g in all_gts[idx] if g[0] == c] for idx in subset_indices}
                else: c_gts = {idx: [list(g) for g in all_gts[idx] if g[0] == c and g[6] == size_target] for idx in subset_indices}
                    
                npos = sum(len(gts) for gts in c_gts.values())
                if npos == 0: continue
                
                tp, fp = np.zeros(len(c_preds)), np.zeros(len(c_preds))
                for i, pred in enumerate(c_preds):
                    img_idx, _, _, px1, py1, px2, py2 = pred
                    pred_box = [px1, py1, px2, py2]
                    gts = c_gts[img_idx]
                    
                    pw, ph = px2 - px1, py2 - py1
                    p_size = get_size_category(pw, ph)
                    if size_target != 'all' and p_size != size_target: continue
                    
                    best_iou, best_idx = 0.5, -1
                    for j, gt in enumerate(gts):
                        if gt[5]: continue 
                        iou = calculate_iou(pred_box, gt[1:5])
                        if iou >= best_iou: best_iou, best_idx = iou, j
                            
                    if best_idx >= 0:
                        tp[i] = 1; gts[best_idx][5] = True
                    else:
                        fp[i] = 1
                        
                fp_cumsum, tp_cumsum = np.cumsum(fp), np.cumsum(tp)
                rec = tp_cumsum / npos
                prec = tp_cumsum / np.maximum(tp_cumsum + fp_cumsum, np.finfo(np.float64).eps)
                metrics[size_target].append(compute_ap(rec, prec))
                
        return {
            "AP50": np.mean(metrics['all']) if metrics['all'] else 0,
            "AP50s": np.mean(metrics['small']) if metrics['small'] else 0,
            "AP50m": np.mean(metrics['medium']) if metrics['medium'] else 0,
            "AP50l": np.mean(metrics['large']) if metrics['large'] else 0,
        }

    result_dict = {}
    for group in ['ALL', 'HR', 'LR']:
        c = stats[group]['count']
        res = calc_metrics_for_subset(stats[group]['indices'])
        res['Img_Cnt'] = c
        res['Avg_Inf_Cnt'] = stats[group]['inf_cnt'] / c if c else 0
        res['Avg_Inf_Time'] = (stats[group]['inf_time'] / c) * 1000 if c else 0
        res['Avg_Tot_Time'] = (stats[group]['total_time'] / c) * 1000 if c else 0
        result_dict[group] = res
        
    result_dict['Peak_VRAM'] = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0
    return result_dict

# =========================================================
# 실행 및 다중 표 그리기
# =========================================================
methods = [
    "UC (2x2 Uniform Crop)", 
    "Ours (DAHI Only)",
    "Ours (Tetris Only)",
    "Ours (DAHI + Tetris)"
]

final_stats = {}
for m in methods: 
    final_stats[m] = run_ablation_benchmark(m)

print("\n" + "="*140)
print(f"🏆 [Ablation Study] Component Contribution Analysis (Total {NUM_TEST_IMAGES} Images) 🏆")
print("="*140)
print(f"{'Method':<35} | {'Type':<4} | {'Img':<4} | {'AP50':<6} | {'AP50s':<6} | {'AP50m':<6} | {'AP50l':<6} | {'Inf Cnt':<7} | {'Inf Time':<9} | {'Tot Time':<9}")
print("-" * 140)

for m, groups in final_stats.items():
    for g in ['ALL', 'HR', 'LR']:
        s = groups[g]
        if s['Img_Cnt'] == 0: continue
        print(f"{m if g == 'ALL' else '':<35} | {g:<4} | {s['Img_Cnt']:<4} | {s['AP50']:.4f} | {s['AP50s']:.4f} | {s['AP50m']:.4f} | {s['AP50l']:.4f} | {s['Avg_Inf_Cnt']:4.1f} /i | {s['Avg_Inf_Time']:5.1f} ms | {s['Avg_Tot_Time']:5.1f} ms")
    print(f"{'':<35} > Peak VRAM: {groups['Peak_VRAM']:.1f} MB")
    print("-" * 140)

🚀 [Ablation Study] 가독성 최적화 버전 벤치마크 시작! (Total 430 images)


⏳ Ours (DAHI + Tetris): 100%|██████████████████████████████| 430/430 [00:43<00:00,  9.97it/s]



🏆 [Ablation Study] Component Contribution Analysis (Total 500 Images) 🏆
Method                              | Type | Img  | AP50   | AP50s  | AP50m  | AP50l  | Inf Cnt | Inf Time  | Tot Time 
--------------------------------------------------------------------------------------------------------------------------------------------
UC (2x2 Uniform Crop)               | ALL  | 430  | 0.0000 | 0.0000 | 0.0000 | 0.0000 |  0.0 /i |   0.0 ms |   0.0 ms
                                    | HR   | 54   | 0.0000 | 0.0000 | 0.0000 | 0.0000 |  0.0 /i |   0.0 ms |   0.0 ms
                                    | LR   | 376  | 0.0000 | 0.0000 | 0.0000 | 0.0000 |  0.0 /i |   0.0 ms |   0.0 ms
                                    > Peak VRAM: 54.0 MB
--------------------------------------------------------------------------------------------------------------------------------------------
Ours (DAHI Only)                    | ALL  | 430  | 0.4167 | 0.2740 | 0.4854 | 0.5471 |  4.7 /i |  52.2 ms |  73.0

In [ ]:
import cv2
import os
import time
import numpy as np
import tqdm
import torch
from ultralytics import YOLO

# =========================================================
# ⚙️ 하이퍼파라미터 (Hyperparameters)
# =========================================================
MODEL_FILTER_PATH = 'model/best_nano.pt'
MODEL_MAIN_PATH = 'model/best_small.pt'

CONF_GLOBAL = 0.3
CONF_FILTER = 0.1     
CONF_DENSE = 0.3
CONF_TETRIS = 0.3
CONF_UC = 0.3

IOU_FILTER_MATCH = 0.1
NMS_CONF_THRESH = 0.3
NMS_IOU_THRESH = 0.4    

DENSE_WINDOW_SIZE = 512
DENSE_STEP = 320      

MERGE_PAD = 16
CROP_PAD_LARGE = 80     
CROP_PAD_SMALL = 16
CROP_PAD_THRESH = 200

CANVAS_SIZE = 960
CANVAS_MARGIN = 2
CANVAS_BG_COLOR = 114

UPSCALE_RATIO = 1.5        
UPSCALE_MAX_THRESH = 200    

NUM_TEST_IMAGES = 5000
HR_THRESHOLD = 1920 * 1080 

# =========================================================
dataset_root = 'data/test'
img_dir, lbl_dir = os.path.join(dataset_root, 'images'), os.path.join(dataset_root, 'labels')
img_list = sorted(os.listdir(img_dir))[:NUM_TEST_IMAGES]

print(f"🚀 [Ablation Study] 필터링 족쇄 해제! 완벽한 재현 시작! (Total {len(img_list)} images)")

m1 = YOLO(MODEL_FILTER_PATH)
m2 = YOLO(MODEL_MAIN_PATH)

def calculate_iou(box1, box2):
    xi1, yi1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    xi2, yi2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])

def get_size_category(w, h):
    area = w * h
    if area < 32 ** 2: return 'small'
    elif area < 96 ** 2: return 'medium'
    else: return 'large'

def merge_clusters_dynamic(boxes, img_w, img_h, merge_pad=MERGE_PAD):
    if not len(boxes): return []
    def get_padded(b, pad): return [max(0, b[0]-pad), max(0, b[1]-pad), min(img_w, b[2]+pad), min(img_h, b[3]+pad)]
    def is_overlap(b1, b2):
        p1, p2 = get_padded(b1, merge_pad), get_padded(b2, merge_pad)
        return (min(p1[2], p2[2]) > max(p1[0], p2[0])) and (min(p1[3], p2[3]) > max(p1[1], p2[1]))
    curr = boxes.copy()
    while True:
        merged, flags = [], [False]*len(curr)
        for i in range(len(curr)):
            if flags[i]: continue
            b = curr[i]
            for j in range(i+1, len(curr)):
                if not flags[j] and is_overlap(b, curr[j]):
                    b = [min(b[0], curr[j][0]), min(b[1], curr[j][1]), max(b[2], curr[j][2]), max(b[3], curr[j][3])]
                    flags[j] = True
            merged.append(b)
        if len(merged) == len(curr): break
        curr = merged
    final_boxes = []
    for b in curr:
        bw, bh = b[2] - b[0], b[3] - b[1]
        crop_pad = CROP_PAD_LARGE if max(bw, bh) < CROP_PAD_THRESH else CROP_PAD_SMALL 
        final_boxes.append(get_padded(b, crop_pad))
    return final_boxes

def run_ablation_benchmark(method_name):
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        
    all_gts = {}; all_preds = []
    stats = {
        'ALL': {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'HR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'LR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()}
    }

    pbar = tqdm.tqdm(img_list, desc=f"⏳ {method_name}", bar_format='{l_bar}{bar:30}{r_bar}')
    for img_idx, img_name in enumerate(pbar):
        img_path, lbl_path = os.path.join(img_dir, img_name), os.path.join(lbl_dir, img_name.replace('.jpg', '.txt'))
        img = cv2.imread(img_path); h, w, _ = img.shape
        
        gts = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    c, xc, yc, bw, bh = map(float, line.split())
                    bw_pix, bh_pix = bw * w, bh * h
                    gts.append([int(c), (xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h, False, get_size_category(bw_pix, bh_pix)])
        all_gts[img_idx] = gts

        t_pipe_start = time.time()
        img_inf_time, img_inf_cnt = 0, 0
        
        # =====================================================================
        # 1. Baseline: UC (2x2 Uniform Crop)
        # =====================================================================
        if method_name == "UC (2x2 Uniform Crop)":
            ch, cw = h // 2, w // 2
            crops, offsets = [img], [(0, 0)]
            for y in [0, ch]:
                for x in [0, cw]:
                    crops.append(img[y:y+ch, x:x+cw])
                    offsets.append((x, y))
            
            t_inf_start = time.time()
            results2 = m2.predict(crops, conf=CONF_UC, verbose=False, batch=5)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 5 
            
            temp_boxes, temp_scores, temp_classes = [], [], []
            for i, res in enumerate(results2):
                ox, oy = offsets[i]
                for b in res.boxes:
                    temp_boxes.append([b.xyxy[0][0]+ox, b.xyxy[0][1]+oy, b.xyxy[0][2]+ox, b.xyxy[0][3]+oy])
                    temp_scores.append(float(b.conf[0])); temp_classes.append(int(b.cls[0]))
                    
            for c in set(temp_classes):
                c_boxes = [b for j, b in enumerate(temp_boxes) if temp_classes[j] == c]
                c_scores = [s for j, s in enumerate(temp_scores) if temp_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 2. 제안 1: Ours (DAHI Only) - 중복필터 완전삭제, 모든 객체 반복 추출
        # =====================================================================
        elif method_name == "Ours (DAHI Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=CONF_GLOBAL, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            t_inf_start = time.time()
            r1 = m1.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                roi_boxes.append([bx1, by1, bx2, by2]) # 💡 중복검사 삭제
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            while len(remaining_boxes) > 0:
                best_count, best_region = -1, None
                for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                        count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                        if count > best_count: 
                            best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                if best_region and best_count >= 1:
                    dense_regions.append(best_region)
                    dx1, dy1, dx2, dy2 = best_region
                    remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                else: break
            
            unified_infer_list = [img[dy1:dy2, dx1:dx2] for dx1, dy1, dx2, dy2 in dense_regions]
            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for idx, (dx1, dy1, dx2, dy2) in enumerate(dense_regions):
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1
                    for b in res_all[idx].boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
            
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 3. 제안 2: Ours (Tetris Only) - 중복필터 완전삭제
        # =====================================================================
        elif method_name == "Ours (Tetris Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=CONF_GLOBAL, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            t_inf_start = time.time()
            r1 = m1.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                roi_boxes.append([bx1, by1, bx2, by2]) # 💡 중복검사 삭제
            
            canvases, canvas_infos = [], []
            if len(roi_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(roi_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

            if len(canvases) > 0:
                t_inf_start = time.time()
                res_pack = m2.predict(canvases, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(canvases)
                
                for c_idx, res in enumerate(res_pack):
                    for b in res.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                        for info in canvas_infos:
                            if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                s = info['scale']
                                orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 4. 제안 3: Ours (DAHI + Tetris) - Top 1 추출, 중복검사 완전삭제
        # =====================================================================
        elif method_name == "Ours (DAHI + Tetris)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=CONF_GLOBAL, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            t_inf_start = time.time()
            r1 = m1.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                roi_boxes.append([bx1, by1, bx2, by2]) # 💡 중복검사 삭제
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            best_count, best_region = -1, None
            for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                    if count > best_count: 
                        best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                        
            if best_region and best_count > 0: 
                dense_regions.append(best_region)
                dx1, dy1, dx2, dy2 = best_region
                remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
            
            unified_infer_list = []
            dense_idx_list = []
            
            for dx1, dy1, dx2, dy2 in dense_regions:
                unified_infer_list.append(img[dy1:dy2, dx1:dx2])
                dense_idx_list.append((len(unified_infer_list) - 1, dx1, dy1, dx2, dy2))

            canvases, canvas_infos = [], []
            canvas_start_idx = -1
            if len(remaining_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(remaining_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

                if len(canvases) > 0:
                    canvas_start_idx = len(unified_infer_list)
                    unified_infer_list.extend(canvases)

            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for d_idx, dx1, dy1, dx2, dy2 in dense_idx_list:
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1; res_dense = res_all[d_idx]
                    for b in res_dense.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                
                if canvas_start_idx != -1:
                    res_pack = res_all[canvas_start_idx:]
                    for c_idx, res in enumerate(res_pack):
                        for b in res.boxes:
                            bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                            bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                            for info in canvas_infos:
                                if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                    if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                    s = info['scale']
                                    orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                    orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                    local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                    break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # 통계 저장
        img_total_time = time.time() - t_pipe_start
        is_hr = (w * h >= HR_THRESHOLD)
        target_keys = ['ALL', 'HR'] if is_hr else ['ALL', 'LR']
        for k in target_keys:
            stats[k]['count'] += 1
            stats[k]['inf_time'] += img_inf_time
            stats[k]['total_time'] += img_total_time
            stats[k]['inf_cnt'] += img_inf_cnt
            stats[k]['indices'].add(img_idx)

    # ---------------------------------------------------------
    # 💡 AP 연산 엔진
    # ---------------------------------------------------------
    def calc_metrics_for_subset(subset_indices):
        if not subset_indices: return {"AP50": 0, "AP50s": 0, "AP50m": 0, "AP50l": 0}
        
        sub_preds = [p for p in all_preds if p[0] in subset_indices]
        sub_preds.sort(key=lambda x: x[2], reverse=True) 
        unique_classes = set([gt[0] for idx in subset_indices for gt in all_gts[idx]])
        metrics = {'all': [], 'small': [], 'medium': [], 'large': []}
        
        for c in unique_classes:
            c_preds = [p for p in sub_preds if p[1] == c]
            for size_target in ['all', 'small', 'medium', 'large']:
                if size_target == 'all': c_gts = {idx: [list(g) for g in all_gts[idx] if g[0] == c] for idx in subset_indices}
                else: c_gts = {idx: [list(g) for g in all_gts[idx] if g[0] == c and g[6] == size_target] for idx in subset_indices}
                    
                npos = sum(len(gts) for gts in c_gts.values())
                if npos == 0: continue
                
                tp, fp = np.zeros(len(c_preds)), np.zeros(len(c_preds))
                for i, pred in enumerate(c_preds):
                    img_idx, _, _, px1, py1, px2, py2 = pred
                    pred_box = [px1, py1, px2, py2]
                    gts = c_gts[img_idx]
                    
                    pw, ph = px2 - px1, py2 - py1
                    p_size = get_size_category(pw, ph)
                    if size_target != 'all' and p_size != size_target: continue
                    
                    best_iou, best_idx = 0.5, -1
                    for j, gt in enumerate(gts):
                        if gt[5]: continue 
                        iou = calculate_iou(pred_box, gt[1:5])
                        if iou >= best_iou: best_iou, best_idx = iou, j
                            
                    if best_idx >= 0:
                        tp[i] = 1; gts[best_idx][5] = True
                    else:
                        fp[i] = 1
                        
                fp_cumsum, tp_cumsum = np.cumsum(fp), np.cumsum(tp)
                rec = tp_cumsum / npos
                prec = tp_cumsum / np.maximum(tp_cumsum + fp_cumsum, np.finfo(np.float64).eps)
                metrics[size_target].append(compute_ap(rec, prec))
                
        return {
            "AP50": np.mean(metrics['all']) if metrics['all'] else 0,
            "AP50s": np.mean(metrics['small']) if metrics['small'] else 0,
            "AP50m": np.mean(metrics['medium']) if metrics['medium'] else 0,
            "AP50l": np.mean(metrics['large']) if metrics['large'] else 0,
        }

    result_dict = {}
    for group in ['ALL', 'HR', 'LR']:
        c = stats[group]['count']
        res = calc_metrics_for_subset(stats[group]['indices'])
        res['Img_Cnt'] = c
        res['Avg_Inf_Cnt'] = stats[group]['inf_cnt'] / c if c else 0
        res['Avg_Inf_Time'] = (stats[group]['inf_time'] / c) * 1000 if c else 0
        res['Avg_Tot_Time'] = (stats[group]['total_time'] / c) * 1000 if c else 0
        result_dict[group] = res
        
    result_dict['Peak_VRAM'] = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0
    return result_dict

# =========================================================
# 실행 및 다중 표 그리기
# =========================================================
methods = [
    "UC (2x2 Uniform Crop)", 
    "Ours (DAHI Only)",
    "Ours (Tetris Only)",
    "Ours (DAHI + Tetris)"
]

final_stats = {}
for m in methods: 
    final_stats[m] = run_ablation_benchmark(m)

print("\n" + "="*140)
print(f"🏆 [Ablation Study] Component Contribution Analysis (Total {NUM_TEST_IMAGES} Images) 🏆")
print("="*140)
print(f"{'Method':<35} | {'Type':<4} | {'Img':<4} | {'AP50':<6} | {'AP50s':<6} | {'AP50m':<6} | {'AP50l':<6} | {'Inf Cnt':<7} | {'Inf Time':<9} | {'Tot Time':<9}")
print("-" * 140)

for m, groups in final_stats.items():
    for g in ['ALL', 'HR', 'LR']:
        s = groups[g]
        if s['Img_Cnt'] == 0: continue
        print(f"{m if g == 'ALL' else '':<35} | {g:<4} | {s['Img_Cnt']:<4} | {s['AP50']:.4f} | {s['AP50s']:.4f} | {s['AP50m']:.4f} | {s['AP50l']:.4f} | {s['Avg_Inf_Cnt']:4.1f} /i | {s['Avg_Inf_Time']:5.1f} ms | {s['Avg_Tot_Time']:5.1f} ms")
    print(f"{'':<35} > Peak VRAM: {groups['Peak_VRAM']:.1f} MB")
    print("-" * 140)

🚀 [Ablation Study] 필터링 족쇄 해제! 완벽한 재현 시작! (Total 430 images)


⏳ Ours (DAHI + Tetris): 100%|██████████████████████████████| 430/430 [00:34<00:00, 12.56it/s]



🏆 [Ablation Study] Component Contribution Analysis (Total 5000 Images) 🏆
Method                              | Type | Img  | AP50   | AP50s  | AP50m  | AP50l  | Inf Cnt | Inf Time  | Tot Time 
--------------------------------------------------------------------------------------------------------------------------------------------
UC (2x2 Uniform Crop)               | ALL  | 430  | 0.4052 | 0.2394 | 0.4945 | 0.5063 |  5.0 /i |  40.3 ms |  68.7 ms
                                    | HR   | 54   | 0.4309 | 0.2468 | 0.4990 | 0.5481 |  5.0 /i |  40.4 ms |  86.1 ms
                                    | LR   | 376  | 0.3993 | 0.2383 | 0.4939 | 0.5133 |  5.0 /i |  40.2 ms |  66.2 ms
                                    > Peak VRAM: 290.6 MB
--------------------------------------------------------------------------------------------------------------------------------------------
Ours (DAHI Only)                    | ALL  | 430  | 0.4214 | 0.2779 | 0.4907 | 0.5613 |  5.7 /i |  58.1 ms |  75

In [ ]:
import cv2
import os
import time
import numpy as np
import tqdm
import torch
from ultralytics import YOLO

# =========================================================
# ⚙️ 하이퍼파라미터 (Hyperparameters)
# =========================================================
MODEL_FILTER_PATH = 'model/best_nano.pt'
MODEL_MAIN_PATH = 'model/best_small.pt'

CONF_GLOBAL = 0.3
CONF_FILTER = 0.1     
CONF_DENSE = 0.3
CONF_TETRIS = 0.3
CONF_UC = 0.3

IOU_FILTER_MATCH = 0.1
NMS_CONF_THRESH = 0.3
NMS_IOU_THRESH = 0.4    

DENSE_WINDOW_SIZE = 512
DENSE_STEP = 320      

MERGE_PAD = 16
CROP_PAD_LARGE = 80     
CROP_PAD_SMALL = 16
CROP_PAD_THRESH = 200

CANVAS_SIZE = 960
CANVAS_MARGIN = 2
CANVAS_BG_COLOR = 114

UPSCALE_RATIO = 1.5        
UPSCALE_MAX_THRESH = 200    

NUM_TEST_IMAGES = 5000
HR_THRESHOLD = 1920 * 1080 

# =========================================================
dataset_root = 'data/test'
img_dir, lbl_dir = os.path.join(dataset_root, 'images'), os.path.join(dataset_root, 'labels')
img_list = sorted(os.listdir(img_dir))[:NUM_TEST_IMAGES]

print(f"🚀 [Ablation Study] 필터링 족쇄 해제! 완벽한 재현 시작! (Total {len(img_list)} images)")

m1 = YOLO(MODEL_FILTER_PATH)
m2 = YOLO(MODEL_MAIN_PATH)

def calculate_iou(box1, box2):
    xi1, yi1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    xi2, yi2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])

def get_size_category(w, h):
    area = w * h
    if area < 32 ** 2: return 'small'
    elif area < 96 ** 2: return 'medium'
    else: return 'large'

def merge_clusters_dynamic(boxes, img_w, img_h, merge_pad=MERGE_PAD):
    if not len(boxes): return []
    def get_padded(b, pad): return [max(0, b[0]-pad), max(0, b[1]-pad), min(img_w, b[2]+pad), min(img_h, b[3]+pad)]
    def is_overlap(b1, b2):
        p1, p2 = get_padded(b1, merge_pad), get_padded(b2, merge_pad)
        return (min(p1[2], p2[2]) > max(p1[0], p2[0])) and (min(p1[3], p2[3]) > max(p1[1], p2[1]))
    curr = boxes.copy()
    while True:
        merged, flags = [], [False]*len(curr)
        for i in range(len(curr)):
            if flags[i]: continue
            b = curr[i]
            for j in range(i+1, len(curr)):
                if not flags[j] and is_overlap(b, curr[j]):
                    b = [min(b[0], curr[j][0]), min(b[1], curr[j][1]), max(b[2], curr[j][2]), max(b[3], curr[j][3])]
                    flags[j] = True
            merged.append(b)
        if len(merged) == len(curr): break
        curr = merged
    final_boxes = []
    for b in curr:
        bw, bh = b[2] - b[0], b[3] - b[1]
        crop_pad = CROP_PAD_LARGE if max(bw, bh) < CROP_PAD_THRESH else CROP_PAD_SMALL 
        final_boxes.append(get_padded(b, crop_pad))
    return final_boxes

def run_ablation_benchmark(method_name):
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        
    all_gts = {}; all_preds = []
    stats = {
        'ALL': {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'HR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'LR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()}
    }

    pbar = tqdm.tqdm(img_list, desc=f"⏳ {method_name}", bar_format='{l_bar}{bar:30}{r_bar}')
    for img_idx, img_name in enumerate(pbar):
        img_path, lbl_path = os.path.join(img_dir, img_name), os.path.join(lbl_dir, img_name.replace('.jpg', '.txt'))
        img = cv2.imread(img_path); h, w, _ = img.shape
        
        gts = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    c, xc, yc, bw, bh = map(float, line.split())
                    bw_pix, bh_pix = bw * w, bh * h
                    gts.append([int(c), (xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h, False, get_size_category(bw_pix, bh_pix)])
        all_gts[img_idx] = gts

        t_pipe_start = time.time()
        img_inf_time, img_inf_cnt = 0, 0
        
        # =====================================================================
        # 1. Baseline: UC (2x2 Uniform Crop)
        # =====================================================================
        if method_name == "UC (2x2 Uniform Crop)":
            ch, cw = h // 2, w // 2
            # crops, offsets = [img], [(0, 0)]
            # for y in [0, ch]:
            #     for x in [0, cw]:
            #         crops.append(img[y:y+ch, x:x+cw])
            #         offsets.append((x, y))
            
            # t_inf_start = time.time()
            # results2 = m2.predict(crops, conf=CONF_UC, verbose=False, batch=5)
            # img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 5 
            
            # temp_boxes, temp_scores, temp_classes = [], [], []
            # for i, res in enumerate(results2):
            #     ox, oy = offsets[i]
            #     for b in res.boxes:
            #         temp_boxes.append([b.xyxy[0][0]+ox, b.xyxy[0][1]+oy, b.xyxy[0][2]+ox, b.xyxy[0][3]+oy])
            #         temp_scores.append(float(b.conf[0])); temp_classes.append(int(b.cls[0]))
                    
            # for c in set(temp_classes):
            #     c_boxes = [b for j, b in enumerate(temp_boxes) if temp_classes[j] == c]
            #     c_scores = [s for j, s in enumerate(temp_scores) if temp_classes[j] == c]
            #     cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
            #     indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
            #     if len(indices) > 0:
            #         for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 2. 제안 1: Ours (DAHI Only) - 중복필터 완전삭제, 모든 객체 반복 추출
        # =====================================================================
        elif method_name == "Ours (DAHI Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=0.1, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            t_inf_start = time.time()
            r1 = res_global
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                roi_boxes.append([bx1, by1, bx2, by2]) # 💡 중복검사 삭제
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            while len(remaining_boxes) > 0:
                best_count, best_region = -1, None
                for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                        count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                        if count > best_count: 
                            best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                if best_region and best_count >= 1:
                    dense_regions.append(best_region)
                    dx1, dy1, dx2, dy2 = best_region
                    remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                else: break
            
            unified_infer_list = [img[dy1:dy2, dx1:dx2] for dx1, dy1, dx2, dy2 in dense_regions]
            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for idx, (dx1, dy1, dx2, dy2) in enumerate(dense_regions):
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1
                    for b in res_all[idx].boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
            
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 3. 제안 2: Ours (Tetris Only) - 중복필터 완전삭제
        # =====================================================================
        elif method_name == "Ours (Tetris Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=0.1, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            t_inf_start = time.time()
            r1 = res_global
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                roi_boxes.append([bx1, by1, bx2, by2]) # 💡 중복검사 삭제
            
            canvases, canvas_infos = [], []
            if len(roi_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(roi_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

            if len(canvases) > 0:
                t_inf_start = time.time()
                res_pack = m2.predict(canvases, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(canvases)
                
                for c_idx, res in enumerate(res_pack):
                    for b in res.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                        for info in canvas_infos:
                            if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                s = info['scale']
                                orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 4. 제안 3: Ours (DAHI + Tetris) - Top 1 추출, 중복검사 완전삭제
        # =====================================================================
        elif method_name == "Ours (DAHI + Tetris)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=0.1, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            t_inf_start = time.time()
            r1 = res_global
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                roi_boxes.append([bx1, by1, bx2, by2]) # 💡 중복검사 삭제
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            best_count, best_region = -1, None
            for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                    if count > best_count: 
                        best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                        
            if best_region and best_count > 0: 
                dense_regions.append(best_region)
                dx1, dy1, dx2, dy2 = best_region
                remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
            
            unified_infer_list = []
            dense_idx_list = []
            
            for dx1, dy1, dx2, dy2 in dense_regions:
                unified_infer_list.append(img[dy1:dy2, dx1:dx2])
                dense_idx_list.append((len(unified_infer_list) - 1, dx1, dy1, dx2, dy2))

            canvases, canvas_infos = [], []
            canvas_start_idx = -1
            if len(remaining_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(remaining_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

                if len(canvases) > 0:
                    canvas_start_idx = len(unified_infer_list)
                    unified_infer_list.extend(canvases)

            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for d_idx, dx1, dy1, dx2, dy2 in dense_idx_list:
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1; res_dense = res_all[d_idx]
                    for b in res_dense.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                
                if canvas_start_idx != -1:
                    res_pack = res_all[canvas_start_idx:]
                    for c_idx, res in enumerate(res_pack):
                        for b in res.boxes:
                            bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                            bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                            for info in canvas_infos:
                                if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                    if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                    s = info['scale']
                                    orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                    orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                    local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                    break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # 통계 저장
        img_total_time = time.time() - t_pipe_start
        is_hr = (w * h >= HR_THRESHOLD)
        target_keys = ['ALL', 'HR'] if is_hr else ['ALL', 'LR']
        for k in target_keys:
            stats[k]['count'] += 1
            stats[k]['inf_time'] += img_inf_time
            stats[k]['total_time'] += img_total_time
            stats[k]['inf_cnt'] += img_inf_cnt
            stats[k]['indices'].add(img_idx)

    # ---------------------------------------------------------
    # 💡 AP 연산 엔진
    # ---------------------------------------------------------
    def calc_metrics_for_subset(subset_indices):
        if not subset_indices: return {"AP50": 0, "AP50s": 0, "AP50m": 0, "AP50l": 0}
        
        sub_preds = [p for p in all_preds if p[0] in subset_indices]
        sub_preds.sort(key=lambda x: x[2], reverse=True) 
        unique_classes = set([gt[0] for idx in subset_indices for gt in all_gts[idx]])
        metrics = {'all': [], 'small': [], 'medium': [], 'large': []}
        
        for c in unique_classes:
            c_preds = [p for p in sub_preds if p[1] == c]
            for size_target in ['all', 'small', 'medium', 'large']:
                if size_target == 'all': c_gts = {idx: [list(g) for g in all_gts[idx] if g[0] == c] for idx in subset_indices}
                else: c_gts = {idx: [list(g) for g in all_gts[idx] if g[0] == c and g[6] == size_target] for idx in subset_indices}
                    
                npos = sum(len(gts) for gts in c_gts.values())
                if npos == 0: continue
                
                tp, fp = np.zeros(len(c_preds)), np.zeros(len(c_preds))
                for i, pred in enumerate(c_preds):
                    img_idx, _, _, px1, py1, px2, py2 = pred
                    pred_box = [px1, py1, px2, py2]
                    gts = c_gts[img_idx]
                    
                    pw, ph = px2 - px1, py2 - py1
                    p_size = get_size_category(pw, ph)
                    if size_target != 'all' and p_size != size_target: continue
                    
                    best_iou, best_idx = 0.5, -1
                    for j, gt in enumerate(gts):
                        if gt[5]: continue 
                        iou = calculate_iou(pred_box, gt[1:5])
                        if iou >= best_iou: best_iou, best_idx = iou, j
                            
                    if best_idx >= 0:
                        tp[i] = 1; gts[best_idx][5] = True
                    else:
                        fp[i] = 1
                        
                fp_cumsum, tp_cumsum = np.cumsum(fp), np.cumsum(tp)
                rec = tp_cumsum / npos
                prec = tp_cumsum / np.maximum(tp_cumsum + fp_cumsum, np.finfo(np.float64).eps)
                metrics[size_target].append(compute_ap(rec, prec))
                
        return {
            "AP50": np.mean(metrics['all']) if metrics['all'] else 0,
            "AP50s": np.mean(metrics['small']) if metrics['small'] else 0,
            "AP50m": np.mean(metrics['medium']) if metrics['medium'] else 0,
            "AP50l": np.mean(metrics['large']) if metrics['large'] else 0,
        }

    result_dict = {}
    for group in ['ALL', 'HR', 'LR']:
        c = stats[group]['count']
        res = calc_metrics_for_subset(stats[group]['indices'])
        res['Img_Cnt'] = c
        res['Avg_Inf_Cnt'] = stats[group]['inf_cnt'] / c if c else 0
        res['Avg_Inf_Time'] = (stats[group]['inf_time'] / c) * 1000 if c else 0
        res['Avg_Tot_Time'] = (stats[group]['total_time'] / c) * 1000 if c else 0
        result_dict[group] = res
        
    result_dict['Peak_VRAM'] = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0
    return result_dict

# =========================================================
# 실행 및 다중 표 그리기
# =========================================================
methods = [
    "UC (2x2 Uniform Crop)", 
    "Ours (DAHI Only)",
    "Ours (Tetris Only)",
    "Ours (DAHI + Tetris)"
]

final_stats = {}
for m in methods: 
    final_stats[m] = run_ablation_benchmark(m)

print("\n" + "="*140)
print(f"🏆 [Ablation Study] Component Contribution Analysis (Total {NUM_TEST_IMAGES} Images) 🏆")
print("="*140)
print(f"{'Method':<35} | {'Type':<4} | {'Img':<4} | {'AP50':<6} | {'AP50s':<6} | {'AP50m':<6} | {'AP50l':<6} | {'Inf Cnt':<7} | {'Inf Time':<9} | {'Tot Time':<9}")
print("-" * 140)

for m, groups in final_stats.items():
    for g in ['ALL', 'HR', 'LR']:
        s = groups[g]
        if s['Img_Cnt'] == 0: continue
        print(f"{m if g == 'ALL' else '':<35} | {g:<4} | {s['Img_Cnt']:<4} | {s['AP50']:.4f} | {s['AP50s']:.4f} | {s['AP50m']:.4f} | {s['AP50l']:.4f} | {s['Avg_Inf_Cnt']:4.1f} /i | {s['Avg_Inf_Time']:5.1f} ms | {s['Avg_Tot_Time']:5.1f} ms")
    print(f"{'':<35} > Peak VRAM: {groups['Peak_VRAM']:.1f} MB")
    print("-" * 140)

🚀 [Ablation Study] 필터링 족쇄 해제! 완벽한 재현 시작! (Total 430 images)


⏳ Ours (DAHI + Tetris): 100%|██████████████████████████████| 430/430 [00:32<00:00, 13.06it/s]



🏆 [Ablation Study] Component Contribution Analysis (Total 5000 Images) 🏆
Method                              | Type | Img  | AP50   | AP50s  | AP50m  | AP50l  | Inf Cnt | Inf Time  | Tot Time 
--------------------------------------------------------------------------------------------------------------------------------------------
UC (2x2 Uniform Crop)               | ALL  | 430  | 0.0000 | 0.0000 | 0.0000 | 0.0000 |  0.0 /i |   0.0 ms |   0.0 ms
                                    | HR   | 54   | 0.0000 | 0.0000 | 0.0000 | 0.0000 |  0.0 /i |   0.0 ms |   0.0 ms
                                    | LR   | 376  | 0.0000 | 0.0000 | 0.0000 | 0.0000 |  0.0 /i |   0.0 ms |   0.0 ms
                                    > Peak VRAM: 121.6 MB
--------------------------------------------------------------------------------------------------------------------------------------------
Ours (DAHI Only)                    | ALL  | 430  | 0.4233 | 0.2775 | 0.4929 | 0.5621 |  5.7 /i |  45.5 ms |  66

In [ ]:
import cv2
import os
import time
import numpy as np
import tqdm
import torch
from ultralytics import YOLO

# =========================================================
# ⚙️ 하이퍼파라미터 (Hyperparameters)
# =========================================================
MODEL_FILTER_PATH = 'model/best_nano.pt'
MODEL_MAIN_PATH = 'model/best_small.pt'

CONF_GLOBAL = 0.3
CONF_FILTER = 0.1     
CONF_DENSE = 0.3
CONF_TETRIS = 0.3
CONF_UC = 0.3

IOU_FILTER_MATCH = 0.97
NMS_CONF_THRESH = 0.3
NMS_IOU_THRESH = 0.4    

DENSE_WINDOW_SIZE = 512
DENSE_STEP = 320      

MERGE_PAD = 16
CROP_PAD_LARGE = 80     
CROP_PAD_SMALL = 16
CROP_PAD_THRESH = 200

CANVAS_SIZE = 960
CANVAS_MARGIN = 2
CANVAS_BG_COLOR = 114

UPSCALE_RATIO = 1.5        
UPSCALE_MAX_THRESH = 200    

NUM_TEST_IMAGES = 5000
HR_THRESHOLD = 1920 * 1080 

# =========================================================
dataset_root = 'data/test'
img_dir, lbl_dir = os.path.join(dataset_root, 'images'), os.path.join(dataset_root, 'labels')
img_list = sorted(os.listdir(img_dir))[:NUM_TEST_IMAGES]

print(f"🚀 [Ablation Study] 필터링 족쇄 해제! 완벽한 재현 시작! (Total {len(img_list)} images)")

m1 = YOLO(MODEL_FILTER_PATH)
m2 = YOLO(MODEL_MAIN_PATH)

def calculate_iou(box1, box2):
    xi1, yi1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    xi2, yi2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])

def get_size_category(w, h):
    area = w * h
    if area < 32 ** 2: return 'small'
    elif area < 96 ** 2: return 'medium'
    else: return 'large'

def merge_clusters_dynamic(boxes, img_w, img_h, merge_pad=MERGE_PAD):
    if not len(boxes): return []
    def get_padded(b, pad): return [max(0, b[0]-pad), max(0, b[1]-pad), min(img_w, b[2]+pad), min(img_h, b[3]+pad)]
    def is_overlap(b1, b2):
        p1, p2 = get_padded(b1, merge_pad), get_padded(b2, merge_pad)
        return (min(p1[2], p2[2]) > max(p1[0], p2[0])) and (min(p1[3], p2[3]) > max(p1[1], p2[1]))
    curr = boxes.copy()
    while True:
        merged, flags = [], [False]*len(curr)
        for i in range(len(curr)):
            if flags[i]: continue
            b = curr[i]
            for j in range(i+1, len(curr)):
                if not flags[j] and is_overlap(b, curr[j]):
                    b = [min(b[0], curr[j][0]), min(b[1], curr[j][1]), max(b[2], curr[j][2]), max(b[3], curr[j][3])]
                    flags[j] = True
            merged.append(b)
        if len(merged) == len(curr): break
        curr = merged
    final_boxes = []
    for b in curr:
        bw, bh = b[2] - b[0], b[3] - b[1]
        crop_pad = CROP_PAD_LARGE if max(bw, bh) < CROP_PAD_THRESH else CROP_PAD_SMALL 
        final_boxes.append(get_padded(b, crop_pad))
    return final_boxes

def run_ablation_benchmark(method_name):
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        
    all_gts = {}; all_preds = []
    stats = {
        'ALL': {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'HR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'LR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()}
    }

    pbar = tqdm.tqdm(img_list, desc=f"⏳ {method_name}", bar_format='{l_bar}{bar:30}{r_bar}')
    for img_idx, img_name in enumerate(pbar):
        img_path, lbl_path = os.path.join(img_dir, img_name), os.path.join(lbl_dir, img_name.replace('.jpg', '.txt'))
        img = cv2.imread(img_path); h, w, _ = img.shape
        
        gts = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    c, xc, yc, bw, bh = map(float, line.split())
                    bw_pix, bh_pix = bw * w, bh * h
                    gts.append([int(c), (xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h, False, get_size_category(bw_pix, bh_pix)])
        all_gts[img_idx] = gts

        t_pipe_start = time.time()
        img_inf_time, img_inf_cnt = 0, 0
        
        # =====================================================================
        # 1. Baseline: UC (2x2 Uniform Crop)
        # =====================================================================
        if method_name == "UC (2x2 Uniform Crop)":
            ch, cw = h // 2, w // 2
            # crops, offsets = [img], [(0, 0)]
            # for y in [0, ch]:
            #     for x in [0, cw]:
            #         crops.append(img[y:y+ch, x:x+cw])
            #         offsets.append((x, y))
            
            # t_inf_start = time.time()
            # results2 = m2.predict(crops, conf=CONF_UC, verbose=False, batch=5)
            # img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 5 
            
            # temp_boxes, temp_scores, temp_classes = [], [], []
            # for i, res in enumerate(results2):
            #     ox, oy = offsets[i]
            #     for b in res.boxes:
            #         temp_boxes.append([b.xyxy[0][0]+ox, b.xyxy[0][1]+oy, b.xyxy[0][2]+ox, b.xyxy[0][3]+oy])
            #         temp_scores.append(float(b.conf[0])); temp_classes.append(int(b.cls[0]))
                    
            # for c in set(temp_classes):
            #     c_boxes = [b for j, b in enumerate(temp_boxes) if temp_classes[j] == c]
            #     c_scores = [s for j, s in enumerate(temp_scores) if temp_classes[j] == c]
            #     cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
            #     indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
            #     if len(indices) > 0:
            #         for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 2. 제안 1: Ours (DAHI Only) - 중복필터 완전삭제, 모든 객체 반복 추출
        # =====================================================================
        elif method_name == "Ours (DAHI Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=0.3, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            t_inf_start = time.time()
            r1 = m1.predict(img, conf=0.1, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                is_found = False
                for gb in global_final_boxes:
                    if calculate_iou([bx1, by1, bx2, by2], gb) > IOU_FILTER_MATCH: is_found = True; break
                if not is_found: roi_boxes.append([bx1, by1, bx2, by2])
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            while len(remaining_boxes) > 0:
                best_count, best_region = -1, None
                for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                        count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                        if count > best_count: 
                            best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                if best_region and best_count >= 1:
                    dense_regions.append(best_region)
                    dx1, dy1, dx2, dy2 = best_region
                    remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                else: break
            
            unified_infer_list = [img[dy1:dy2, dx1:dx2] for dx1, dy1, dx2, dy2 in dense_regions]
            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for idx, (dx1, dy1, dx2, dy2) in enumerate(dense_regions):
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1
                    for b in res_all[idx].boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
            
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 3. 제안 2: Ours (Tetris Only) - 중복필터 완전삭제
        # =====================================================================
        elif method_name == "Ours (Tetris Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=0.3, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            t_inf_start = time.time()
            r1 = m1.predict(img, conf=0.1, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                is_found = False
                for gb in global_final_boxes:
                    if calculate_iou([bx1, by1, bx2, by2], gb) > IOU_FILTER_MATCH: is_found = True; break
                if not is_found: roi_boxes.append([bx1, by1, bx2, by2])
            
            canvases, canvas_infos = [], []
            if len(roi_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(roi_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

            if len(canvases) > 0:
                t_inf_start = time.time()
                res_pack = m2.predict(canvases, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(canvases)
                
                for c_idx, res in enumerate(res_pack):
                    for b in res.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                        for info in canvas_infos:
                            if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                s = info['scale']
                                orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 4. 제안 3: Ours (DAHI + Tetris) - Top 1 추출, 중복검사 완전삭제
        # =====================================================================
        elif method_name == "Ours (DAHI + Tetris)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=0.3, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            t_inf_start = time.time()
            r1 = m1.predict(img, conf=0.1, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                is_found = False
                for gb in global_final_boxes:
                    if calculate_iou([bx1, by1, bx2, by2], gb) > IOU_FILTER_MATCH: is_found = True; break
                if not is_found: roi_boxes.append([bx1, by1, bx2, by2])
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            best_count, best_region = -1, None
            for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                    if count > best_count: 
                        best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                        
            if best_region and best_count > 0: 
                dense_regions.append(best_region)
                dx1, dy1, dx2, dy2 = best_region
                remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
            
            unified_infer_list = []
            dense_idx_list = []
            
            for dx1, dy1, dx2, dy2 in dense_regions:
                unified_infer_list.append(img[dy1:dy2, dx1:dx2])
                dense_idx_list.append((len(unified_infer_list) - 1, dx1, dy1, dx2, dy2))

            canvases, canvas_infos = [], []
            canvas_start_idx = -1
            if len(remaining_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(remaining_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

                if len(canvases) > 0:
                    canvas_start_idx = len(unified_infer_list)
                    unified_infer_list.extend(canvases)

            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for d_idx, dx1, dy1, dx2, dy2 in dense_idx_list:
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1; res_dense = res_all[d_idx]
                    for b in res_dense.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                
                if canvas_start_idx != -1:
                    res_pack = res_all[canvas_start_idx:]
                    for c_idx, res in enumerate(res_pack):
                        for b in res.boxes:
                            bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                            bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                            for info in canvas_infos:
                                if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                    if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                    s = info['scale']
                                    orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                    orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                    local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                    break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # 통계 저장
        img_total_time = time.time() - t_pipe_start
        is_hr = (w * h >= HR_THRESHOLD)
        target_keys = ['ALL', 'HR'] if is_hr else ['ALL', 'LR']
        for k in target_keys:
            stats[k]['count'] += 1
            stats[k]['inf_time'] += img_inf_time
            stats[k]['total_time'] += img_total_time
            stats[k]['inf_cnt'] += img_inf_cnt
            stats[k]['indices'].add(img_idx)

    # ---------------------------------------------------------
    # 💡 AP 연산 엔진
    # ---------------------------------------------------------
    def calc_metrics_for_subset(subset_indices):
        if not subset_indices: return {"AP50": 0, "AP50s": 0, "AP50m": 0, "AP50l": 0}
        
        sub_preds = [p for p in all_preds if p[0] in subset_indices]
        sub_preds.sort(key=lambda x: x[2], reverse=True) 
        unique_classes = set([gt[0] for idx in subset_indices for gt in all_gts[idx]])
        metrics = {'all': [], 'small': [], 'medium': [], 'large': []}
        
        for c in unique_classes:
            c_preds = [p for p in sub_preds if p[1] == c]
            for size_target in ['all', 'small', 'medium', 'large']:
                if size_target == 'all': c_gts = {idx: [list(g) for g in all_gts[idx] if g[0] == c] for idx in subset_indices}
                else: c_gts = {idx: [list(g) for g in all_gts[idx] if g[0] == c and g[6] == size_target] for idx in subset_indices}
                    
                npos = sum(len(gts) for gts in c_gts.values())
                if npos == 0: continue
                
                tp, fp = np.zeros(len(c_preds)), np.zeros(len(c_preds))
                for i, pred in enumerate(c_preds):
                    img_idx, _, _, px1, py1, px2, py2 = pred
                    pred_box = [px1, py1, px2, py2]
                    gts = c_gts[img_idx]
                    
                    pw, ph = px2 - px1, py2 - py1
                    p_size = get_size_category(pw, ph)
                    if size_target != 'all' and p_size != size_target: continue
                    
                    best_iou, best_idx = 0.5, -1
                    for j, gt in enumerate(gts):
                        if gt[5]: continue 
                        iou = calculate_iou(pred_box, gt[1:5])
                        if iou >= best_iou: best_iou, best_idx = iou, j
                            
                    if best_idx >= 0:
                        tp[i] = 1; gts[best_idx][5] = True
                    else:
                        fp[i] = 1
                        
                fp_cumsum, tp_cumsum = np.cumsum(fp), np.cumsum(tp)
                rec = tp_cumsum / npos
                prec = tp_cumsum / np.maximum(tp_cumsum + fp_cumsum, np.finfo(np.float64).eps)
                metrics[size_target].append(compute_ap(rec, prec))
                
        return {
            "AP50": np.mean(metrics['all']) if metrics['all'] else 0,
            "AP50s": np.mean(metrics['small']) if metrics['small'] else 0,
            "AP50m": np.mean(metrics['medium']) if metrics['medium'] else 0,
            "AP50l": np.mean(metrics['large']) if metrics['large'] else 0,
        }

    result_dict = {}
    for group in ['ALL', 'HR', 'LR']:
        c = stats[group]['count']
        res = calc_metrics_for_subset(stats[group]['indices'])
        res['Img_Cnt'] = c
        res['Avg_Inf_Cnt'] = stats[group]['inf_cnt'] / c if c else 0
        res['Avg_Inf_Time'] = (stats[group]['inf_time'] / c) * 1000 if c else 0
        res['Avg_Tot_Time'] = (stats[group]['total_time'] / c) * 1000 if c else 0
        result_dict[group] = res
        
    result_dict['Peak_VRAM'] = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0
    return result_dict

# =========================================================
# 실행 및 다중 표 그리기
# =========================================================
methods = [
    "UC (2x2 Uniform Crop)", 
    "Ours (DAHI Only)",
    "Ours (Tetris Only)",
    "Ours (DAHI + Tetris)"
]

final_stats = {}
for m in methods: 
    final_stats[m] = run_ablation_benchmark(m)

print("\n" + "="*140)
print(f"🏆 [Ablation Study] Component Contribution Analysis (Total {NUM_TEST_IMAGES} Images) 🏆")
print("="*140)
print(f"{'Method':<35} | {'Type':<4} | {'Img':<4} | {'AP50':<6} | {'AP50s':<6} | {'AP50m':<6} | {'AP50l':<6} | {'Inf Cnt':<7} | {'Inf Time':<9} | {'Tot Time':<9}")
print("-" * 140)

for m, groups in final_stats.items():
    for g in ['ALL', 'HR', 'LR']:
        s = groups[g]
        if s['Img_Cnt'] == 0: continue
        print(f"{m if g == 'ALL' else '':<35} | {g:<4} | {s['Img_Cnt']:<4} | {s['AP50']:.4f} | {s['AP50s']:.4f} | {s['AP50m']:.4f} | {s['AP50l']:.4f} | {s['Avg_Inf_Cnt']:4.1f} /i | {s['Avg_Inf_Time']:5.1f} ms | {s['Avg_Tot_Time']:5.1f} ms")
    print(f"{'':<35} > Peak VRAM: {groups['Peak_VRAM']:.1f} MB")
    print("-" * 140)

🚀 [Ablation Study] 필터링 족쇄 해제! 완벽한 재현 시작! (Total 430 images)


⏳ Ours (DAHI + Tetris): 100%|██████████████████████████████| 430/430 [00:37<00:00, 11.61it/s]



🏆 [Ablation Study] Component Contribution Analysis (Total 5000 Images) 🏆
Method                              | Type | Img  | AP50   | AP50s  | AP50m  | AP50l  | Inf Cnt | Inf Time  | Tot Time 
--------------------------------------------------------------------------------------------------------------------------------------------
UC (2x2 Uniform Crop)               | ALL  | 430  | 0.0000 | 0.0000 | 0.0000 | 0.0000 |  0.0 /i |   0.0 ms |   0.0 ms
                                    | HR   | 54   | 0.0000 | 0.0000 | 0.0000 | 0.0000 |  0.0 /i |   0.0 ms |   0.0 ms
                                    | LR   | 376  | 0.0000 | 0.0000 | 0.0000 | 0.0000 |  0.0 /i |   0.0 ms |   0.0 ms
                                    > Peak VRAM: 176.4 MB
--------------------------------------------------------------------------------------------------------------------------------------------
Ours (DAHI Only)                    | ALL  | 430  | 0.4215 | 0.2779 | 0.4907 | 0.5617 |  5.7 /i |  58.3 ms |  80

In [ ]:
import cv2
import os
import time
import numpy as np
import tqdm
import torch
from ultralytics import YOLO

# =========================================================
# ⚙️ 하이퍼파라미터 (Hyperparameters)
# =========================================================
MODEL_FILTER_PATH = 'model/best_nano.pt'
MODEL_MAIN_PATH = 'model/best_small.pt'

CONF_GLOBAL = 0.3
CONF_FILTER = 0.1     
CONF_DENSE = 0.3
CONF_TETRIS = 0.3
CONF_UC = 0.3

IOU_FILTER_MATCH = 0.1
NMS_CONF_THRESH = 0.3
NMS_IOU_THRESH = 0.4    

DENSE_WINDOW_SIZE = 512
DENSE_STEP = 320      

MERGE_PAD = 16
CROP_PAD_LARGE = 80     
CROP_PAD_SMALL = 16
CROP_PAD_THRESH = 200

CANVAS_SIZE = 960
CANVAS_MARGIN = 2
CANVAS_BG_COLOR = 114

UPSCALE_RATIO = 1.5        
UPSCALE_MAX_THRESH = 200    

NUM_TEST_IMAGES = 5000
HR_THRESHOLD = 1920 * 1080 

# =========================================================
dataset_root = 'data/valid'
img_dir, lbl_dir = os.path.join(dataset_root, 'images'), os.path.join(dataset_root, 'labels')
img_list = sorted(os.listdir(img_dir))[:NUM_TEST_IMAGES]

print(f"🚀 [Ablation Study] 필터링 족쇄 해제! 완벽한 재현 시작! (Total {len(img_list)} images)")

m1 = YOLO(MODEL_FILTER_PATH)
m2 = YOLO(MODEL_MAIN_PATH)

def calculate_iou(box1, box2):
    xi1, yi1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    xi2, yi2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])

def get_size_category(w, h):
    area = w * h
    if area < 32 ** 2: return 'small'
    elif area < 96 ** 2: return 'medium'
    else: return 'large'

def merge_clusters_dynamic(boxes, img_w, img_h, merge_pad=MERGE_PAD):
    if not len(boxes): return []
    def get_padded(b, pad): return [max(0, b[0]-pad), max(0, b[1]-pad), min(img_w, b[2]+pad), min(img_h, b[3]+pad)]
    def is_overlap(b1, b2):
        p1, p2 = get_padded(b1, merge_pad), get_padded(b2, merge_pad)
        return (min(p1[2], p2[2]) > max(p1[0], p2[0])) and (min(p1[3], p2[3]) > max(p1[1], p2[1]))
    curr = boxes.copy()
    while True:
        merged, flags = [], [False]*len(curr)
        for i in range(len(curr)):
            if flags[i]: continue
            b = curr[i]
            for j in range(i+1, len(curr)):
                if not flags[j] and is_overlap(b, curr[j]):
                    b = [min(b[0], curr[j][0]), min(b[1], curr[j][1]), max(b[2], curr[j][2]), max(b[3], curr[j][3])]
                    flags[j] = True
            merged.append(b)
        if len(merged) == len(curr): break
        curr = merged
    final_boxes = []
    for b in curr:
        bw, bh = b[2] - b[0], b[3] - b[1]
        crop_pad = CROP_PAD_LARGE if max(bw, bh) < CROP_PAD_THRESH else CROP_PAD_SMALL 
        final_boxes.append(get_padded(b, crop_pad))
    return final_boxes

def run_ablation_benchmark(method_name):
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        
    all_gts = {}; all_preds = []
    stats = {
        'ALL': {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'HR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'LR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()}
    }

    pbar = tqdm.tqdm(img_list, desc=f"⏳ {method_name}", bar_format='{l_bar}{bar:30}{r_bar}')
    for img_idx, img_name in enumerate(pbar):
        img_path, lbl_path = os.path.join(img_dir, img_name), os.path.join(lbl_dir, img_name.replace('.jpg', '.txt'))
        img = cv2.imread(img_path); h, w, _ = img.shape
        
        gts = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    c, xc, yc, bw, bh = map(float, line.split())
                    bw_pix, bh_pix = bw * w, bh * h
                    gts.append([int(c), (xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h, False, get_size_category(bw_pix, bh_pix)])
        all_gts[img_idx] = gts

        t_pipe_start = time.time()
        img_inf_time, img_inf_cnt = 0, 0
        
        # =====================================================================
        # 1. Baseline: UC (2x2 Uniform Crop)
        # =====================================================================
        if method_name == "UC (2x2 Uniform Crop)":
            ch, cw = h // 2, w // 2
            # crops, offsets = [img], [(0, 0)]
            # for y in [0, ch]:
            #     for x in [0, cw]:
            #         crops.append(img[y:y+ch, x:x+cw])
            #         offsets.append((x, y))
            
            # t_inf_start = time.time()
            # results2 = m2.predict(crops, conf=CONF_UC, verbose=False, batch=5)
            # img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 5 
            
            # temp_boxes, temp_scores, temp_classes = [], [], []
            # for i, res in enumerate(results2):
            #     ox, oy = offsets[i]
            #     for b in res.boxes:
            #         temp_boxes.append([b.xyxy[0][0]+ox, b.xyxy[0][1]+oy, b.xyxy[0][2]+ox, b.xyxy[0][3]+oy])
            #         temp_scores.append(float(b.conf[0])); temp_classes.append(int(b.cls[0]))
                    
            # for c in set(temp_classes):
            #     c_boxes = [b for j, b in enumerate(temp_boxes) if temp_classes[j] == c]
            #     c_scores = [s for j, s in enumerate(temp_scores) if temp_classes[j] == c]
            #     cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
            #     indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
            #     if len(indices) > 0:
            #         for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 2. 제안 1: Ours (DAHI Only) - 중복필터 완전삭제, 모든 객체 반복 추출
        # =====================================================================
        elif method_name == "Ours (DAHI Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=0.1, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            t_inf_start = time.time()
            r1 = res_global
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                roi_boxes.append([bx1, by1, bx2, by2]) # 💡 중복검사 삭제
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            while len(remaining_boxes) > 0:
                best_count, best_region = -1, None
                for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                        count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                        if count > best_count: 
                            best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                if best_region and best_count >= 1:
                    dense_regions.append(best_region)
                    dx1, dy1, dx2, dy2 = best_region
                    remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                else: break
            
            unified_infer_list = [img[dy1:dy2, dx1:dx2] for dx1, dy1, dx2, dy2 in dense_regions]
            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for idx, (dx1, dy1, dx2, dy2) in enumerate(dense_regions):
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1
                    for b in res_all[idx].boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
            
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 3. 제안 2: Ours (Tetris Only) - 중복필터 완전삭제
        # =====================================================================
        elif method_name == "Ours (Tetris Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=0.1, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            t_inf_start = time.time()
            r1 = res_global
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                roi_boxes.append([bx1, by1, bx2, by2]) # 💡 중복검사 삭제
            
            canvases, canvas_infos = [], []
            if len(roi_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(roi_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

            if len(canvases) > 0:
                t_inf_start = time.time()
                res_pack = m2.predict(canvases, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(canvases)
                
                for c_idx, res in enumerate(res_pack):
                    for b in res.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                        for info in canvas_infos:
                            if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                s = info['scale']
                                orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 4. 제안 3: Ours (DAHI + Tetris) - Top 1 추출, 중복검사 완전삭제
        # =====================================================================
        elif method_name == "Ours (DAHI + Tetris)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=0.1, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            t_inf_start = time.time()
            r1 = res_global
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                roi_boxes.append([bx1, by1, bx2, by2]) # 💡 중복검사 삭제
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            best_count, best_region = -1, None
            for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                    if count > best_count: 
                        best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                        
            if best_region and best_count > 0: 
                dense_regions.append(best_region)
                dx1, dy1, dx2, dy2 = best_region
                remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
            
            unified_infer_list = []
            dense_idx_list = []
            
            for dx1, dy1, dx2, dy2 in dense_regions:
                unified_infer_list.append(img[dy1:dy2, dx1:dx2])
                dense_idx_list.append((len(unified_infer_list) - 1, dx1, dy1, dx2, dy2))

            canvases, canvas_infos = [], []
            canvas_start_idx = -1
            if len(remaining_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(remaining_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

                if len(canvases) > 0:
                    canvas_start_idx = len(unified_infer_list)
                    unified_infer_list.extend(canvases)

            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for d_idx, dx1, dy1, dx2, dy2 in dense_idx_list:
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1; res_dense = res_all[d_idx]
                    for b in res_dense.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                
                if canvas_start_idx != -1:
                    res_pack = res_all[canvas_start_idx:]
                    for c_idx, res in enumerate(res_pack):
                        for b in res.boxes:
                            bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                            bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                            for info in canvas_infos:
                                if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                    if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                    s = info['scale']
                                    orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                    orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                    local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                    break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # 통계 저장
        img_total_time = time.time() - t_pipe_start
        is_hr = (w * h >= HR_THRESHOLD)
        target_keys = ['ALL', 'HR'] if is_hr else ['ALL', 'LR']
        for k in target_keys:
            stats[k]['count'] += 1
            stats[k]['inf_time'] += img_inf_time
            stats[k]['total_time'] += img_total_time
            stats[k]['inf_cnt'] += img_inf_cnt
            stats[k]['indices'].add(img_idx)

    # ---------------------------------------------------------
    # 💡 AP 연산 엔진
    # ---------------------------------------------------------
    def calc_metrics_for_subset(subset_indices):
        if not subset_indices: return {"AP50": 0, "AP50s": 0, "AP50m": 0, "AP50l": 0}
        
        sub_preds = [p for p in all_preds if p[0] in subset_indices]
        sub_preds.sort(key=lambda x: x[2], reverse=True) 
        unique_classes = set([gt[0] for idx in subset_indices for gt in all_gts[idx]])
        metrics = {'all': [], 'small': [], 'medium': [], 'large': []}
        
        for c in unique_classes:
            c_preds = [p for p in sub_preds if p[1] == c]
            for size_target in ['all', 'small', 'medium', 'large']:
                if size_target == 'all': c_gts = {idx: [list(g) for g in all_gts[idx] if g[0] == c] for idx in subset_indices}
                else: c_gts = {idx: [list(g) for g in all_gts[idx] if g[0] == c and g[6] == size_target] for idx in subset_indices}
                    
                npos = sum(len(gts) for gts in c_gts.values())
                if npos == 0: continue
                
                tp, fp = np.zeros(len(c_preds)), np.zeros(len(c_preds))
                for i, pred in enumerate(c_preds):
                    img_idx, _, _, px1, py1, px2, py2 = pred
                    pred_box = [px1, py1, px2, py2]
                    gts = c_gts[img_idx]
                    
                    pw, ph = px2 - px1, py2 - py1
                    p_size = get_size_category(pw, ph)
                    if size_target != 'all' and p_size != size_target: continue
                    
                    best_iou, best_idx = 0.5, -1
                    for j, gt in enumerate(gts):
                        if gt[5]: continue 
                        iou = calculate_iou(pred_box, gt[1:5])
                        if iou >= best_iou: best_iou, best_idx = iou, j
                            
                    if best_idx >= 0:
                        tp[i] = 1; gts[best_idx][5] = True
                    else:
                        fp[i] = 1
                        
                fp_cumsum, tp_cumsum = np.cumsum(fp), np.cumsum(tp)
                rec = tp_cumsum / npos
                prec = tp_cumsum / np.maximum(tp_cumsum + fp_cumsum, np.finfo(np.float64).eps)
                metrics[size_target].append(compute_ap(rec, prec))
                
        return {
            "AP50": np.mean(metrics['all']) if metrics['all'] else 0,
            "AP50s": np.mean(metrics['small']) if metrics['small'] else 0,
            "AP50m": np.mean(metrics['medium']) if metrics['medium'] else 0,
            "AP50l": np.mean(metrics['large']) if metrics['large'] else 0,
        }

    result_dict = {}
    for group in ['ALL', 'HR', 'LR']:
        c = stats[group]['count']
        res = calc_metrics_for_subset(stats[group]['indices'])
        res['Img_Cnt'] = c
        res['Avg_Inf_Cnt'] = stats[group]['inf_cnt'] / c if c else 0
        res['Avg_Inf_Time'] = (stats[group]['inf_time'] / c) * 1000 if c else 0
        res['Avg_Tot_Time'] = (stats[group]['total_time'] / c) * 1000 if c else 0
        result_dict[group] = res
        
    result_dict['Peak_VRAM'] = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0
    return result_dict

# =========================================================
# 실행 및 다중 표 그리기
# =========================================================
methods = [
    "UC (2x2 Uniform Crop)", 
    "Ours (DAHI Only)",
    "Ours (Tetris Only)",
    "Ours (DAHI + Tetris)"
]

final_stats = {}
for m in methods: 
    final_stats[m] = run_ablation_benchmark(m)

print("\n" + "="*140)
print(f"🏆 [Ablation Study] Component Contribution Analysis (Total {NUM_TEST_IMAGES} Images) 🏆")
print("="*140)
print(f"{'Method':<35} | {'Type':<4} | {'Img':<4} | {'AP50':<6} | {'AP50s':<6} | {'AP50m':<6} | {'AP50l':<6} | {'Inf Cnt':<7} | {'Inf Time':<9} | {'Tot Time':<9}")
print("-" * 140)

for m, groups in final_stats.items():
    for g in ['ALL', 'HR', 'LR']:
        s = groups[g]
        if s['Img_Cnt'] == 0: continue
        print(f"{m if g == 'ALL' else '':<35} | {g:<4} | {s['Img_Cnt']:<4} | {s['AP50']:.4f} | {s['AP50s']:.4f} | {s['AP50m']:.4f} | {s['AP50l']:.4f} | {s['Avg_Inf_Cnt']:4.1f} /i | {s['Avg_Inf_Time']:5.1f} ms | {s['Avg_Tot_Time']:5.1f} ms")
    print(f"{'':<35} > Peak VRAM: {groups['Peak_VRAM']:.1f} MB")
    print("-" * 140)

🚀 [Ablation Study] 필터링 족쇄 해제! 완벽한 재현 시작! (Total 1294 images)


⏳ Ours (DAHI + Tetris): 100%|██████████████████████████████| 1294/1294 [01:34<00:00, 13.67it/s]



🏆 [Ablation Study] Component Contribution Analysis (Total 5000 Images) 🏆
Method                              | Type | Img  | AP50   | AP50s  | AP50m  | AP50l  | Inf Cnt | Inf Time  | Tot Time 
--------------------------------------------------------------------------------------------------------------------------------------------
UC (2x2 Uniform Crop)               | ALL  | 1294 | 0.0000 | 0.0000 | 0.0000 | 0.0000 |  0.0 /i |   0.0 ms |   0.0 ms
                                    | HR   | 185  | 0.0000 | 0.0000 | 0.0000 | 0.0000 |  0.0 /i |   0.0 ms |   0.0 ms
                                    | LR   | 1109 | 0.0000 | 0.0000 | 0.0000 | 0.0000 |  0.0 /i |   0.0 ms |   0.0 ms
                                    > Peak VRAM: 42.8 MB
--------------------------------------------------------------------------------------------------------------------------------------------
Ours (DAHI Only)                    | ALL  | 1294 | 0.4029 | 0.2622 | 0.4603 | 0.4690 |  5.8 /i |  48.5 ms |  72.

In [ ]:
import cv2
import os
import time
import numpy as np
import tqdm
import torch
from ultralytics import YOLO

# =========================================================
# ⚙️ 하이퍼파라미터 (Hyperparameters)
# =========================================================
MODEL_FILTER_PATH = 'model/best_nano.pt'
MODEL_MAIN_PATH = 'model/best_small.pt'

CONF_GLOBAL = 0.3
CONF_FILTER = 0.1     
CONF_DENSE = 0.3
CONF_TETRIS = 0.3
CONF_UC = 0.3

IOU_FILTER_MATCH = 0.1
NMS_CONF_THRESH = 0.3
NMS_IOU_THRESH = 0.4    

DENSE_WINDOW_SIZE = 512
DENSE_STEP = 320      

MERGE_PAD = 16
CROP_PAD_LARGE = 80     
CROP_PAD_SMALL = 16
CROP_PAD_THRESH = 200

CANVAS_SIZE = 960
CANVAS_MARGIN = 2
CANVAS_BG_COLOR = 114

UPSCALE_RATIO = 1.5        
UPSCALE_MAX_THRESH = 200    

NUM_TEST_IMAGES = 5000
HR_THRESHOLD = 1920 * 1080 

# =========================================================
dataset_root = 'data/valid'
img_dir, lbl_dir = os.path.join(dataset_root, 'images'), os.path.join(dataset_root, 'labels')
img_list = sorted(os.listdir(img_dir))[:NUM_TEST_IMAGES]

print(f"🚀 [Ablation Study] 필터링 족쇄 해제! 완벽한 재현 시작! (Total {len(img_list)} images)")

m1 = YOLO(MODEL_FILTER_PATH)
m2 = YOLO(MODEL_MAIN_PATH)

def calculate_iou(box1, box2):
    xi1, yi1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    xi2, yi2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])

def get_size_category(w, h):
    area = w * h
    if area < 32 ** 2: return 'small'
    elif area < 96 ** 2: return 'medium'
    else: return 'large'

def merge_clusters_dynamic(boxes, img_w, img_h, merge_pad=MERGE_PAD):
    if not len(boxes): return []
    def get_padded(b, pad): return [max(0, b[0]-pad), max(0, b[1]-pad), min(img_w, b[2]+pad), min(img_h, b[3]+pad)]
    def is_overlap(b1, b2):
        p1, p2 = get_padded(b1, merge_pad), get_padded(b2, merge_pad)
        return (min(p1[2], p2[2]) > max(p1[0], p2[0])) and (min(p1[3], p2[3]) > max(p1[1], p2[1]))
    curr = boxes.copy()
    while True:
        merged, flags = [], [False]*len(curr)
        for i in range(len(curr)):
            if flags[i]: continue
            b = curr[i]
            for j in range(i+1, len(curr)):
                if not flags[j] and is_overlap(b, curr[j]):
                    b = [min(b[0], curr[j][0]), min(b[1], curr[j][1]), max(b[2], curr[j][2]), max(b[3], curr[j][3])]
                    flags[j] = True
            merged.append(b)
        if len(merged) == len(curr): break
        curr = merged
    final_boxes = []
    for b in curr:
        bw, bh = b[2] - b[0], b[3] - b[1]
        crop_pad = CROP_PAD_LARGE if max(bw, bh) < CROP_PAD_THRESH else CROP_PAD_SMALL 
        final_boxes.append(get_padded(b, crop_pad))
    return final_boxes

def run_ablation_benchmark(method_name):
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        
    all_gts = {}; all_preds = []
    stats = {
        'ALL': {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'HR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'LR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()}
    }

    pbar = tqdm.tqdm(img_list, desc=f"⏳ {method_name}", bar_format='{l_bar}{bar:30}{r_bar}')
    for img_idx, img_name in enumerate(pbar):
        img_path, lbl_path = os.path.join(img_dir, img_name), os.path.join(lbl_dir, img_name.replace('.jpg', '.txt'))
        img = cv2.imread(img_path); h, w, _ = img.shape
        
        gts = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    c, xc, yc, bw, bh = map(float, line.split())
                    bw_pix, bh_pix = bw * w, bh * h
                    gts.append([int(c), (xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h, False, get_size_category(bw_pix, bh_pix)])
        all_gts[img_idx] = gts

        t_pipe_start = time.time()
        img_inf_time, img_inf_cnt = 0, 0
        
        # =====================================================================
        # 1. Baseline: UC (2x2 Uniform Crop)
        # =====================================================================
        if method_name == "UC (2x2 Uniform Crop)":
            ch, cw = h // 2, w // 2
            # crops, offsets = [img], [(0, 0)]
            # for y in [0, ch]:
            #     for x in [0, cw]:
            #         crops.append(img[y:y+ch, x:x+cw])
            #         offsets.append((x, y))
            
            # t_inf_start = time.time()
            # results2 = m2.predict(crops, conf=CONF_UC, verbose=False, batch=5)
            # img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 5 
            
            # temp_boxes, temp_scores, temp_classes = [], [], []
            # for i, res in enumerate(results2):
            #     ox, oy = offsets[i]
            #     for b in res.boxes:
            #         temp_boxes.append([b.xyxy[0][0]+ox, b.xyxy[0][1]+oy, b.xyxy[0][2]+ox, b.xyxy[0][3]+oy])
            #         temp_scores.append(float(b.conf[0])); temp_classes.append(int(b.cls[0]))
                    
            # for c in set(temp_classes):
            #     c_boxes = [b for j, b in enumerate(temp_boxes) if temp_classes[j] == c]
            #     c_scores = [s for j, s in enumerate(temp_scores) if temp_classes[j] == c]
            #     cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
            #     indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
            #     if len(indices) > 0:
            #         for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 2. 제안 1: Ours (DAHI Only) - 중복필터 완전삭제, 모든 객체 반복 추출
        # =====================================================================
        elif method_name == "Ours (DAHI Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=0.3, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            t_inf_start = time.time()
            r1 = m1.predict(img, conf=0.1, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                roi_boxes.append([bx1, by1, bx2, by2]) # 💡 중복검사 삭제
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            while len(remaining_boxes) > 0:
                best_count, best_region = -1, None
                for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                        count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                        if count > best_count: 
                            best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                if best_region and best_count >= 1:
                    dense_regions.append(best_region)
                    dx1, dy1, dx2, dy2 = best_region
                    remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                else: break
            
            unified_infer_list = [img[dy1:dy2, dx1:dx2] for dx1, dy1, dx2, dy2 in dense_regions]
            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for idx, (dx1, dy1, dx2, dy2) in enumerate(dense_regions):
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1
                    for b in res_all[idx].boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
            
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 3. 제안 2: Ours (Tetris Only) - 중복필터 완전삭제
        # =====================================================================
        elif method_name == "Ours (Tetris Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=0.3, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            t_inf_start = time.time()
            r1 = m1.predict(img, conf=0.1, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                roi_boxes.append([bx1, by1, bx2, by2]) # 💡 중복검사 삭제
            
            canvases, canvas_infos = [], []
            if len(roi_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(roi_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

            if len(canvases) > 0:
                t_inf_start = time.time()
                res_pack = m2.predict(canvases, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(canvases)
                
                for c_idx, res in enumerate(res_pack):
                    for b in res.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                        for info in canvas_infos:
                            if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                s = info['scale']
                                orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 4. 제안 3: Ours (DAHI + Tetris) - Top 1 추출, 중복검사 완전삭제
        # =====================================================================
        elif method_name == "Ours (DAHI + Tetris)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=0.1, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            t_inf_start = time.time()
            r1 = m1.predict(img, conf=0.1, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                roi_boxes.append([bx1, by1, bx2, by2]) # 💡 중복검사 삭제
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            best_count, best_region = -1, None
            for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                    if count > best_count: 
                        best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                        
            if best_region and best_count > 0: 
                dense_regions.append(best_region)
                dx1, dy1, dx2, dy2 = best_region
                remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
            
            unified_infer_list = []
            dense_idx_list = []
            
            for dx1, dy1, dx2, dy2 in dense_regions:
                unified_infer_list.append(img[dy1:dy2, dx1:dx2])
                dense_idx_list.append((len(unified_infer_list) - 1, dx1, dy1, dx2, dy2))

            canvases, canvas_infos = [], []
            canvas_start_idx = -1
            if len(remaining_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(remaining_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

                if len(canvases) > 0:
                    canvas_start_idx = len(unified_infer_list)
                    unified_infer_list.extend(canvases)

            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for d_idx, dx1, dy1, dx2, dy2 in dense_idx_list:
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1; res_dense = res_all[d_idx]
                    for b in res_dense.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                
                if canvas_start_idx != -1:
                    res_pack = res_all[canvas_start_idx:]
                    for c_idx, res in enumerate(res_pack):
                        for b in res.boxes:
                            bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                            bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                            for info in canvas_infos:
                                if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                    if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                    s = info['scale']
                                    orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                    orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                    local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                    break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # 통계 저장
        img_total_time = time.time() - t_pipe_start
        is_hr = (w * h >= HR_THRESHOLD)
        target_keys = ['ALL', 'HR'] if is_hr else ['ALL', 'LR']
        for k in target_keys:
            stats[k]['count'] += 1
            stats[k]['inf_time'] += img_inf_time
            stats[k]['total_time'] += img_total_time
            stats[k]['inf_cnt'] += img_inf_cnt
            stats[k]['indices'].add(img_idx)

    # ---------------------------------------------------------
    # 💡 AP 연산 엔진
    # ---------------------------------------------------------
    def calc_metrics_for_subset(subset_indices):
        if not subset_indices: return {"AP50": 0, "AP50s": 0, "AP50m": 0, "AP50l": 0}
        
        sub_preds = [p for p in all_preds if p[0] in subset_indices]
        sub_preds.sort(key=lambda x: x[2], reverse=True) 
        unique_classes = set([gt[0] for idx in subset_indices for gt in all_gts[idx]])
        metrics = {'all': [], 'small': [], 'medium': [], 'large': []}
        
        for c in unique_classes:
            c_preds = [p for p in sub_preds if p[1] == c]
            for size_target in ['all', 'small', 'medium', 'large']:
                if size_target == 'all': c_gts = {idx: [list(g) for g in all_gts[idx] if g[0] == c] for idx in subset_indices}
                else: c_gts = {idx: [list(g) for g in all_gts[idx] if g[0] == c and g[6] == size_target] for idx in subset_indices}
                    
                npos = sum(len(gts) for gts in c_gts.values())
                if npos == 0: continue
                
                tp, fp = np.zeros(len(c_preds)), np.zeros(len(c_preds))
                for i, pred in enumerate(c_preds):
                    img_idx, _, _, px1, py1, px2, py2 = pred
                    pred_box = [px1, py1, px2, py2]
                    gts = c_gts[img_idx]
                    
                    pw, ph = px2 - px1, py2 - py1
                    p_size = get_size_category(pw, ph)
                    if size_target != 'all' and p_size != size_target: continue
                    
                    best_iou, best_idx = 0.5, -1
                    for j, gt in enumerate(gts):
                        if gt[5]: continue 
                        iou = calculate_iou(pred_box, gt[1:5])
                        if iou >= best_iou: best_iou, best_idx = iou, j
                            
                    if best_idx >= 0:
                        tp[i] = 1; gts[best_idx][5] = True
                    else:
                        fp[i] = 1
                        
                fp_cumsum, tp_cumsum = np.cumsum(fp), np.cumsum(tp)
                rec = tp_cumsum / npos
                prec = tp_cumsum / np.maximum(tp_cumsum + fp_cumsum, np.finfo(np.float64).eps)
                metrics[size_target].append(compute_ap(rec, prec))
                
        return {
            "AP50": np.mean(metrics['all']) if metrics['all'] else 0,
            "AP50s": np.mean(metrics['small']) if metrics['small'] else 0,
            "AP50m": np.mean(metrics['medium']) if metrics['medium'] else 0,
            "AP50l": np.mean(metrics['large']) if metrics['large'] else 0,
        }

    result_dict = {}
    for group in ['ALL', 'HR', 'LR']:
        c = stats[group]['count']
        res = calc_metrics_for_subset(stats[group]['indices'])
        res['Img_Cnt'] = c
        res['Avg_Inf_Cnt'] = stats[group]['inf_cnt'] / c if c else 0
        res['Avg_Inf_Time'] = (stats[group]['inf_time'] / c) * 1000 if c else 0
        res['Avg_Tot_Time'] = (stats[group]['total_time'] / c) * 1000 if c else 0
        result_dict[group] = res
        
    result_dict['Peak_VRAM'] = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0
    return result_dict

# =========================================================
# 실행 및 다중 표 그리기
# =========================================================
methods = [
    "UC (2x2 Uniform Crop)", 
    "Ours (DAHI Only)",
    "Ours (Tetris Only)",
    "Ours (DAHI + Tetris)"
]

final_stats = {}
for m in methods: 
    final_stats[m] = run_ablation_benchmark(m)

print("\n" + "="*140)
print(f"🏆 [Ablation Study] Component Contribution Analysis (Total {NUM_TEST_IMAGES} Images) 🏆")
print("="*140)
print(f"{'Method':<35} | {'Type':<4} | {'Img':<4} | {'AP50':<6} | {'AP50s':<6} | {'AP50m':<6} | {'AP50l':<6} | {'Inf Cnt':<7} | {'Inf Time':<9} | {'Tot Time':<9}")
print("-" * 140)

for m, groups in final_stats.items():
    for g in ['ALL', 'HR', 'LR']:
        s = groups[g]
        if s['Img_Cnt'] == 0: continue
        print(f"{m if g == 'ALL' else '':<35} | {g:<4} | {s['Img_Cnt']:<4} | {s['AP50']:.4f} | {s['AP50s']:.4f} | {s['AP50m']:.4f} | {s['AP50l']:.4f} | {s['Avg_Inf_Cnt']:4.1f} /i | {s['Avg_Inf_Time']:5.1f} ms | {s['Avg_Tot_Time']:5.1f} ms")
    print(f"{'':<35} > Peak VRAM: {groups['Peak_VRAM']:.1f} MB")
    print("-" * 140)

🚀 [Ablation Study] 필터링 족쇄 해제! 완벽한 재현 시작! (Total 1294 images)


⏳ Ours (DAHI + Tetris): 100%|██████████████████████████████| 1294/1294 [02:06<00:00, 10.23it/s]



🏆 [Ablation Study] Component Contribution Analysis (Total 5000 Images) 🏆
Method                              | Type | Img  | AP50   | AP50s  | AP50m  | AP50l  | Inf Cnt | Inf Time  | Tot Time 
--------------------------------------------------------------------------------------------------------------------------------------------
UC (2x2 Uniform Crop)               | ALL  | 1294 | 0.0000 | 0.0000 | 0.0000 | 0.0000 |  0.0 /i |   0.0 ms |   0.0 ms
                                    | HR   | 185  | 0.0000 | 0.0000 | 0.0000 | 0.0000 |  0.0 /i |   0.0 ms |   0.0 ms
                                    | LR   | 1109 | 0.0000 | 0.0000 | 0.0000 | 0.0000 |  0.0 /i |   0.0 ms |   0.0 ms
                                    > Peak VRAM: 42.8 MB
--------------------------------------------------------------------------------------------------------------------------------------------
Ours (DAHI Only)                    | ALL  | 1294 | 0.3998 | 0.2610 | 0.4550 | 0.4675 |  5.8 /i |  59.4 ms |  77.

In [ ]:
pip install pycocotools

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import cv2
import os
import time
import numpy as np
import tqdm
import torch
from ultralytics import YOLO

import contextlib
import io

# 💡 [NEW] 공식 COCO API 임포트
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# =========================================================
# ⚙️ 하이퍼파라미터 (Hyperparameters)
# =========================================================
MODEL_FILTER_PATH = 'model/best_nano.pt'
MODEL_MAIN_PATH = 'model/best_small.pt'

CONF_GLOBAL = 0.3
CONF_FILTER = 0.1     
CONF_DENSE = 0.3
CONF_TETRIS = 0.3
CONF_UC = 0.3

IOU_FILTER_MATCH = 0.97
NMS_CONF_THRESH = 0.3
NMS_IOU_THRESH = 0.4    

DENSE_WINDOW_SIZE = 512
DENSE_STEP = 320      

MERGE_PAD = 16
CROP_PAD_LARGE = 80     
CROP_PAD_SMALL = 16
CROP_PAD_THRESH = 1000

CANVAS_SIZE = 960
CANVAS_MARGIN = 2
CANVAS_BG_COLOR = 114

UPSCALE_RATIO = 1.5        
UPSCALE_MAX_THRESH = 200    

NUM_TEST_IMAGES = 5000
HR_THRESHOLD = 1920 * 1080 

# =========================================================
dataset_root = 'data/test'
img_dir, lbl_dir = os.path.join(dataset_root, 'images'), os.path.join(dataset_root, 'labels')
img_list = sorted(os.listdir(img_dir))[:NUM_TEST_IMAGES]

print(f"🚀 [Ablation Study] 필터링 족쇄 해제! 완벽한 재현 시작! (Total {len(img_list)} images)")

m1 = YOLO(MODEL_FILTER_PATH)
m2 = YOLO(MODEL_MAIN_PATH)

def calculate_iou(box1, box2):
    xi1, yi1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    xi2, yi2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])

def get_size_category(w, h):
    area = w * h
    if area < 32 ** 2: return 'small'
    elif area < 96 ** 2: return 'medium'
    else: return 'large'

def merge_clusters_dynamic(boxes, img_w, img_h, merge_pad=MERGE_PAD):
    if not len(boxes): return []
    def get_padded(b, pad): return [max(0, b[0]-pad), max(0, b[1]-pad), min(img_w, b[2]+pad), min(img_h, b[3]+pad)]
    def is_overlap(b1, b2):
        p1, p2 = get_padded(b1, merge_pad), get_padded(b2, merge_pad)
        return (min(p1[2], p2[2]) > max(p1[0], p2[0])) and (min(p1[3], p2[3]) > max(p1[1], p2[1]))
    curr = boxes.copy()
    while True:
        merged, flags = [], [False]*len(curr)
        for i in range(len(curr)):
            if flags[i]: continue
            b = curr[i]
            for j in range(i+1, len(curr)):
                if not flags[j] and is_overlap(b, curr[j]):
                    b = [min(b[0], curr[j][0]), min(b[1], curr[j][1]), max(b[2], curr[j][2]), max(b[3], curr[j][3])]
                    flags[j] = True
            merged.append(b)
        if len(merged) == len(curr): break
        curr = merged
    final_boxes = []
    for b in curr:
        bw, bh = b[2] - b[0], b[3] - b[1]
        crop_pad = CROP_PAD_LARGE if max(bw, bh) < CROP_PAD_THRESH else CROP_PAD_SMALL 
        final_boxes.append(get_padded(b, crop_pad))
    return final_boxes

def run_official_ablation_benchmark(method_name):
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        
    all_gts = {}; all_preds = []
    
    # 💡 [NEW] 이미지 메타데이터 저장을 위한 변수 추가 (COCO 포맷용)
    img_infos = {} 
    
    stats = {
        'ALL': {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'HR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'LR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()}
    }

    pbar = tqdm.tqdm(img_list, desc=f"⏳ {method_name}", bar_format='{l_bar}{bar:30}{r_bar}')
    for img_idx, img_name in enumerate(pbar):
        img_path, lbl_path = os.path.join(img_dir, img_name), os.path.join(lbl_dir, img_name.replace('.jpg', '.txt'))
        img = cv2.imread(img_path); h, w, _ = img.shape
        
        # COCO 포맷 생성을 위해 이미지 정보 저장
        img_infos[img_idx] = {'w': w, 'h': h, 'name': img_name}
        
        gts = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    c, xc, yc, bw, bh = map(float, line.split())
                    gts.append([int(c), (xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h]) # visited, size_cat 삭제 (COCO가 알아서 함)
        all_gts[img_idx] = gts

        t_pipe_start = time.time()
        img_inf_time, img_inf_cnt = 0, 0
        
        # =====================================================================
        # 🤖 여기에 기존 추론(Inference) 코드 블록 (UC, DAHI, Tetris 등)이 
        # 이전 코드와 100% 동일하게 들어갑니다. (코드 길이상 생략)
        # =====================================================================

                # =====================================================================
        # 1. Baseline: UC (2x2 Uniform Crop)
        # =====================================================================
        if method_name == "UC (2x2 Uniform Crop)":
            ch, cw = h // 2, w // 2
            # crops, offsets = [img], [(0, 0)]
            # for y in [0, ch]:
            #     for x in [0, cw]:
            #         crops.append(img[y:y+ch, x:x+cw])
            #         offsets.append((x, y))
            
            # t_inf_start = time.time()
            # results2 = m2.predict(crops, conf=CONF_UC, verbose=False, batch=5)
            # img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 5 
            
            # temp_boxes, temp_scores, temp_classes = [], [], []
            # for i, res in enumerate(results2):
            #     ox, oy = offsets[i]
            #     for b in res.boxes:
            #         temp_boxes.append([b.xyxy[0][0]+ox, b.xyxy[0][1]+oy, b.xyxy[0][2]+ox, b.xyxy[0][3]+oy])
            #         temp_scores.append(float(b.conf[0])); temp_classes.append(int(b.cls[0]))
                    
            # for c in set(temp_classes):
            #     c_boxes = [b for j, b in enumerate(temp_boxes) if temp_classes[j] == c]
            #     c_scores = [s for j, s in enumerate(temp_scores) if temp_classes[j] == c]
            #     cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
            #     indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
            #     if len(indices) > 0:
            #         for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 2. 제안 1: Ours (DAHI Only) - 중복필터 완전삭제, 모든 객체 반복 추출
        # =====================================================================
        elif method_name == "Ours (DAHI Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=0.3, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            t_inf_start = time.time()
            r1 = m1.predict(img, conf=0.1, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                is_found = False
                for gb in global_final_boxes:
                    if calculate_iou([bx1, by1, bx2, by2], gb) > IOU_FILTER_MATCH: is_found = True; break
                if not is_found: roi_boxes.append([bx1, by1, bx2, by2])
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            while len(remaining_boxes) > 0:
                best_count, best_region = -1, None
                for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                        count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                        if count > best_count: 
                            best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                if best_region and best_count >= 1:
                    dense_regions.append(best_region)
                    dx1, dy1, dx2, dy2 = best_region
                    remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                else: break
            
            unified_infer_list = [img[dy1:dy2, dx1:dx2] for dx1, dy1, dx2, dy2 in dense_regions]
            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for idx, (dx1, dy1, dx2, dy2) in enumerate(dense_regions):
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1
                    for b in res_all[idx].boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
            
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 3. 제안 2: Ours (Tetris Only) - 중복필터 완전삭제
        # =====================================================================
        elif method_name == "Ours (Tetris Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=0.3, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            t_inf_start = time.time()
            r1 = m1.predict(img, conf=0.1, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                is_found = False
                for gb in global_final_boxes:
                    if calculate_iou([bx1, by1, bx2, by2], gb) > IOU_FILTER_MATCH: is_found = True; break
                if not is_found: roi_boxes.append([bx1, by1, bx2, by2])
            
            canvases, canvas_infos = [], []
            if len(roi_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(roi_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

            if len(canvases) > 0:
                t_inf_start = time.time()
                res_pack = m2.predict(canvases, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(canvases)
                
                for c_idx, res in enumerate(res_pack):
                    for b in res.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                        for info in canvas_infos:
                            if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                s = info['scale']
                                orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 4. 제안 3: Ours (DAHI + Tetris) - Top 1 추출, 중복검사 완전삭제
        # =====================================================================
        elif method_name == "Ours (DAHI + Tetris)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=0.3, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            t_inf_start = time.time()
            r1 = m1.predict(img, conf=0.1, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                is_found = False
                for gb in global_final_boxes:
                    if calculate_iou([bx1, by1, bx2, by2], gb) > IOU_FILTER_MATCH: is_found = True; break
                if not is_found: roi_boxes.append([bx1, by1, bx2, by2])
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            best_count, best_region = -1, None
            for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                    if count > best_count: 
                        best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                        
            if best_region and best_count > 0: 
                dense_regions.append(best_region)
                dx1, dy1, dx2, dy2 = best_region
                remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
            
            unified_infer_list = []
            dense_idx_list = []
            
            for dx1, dy1, dx2, dy2 in dense_regions:
                unified_infer_list.append(img[dy1:dy2, dx1:dx2])
                dense_idx_list.append((len(unified_infer_list) - 1, dx1, dy1, dx2, dy2))

            canvases, canvas_infos = [], []
            canvas_start_idx = -1
            if len(remaining_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(remaining_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

                if len(canvases) > 0:
                    canvas_start_idx = len(unified_infer_list)
                    unified_infer_list.extend(canvases)

            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for d_idx, dx1, dy1, dx2, dy2 in dense_idx_list:
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1; res_dense = res_all[d_idx]
                    for b in res_dense.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                
                if canvas_start_idx != -1:
                    res_pack = res_all[canvas_start_idx:]
                    for c_idx, res in enumerate(res_pack):
                        for b in res.boxes:
                            bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                            bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                            for info in canvas_infos:
                                if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                    if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                    s = info['scale']
                                    orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                    orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                    local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                    break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])
        
        # 통계 저장
        img_total_time = time.time() - t_pipe_start
        is_hr = (w * h >= HR_THRESHOLD)
        target_keys = ['ALL', 'HR'] if is_hr else ['ALL', 'LR']
        for k in target_keys:
            stats[k]['count'] += 1
            stats[k]['inf_time'] += img_inf_time
            stats[k]['total_time'] += img_total_time
            stats[k]['inf_cnt'] += img_inf_cnt
            stats[k]['indices'].add(img_idx)

    # ---------------------------------------------------------
    # 💡 [NEW] 공식 COCO API 연산 엔진 도입
    # ---------------------------------------------------------
    def calc_official_coco_metrics(subset_indices):
        if not subset_indices: return {"AP50:95": 0, "AP50": 0, "AP50s": 0, "AP50m": 0, "AP50l": 0}
        
        # 1. COCO GT (Ground Truth) 딕셔너리 생성
        gt_dict = {"images": [], "annotations": [], "categories": []}
        for i in range(10): # VisDrone 기본 10개 클래스 가정
            gt_dict["categories"].append({"id": i, "name": f"class_{i}"})
            
        ann_id = 1
        for img_idx in subset_indices:
            info = img_infos[img_idx]
            gt_dict["images"].append({"id": img_idx, "width": info['w'], "height": info['h'], "file_name": info['name']})
            for gt in all_gts[img_idx]:
                c, x1, y1, x2, y2 = gt
                bw, bh = x2 - x1, y2 - y1
                gt_dict["annotations"].append({
                    "id": ann_id, "image_id": img_idx, "category_id": int(c),
                    "bbox": [x1, y1, bw, bh], "area": bw * bh, "iscrowd": 0
                })
                ann_id += 1

        cocoGt = COCO()
        cocoGt.dataset = gt_dict
        cocoGt.createIndex()

        # 2. COCO Pred (Predictions) 리스트 생성
        sub_preds = [p for p in all_preds if p[0] in subset_indices]
        pred_list = []
        for pred in sub_preds:
            img_idx, c, score, x1, y1, x2, y2 = pred
            bw, bh = x2 - x1, y2 - y1
            pred_list.append({
                "image_id": img_idx, "category_id": int(c),
                "bbox": [x1, y1, bw, bh], "score": float(score)
            })

        if not pred_list: return {"AP50:95": 0, "AP50": 0, "AP50s": 0, "AP50m": 0, "AP50l": 0}

        cocoDt = cocoGt.loadRes(pred_list)

        # 3. COCO 평가 객체 초기화 및 💡 VisDrone 특화 세팅 (maxDets)
        cocoEval = COCOeval(cocoGt, cocoDt, 'bbox')
        
        # [중요] 논문용 세팅: VisDrone은 한 장에 객체가 수백 개이므로 maxDets를 500으로 상향!
        cocoEval.params.maxDets = [100, 300, 500] 
        
        cocoEval.evaluate()
        cocoEval.accumulate()
        
        # 💡 [FIX] summarize()를 반드시 호출해야 stats 배열이 생성됩니다!
        # 콘솔 창이 지저분해지는 것을 막기 위해 출력을 잠시 막아둡니다(Redirect).
        with contextlib.redirect_stdout(io.StringIO()):
            cocoEval.summarize()

        # 4. 평가 지표 추출 (cocoEval.stats 인덱스 매핑)
        # 만약 객체가 아예 없어서 stats가 12개가 안 채워졌을 경우를 대비한 안전장치 추가
        if len(cocoEval.stats) < 12:
            return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}

        return {
            "AP50:95": cocoEval.stats[0],
            "AP50": cocoEval.stats[1],
            "AP_small": cocoEval.stats[3],  
            "AP_medium": cocoEval.stats[4], 
            "AP_large": cocoEval.stats[5],
        }

    # 결과 취합
    result_dict = {}
    for group in ['ALL', 'HR', 'LR']:
        c = stats[group]['count']
        res = calc_official_coco_metrics(stats[group]['indices'])
        res['Img_Cnt'] = c
        res['Avg_Inf_Cnt'] = stats[group]['inf_cnt'] / c if c else 0
        res['Avg_Inf_Time'] = (stats[group]['inf_time'] / c) * 1000 if c else 0
        res['Avg_Tot_Time'] = (stats[group]['total_time'] / c) * 1000 if c else 0
        result_dict[group] = res
        
    result_dict['Peak_VRAM'] = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0
    return result_dict

# =========================================================
# 실행 및 다중 표 그리기 (Official COCO Protocol 반영)
# =========================================================
methods = [
    "UC (2x2 Uniform Crop)", 
    "Ours (DAHI Only)",
    "Ours (Tetris Only)",
    "Ours (DAHI + Tetris)"
]

final_stats = {}
for m in methods: 
    final_stats[m] = run_official_ablation_benchmark(m)

print("\n" + "="*145)
print(f"🏆 [Ablation Study] Official COCO Protocol Evaluation (Total {NUM_TEST_IMAGES} Images) 🏆")
print("="*145)
# 💡 COCO 핵심 지표인 mAP (AP50:95)를 추가하고, 크기별 지표명을 COCO 표준(APs, APm, APl)으로 변경
print(f"{'Method':<32} | {'Type':<4} | {'Img':<4} | {'mAP':<6} | {'AP50':<6} | {'APs':<6} | {'APm':<6} | {'APl':<6} | {'Inf Cnt':<7} | {'Inf Time':<9} | {'Tot Time':<9}")
print("-" * 145)

for m, groups in final_stats.items():
    for g in ['ALL', 'HR', 'LR']:
        s = groups[g]
        if s['Img_Cnt'] == 0: continue
        
        # 💡 리턴된 COCO 딕셔너리 키값 매핑
        mAP  = s.get('AP50:95', 0.0)
        ap50 = s.get('AP50', 0.0)
        aps  = s.get('AP_small', 0.0)
        apm  = s.get('AP_medium', 0.0)
        apl  = s.get('AP_large', 0.0)
        
        print(f"{m if g == 'ALL' else '':<32} | {g:<4} | {s['Img_Cnt']:<4} | {mAP:.4f} | {ap50:.4f} | {aps:.4f} | {apm:.4f} | {apl:.4f} | {s['Avg_Inf_Cnt']:4.1f} /i | {s['Avg_Inf_Time']:5.1f} ms | {s['Avg_Tot_Time']:5.1f} ms")
    
    print(f"{'':<32} > Peak VRAM: {groups.get('Peak_VRAM', 0.0):.1f} MB")
    print("-" * 145)

🚀 [Ablation Study] 필터링 족쇄 해제! 완벽한 재현 시작! (Total 430 images)


⏳ UC (2x2 Uniform Crop): 100%|██████████████████████████████| 430/430 [00:05<00:00, 78.71it/s] 


creating index...
index created!
creating index...
index created!
creating index...
index created!


⏳ Ours (DAHI Only): 100%|██████████████████████████████| 430/430 [00:40<00:00, 10.61it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.16s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=10.32s).
Accumulating evaluation results...
DONE (t=0.46s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=3.40s).
Accumulating evaluation results...
DONE (t=0.12s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.15s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=6.94s).
Accumulating evaluation results...
DONE (t=0.37s).


⏳ Ours (Tetris Only): 100%|██████████████████████████████| 430/430 [00:35<00:00, 12.10it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.02s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=9.34s).
Accumulating evaluation results...
DONE (t=0.43s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=3.04s).
Accumulating evaluation results...
DONE (t=0.11s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=6.52s).
Accumulating evaluation results...
DONE (t=0.35s).


⏳ Ours (DAHI + Tetris): 100%|██████████████████████████████| 430/430 [00:36<00:00, 11.68it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.16s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=10.19s).
Accumulating evaluation results...
DONE (t=0.47s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=3.13s).
Accumulating evaluation results...
DONE (t=0.12s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=7.06s).
Accumulating evaluation results...
DONE (t=0.37s).

🏆 [Ablation Study] Official COCO Protocol Evaluation (Total 5000 Images) 🏆
Method                           | Type | Img  | mAP    | AP50   | APs    | APm    | APl    | Inf Cnt | Inf Time  | Tot Time 
--------------------------------------------------------------

In [ ]:
import cv2
import os
import time
import numpy as np
import tqdm
import torch
from ultralytics import YOLO

import contextlib
import io

# 💡 [NEW] 공식 COCO API 임포트
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# =========================================================
# ⚙️ 하이퍼파라미터 (Hyperparameters) - Single Model Edition
# =========================================================
# MODEL_FILTER_PATH = 'model/best_nano.pt' # 💡 보조 모델 완전히 삭제!
MODEL_MAIN_PATH = 'model/best_small.pt'

CONF_GLOBAL = 0.3
CONF_FILTER = 0.1     
CONF_DENSE = 0.3
CONF_TETRIS = 0.3
CONF_UC = 0.3

# IOU_FILTER_MATCH = 0.97 # 💡 사용 안함 (중복검사 삭제됨)
NMS_CONF_THRESH = 0.3
NMS_IOU_THRESH = 0.4    

DENSE_WINDOW_SIZE = 512
DENSE_STEP = 320      

MERGE_PAD = 16
CROP_PAD_LARGE = 80     
CROP_PAD_SMALL = 16
CROP_PAD_THRESH = 200

CANVAS_SIZE = 960
CANVAS_MARGIN = 2
CANVAS_BG_COLOR = 114

UPSCALE_RATIO = 1.5     
UPSCALE_MAX_THRESH = 200

NUM_TEST_IMAGES = 5000
HR_THRESHOLD = 1920 * 1080 

# =========================================================
dataset_root = 'data/test'
img_dir, lbl_dir = os.path.join(dataset_root, 'images'), os.path.join(dataset_root, 'labels')
img_list = sorted(os.listdir(img_dir))[:NUM_TEST_IMAGES]

print(f"🚀 [Ablation Study] Single-Model Pipeline! 완벽한 재현 시작! (Total {len(img_list)} images)")

# m1 = YOLO(MODEL_FILTER_PATH) # 💡 보조 모델 삭제
m2 = YOLO(MODEL_MAIN_PATH)

def calculate_iou(box1, box2):
    xi1, yi1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    xi2, yi2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])

def get_size_category(w, h):
    area = w * h
    if area < 32 ** 2: return 'small'
    elif area < 96 ** 2: return 'medium'
    else: return 'large'

def merge_clusters_dynamic(boxes, img_w, img_h, merge_pad=MERGE_PAD):
    if not len(boxes): return []
    def get_padded(b, pad): return [max(0, b[0]-pad), max(0, b[1]-pad), min(img_w, b[2]+pad), min(img_h, b[3]+pad)]
    def is_overlap(b1, b2):
        p1, p2 = get_padded(b1, merge_pad), get_padded(b2, merge_pad)
        return (min(p1[2], p2[2]) > max(p1[0], p2[0])) and (min(p1[3], p2[3]) > max(p1[1], p2[1]))
    curr = boxes.copy()
    while True:
        merged, flags = [], [False]*len(curr)
        for i in range(len(curr)):
            if flags[i]: continue
            b = curr[i]
            for j in range(i+1, len(curr)):
                if not flags[j] and is_overlap(b, curr[j]):
                    b = [min(b[0], curr[j][0]), min(b[1], curr[j][1]), max(b[2], curr[j][2]), max(b[3], curr[j][3])]
                    flags[j] = True
            merged.append(b)
        if len(merged) == len(curr): break
        curr = merged
    final_boxes = []
    for b in curr:
        bw, bh = b[2] - b[0], b[3] - b[1]
        crop_pad = CROP_PAD_LARGE if max(bw, bh) < CROP_PAD_THRESH else CROP_PAD_SMALL 
        final_boxes.append(get_padded(b, crop_pad))
    return final_boxes

def run_official_ablation_benchmark(method_name):
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        
    all_gts = {}; all_preds = []
    img_infos = {} 
    
    stats = {
        'ALL': {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'HR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'LR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()}
    }

    pbar = tqdm.tqdm(img_list, desc=f"⏳ {method_name}", bar_format='{l_bar}{bar:30}{r_bar}')
    for img_idx, img_name in enumerate(pbar):
        img_path, lbl_path = os.path.join(img_dir, img_name), os.path.join(lbl_dir, img_name.replace('.jpg', '.txt'))
        img = cv2.imread(img_path); h, w, _ = img.shape
        
        img_infos[img_idx] = {'w': w, 'h': h, 'name': img_name}
        
        gts = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    c, xc, yc, bw, bh = map(float, line.split())
                    gts.append([int(c), (xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h]) 
        all_gts[img_idx] = gts

        t_pipe_start = time.time()
        img_inf_time, img_inf_cnt = 0, 0
        
        # =====================================================================
        # 1. Baseline: UC (2x2 Uniform Crop)
        # =====================================================================
        if method_name == "UC (2x2 Uniform Crop)":
            # UC 평가가 너무 오래 걸려 비활성화
            pass 

        # =====================================================================
        # 2. 제안 1: Ours (DAHI Only)
        # =====================================================================
        elif method_name == "Ours (DAHI Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            roi_boxes = []
            
            # 💡 [핵심 최적화] 메인 모델(m2)로 CONF_FILTER(0.1) 기준 단 1번만 스캔
            t_inf_start = time.time()
            res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global_all[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = float(b.conf[0])
                
                # 1. 글로벌 확정 박스 (0.3 이상)
                if conf >= CONF_GLOBAL:
                    boosted_conf = min(1.0, conf * 1.10)
                    global_final_boxes.append([bx1, by1, bx2, by2])
                    global_final_scores.append(boosted_conf)
                    global_final_classes.append(int(b.cls[0]))
                
                # 2. ROI 박스 등록 (모든 탐지 객체 재검사)
                roi_boxes.append([bx1, by1, bx2, by2])
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            while len(remaining_boxes) > 0:
                best_count, best_region = -1, None
                for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                        count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                        if count > best_count: 
                            best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                if best_region and best_count >= 1:
                    dense_regions.append(best_region)
                    dx1, dy1, dx2, dy2 = best_region
                    remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                else: break
            
            unified_infer_list = [img[dy1:dy2, dx1:dx2] for dx1, dy1, dx2, dy2 in dense_regions]
            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for idx, (dx1, dy1, dx2, dy2) in enumerate(dense_regions):
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1
                    for b in res_all[idx].boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
            
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 3. 제안 2: Ours (Tetris Only)
        # =====================================================================
        elif method_name == "Ours (Tetris Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            roi_boxes = []
            
            t_inf_start = time.time()
            res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global_all[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = float(b.conf[0])
                if conf >= CONF_GLOBAL:
                    global_final_boxes.append([bx1, by1, bx2, by2])
                    global_final_scores.append(min(1.0, conf * 1.10))
                    global_final_classes.append(int(b.cls[0]))
                roi_boxes.append([bx1, by1, bx2, by2])
            
            canvases, canvas_infos = [], []
            if len(roi_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(roi_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

            if len(canvases) > 0:
                t_inf_start = time.time()
                res_pack = m2.predict(canvases, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(canvases)
                
                for c_idx, res in enumerate(res_pack):
                    for b in res.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                        for info in canvas_infos:
                            if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                s = info['scale']
                                orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 4. 제안 3: Ours (DAHI + Tetris)
        # =====================================================================
        elif method_name == "Ours (DAHI + Tetris)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            roi_boxes = []
            
            t_inf_start = time.time()
            res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global_all[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = float(b.conf[0])
                if conf >= CONF_GLOBAL:
                    global_final_boxes.append([bx1, by1, bx2, by2])
                    global_final_scores.append(min(1.0, conf * 1.10))
                    global_final_classes.append(int(b.cls[0]))
                roi_boxes.append([bx1, by1, bx2, by2])
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            best_count, best_region = -1, None
            for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                    if count > best_count: 
                        best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                        
            if best_region and best_count > 0: 
                dense_regions.append(best_region)
                dx1, dy1, dx2, dy2 = best_region
                remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
            
            unified_infer_list = []
            dense_idx_list = []
            
            for dx1, dy1, dx2, dy2 in dense_regions:
                unified_infer_list.append(img[dy1:dy2, dx1:dx2])
                dense_idx_list.append((len(unified_infer_list) - 1, dx1, dy1, dx2, dy2))

            canvases, canvas_infos = [], []
            canvas_start_idx = -1
            if len(remaining_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(remaining_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

                if len(canvases) > 0:
                    canvas_start_idx = len(unified_infer_list)
                    unified_infer_list.extend(canvases)

            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for d_idx, dx1, dy1, dx2, dy2 in dense_idx_list:
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1; res_dense = res_all[d_idx]
                    for b in res_dense.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                
                if canvas_start_idx != -1:
                    res_pack = res_all[canvas_start_idx:]
                    for c_idx, res in enumerate(res_pack):
                        for b in res.boxes:
                            bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                            bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                            for info in canvas_infos:
                                if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                    if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                    s = info['scale']
                                    orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                    orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                    local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                    break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # 통계 저장
        img_total_time = time.time() - t_pipe_start
        is_hr = (w * h >= HR_THRESHOLD)
        target_keys = ['ALL', 'HR'] if is_hr else ['ALL', 'LR']
        for k in target_keys:
            stats[k]['count'] += 1
            stats[k]['inf_time'] += img_inf_time
            stats[k]['total_time'] += img_total_time
            stats[k]['inf_cnt'] += img_inf_cnt
            stats[k]['indices'].add(img_idx)

    # ---------------------------------------------------------
    # 💡 공식 COCO API 연산 엔진
    # ---------------------------------------------------------
    def calc_official_coco_metrics(subset_indices):
        if not subset_indices: return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}
        
        gt_dict = {"images": [], "annotations": [], "categories": []}
        for i in range(10): gt_dict["categories"].append({"id": i, "name": f"class_{i}"})
            
        ann_id = 1
        for img_idx in subset_indices:
            info = img_infos[img_idx]
            gt_dict["images"].append({"id": img_idx, "width": info['w'], "height": info['h'], "file_name": info['name']})
            for gt in all_gts[img_idx]:
                c, x1, y1, x2, y2 = gt
                bw, bh = x2 - x1, y2 - y1
                gt_dict["annotations"].append({"id": ann_id, "image_id": img_idx, "category_id": int(c), "bbox": [x1, y1, bw, bh], "area": bw * bh, "iscrowd": 0})
                ann_id += 1

        cocoGt = COCO()
        cocoGt.dataset = gt_dict
        cocoGt.createIndex()

        sub_preds = [p for p in all_preds if p[0] in subset_indices]
        pred_list = []
        for pred in sub_preds:
            img_idx, c, score, x1, y1, x2, y2 = pred
            bw, bh = x2 - x1, y2 - y1
            pred_list.append({"image_id": img_idx, "category_id": int(c), "bbox": [x1, y1, bw, bh], "score": float(score)})

        if not pred_list: return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}

        cocoDt = cocoGt.loadRes(pred_list)

        cocoEval = COCOeval(cocoGt, cocoDt, 'bbox')
        cocoEval.params.maxDets = [100, 300, 500] 
        
        cocoEval.evaluate()
        cocoEval.accumulate()
        
        with contextlib.redirect_stdout(io.StringIO()):
            cocoEval.summarize()

        if len(cocoEval.stats) < 12:
            return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}

        return {
            "AP50:95": cocoEval.stats[0],
            "AP50": cocoEval.stats[1],
            "AP_small": cocoEval.stats[3],  
            "AP_medium": cocoEval.stats[4], 
            "AP_large": cocoEval.stats[5],
        }

    result_dict = {}
    for group in ['ALL', 'HR', 'LR']:
        c = stats[group]['count']
        res = calc_official_coco_metrics(stats[group]['indices'])
        res['Img_Cnt'] = c
        res['Avg_Inf_Cnt'] = stats[group]['inf_cnt'] / c if c else 0
        res['Avg_Inf_Time'] = (stats[group]['inf_time'] / c) * 1000 if c else 0
        res['Avg_Tot_Time'] = (stats[group]['total_time'] / c) * 1000 if c else 0
        result_dict[group] = res
        
    result_dict['Peak_VRAM'] = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0
    return result_dict

# =========================================================
# 실행 및 다중 표 그리기 (Official COCO Protocol)
# =========================================================
methods = [
    "UC (2x2 Uniform Crop)", 
    "Ours (DAHI Only)",
    "Ours (Tetris Only)",
    "Ours (DAHI + Tetris)"
]

final_stats = {}
for m in methods: 
    final_stats[m] = run_official_ablation_benchmark(m)

print("\n" + "="*145)
print(f"🏆 [Ablation Study] Single Model & Official COCO Protocol Evaluation (Total {NUM_TEST_IMAGES} Images) 🏆")
print("="*145)
print(f"{'Method':<32} | {'Type':<4} | {'Img':<4} | {'mAP':<6} | {'AP50':<6} | {'APs':<6} | {'APm':<6} | {'APl':<6} | {'Inf Cnt':<7} | {'Inf Time':<9} | {'Tot Time':<9}")
print("-" * 145)

for m, groups in final_stats.items():
    for g in ['ALL', 'HR', 'LR']:
        s = groups[g]
        if s['Img_Cnt'] == 0: continue
        
        mAP  = s.get('AP50:95', 0.0)
        ap50 = s.get('AP50', 0.0)
        aps  = s.get('AP_small', 0.0)
        apm  = s.get('AP_medium', 0.0)
        apl  = s.get('AP_large', 0.0)
        
        print(f"{m if g == 'ALL' else '':<32} | {g:<4} | {s['Img_Cnt']:<4} | {mAP:.4f} | {ap50:.4f} | {aps:.4f} | {apm:.4f} | {apl:.4f} | {s['Avg_Inf_Cnt']:4.1f} /i | {s['Avg_Inf_Time']:5.1f} ms | {s['Avg_Tot_Time']:5.1f} ms")
    
    print(f"{'':<32} > Peak VRAM: {groups.get('Peak_VRAM', 0.0):.1f} MB")
    print("-" * 145)

🚀 [Ablation Study] Single-Model Pipeline! 완벽한 재현 시작! (Total 430 images)


⏳ UC (2x2 Uniform Crop):  31%|█████████▎                    | 134/430 [00:02<00:04, 66.53it/s]


KeyboardInterrupt: 

In [ ]:
import cv2
import os
import time
import numpy as np
import tqdm
import torch
from ultralytics import YOLO

import contextlib
import io

# 💡 [NEW] 공식 COCO API 임포트
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# =========================================================
# ⚙️ 하이퍼파라미터 (Hyperparameters)
# =========================================================
MODEL_FILTER_PATH = 'model/best_nano.pt'
MODEL_MAIN_PATH = 'model/best_small.pt'

CONF_GLOBAL = 0.3
CONF_FILTER = 0.1     
CONF_DENSE = 0.3
CONF_TETRIS = 0.3
CONF_UC = 0.3

IOU_FILTER_MATCH = 0.97
NMS_CONF_THRESH = 0.3
NMS_IOU_THRESH = 0.4    

DENSE_WINDOW_SIZE = 512
DENSE_STEP = 320      

MERGE_PAD = 16
CROP_PAD_LARGE = 80     
CROP_PAD_SMALL = 16
CROP_PAD_THRESH = 200

CANVAS_SIZE = 960
CANVAS_MARGIN = 2
CANVAS_BG_COLOR = 114

UPSCALE_RATIO = 1.5        
UPSCALE_MAX_THRESH = 200    

NUM_TEST_IMAGES = 5000
HR_THRESHOLD = 1920 * 1080 

# =========================================================
dataset_root = 'data/valid'
img_dir, lbl_dir = os.path.join(dataset_root, 'images'), os.path.join(dataset_root, 'labels')
img_list = sorted(os.listdir(img_dir))[:NUM_TEST_IMAGES]

print(f"🚀 [Ablation Study] 필터링 족쇄 해제! 완벽한 재현 시작! (Total {len(img_list)} images)")

m1 = YOLO(MODEL_FILTER_PATH)
m2 = YOLO(MODEL_MAIN_PATH)

def calculate_iou(box1, box2):
    xi1, yi1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    xi2, yi2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])

def get_size_category(w, h):
    area = w * h
    if area < 32 ** 2: return 'small'
    elif area < 96 ** 2: return 'medium'
    else: return 'large'

def merge_clusters_dynamic(boxes, img_w, img_h, merge_pad=MERGE_PAD):
    if not len(boxes): return []
    def get_padded(b, pad): return [max(0, b[0]-pad), max(0, b[1]-pad), min(img_w, b[2]+pad), min(img_h, b[3]+pad)]
    def is_overlap(b1, b2):
        p1, p2 = get_padded(b1, merge_pad), get_padded(b2, merge_pad)
        return (min(p1[2], p2[2]) > max(p1[0], p2[0])) and (min(p1[3], p2[3]) > max(p1[1], p2[1]))
    curr = boxes.copy()
    while True:
        merged, flags = [], [False]*len(curr)
        for i in range(len(curr)):
            if flags[i]: continue
            b = curr[i]
            for j in range(i+1, len(curr)):
                if not flags[j] and is_overlap(b, curr[j]):
                    b = [min(b[0], curr[j][0]), min(b[1], curr[j][1]), max(b[2], curr[j][2]), max(b[3], curr[j][3])]
                    flags[j] = True
            merged.append(b)
        if len(merged) == len(curr): break
        curr = merged
    final_boxes = []
    for b in curr:
        bw, bh = b[2] - b[0], b[3] - b[1]
        crop_pad = CROP_PAD_LARGE if max(bw, bh) < CROP_PAD_THRESH else CROP_PAD_SMALL 
        final_boxes.append(get_padded(b, crop_pad))
    return final_boxes

def run_official_ablation_benchmark(method_name):
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        
    all_gts = {}; all_preds = []
    
    # 💡 [NEW] 이미지 메타데이터 저장을 위한 변수 추가 (COCO 포맷용)
    img_infos = {} 
    
    stats = {
        'ALL': {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'HR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'LR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()}
    }

    pbar = tqdm.tqdm(img_list, desc=f"⏳ {method_name}", bar_format='{l_bar}{bar:30}{r_bar}')
    for img_idx, img_name in enumerate(pbar):
        img_path, lbl_path = os.path.join(img_dir, img_name), os.path.join(lbl_dir, img_name.replace('.jpg', '.txt'))
        img = cv2.imread(img_path); h, w, _ = img.shape
        
        # COCO 포맷 생성을 위해 이미지 정보 저장
        img_infos[img_idx] = {'w': w, 'h': h, 'name': img_name}
        
        gts = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    c, xc, yc, bw, bh = map(float, line.split())
                    gts.append([int(c), (xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h]) # visited, size_cat 삭제 (COCO가 알아서 함)
        all_gts[img_idx] = gts

        t_pipe_start = time.time()
        img_inf_time, img_inf_cnt = 0, 0
        
        # =====================================================================
        # 🤖 여기에 기존 추론(Inference) 코드 블록 (UC, DAHI, Tetris 등)이 
        # 이전 코드와 100% 동일하게 들어갑니다. (코드 길이상 생략)
        # =====================================================================

        # =====================================================================
        # 1. Baseline: UC (2x2 Uniform Crop)
        # =====================================================================
        if method_name == "UC (2x2 Uniform Crop)":
            ch, cw = h // 2, w // 2
            crops, offsets = [img], [(0, 0)]
            for y in [0, ch]:
                for x in [0, cw]:
                    crops.append(img[y:y+ch, x:x+cw])
                    offsets.append((x, y))
            
            t_inf_start = time.time()
            results2 = m2.predict(crops, conf=CONF_UC, verbose=False, batch=5)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 5 
            
            temp_boxes, temp_scores, temp_classes = [], [], []
            for i, res in enumerate(results2):
                ox, oy = offsets[i]
                for b in res.boxes:
                    # 💡 [FIX] .item() 을 사용하여 텐서를 완벽하게 파이썬 float으로 변환!
                    bx1 = b.xyxy[0][0].item() + ox
                    by1 = b.xyxy[0][1].item() + oy
                    bx2 = b.xyxy[0][2].item() + ox
                    by2 = b.xyxy[0][3].item() + oy
                    
                    temp_boxes.append([bx1, by1, bx2, by2])
                    temp_scores.append(float(b.conf[0]))
                    temp_classes.append(int(b.cls[0]))
                    
            for c in set(temp_classes):
                c_boxes = [b for j, b in enumerate(temp_boxes) if temp_classes[j] == c]
                c_scores = [s for j, s in enumerate(temp_scores) if temp_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 2. 제안 1: Ours (DAHI Only) - 중복필터 완전삭제, 모든 객체 반복 추출
        # =====================================================================
        elif method_name == "Ours (DAHI Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=0.3, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            t_inf_start = time.time()
            r1 = m1.predict(img, conf=0.1, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                is_found = False
                for gb in global_final_boxes:
                    if calculate_iou([bx1, by1, bx2, by2], gb) > IOU_FILTER_MATCH: is_found = True; break
                if not is_found: roi_boxes.append([bx1, by1, bx2, by2])
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            while len(remaining_boxes) > 0:
                best_count, best_region = -1, None
                for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                        count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                        if count > best_count: 
                            best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                if best_region and best_count >= 1:
                    dense_regions.append(best_region)
                    dx1, dy1, dx2, dy2 = best_region
                    remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                else: break
            
            unified_infer_list = [img[dy1:dy2, dx1:dx2] for dx1, dy1, dx2, dy2 in dense_regions]
            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for idx, (dx1, dy1, dx2, dy2) in enumerate(dense_regions):
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1
                    for b in res_all[idx].boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
            
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 3. 제안 2: Ours (Tetris Only) - 중복필터 완전삭제
        # =====================================================================
        elif method_name == "Ours (Tetris Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=0.3, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            t_inf_start = time.time()
            r1 = m1.predict(img, conf=0.1, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                is_found = False
                for gb in global_final_boxes:
                    if calculate_iou([bx1, by1, bx2, by2], gb) > IOU_FILTER_MATCH: is_found = True; break
                if not is_found: roi_boxes.append([bx1, by1, bx2, by2])
            
            canvases, canvas_infos = [], []
            if len(roi_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(roi_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

            if len(canvases) > 0:
                t_inf_start = time.time()
                res_pack = m2.predict(canvases, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(canvases)
                
                for c_idx, res in enumerate(res_pack):
                    for b in res.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                        for info in canvas_infos:
                            if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                s = info['scale']
                                orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 4. 제안 3: Ours (DAHI + Tetris) - Top 1 추출, 중복검사 완전삭제
        # =====================================================================
        elif method_name == "Ours (DAHI + Tetris)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            
            t_inf_start = time.time()
            res_global = m2.predict(img, conf=0.3, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            for b in res_global[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = min(1.0, float(b.conf[0]) * 1.10)
                global_final_boxes.append([bx1, by1, bx2, by2]); global_final_scores.append(conf); global_final_classes.append(int(b.cls[0]))

            t_inf_start = time.time()
            r1 = m1.predict(img, conf=0.1, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            roi_boxes = []
            for b1 in r1[0].boxes:
                bx1, by1, bx2, by2 = b1.xyxy[0].tolist()
                is_found = False
                for gb in global_final_boxes:
                    if calculate_iou([bx1, by1, bx2, by2], gb) > IOU_FILTER_MATCH: is_found = True; break
                if not is_found: roi_boxes.append([bx1, by1, bx2, by2])
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            best_count, best_region = -1, None
            for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                    if count > best_count: 
                        best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                        
            if best_region and best_count > 0: 
                dense_regions.append(best_region)
                dx1, dy1, dx2, dy2 = best_region
                remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
            
            unified_infer_list = []
            dense_idx_list = []
            
            for dx1, dy1, dx2, dy2 in dense_regions:
                unified_infer_list.append(img[dy1:dy2, dx1:dx2])
                dense_idx_list.append((len(unified_infer_list) - 1, dx1, dy1, dx2, dy2))

            canvases, canvas_infos = [], []
            canvas_start_idx = -1
            if len(remaining_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(remaining_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

                if len(canvases) > 0:
                    canvas_start_idx = len(unified_infer_list)
                    unified_infer_list.extend(canvases)

            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for d_idx, dx1, dy1, dx2, dy2 in dense_idx_list:
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1; res_dense = res_all[d_idx]
                    for b in res_dense.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                
                if canvas_start_idx != -1:
                    res_pack = res_all[canvas_start_idx:]
                    for c_idx, res in enumerate(res_pack):
                        for b in res.boxes:
                            bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                            bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                            for info in canvas_infos:
                                if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                    if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                    s = info['scale']
                                    orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                    orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                    local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                    break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])
        
        # 통계 저장
        img_total_time = time.time() - t_pipe_start
        is_hr = (w * h >= HR_THRESHOLD)
        target_keys = ['ALL', 'HR'] if is_hr else ['ALL', 'LR']
        for k in target_keys:
            stats[k]['count'] += 1
            stats[k]['inf_time'] += img_inf_time
            stats[k]['total_time'] += img_total_time
            stats[k]['inf_cnt'] += img_inf_cnt
            stats[k]['indices'].add(img_idx)

    # ---------------------------------------------------------
    # 💡 [NEW] 공식 COCO API 연산 엔진 도입
    # ---------------------------------------------------------
    def calc_official_coco_metrics(subset_indices):
        if not subset_indices: return {"AP50:95": 0, "AP50": 0, "AP50s": 0, "AP50m": 0, "AP50l": 0}
        
        # 1. COCO GT (Ground Truth) 딕셔너리 생성
        gt_dict = {"images": [], "annotations": [], "categories": []}
        for i in range(10): # VisDrone 기본 10개 클래스 가정
            gt_dict["categories"].append({"id": i, "name": f"class_{i}"})
            
        ann_id = 1
        for img_idx in subset_indices:
            info = img_infos[img_idx]
            gt_dict["images"].append({"id": img_idx, "width": info['w'], "height": info['h'], "file_name": info['name']})
            for gt in all_gts[img_idx]:
                c, x1, y1, x2, y2 = gt
                bw, bh = x2 - x1, y2 - y1
                gt_dict["annotations"].append({
                    "id": ann_id, "image_id": img_idx, "category_id": int(c),
                    "bbox": [x1, y1, bw, bh], "area": bw * bh, "iscrowd": 0
                })
                ann_id += 1

        cocoGt = COCO()
        cocoGt.dataset = gt_dict
        cocoGt.createIndex()

        # 2. COCO Pred (Predictions) 리스트 생성
        sub_preds = [p for p in all_preds if p[0] in subset_indices]
        pred_list = []
        for pred in sub_preds:
            img_idx, c, score, x1, y1, x2, y2 = pred
            bw, bh = x2 - x1, y2 - y1
            pred_list.append({
                "image_id": img_idx, "category_id": int(c),
                "bbox": [x1, y1, bw, bh], "score": float(score)
            })

        if not pred_list: return {"AP50:95": 0, "AP50": 0, "AP50s": 0, "AP50m": 0, "AP50l": 0}

        cocoDt = cocoGt.loadRes(pred_list)

        # 3. COCO 평가 객체 초기화 및 💡 VisDrone 특화 세팅 (maxDets)
        cocoEval = COCOeval(cocoGt, cocoDt, 'bbox')
        
        # [중요] 논문용 세팅: VisDrone은 한 장에 객체가 수백 개이므로 maxDets를 500으로 상향!
        cocoEval.params.maxDets = [100, 300, 500] 
        
        cocoEval.evaluate()
        cocoEval.accumulate()
        
        # 💡 [FIX] summarize()를 반드시 호출해야 stats 배열이 생성됩니다!
        # 콘솔 창이 지저분해지는 것을 막기 위해 출력을 잠시 막아둡니다(Redirect).
        with contextlib.redirect_stdout(io.StringIO()):
            cocoEval.summarize()

        # 4. 평가 지표 추출 (cocoEval.stats 인덱스 매핑)
        # 만약 객체가 아예 없어서 stats가 12개가 안 채워졌을 경우를 대비한 안전장치 추가
        if len(cocoEval.stats) < 12:
            return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}

        return {
            "AP50:95": cocoEval.stats[0],
            "AP50": cocoEval.stats[1],
            "AP_small": cocoEval.stats[3],  
            "AP_medium": cocoEval.stats[4], 
            "AP_large": cocoEval.stats[5],
        }

    # 결과 취합
    result_dict = {}
    for group in ['ALL', 'HR', 'LR']:
        c = stats[group]['count']
        res = calc_official_coco_metrics(stats[group]['indices'])
        res['Img_Cnt'] = c
        res['Avg_Inf_Cnt'] = stats[group]['inf_cnt'] / c if c else 0
        res['Avg_Inf_Time'] = (stats[group]['inf_time'] / c) * 1000 if c else 0
        res['Avg_Tot_Time'] = (stats[group]['total_time'] / c) * 1000 if c else 0
        result_dict[group] = res
        
    result_dict['Peak_VRAM'] = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0
    return result_dict

# =========================================================
# 실행 및 다중 표 그리기 (Official COCO Protocol 반영)
# =========================================================
methods = [
    "UC (2x2 Uniform Crop)", 
    "Ours (DAHI Only)",
    "Ours (Tetris Only)",
    "Ours (DAHI + Tetris)"
]

final_stats = {}
for m in methods: 
    final_stats[m] = run_official_ablation_benchmark(m)

print("\n" + "="*145)
print(f"🏆 [Ablation Study] Official COCO Protocol Evaluation (Total {NUM_TEST_IMAGES} Images) 🏆")
print("="*145)
# 💡 COCO 핵심 지표인 mAP (AP50:95)를 추가하고, 크기별 지표명을 COCO 표준(APs, APm, APl)으로 변경
print(f"{'Method':<32} | {'Type':<4} | {'Img':<4} | {'mAP':<6} | {'AP50':<6} | {'APs':<6} | {'APm':<6} | {'APl':<6} | {'Inf Cnt':<7} | {'Inf Time':<9} | {'Tot Time':<9}")
print("-" * 145)

for m, groups in final_stats.items():
    for g in ['ALL', 'HR', 'LR']:
        s = groups[g]
        if s['Img_Cnt'] == 0: continue
        
        # 💡 리턴된 COCO 딕셔너리 키값 매핑
        mAP  = s.get('AP50:95', 0.0)
        ap50 = s.get('AP50', 0.0)
        aps  = s.get('AP_small', 0.0)
        apm  = s.get('AP_medium', 0.0)
        apl  = s.get('AP_large', 0.0)
        
        print(f"{m if g == 'ALL' else '':<32} | {g:<4} | {s['Img_Cnt']:<4} | {mAP:.4f} | {ap50:.4f} | {aps:.4f} | {apm:.4f} | {apl:.4f} | {s['Avg_Inf_Cnt']:4.1f} /i | {s['Avg_Inf_Time']:5.1f} ms | {s['Avg_Tot_Time']:5.1f} ms")
    
    print(f"{'':<32} > Peak VRAM: {groups.get('Peak_VRAM', 0.0):.1f} MB")
    print("-" * 145)

🚀 [Ablation Study] 필터링 족쇄 해제! 완벽한 재현 시작! (Total 1294 images)


⏳ UC (2x2 Uniform Crop): 100%|██████████████████████████████| 1294/1294 [01:31<00:00, 14.21it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=30.88s).
Accumulating evaluation results...
DONE (t=1.43s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=6.44s).
Accumulating evaluation results...
DONE (t=0.27s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.04s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=24.93s).
Accumulating evaluation results...
DONE (t=1.13s).


⏳ Ours (DAHI Only): 100%|██████████████████████████████| 1294/1294 [02:00<00:00, 10.73it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.54s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=32.17s).
Accumulating evaluation results...
DONE (t=1.51s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=7.45s).
Accumulating evaluation results...
DONE (t=0.30s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.54s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=25.04s).
Accumulating evaluation results...
DONE (t=1.18s).


⏳ Ours (Tetris Only): 100%|██████████████████████████████| 1294/1294 [01:44<00:00, 12.33it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=29.12s).
Accumulating evaluation results...
DONE (t=1.32s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=6.31s).
Accumulating evaluation results...
DONE (t=0.27s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.04s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=23.43s).
Accumulating evaluation results...
DONE (t=1.11s).


⏳ Ours (DAHI + Tetris): 100%|██████████████████████████████| 1294/1294 [01:52<00:00, 11.55it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.06s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=33.19s).
Accumulating evaluation results...
DONE (t=1.49s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=7.10s).
Accumulating evaluation results...
DONE (t=0.28s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.04s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=25.67s).
Accumulating evaluation results...
DONE (t=1.24s).

🏆 [Ablation Study] Official COCO Protocol Evaluation (Total 5000 Images) 🏆
Method                           | Type | Img  | mAP    | AP50   | APs    | APm    | APl    | Inf Cnt | Inf Time  | Tot Time 
-------------------------------------------------------------

In [ ]:
import cv2
import os
import time
import numpy as np
import tqdm
import torch
from ultralytics import YOLO

import contextlib
import io

# 💡 [NEW] 공식 COCO API 임포트
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# =========================================================
# ⚙️ 하이퍼파라미터 (Hyperparameters) - Single Model Edition
# =========================================================
# MODEL_FILTER_PATH = 'model/best_nano.pt' # 💡 보조 모델 완전히 삭제!
MODEL_MAIN_PATH = 'model/best_small.pt'

CONF_GLOBAL = 0.3
CONF_FILTER = 0.1     
CONF_DENSE = 0.3
CONF_TETRIS = 0.3
CONF_UC = 0.3

# IOU_FILTER_MATCH = 0.97 # 💡 사용 안함 (중복검사 삭제됨)
NMS_CONF_THRESH = 0.3
NMS_IOU_THRESH = 0.4    

DENSE_WINDOW_SIZE = 512
DENSE_STEP = 320      

MERGE_PAD = 16
CROP_PAD_LARGE = 80     
CROP_PAD_SMALL = 16
CROP_PAD_THRESH = 200

CANVAS_SIZE = 960
CANVAS_MARGIN = 2
CANVAS_BG_COLOR = 114

UPSCALE_RATIO = 1.5        
UPSCALE_MAX_THRESH = 200    

NUM_TEST_IMAGES = 5000
HR_THRESHOLD = 1920 * 1080 

# =========================================================
dataset_root = 'data/valid'
img_dir, lbl_dir = os.path.join(dataset_root, 'images'), os.path.join(dataset_root, 'labels')
img_list = sorted(os.listdir(img_dir))[:NUM_TEST_IMAGES]

print(f"🚀 [Ablation Study] Single-Model Pipeline! 완벽한 재현 시작! (Total {len(img_list)} images)")

# m1 = YOLO(MODEL_FILTER_PATH) # 💡 보조 모델 삭제
m2 = YOLO(MODEL_MAIN_PATH)

def calculate_iou(box1, box2):
    xi1, yi1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    xi2, yi2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])

def get_size_category(w, h):
    area = w * h
    if area < 32 ** 2: return 'small'
    elif area < 96 ** 2: return 'medium'
    else: return 'large'

def merge_clusters_dynamic(boxes, img_w, img_h, merge_pad=MERGE_PAD):
    if not len(boxes): return []
    def get_padded(b, pad): return [max(0, b[0]-pad), max(0, b[1]-pad), min(img_w, b[2]+pad), min(img_h, b[3]+pad)]
    def is_overlap(b1, b2):
        p1, p2 = get_padded(b1, merge_pad), get_padded(b2, merge_pad)
        return (min(p1[2], p2[2]) > max(p1[0], p2[0])) and (min(p1[3], p2[3]) > max(p1[1], p2[1]))
    curr = boxes.copy()
    while True:
        merged, flags = [], [False]*len(curr)
        for i in range(len(curr)):
            if flags[i]: continue
            b = curr[i]
            for j in range(i+1, len(curr)):
                if not flags[j] and is_overlap(b, curr[j]):
                    b = [min(b[0], curr[j][0]), min(b[1], curr[j][1]), max(b[2], curr[j][2]), max(b[3], curr[j][3])]
                    flags[j] = True
            merged.append(b)
        if len(merged) == len(curr): break
        curr = merged
    final_boxes = []
    for b in curr:
        bw, bh = b[2] - b[0], b[3] - b[1]
        crop_pad = CROP_PAD_LARGE if max(bw, bh) < CROP_PAD_THRESH else CROP_PAD_SMALL 
        final_boxes.append(get_padded(b, crop_pad))
    return final_boxes

def run_official_ablation_benchmark(method_name):
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        
    all_gts = {}; all_preds = []
    img_infos = {} 
    
    stats = {
        'ALL': {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'HR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'LR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()}
    }

    pbar = tqdm.tqdm(img_list, desc=f"⏳ {method_name}", bar_format='{l_bar}{bar:30}{r_bar}')
    for img_idx, img_name in enumerate(pbar):
        img_path, lbl_path = os.path.join(img_dir, img_name), os.path.join(lbl_dir, img_name.replace('.jpg', '.txt'))
        img = cv2.imread(img_path); h, w, _ = img.shape
        
        img_infos[img_idx] = {'w': w, 'h': h, 'name': img_name}
        
        gts = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    c, xc, yc, bw, bh = map(float, line.split())
                    gts.append([int(c), (xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h]) 
        all_gts[img_idx] = gts

        t_pipe_start = time.time()
        img_inf_time, img_inf_cnt = 0, 0
        
        # =====================================================================
        # 1. Baseline: UC (2x2 Uniform Crop)
        # =====================================================================
        if method_name == "UC (2x2 Uniform Crop)":
            # UC 평가가 너무 오래 걸려 비활성화
            pass 

        # =====================================================================
        # 2. 제안 1: Ours (DAHI Only)
        # =====================================================================
        elif method_name == "Ours (DAHI Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            roi_boxes = []
            
            # 💡 [핵심 최적화] 메인 모델(m2)로 CONF_FILTER(0.1) 기준 단 1번만 스캔
            t_inf_start = time.time()
            res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global_all[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = float(b.conf[0])
                
                # 1. 글로벌 확정 박스 (0.3 이상)
                if conf >= CONF_GLOBAL:
                    boosted_conf = min(1.0, conf * 1.10)
                    global_final_boxes.append([bx1, by1, bx2, by2])
                    global_final_scores.append(boosted_conf)
                    global_final_classes.append(int(b.cls[0]))
                
                # 2. ROI 박스 등록 (모든 탐지 객체 재검사)
                roi_boxes.append([bx1, by1, bx2, by2])
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            while len(remaining_boxes) > 0:
                best_count, best_region = -1, None
                for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                        count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                        if count > best_count: 
                            best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                if best_region and best_count >= 1:
                    dense_regions.append(best_region)
                    dx1, dy1, dx2, dy2 = best_region
                    remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                else: break
            
            unified_infer_list = [img[dy1:dy2, dx1:dx2] for dx1, dy1, dx2, dy2 in dense_regions]
            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for idx, (dx1, dy1, dx2, dy2) in enumerate(dense_regions):
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1
                    for b in res_all[idx].boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
            
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 3. 제안 2: Ours (Tetris Only)
        # =====================================================================
        elif method_name == "Ours (Tetris Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            roi_boxes = []
            
            t_inf_start = time.time()
            res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global_all[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = float(b.conf[0])
                if conf >= CONF_GLOBAL:
                    global_final_boxes.append([bx1, by1, bx2, by2])
                    global_final_scores.append(min(1.0, conf * 1.10))
                    global_final_classes.append(int(b.cls[0]))
                roi_boxes.append([bx1, by1, bx2, by2])
            
            canvases, canvas_infos = [], []
            if len(roi_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(roi_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

            if len(canvases) > 0:
                t_inf_start = time.time()
                res_pack = m2.predict(canvases, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(canvases)
                
                for c_idx, res in enumerate(res_pack):
                    for b in res.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                        for info in canvas_infos:
                            if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                s = info['scale']
                                orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 4. 제안 3: Ours (DAHI + Tetris)
        # =====================================================================
        elif method_name == "Ours (DAHI + Tetris)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            roi_boxes = []
            
            t_inf_start = time.time()
            res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global_all[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = float(b.conf[0])
                if conf >= CONF_GLOBAL:
                    global_final_boxes.append([bx1, by1, bx2, by2])
                    global_final_scores.append(min(1.0, conf * 1.10))
                    global_final_classes.append(int(b.cls[0]))
                roi_boxes.append([bx1, by1, bx2, by2])
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            best_count, best_region = -1, None
            for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                    if count > best_count: 
                        best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                        
            if best_region and best_count > 0: 
                dense_regions.append(best_region)
                dx1, dy1, dx2, dy2 = best_region
                remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
            
            unified_infer_list = []
            dense_idx_list = []
            
            for dx1, dy1, dx2, dy2 in dense_regions:
                unified_infer_list.append(img[dy1:dy2, dx1:dx2])
                dense_idx_list.append((len(unified_infer_list) - 1, dx1, dy1, dx2, dy2))

            canvases, canvas_infos = [], []
            canvas_start_idx = -1
            if len(remaining_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(remaining_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

                if len(canvases) > 0:
                    canvas_start_idx = len(unified_infer_list)
                    unified_infer_list.extend(canvases)

            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for d_idx, dx1, dy1, dx2, dy2 in dense_idx_list:
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1; res_dense = res_all[d_idx]
                    for b in res_dense.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                
                if canvas_start_idx != -1:
                    res_pack = res_all[canvas_start_idx:]
                    for c_idx, res in enumerate(res_pack):
                        for b in res.boxes:
                            bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                            bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                            for info in canvas_infos:
                                if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                    if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                    s = info['scale']
                                    orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                    orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                    local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                    break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # 통계 저장
        img_total_time = time.time() - t_pipe_start
        is_hr = (w * h >= HR_THRESHOLD)
        target_keys = ['ALL', 'HR'] if is_hr else ['ALL', 'LR']
        for k in target_keys:
            stats[k]['count'] += 1
            stats[k]['inf_time'] += img_inf_time
            stats[k]['total_time'] += img_total_time
            stats[k]['inf_cnt'] += img_inf_cnt
            stats[k]['indices'].add(img_idx)

    # ---------------------------------------------------------
    # 💡 공식 COCO API 연산 엔진
    # ---------------------------------------------------------
    def calc_official_coco_metrics(subset_indices):
        if not subset_indices: return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}
        
        gt_dict = {"images": [], "annotations": [], "categories": []}
        for i in range(10): gt_dict["categories"].append({"id": i, "name": f"class_{i}"})
            
        ann_id = 1
        for img_idx in subset_indices:
            info = img_infos[img_idx]
            gt_dict["images"].append({"id": img_idx, "width": info['w'], "height": info['h'], "file_name": info['name']})
            for gt in all_gts[img_idx]:
                c, x1, y1, x2, y2 = gt
                bw, bh = x2 - x1, y2 - y1
                gt_dict["annotations"].append({"id": ann_id, "image_id": img_idx, "category_id": int(c), "bbox": [x1, y1, bw, bh], "area": bw * bh, "iscrowd": 0})
                ann_id += 1

        cocoGt = COCO()
        cocoGt.dataset = gt_dict
        cocoGt.createIndex()

        sub_preds = [p for p in all_preds if p[0] in subset_indices]
        pred_list = []
        for pred in sub_preds:
            img_idx, c, score, x1, y1, x2, y2 = pred
            bw, bh = x2 - x1, y2 - y1
            pred_list.append({"image_id": img_idx, "category_id": int(c), "bbox": [x1, y1, bw, bh], "score": float(score)})

        if not pred_list: return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}

        cocoDt = cocoGt.loadRes(pred_list)

        cocoEval = COCOeval(cocoGt, cocoDt, 'bbox')
        cocoEval.params.maxDets = [100, 300, 500] 
        
        cocoEval.evaluate()
        cocoEval.accumulate()
        
        with contextlib.redirect_stdout(io.StringIO()):
            cocoEval.summarize()

        if len(cocoEval.stats) < 12:
            return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}

        return {
            "AP50:95": cocoEval.stats[0],
            "AP50": cocoEval.stats[1],
            "AP_small": cocoEval.stats[3],  
            "AP_medium": cocoEval.stats[4], 
            "AP_large": cocoEval.stats[5],
        }

    result_dict = {}
    for group in ['ALL', 'HR', 'LR']:
        c = stats[group]['count']
        res = calc_official_coco_metrics(stats[group]['indices'])
        res['Img_Cnt'] = c
        res['Avg_Inf_Cnt'] = stats[group]['inf_cnt'] / c if c else 0
        res['Avg_Inf_Time'] = (stats[group]['inf_time'] / c) * 1000 if c else 0
        res['Avg_Tot_Time'] = (stats[group]['total_time'] / c) * 1000 if c else 0
        result_dict[group] = res
        
    result_dict['Peak_VRAM'] = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0
    return result_dict

# =========================================================
# 실행 및 다중 표 그리기 (Official COCO Protocol)
# =========================================================
methods = [
    "UC (2x2 Uniform Crop)", 
    "Ours (DAHI Only)",
    "Ours (Tetris Only)",
    "Ours (DAHI + Tetris)"
]

final_stats = {}
for m in methods: 
    final_stats[m] = run_official_ablation_benchmark(m)

print("\n" + "="*145)
print(f"🏆 [Ablation Study] Single Model & Official COCO Protocol Evaluation (Total {NUM_TEST_IMAGES} Images) 🏆")
print("="*145)
print(f"{'Method':<32} | {'Type':<4} | {'Img':<4} | {'mAP':<6} | {'AP50':<6} | {'APs':<6} | {'APm':<6} | {'APl':<6} | {'Inf Cnt':<7} | {'Inf Time':<9} | {'Tot Time':<9}")
print("-" * 145)

for m, groups in final_stats.items():
    for g in ['ALL', 'HR', 'LR']:
        s = groups[g]
        if s['Img_Cnt'] == 0: continue
        
        mAP  = s.get('AP50:95', 0.0)
        ap50 = s.get('AP50', 0.0)
        aps  = s.get('AP_small', 0.0)
        apm  = s.get('AP_medium', 0.0)
        apl  = s.get('AP_large', 0.0)
        
        print(f"{m if g == 'ALL' else '':<32} | {g:<4} | {s['Img_Cnt']:<4} | {mAP:.4f} | {ap50:.4f} | {aps:.4f} | {apm:.4f} | {apl:.4f} | {s['Avg_Inf_Cnt']:4.1f} /i | {s['Avg_Inf_Time']:5.1f} ms | {s['Avg_Tot_Time']:5.1f} ms")
    
    print(f"{'':<32} > Peak VRAM: {groups.get('Peak_VRAM', 0.0):.1f} MB")
    print("-" * 145)

🚀 [Ablation Study] Single-Model Pipeline! 완벽한 재현 시작! (Total 1294 images)


⏳ UC (2x2 Uniform Crop): 100%|██████████████████████████████| 1294/1294 [00:15<00:00, 81.04it/s]


creating index...
index created!
creating index...
index created!
creating index...
index created!


⏳ Ours (DAHI Only): 100%|██████████████████████████████| 1294/1294 [01:38<00:00, 13.19it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.06s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=34.65s).
Accumulating evaluation results...
DONE (t=1.47s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=8.00s).
Accumulating evaluation results...
DONE (t=0.29s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.04s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=26.56s).
Accumulating evaluation results...
DONE (t=1.13s).


⏳ Ours (Tetris Only): 100%|██████████████████████████████| 1294/1294 [01:20<00:00, 16.08it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=31.22s).
Accumulating evaluation results...
DONE (t=1.32s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=6.61s).
Accumulating evaluation results...
DONE (t=0.26s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.04s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=24.26s).
Accumulating evaluation results...
DONE (t=1.03s).


⏳ Ours (DAHI + Tetris): 100%|██████████████████████████████| 1294/1294 [01:26<00:00, 15.01it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.34s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=34.68s).
Accumulating evaluation results...
DONE (t=1.60s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=7.36s).
Accumulating evaluation results...
DONE (t=0.28s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.31s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=26.75s).
Accumulating evaluation results...
DONE (t=1.15s).

🏆 [Ablation Study] Single Model & Official COCO Protocol Evaluation (Total 5000 Images) 🏆
Method                           | Type | Img  | mAP    | AP50   | APs    | APm    | APl    | Inf Cnt | Inf Time  | Tot Time 
----------------------------------------------

In [ ]:
# 라우팅

import cv2
import os
import time
import numpy as np
import tqdm
import torch
from ultralytics import YOLO

import contextlib
import io

# 💡 [NEW] 공식 COCO API 임포트
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# =========================================================
# ⚙️ 하이퍼파라미터 (Hyperparameters) - Single Model Edition
# =========================================================
# MODEL_FILTER_PATH = 'model/best_nano.pt' # 💡 보조 모델 완전히 삭제!
MODEL_MAIN_PATH = 'model/best_small.pt'

CONF_GLOBAL = 0.3
CONF_FILTER = 0.1     
CONF_DENSE = 0.3
CONF_TETRIS = 0.3
CONF_UC = 0.3

DENSE_RATIO_THRESH = 0.30  # 💡 [NEW] 라우팅 임계값: 전체 소형 객체의 50% 이상 밀집 시 DAHI 구역 인정
# IOU_FILTER_MATCH = 0.97 # 💡 사용 안함 (중복검사 삭제됨)
NMS_CONF_THRESH = 0.3
NMS_IOU_THRESH = 0.4    

DENSE_WINDOW_SIZE = 512
DENSE_STEP = 320      

MERGE_PAD = 16
CROP_PAD_LARGE = 80     
CROP_PAD_SMALL = 16
CROP_PAD_THRESH = 200

CANVAS_SIZE = 960
CANVAS_MARGIN = 2
CANVAS_BG_COLOR = 114

UPSCALE_RATIO = 1.5        
UPSCALE_MAX_THRESH = 200    

NUM_TEST_IMAGES = 5000
HR_THRESHOLD = 1920 * 1080 

# =========================================================
dataset_root = 'data/valid'
img_dir, lbl_dir = os.path.join(dataset_root, 'images'), os.path.join(dataset_root, 'labels')
img_list = sorted(os.listdir(img_dir))[:NUM_TEST_IMAGES]

print(f"🚀 [Ablation Study] Single-Model Pipeline! 완벽한 재현 시작! (Total {len(img_list)} images)")

# m1 = YOLO(MODEL_FILTER_PATH) # 💡 보조 모델 삭제
m2 = YOLO(MODEL_MAIN_PATH)

def calculate_iou(box1, box2):
    xi1, yi1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    xi2, yi2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])

def get_size_category(w, h):
    area = w * h
    if area < 32 ** 2: return 'small'
    elif area < 96 ** 2: return 'medium'
    else: return 'large'

def merge_clusters_dynamic(boxes, img_w, img_h, merge_pad=MERGE_PAD):
    if not len(boxes): return []
    def get_padded(b, pad): return [max(0, b[0]-pad), max(0, b[1]-pad), min(img_w, b[2]+pad), min(img_h, b[3]+pad)]
    def is_overlap(b1, b2):
        p1, p2 = get_padded(b1, merge_pad), get_padded(b2, merge_pad)
        return (min(p1[2], p2[2]) > max(p1[0], p2[0])) and (min(p1[3], p2[3]) > max(p1[1], p2[1]))
    curr = boxes.copy()
    while True:
        merged, flags = [], [False]*len(curr)
        for i in range(len(curr)):
            if flags[i]: continue
            b = curr[i]
            for j in range(i+1, len(curr)):
                if not flags[j] and is_overlap(b, curr[j]):
                    b = [min(b[0], curr[j][0]), min(b[1], curr[j][1]), max(b[2], curr[j][2]), max(b[3], curr[j][3])]
                    flags[j] = True
            merged.append(b)
        if len(merged) == len(curr): break
        curr = merged
    final_boxes = []
    for b in curr:
        bw, bh = b[2] - b[0], b[3] - b[1]
        crop_pad = CROP_PAD_LARGE if max(bw, bh) < CROP_PAD_THRESH else CROP_PAD_SMALL 
        final_boxes.append(get_padded(b, crop_pad))
    return final_boxes

def run_official_ablation_benchmark(method_name):
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        
    all_gts = {}; all_preds = []
    img_infos = {} 
    
    stats = {
        'ALL': {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'HR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'LR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()}
    }

    pbar = tqdm.tqdm(img_list, desc=f"⏳ {method_name}", bar_format='{l_bar}{bar:30}{r_bar}')
    for img_idx, img_name in enumerate(pbar):
        img_path, lbl_path = os.path.join(img_dir, img_name), os.path.join(lbl_dir, img_name.replace('.jpg', '.txt'))
        img = cv2.imread(img_path); h, w, _ = img.shape
        
        img_infos[img_idx] = {'w': w, 'h': h, 'name': img_name}
        
        gts = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    c, xc, yc, bw, bh = map(float, line.split())
                    gts.append([int(c), (xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h]) 
        all_gts[img_idx] = gts

        t_pipe_start = time.time()
        img_inf_time, img_inf_cnt = 0, 0
        
        # =====================================================================
        # 1. Baseline: UC (2x2 Uniform Crop)
        # =====================================================================
        if method_name == "UC (2x2 Uniform Crop)":
            # UC 평가가 너무 오래 걸려 비활성화
            ch, cw = h // 2, w // 2
            crops, offsets = [img], [(0, 0)]
            for y in [0, ch]:
                for x in [0, cw]:
                    crops.append(img[y:y+ch, x:x+cw])
                    offsets.append((x, y))
            
            t_inf_start = time.time()
            results2 = m2.predict(crops, conf=CONF_UC, verbose=False, batch=5)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 5 
            
            temp_boxes, temp_scores, temp_classes = [], [], []
            for i, res in enumerate(results2):
                ox, oy = offsets[i]
                for b in res.boxes:
                    # 💡 [FIX] .item() 을 사용하여 텐서를 완벽하게 파이썬 float으로 변환!
                    bx1 = b.xyxy[0][0].item() + ox
                    by1 = b.xyxy[0][1].item() + oy
                    bx2 = b.xyxy[0][2].item() + ox
                    by2 = b.xyxy[0][3].item() + oy
                    
                    temp_boxes.append([bx1, by1, bx2, by2])
                    temp_scores.append(float(b.conf[0]))
                    temp_classes.append(int(b.cls[0]))
                    
            for c in set(temp_classes):
                c_boxes = [b for j, b in enumerate(temp_boxes) if temp_classes[j] == c]
                c_scores = [s for j, s in enumerate(temp_scores) if temp_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 2. 제안 1: Ours (DAHI Only)
        # =====================================================================
        elif method_name == "Ours (DAHI Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            roi_boxes = []
            
            # 💡 [핵심 최적화] 메인 모델(m2)로 CONF_FILTER(0.1) 기준 단 1번만 스캔
            t_inf_start = time.time()
            res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global_all[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = float(b.conf[0])
                
                # 1. 글로벌 확정 박스 (0.3 이상)
                if conf >= CONF_GLOBAL:
                    boosted_conf = min(1.0, conf * 1.10)
                    global_final_boxes.append([bx1, by1, bx2, by2])
                    global_final_scores.append(boosted_conf)
                    global_final_classes.append(int(b.cls[0]))
                
                # 2. ROI 박스 등록 (모든 탐지 객체 재검사)
                roi_boxes.append([bx1, by1, bx2, by2])
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            while len(remaining_boxes) > 0:
                best_count, best_region = -1, None
                for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                        count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                        if count > best_count: 
                            best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                if best_region and best_count >= 1:
                    dense_regions.append(best_region)
                    dx1, dy1, dx2, dy2 = best_region
                    remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                else: break
            
            unified_infer_list = [img[dy1:dy2, dx1:dx2] for dx1, dy1, dx2, dy2 in dense_regions]
            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for idx, (dx1, dy1, dx2, dy2) in enumerate(dense_regions):
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1
                    for b in res_all[idx].boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
            
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 3. 제안 2: Ours (Tetris Only)
        # =====================================================================
        elif method_name == "Ours (Tetris Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            roi_boxes = []
            
            t_inf_start = time.time()
            res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global_all[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = float(b.conf[0])
                if conf >= CONF_GLOBAL:
                    global_final_boxes.append([bx1, by1, bx2, by2])
                    global_final_scores.append(min(1.0, conf * 1.10))
                    global_final_classes.append(int(b.cls[0]))
                roi_boxes.append([bx1, by1, bx2, by2])
            
            canvases, canvas_infos = [], []
            if len(roi_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(roi_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

            if len(canvases) > 0:
                t_inf_start = time.time()
                res_pack = m2.predict(canvases, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(canvases)
                
                for c_idx, res in enumerate(res_pack):
                    for b in res.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                        for info in canvas_infos:
                            if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                s = info['scale']
                                orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 4. 제안 3: Ours (DAHI + Tetris)
        # =====================================================================
        elif method_name == "Ours (DAHI + Tetris)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            roi_boxes = []
            
            t_inf_start = time.time()
            res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global_all[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = float(b.conf[0])
                if conf >= CONF_GLOBAL:
                    global_final_boxes.append([bx1, by1, bx2, by2])
                    global_final_scores.append(min(1.0, conf * 1.10))
                    global_final_classes.append(int(b.cls[0]))
                roi_boxes.append([bx1, by1, bx2, by2])
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            # 💡 [NEW] CAD-Router (Class-Aware Density Router) 로직 시작
            # 1. 라우팅 판단을 위한 "소형 객체" 중심점만 추출 (중/대형 객체 배제)
            small_centroids = []
            for b in remaining_boxes:
                bx1, by1, bx2, by2 = b
                bw, bh = bx2 - bx1, by2 - by1
                if get_size_category(bw, bh) == 'small':
                    cx, cy = (bx1 + bx2) / 2, (by1 + by2) / 2
                    small_centroids.append((cx, cy))
            
            total_small_objs = len(small_centroids)
            # 기준점: 소형 객체 총합의 DENSE_RATIO_THRESH (50%) 이상 (단, 최소 1개 보장)
            route_threshold = max(1, int(total_small_objs * DENSE_RATIO_THRESH)) 
            
            if total_small_objs > 0:
                current_small_centroids = small_centroids.copy()
                
                # 밀집 구역이 기준을 충족하는 한 계속해서 DAHI 영역을 추출 (Spatial NMS)
                while len(current_small_centroids) > 0:
                    best_count, best_region = -1, None
                    
                    # 2. 슬라이딩 탐색으로 최적의 밀집 구역 1개 찾기
                    for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                        for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                            # 해당 창 내부에 있는 '소형 객체' 점의 개수만 카운트
                            count = sum(1 for cx, cy in current_small_centroids if x <= cx <= x + DENSE_WINDOW_SIZE and y <= cy <= y + DENSE_WINDOW_SIZE)
                            if count > best_count: 
                                best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                    
                    # 3. 라우팅 조건 판별
                    if best_region and best_count >= route_threshold:
                        # 기준을 넘었으므로 DAHI (밀집 구역) 리스트에 추가
                        dense_regions.append(best_region)
                        dx1, dy1, dx2, dy2 = best_region
                        
                        # Spatial NMS: 방금 채택된 영역에 속한 중심점들은 계산에서 제외
                        current_small_centroids = [(cx, cy) for cx, cy in current_small_centroids if not (dx1 <= cx <= dx2 and dy1 <= cy <= dy2)]
                        
                        # 실제 추론 큐(remaining_boxes)에서도 해당 영역에 속한 '모든 객체(대/중/소)'를 DAHI로 넘김 (제거)
                        remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                    else:
                        # 더 이상 임계값(n%)을 넘는 밀집 구역이 없다면 탐색 즉시 중단
                        # 남은 remaining_boxes는 아래쪽 코드에 의해 자연스럽게 Tetris(Spatial Packing)로 라우팅 됨
                        break
            
            # (이후 코드는 기존과 동일하게 unified_infer_list 생성 및 Tetris Canvas 생성 로직으로 이어짐)
            unified_infer_list = []
            dense_idx_list = []
            
            for dx1, dy1, dx2, dy2 in dense_regions:
                unified_infer_list.append(img[dy1:dy2, dx1:dx2])
                dense_idx_list.append((len(unified_infer_list) - 1, dx1, dy1, dx2, dy2))

            canvases, canvas_infos = [], []
            canvas_start_idx = -1
            if len(remaining_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(remaining_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

                if len(canvases) > 0:
                    canvas_start_idx = len(unified_infer_list)
                    unified_infer_list.extend(canvases)

            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for d_idx, dx1, dy1, dx2, dy2 in dense_idx_list:
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1; res_dense = res_all[d_idx]
                    for b in res_dense.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                
                if canvas_start_idx != -1:
                    res_pack = res_all[canvas_start_idx:]
                    for c_idx, res in enumerate(res_pack):
                        for b in res.boxes:
                            bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                            bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                            for info in canvas_infos:
                                if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                    if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                    s = info['scale']
                                    orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                    orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                    local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                    break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # 통계 저장
        img_total_time = time.time() - t_pipe_start
        is_hr = (w * h >= HR_THRESHOLD)
        target_keys = ['ALL', 'HR'] if is_hr else ['ALL', 'LR']
        for k in target_keys:
            stats[k]['count'] += 1
            stats[k]['inf_time'] += img_inf_time
            stats[k]['total_time'] += img_total_time
            stats[k]['inf_cnt'] += img_inf_cnt
            stats[k]['indices'].add(img_idx)

    # ---------------------------------------------------------
    # 💡 공식 COCO API 연산 엔진
    # ---------------------------------------------------------
    def calc_official_coco_metrics(subset_indices):
        if not subset_indices: return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}
        
        gt_dict = {"images": [], "annotations": [], "categories": []}
        for i in range(10): gt_dict["categories"].append({"id": i, "name": f"class_{i}"})
            
        ann_id = 1
        for img_idx in subset_indices:
            info = img_infos[img_idx]
            gt_dict["images"].append({"id": img_idx, "width": info['w'], "height": info['h'], "file_name": info['name']})
            for gt in all_gts[img_idx]:
                c, x1, y1, x2, y2 = gt
                bw, bh = x2 - x1, y2 - y1
                gt_dict["annotations"].append({"id": ann_id, "image_id": img_idx, "category_id": int(c), "bbox": [x1, y1, bw, bh], "area": bw * bh, "iscrowd": 0})
                ann_id += 1

        cocoGt = COCO()
        cocoGt.dataset = gt_dict
        cocoGt.createIndex()

        sub_preds = [p for p in all_preds if p[0] in subset_indices]
        pred_list = []
        for pred in sub_preds:
            img_idx, c, score, x1, y1, x2, y2 = pred
            bw, bh = x2 - x1, y2 - y1
            pred_list.append({"image_id": img_idx, "category_id": int(c), "bbox": [x1, y1, bw, bh], "score": float(score)})

        if not pred_list: return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}

        cocoDt = cocoGt.loadRes(pred_list)

        cocoEval = COCOeval(cocoGt, cocoDt, 'bbox')
        cocoEval.params.maxDets = [100, 300, 500] 
        
        cocoEval.evaluate()
        cocoEval.accumulate()
        
        with contextlib.redirect_stdout(io.StringIO()):
            cocoEval.summarize()

        if len(cocoEval.stats) < 12:
            return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}

        return {
            "AP50:95": cocoEval.stats[0],
            "AP50": cocoEval.stats[1],
            "AP_small": cocoEval.stats[3],  
            "AP_medium": cocoEval.stats[4], 
            "AP_large": cocoEval.stats[5],
        }

    result_dict = {}
    for group in ['ALL', 'HR', 'LR']:
        c = stats[group]['count']
        res = calc_official_coco_metrics(stats[group]['indices'])
        res['Img_Cnt'] = c
        res['Avg_Inf_Cnt'] = stats[group]['inf_cnt'] / c if c else 0
        res['Avg_Inf_Time'] = (stats[group]['inf_time'] / c) * 1000 if c else 0
        res['Avg_Tot_Time'] = (stats[group]['total_time'] / c) * 1000 if c else 0
        result_dict[group] = res
        
    result_dict['Peak_VRAM'] = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0
    return result_dict

# =========================================================
# 실행 및 다중 표 그리기 (Official COCO Protocol)
# =========================================================
methods = [
    "UC (2x2 Uniform Crop)", 
    "Ours (DAHI Only)",
    "Ours (Tetris Only)",
    "Ours (DAHI + Tetris)"
]

final_stats = {}
for m in methods: 
    final_stats[m] = run_official_ablation_benchmark(m)

print("\n" + "="*145)
print(f"🏆 [Ablation Study] Single Model & Official COCO Protocol Evaluation (Total {NUM_TEST_IMAGES} Images) 🏆")
print("="*145)
print(f"{'Method':<32} | {'Type':<4} | {'Img':<4} | {'mAP':<6} | {'AP50':<6} | {'APs':<6} | {'APm':<6} | {'APl':<6} | {'Inf Cnt':<7} | {'Inf Time':<9} | {'Tot Time':<9}")
print("-" * 145)

for m, groups in final_stats.items():
    for g in ['ALL', 'HR', 'LR']:
        s = groups[g]
        if s['Img_Cnt'] == 0: continue
        
        mAP  = s.get('AP50:95', 0.0)
        ap50 = s.get('AP50', 0.0)
        aps  = s.get('AP_small', 0.0)
        apm  = s.get('AP_medium', 0.0)
        apl  = s.get('AP_large', 0.0)
        
        print(f"{m if g == 'ALL' else '':<32} | {g:<4} | {s['Img_Cnt']:<4} | {mAP:.4f} | {ap50:.4f} | {aps:.4f} | {apm:.4f} | {apl:.4f} | {s['Avg_Inf_Cnt']:4.1f} /i | {s['Avg_Inf_Time']:5.1f} ms | {s['Avg_Tot_Time']:5.1f} ms")
    
    print(f"{'':<32} > Peak VRAM: {groups.get('Peak_VRAM', 0.0):.1f} MB")
    print("-" * 145)

🚀 [Ablation Study] Single-Model Pipeline! 완벽한 재현 시작! (Total 1294 images)


⏳ UC (2x2 Uniform Crop): 100%|██████████████████████████████| 1294/1294 [01:31<00:00, 14.11it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=31.67s).
Accumulating evaluation results...
DONE (t=1.44s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=6.48s).
Accumulating evaluation results...
DONE (t=0.26s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.04s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=25.01s).
Accumulating evaluation results...
DONE (t=1.18s).


⏳ Ours (DAHI Only): 100%|██████████████████████████████| 1294/1294 [01:38<00:00, 13.08it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=32.79s).
Accumulating evaluation results...
DONE (t=1.52s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=7.50s).
Accumulating evaluation results...
DONE (t=0.30s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.24s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=25.09s).
Accumulating evaluation results...
DONE (t=1.40s).


⏳ Ours (Tetris Only): 100%|██████████████████████████████| 1294/1294 [01:22<00:00, 15.65it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.27s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=29.53s).
Accumulating evaluation results...
DONE (t=1.35s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=6.55s).
Accumulating evaluation results...
DONE (t=0.25s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.04s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=23.24s).
Accumulating evaluation results...
DONE (t=1.09s).


⏳ Ours (DAHI + Tetris): 100%|██████████████████████████████| 1294/1294 [01:28<00:00, 14.60it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.06s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=33.01s).
Accumulating evaluation results...
DONE (t=1.50s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=7.08s).
Accumulating evaluation results...
DONE (t=0.28s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=25.54s).
Accumulating evaluation results...
DONE (t=1.17s).

🏆 [Ablation Study] Single Model & Official COCO Protocol Evaluation (Total 5000 Images) 🏆
Method                           | Type | Img  | mAP    | AP50   | APs    | APm    | APl    | Inf Cnt | Inf Time  | Tot Time 
----------------------------------------------

In [ ]:
# 라우팅

import cv2
import os
import time
import numpy as np
import tqdm
import torch
from ultralytics import YOLO

import contextlib
import io

# 💡 [NEW] 공식 COCO API 임포트
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# =========================================================
# ⚙️ 하이퍼파라미터 (Hyperparameters) - Single Model Edition
# =========================================================
# MODEL_FILTER_PATH = 'model/best_nano.pt' # 💡 보조 모델 완전히 삭제!
MODEL_MAIN_PATH = 'model/best_small.pt'

CONF_GLOBAL = 0.3
CONF_FILTER = 0.1     
CONF_DENSE = 0.3
CONF_TETRIS = 0.3
CONF_UC = 0.3

DENSE_RATIO_THRESH = 0.30  # 💡 [NEW] 라우팅 임계값: 전체 소형 객체의 50% 이상 밀집 시 DAHI 구역 인정
# IOU_FILTER_MATCH = 0.97 # 💡 사용 안함 (중복검사 삭제됨)
NMS_CONF_THRESH = 0.3
NMS_IOU_THRESH = 0.4    

DENSE_WINDOW_SIZE = 512
DENSE_STEP = 320      

MERGE_PAD = 16
CROP_PAD_LARGE = 80     
CROP_PAD_SMALL = 16
CROP_PAD_THRESH = 200

CANVAS_SIZE = 960
CANVAS_MARGIN = 2
CANVAS_BG_COLOR = 114

UPSCALE_RATIO = 1.5        
UPSCALE_MAX_THRESH = 200    

NUM_TEST_IMAGES = 5000
HR_THRESHOLD = 1920 * 1080 

# =========================================================
dataset_root = 'data/valid'
img_dir, lbl_dir = os.path.join(dataset_root, 'images'), os.path.join(dataset_root, 'labels')
img_list = sorted(os.listdir(img_dir))[:NUM_TEST_IMAGES]

print(f"🚀 [Ablation Study] Single-Model Pipeline! 완벽한 재현 시작! (Total {len(img_list)} images)")

# m1 = YOLO(MODEL_FILTER_PATH) # 💡 보조 모델 삭제
m2 = YOLO(MODEL_MAIN_PATH)

def calculate_iou(box1, box2):
    xi1, yi1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    xi2, yi2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])

def get_size_category(w, h):
    area = w * h
    if area < 32 ** 2: return 'small'
    elif area < 96 ** 2: return 'medium'
    else: return 'large'

def merge_clusters_dynamic(boxes, img_w, img_h, merge_pad=MERGE_PAD):
    if not len(boxes): return []
    def get_padded(b, pad): return [max(0, b[0]-pad), max(0, b[1]-pad), min(img_w, b[2]+pad), min(img_h, b[3]+pad)]
    def is_overlap(b1, b2):
        p1, p2 = get_padded(b1, merge_pad), get_padded(b2, merge_pad)
        return (min(p1[2], p2[2]) > max(p1[0], p2[0])) and (min(p1[3], p2[3]) > max(p1[1], p2[1]))
    curr = boxes.copy()
    while True:
        merged, flags = [], [False]*len(curr)
        for i in range(len(curr)):
            if flags[i]: continue
            b = curr[i]
            for j in range(i+1, len(curr)):
                if not flags[j] and is_overlap(b, curr[j]):
                    b = [min(b[0], curr[j][0]), min(b[1], curr[j][1]), max(b[2], curr[j][2]), max(b[3], curr[j][3])]
                    flags[j] = True
            merged.append(b)
        if len(merged) == len(curr): break
        curr = merged
    final_boxes = []
    for b in curr:
        bw, bh = b[2] - b[0], b[3] - b[1]
        crop_pad = CROP_PAD_LARGE if max(bw, bh) < CROP_PAD_THRESH else CROP_PAD_SMALL 
        final_boxes.append(get_padded(b, crop_pad))
    return final_boxes

def run_official_ablation_benchmark(method_name):
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        
    all_gts = {}; all_preds = []
    img_infos = {} 
    
    stats = {
        'ALL': {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'HR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'LR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()}
    }

    pbar = tqdm.tqdm(img_list, desc=f"⏳ {method_name}", bar_format='{l_bar}{bar:30}{r_bar}')
    for img_idx, img_name in enumerate(pbar):
        img_path, lbl_path = os.path.join(img_dir, img_name), os.path.join(lbl_dir, img_name.replace('.jpg', '.txt'))
        img = cv2.imread(img_path); h, w, _ = img.shape
        
        img_infos[img_idx] = {'w': w, 'h': h, 'name': img_name}
        
        gts = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    c, xc, yc, bw, bh = map(float, line.split())
                    gts.append([int(c), (xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h]) 
        all_gts[img_idx] = gts

        t_pipe_start = time.time()
        img_inf_time, img_inf_cnt = 0, 0
        
        # =====================================================================
        # 1. Baseline: UC (2x2 Uniform Crop)
        # =====================================================================
        if method_name == "UC (2x2 Uniform Crop)":
            # UC 평가가 너무 오래 걸려 비활성화
            ch, cw = h // 2, w // 2
            crops, offsets = [img], [(0, 0)]
            for y in [0, ch]:
                for x in [0, cw]:
                    crops.append(img[y:y+ch, x:x+cw])
                    offsets.append((x, y))
            
            t_inf_start = time.time()
            results2 = m2.predict(crops, conf=CONF_UC, verbose=False, batch=5)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 5 
            
            temp_boxes, temp_scores, temp_classes = [], [], []
            for i, res in enumerate(results2):
                ox, oy = offsets[i]
                for b in res.boxes:
                    # 💡 [FIX] .item() 을 사용하여 텐서를 완벽하게 파이썬 float으로 변환!
                    bx1 = b.xyxy[0][0].item() + ox
                    by1 = b.xyxy[0][1].item() + oy
                    bx2 = b.xyxy[0][2].item() + ox
                    by2 = b.xyxy[0][3].item() + oy
                    
                    temp_boxes.append([bx1, by1, bx2, by2])
                    temp_scores.append(float(b.conf[0]))
                    temp_classes.append(int(b.cls[0]))
                    
            for c in set(temp_classes):
                c_boxes = [b for j, b in enumerate(temp_boxes) if temp_classes[j] == c]
                c_scores = [s for j, s in enumerate(temp_scores) if temp_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 2. 제안 1: Ours (DAHI Only)
        # =====================================================================
        elif method_name == "Ours (DAHI Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            roi_boxes = []
            
            # 💡 [핵심 최적화] 메인 모델(m2)로 CONF_FILTER(0.1) 기준 단 1번만 스캔
            t_inf_start = time.time()
            res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global_all[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = float(b.conf[0])
                
                # 1. 글로벌 확정 박스 (0.3 이상)
                if conf >= CONF_GLOBAL:
                    boosted_conf = min(1.0, conf * 1.10)
                    global_final_boxes.append([bx1, by1, bx2, by2])
                    global_final_scores.append(boosted_conf)
                    global_final_classes.append(int(b.cls[0]))
                
                # 2. ROI 박스 등록 (모든 탐지 객체 재검사)
                roi_boxes.append([bx1, by1, bx2, by2])
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            while len(remaining_boxes) > 0:
                best_count, best_region = -1, None
                for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                        count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                        if count > best_count: 
                            best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                if best_region and best_count >= 1:
                    dense_regions.append(best_region)
                    dx1, dy1, dx2, dy2 = best_region
                    remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                else: break
            
            unified_infer_list = [img[dy1:dy2, dx1:dx2] for dx1, dy1, dx2, dy2 in dense_regions]
            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for idx, (dx1, dy1, dx2, dy2) in enumerate(dense_regions):
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1
                    for b in res_all[idx].boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
            
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 3. 제안 2: Ours (Tetris Only)
        # =====================================================================
        elif method_name == "Ours (Tetris Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            roi_boxes = []
            
            t_inf_start = time.time()
            res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global_all[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = float(b.conf[0])
                if conf >= CONF_GLOBAL:
                    global_final_boxes.append([bx1, by1, bx2, by2])
                    global_final_scores.append(min(1.0, conf * 1.10))
                    global_final_classes.append(int(b.cls[0]))
                roi_boxes.append([bx1, by1, bx2, by2])
            
            canvases, canvas_infos = [], []
            if len(roi_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(roi_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

            if len(canvases) > 0:
                t_inf_start = time.time()
                res_pack = m2.predict(canvases, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(canvases)
                
                for c_idx, res in enumerate(res_pack):
                    for b in res.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                        for info in canvas_infos:
                            if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                s = info['scale']
                                orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 4. 제안 3: Ours (DAHI + Tetris)
        # =====================================================================
        elif method_name == "Ours (DAHI + Tetris)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            roi_boxes = []
            
            t_inf_start = time.time()
            res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global_all[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = float(b.conf[0])
                if conf >= CONF_GLOBAL:
                    global_final_boxes.append([bx1, by1, bx2, by2])
                    global_final_scores.append(min(1.0, conf * 1.10))
                    global_final_classes.append(int(b.cls[0]))
                roi_boxes.append([bx1, by1, bx2, by2])
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            # 💡 [NEW] CAD-Router (Class-Aware Density Router) 로직 시작
            # 1. 라우팅 판단을 위한 "소형 객체" 중심점만 추출 (중/대형 객체 배제)
            small_centroids = []
            for b in remaining_boxes:
                bx1, by1, bx2, by2 = b
                bw, bh = bx2 - bx1, by2 - by1
                if get_size_category(bw, bh) == 'small':
                    cx, cy = (bx1 + bx2) / 2, (by1 + by2) / 2
                    small_centroids.append((cx, cy))
            
            total_small_objs = len(small_centroids)
            # 기준점: 소형 객체 총합의 DENSE_RATIO_THRESH (50%) 이상 (단, 최소 1개 보장)
            route_threshold = max(1, int(total_small_objs * DENSE_RATIO_THRESH)) 
            
            if total_small_objs > 0:
                current_small_centroids = small_centroids.copy()
                
                # 밀집 구역이 기준을 충족하는 한 계속해서 DAHI 영역을 추출 (Spatial NMS)
                while len(current_small_centroids) > 0:
                    best_count, best_region = -1, None
                    
                    # 2. 슬라이딩 탐색으로 최적의 밀집 구역 1개 찾기
                    for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                        for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                            # 해당 창 내부에 있는 '소형 객체' 점의 개수만 카운트
                            count = sum(1 for cx, cy in current_small_centroids if x <= cx <= x + DENSE_WINDOW_SIZE and y <= cy <= y + DENSE_WINDOW_SIZE)
                            if count > best_count: 
                                best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                    
                    # 3. 라우팅 조건 판별
                    if best_region and best_count >= route_threshold:
                        # 기준을 넘었으므로 DAHI (밀집 구역) 리스트에 추가
                        dense_regions.append(best_region)
                        dx1, dy1, dx2, dy2 = best_region
                        
                        # Spatial NMS: 방금 채택된 영역에 속한 중심점들은 계산에서 제외
                        current_small_centroids = [(cx, cy) for cx, cy in current_small_centroids if not (dx1 <= cx <= dx2 and dy1 <= cy <= dy2)]
                        
                        # 실제 추론 큐(remaining_boxes)에서도 해당 영역에 속한 '모든 객체(대/중/소)'를 DAHI로 넘김 (제거)
                        remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                    else:
                        # 더 이상 임계값(n%)을 넘는 밀집 구역이 없다면 탐색 즉시 중단
                        # 남은 remaining_boxes는 아래쪽 코드에 의해 자연스럽게 Tetris(Spatial Packing)로 라우팅 됨
                        break
            
            # (이후 코드는 기존과 동일하게 unified_infer_list 생성 및 Tetris Canvas 생성 로직으로 이어짐)
            unified_infer_list = []
            dense_idx_list = []
            
            for dx1, dy1, dx2, dy2 in dense_regions:
                unified_infer_list.append(img[dy1:dy2, dx1:dx2])
                dense_idx_list.append((len(unified_infer_list) - 1, dx1, dy1, dx2, dy2))

            canvases, canvas_infos = [], []
            canvas_start_idx = -1
            if len(remaining_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(remaining_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

                if len(canvases) > 0:
                    canvas_start_idx = len(unified_infer_list)
                    unified_infer_list.extend(canvases)

            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for d_idx, dx1, dy1, dx2, dy2 in dense_idx_list:
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1; res_dense = res_all[d_idx]
                    for b in res_dense.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                
                if canvas_start_idx != -1:
                    res_pack = res_all[canvas_start_idx:]
                    for c_idx, res in enumerate(res_pack):
                        for b in res.boxes:
                            bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                            bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                            for info in canvas_infos:
                                if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                    if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                    s = info['scale']
                                    orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                    orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                    local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                    break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # 통계 저장
        img_total_time = time.time() - t_pipe_start
        is_hr = (w * h >= HR_THRESHOLD)
        target_keys = ['ALL', 'HR'] if is_hr else ['ALL', 'LR']
        for k in target_keys:
            stats[k]['count'] += 1
            stats[k]['inf_time'] += img_inf_time
            stats[k]['total_time'] += img_total_time
            stats[k]['inf_cnt'] += img_inf_cnt
            stats[k]['indices'].add(img_idx)

    # ---------------------------------------------------------
    # 💡 공식 COCO API 연산 엔진
    # ---------------------------------------------------------
    def calc_official_coco_metrics(subset_indices):
        if not subset_indices: return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}
        
        gt_dict = {"images": [], "annotations": [], "categories": []}
        for i in range(10): gt_dict["categories"].append({"id": i, "name": f"class_{i}"})
            
        ann_id = 1
        for img_idx in subset_indices:
            info = img_infos[img_idx]
            gt_dict["images"].append({"id": img_idx, "width": info['w'], "height": info['h'], "file_name": info['name']})
            for gt in all_gts[img_idx]:
                c, x1, y1, x2, y2 = gt
                bw, bh = x2 - x1, y2 - y1
                gt_dict["annotations"].append({"id": ann_id, "image_id": img_idx, "category_id": int(c), "bbox": [x1, y1, bw, bh], "area": bw * bh, "iscrowd": 0})
                ann_id += 1

        cocoGt = COCO()
        cocoGt.dataset = gt_dict
        cocoGt.createIndex()

        sub_preds = [p for p in all_preds if p[0] in subset_indices]
        pred_list = []
        for pred in sub_preds:
            img_idx, c, score, x1, y1, x2, y2 = pred
            bw, bh = x2 - x1, y2 - y1
            pred_list.append({"image_id": img_idx, "category_id": int(c), "bbox": [x1, y1, bw, bh], "score": float(score)})

        if not pred_list: return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}

        cocoDt = cocoGt.loadRes(pred_list)

        cocoEval = COCOeval(cocoGt, cocoDt, 'bbox')
        cocoEval.params.maxDets = [100, 300, 500] 
        
        cocoEval.evaluate()
        cocoEval.accumulate()
        
        with contextlib.redirect_stdout(io.StringIO()):
            cocoEval.summarize()

        if len(cocoEval.stats) < 12:
            return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0, "precision": None} # None 추가

        return {
            "AP50:95": cocoEval.stats[0],
            "AP50": cocoEval.stats[1],
            "AP_small": cocoEval.stats[3],  
            "AP_medium": cocoEval.stats[4], 
            "AP_large": cocoEval.stats[5],
            "precision": cocoEval.eval['precision'] # 💡 [NEW] PR 커브용 Raw 데이터 추가
        }

    result_dict = {}
    for group in ['ALL', 'HR', 'LR']:
        c = stats[group]['count']
        res = calc_official_coco_metrics(stats[group]['indices'])
        res['Img_Cnt'] = c
        res['Avg_Inf_Cnt'] = stats[group]['inf_cnt'] / c if c else 0
        res['Avg_Inf_Time'] = (stats[group]['inf_time'] / c) * 1000 if c else 0
        res['Avg_Tot_Time'] = (stats[group]['total_time'] / c) * 1000 if c else 0
        result_dict[group] = res
        
    result_dict['Peak_VRAM'] = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0
    return result_dict

# =========================================================
# 실행 및 다중 표 그리기 (Official COCO Protocol)
# =========================================================
methods = [
    "UC (2x2 Uniform Crop)", 
    "Ours (DAHI Only)",
    "Ours (Tetris Only)",
    "Ours (DAHI + Tetris)"
]

final_stats = {}
for m in methods: 
    final_stats[m] = run_official_ablation_benchmark(m)

print("\n" + "="*145)
print(f"🏆 [Ablation Study] Single Model & Official COCO Protocol Evaluation (Total {NUM_TEST_IMAGES} Images) 🏆")
print("="*145)
print(f"{'Method':<32} | {'Type':<4} | {'Img':<4} | {'mAP':<6} | {'AP50':<6} | {'APs':<6} | {'APm':<6} | {'APl':<6} | {'Inf Cnt':<7} | {'Inf Time':<9} | {'Tot Time':<9}")
print("-" * 145)

for m, groups in final_stats.items():
    for g in ['ALL', 'HR', 'LR']:
        s = groups[g]
        if s['Img_Cnt'] == 0: continue
        
        mAP  = s.get('AP50:95', 0.0)
        ap50 = s.get('AP50', 0.0)
        aps  = s.get('AP_small', 0.0)
        apm  = s.get('AP_medium', 0.0)
        apl  = s.get('AP_large', 0.0)
        
        print(f"{m if g == 'ALL' else '':<32} | {g:<4} | {s['Img_Cnt']:<4} | {mAP:.4f} | {ap50:.4f} | {aps:.4f} | {apm:.4f} | {apl:.4f} | {s['Avg_Inf_Cnt']:4.1f} /i | {s['Avg_Inf_Time']:5.1f} ms | {s['Avg_Tot_Time']:5.1f} ms")
    
    print(f"{'':<32} > Peak VRAM: {groups.get('Peak_VRAM', 0.0):.1f} MB")
    print("-" * 145)


import matplotlib.pyplot as plt
import numpy as np

# 논문용 영문 폰트 및 스타일 세팅 (한글 깨짐 문제 원천 차단)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-whitegrid')

# 색상 및 마커 지정
colors = ['#7f8c8d', '#e74c3c', '#3498db', '#2ecc71']
markers = ['s', 'o', '^', 'D']
methods = list(final_stats.keys())

# 💡 [NEW] 논문용 세련된 네이밍 매핑
display_names = {
    "UC (2x2 Uniform Crop)": "UC",
    "Ours (DAHI Only)": "DAHI Only",
    "Ours (Tetris Only)": "Tetris Only",
    "Ours (DAHI + Tetris)": "HAHI"
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Ablation Study of Inference Optimization Framework', fontsize=22, fontweight='bold')

# ==========================================
# 1. mAP vs Latency (Total Time) Trade-off
# ==========================================
ax = axes[0, 0]
for idx, m in enumerate(methods):
    if final_stats[m]['ALL']['Img_Cnt'] == 0: continue
    latency = final_stats[m]['ALL']['Avg_Tot_Time']
    mAP = final_stats[m]['ALL']['AP50:95']
    name = display_names[m]
    
    ax.scatter(latency, mAP, color=colors[idx], marker=markers[idx], s=200, label=name, edgecolor='black', zorder=5)
    ax.annotate(name, (latency, mAP), xytext=(10, -5), textcoords='offset points', fontsize=12, fontweight='medium')

ax.set_title('Overall mAP vs Total Latency', fontsize=15, pad=10)
ax.set_xlabel('Average Total Time (ms/img)', fontsize=13)
ax.set_ylabel('COCO mAP (AP50:95)', fontsize=13)
ax.grid(True, linestyle='--', alpha=0.7)

# ==========================================
# 2. AP_small vs Latency Trade-off (핵심 지표)
# ==========================================
ax = axes[0, 1]
for idx, m in enumerate(methods):
    if final_stats[m]['ALL']['Img_Cnt'] == 0: continue
    latency = final_stats[m]['ALL']['Avg_Tot_Time']
    aps = final_stats[m]['ALL']['AP_small']
    name = display_names[m]
    
    ax.scatter(latency, aps, color=colors[idx], marker=markers[idx], s=200, label=name, edgecolor='black', zorder=5)
    ax.annotate(name, (latency, aps), xytext=(10, -5), textcoords='offset points', fontsize=12, fontweight='medium')
    
ax.set_title('Small Object Accuracy (AP_small) vs Total Latency', fontsize=15, pad=10)
ax.set_xlabel('Average Total Time (ms/img)', fontsize=13)
ax.set_ylabel('AP_small', fontsize=13)
ax.grid(True, linestyle='--', alpha=0.7)

# ==========================================
# 3. Peak VRAM Usage Comparison
# ==========================================
ax = axes[1, 0]
vram_usages = [final_stats[m].get('Peak_VRAM', 0.0) for m in methods]
x_labels = [display_names[m] for m in methods]

bars = ax.bar(x_labels, vram_usages, color=colors, edgecolor='black', width=0.6)

ax.set_title('Peak VRAM Usage', fontsize=15, pad=10)
ax.set_ylabel('VRAM (MB)', fontsize=13)
ax.tick_params(axis='x', labelsize=12)

for bar in bars:
    yval = bar.get_height()
    if yval > 0:
        ax.text(bar.get_x() + bar.get_width()/2, yval + 5, f'{yval:.1f} MB', ha='center', va='bottom', fontsize=12, fontweight='bold')

# ==========================================
# 4. Precision-Recall Curve (for ALL area)
# ==========================================
ax = axes[1, 1]
for idx, m in enumerate(methods):
    precision_array = final_stats[m]['ALL'].get('precision', None)
    if precision_array is not None:
        pr_iou50 = precision_array[0, :, :, 0, -1] 
        pr_mean = np.mean([p[p > -1] for p in pr_iou50.T if len(p[p > -1]) > 0], axis=0) if len(pr_iou50) > 0 else []
        
        if len(pr_mean) == 101:
            recalls = np.linspace(0.0, 1.0, 101)
            # 💡 [NEW] drawstyle='steps-post'를 사용하여 학술적인 형태의 조밀한 계단식 커브 생성
            ax.plot(recalls, pr_mean, color=colors[idx], label=display_names[m], linewidth=2.5, drawstyle='steps-post', alpha=0.9)

ax.set_title('Precision-Recall Curve (IoU=0.50)', fontsize=15, pad=10)
ax.set_xlabel('Recall', fontsize=13)
ax.set_ylabel('Precision', fontsize=13)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.5, 1.05]) # 💡 [NEW] Y축을 0.5부터 시작하여 곡선 간의 차이를 부각
ax.legend(loc='lower left', fontsize=11, frameon=True, shadow=True)
ax.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.subplots_adjust(top=0.90)

# 저장 및 출력
plt.savefig('ablation_results_hahi.png', dpi=300, bbox_inches='tight')
print("\n✅ 영문 폰트 및 HAHI 네이밍이 적용된 그래프가 'ablation_results_hahi.png'로 저장되었습니다.")
plt.show()

🚀 [Ablation Study] Single-Model Pipeline! 완벽한 재현 시작! (Total 1294 images)


⏳ UC (2x2 Uniform Crop): 100%|██████████████████████████████| 1294/1294 [01:56<00:00, 11.08it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=32.81s).
Accumulating evaluation results...
DONE (t=1.55s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=6.68s).
Accumulating evaluation results...
DONE (t=0.27s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.04s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=26.30s).
Accumulating evaluation results...
DONE (t=1.21s).


⏳ Ours (DAHI Only): 100%|██████████████████████████████| 1294/1294 [01:52<00:00, 11.49it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.27s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=34.96s).
Accumulating evaluation results...
DONE (t=1.82s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=8.16s).
Accumulating evaluation results...
DONE (t=0.32s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=25.79s).
Accumulating evaluation results...
DONE (t=1.15s).


...

⏳ Ours (DAHI + Tetris): 100%|██████████████████████████████| 1294/1294 [01:32<00:00, 13.99it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.25s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=33.15s).
Accumulating evaluation results...
DONE (t=1.52s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.15s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=7.15s).
Accumulating evaluation results...
DONE (t=0.30s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.22s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=25.88s).
Accumulating evaluation results...
DONE (t=1.77s).

🏆 [Ablation Study] Single Model & Official COCO Protocol Evaluation (Total 5000 Images) 🏆
Method                           | Type | Img  | mAP    | AP50   | APs    | APm    | APl    | Inf Cnt | Inf Time  | Tot Time 
----------------------------------------------

C:\Users\<user>\AppData\Local\Temp\ipykernel_20736\816358959.py:753: UserWarning: Glyph 52628 (\N{HANGUL SYLLABLE CU}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\<user>\AppData\Local\Temp\ipykernel_20736\816358959.py:753: UserWarning: Glyph 47200 (\N{HANGUL SYLLABLE RON}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\<user>\AppData\Local\Temp\ipykernel_20736\816358959.py:753: UserWarning: Glyph 52572 (\N{HANGUL SYLLABLE COE}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\<user>\AppData\Local\Temp\ipykernel_20736\816358959.py:753: UserWarning: Glyph 51201 (\N{HANGUL SYLLABLE JEOG}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\<user>\AppData\Local\Temp\ipykernel_20736\816358959.py:753: UserWarning: Glyph 54868 (\N{HANGUL SYLLABLE HWA}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\<user>\AppData\Local\Temp\ipykernel_20736\816358959.py:753: UserWarning: Glyph 54532 (\N{HANGUL SYLLABLE PEU}) missing from font(s) Arial.
  plt.t


✅ 시각화 그래프가 'ablation_results.png'로 저장되었습니다.


f:\notebook\workspace\sd-scripts\venv_5050\lib\site-packages\IPython\core\pylabtools.py:170: UserWarning: Glyph 52628 (\N{HANGUL SYLLABLE CU}) missing from font(s) Arial.
  fig.canvas.print_figure(bytes_io, **kw)
f:\notebook\workspace\sd-scripts\venv_5050\lib\site-packages\IPython\core\pylabtools.py:170: UserWarning: Glyph 47200 (\N{HANGUL SYLLABLE RON}) missing from font(s) Arial.
  fig.canvas.print_figure(bytes_io, **kw)
f:\notebook\workspace\sd-scripts\venv_5050\lib\site-packages\IPython\core\pylabtools.py:170: UserWarning: Glyph 52572 (\N{HANGUL SYLLABLE COE}) missing from font(s) Arial.
  fig.canvas.print_figure(bytes_io, **kw)
f:\notebook\workspace\sd-scripts\venv_5050\lib\site-packages\IPython\core\pylabtools.py:170: UserWarning: Glyph 51201 (\N{HANGUL SYLLABLE JEOG}) missing from font(s) Arial.
  fig.canvas.print_figure(bytes_io, **kw)
f:\notebook\workspace\sd-scripts\venv_5050\lib\site-packages\IPython\core\pylabtools.py:170: UserWarning: Glyph 54868 (\N{HANGUL SYLLABLE HWA}) 

In [ ]:
# 라우팅

import cv2
import os
import time
import numpy as np
import tqdm
import torch
from ultralytics import YOLO

import contextlib
import io

# 💡 [NEW] 공식 COCO API 임포트
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# =========================================================
# ⚙️ 하이퍼파라미터 (Hyperparameters) - Single Model Edition
# =========================================================
# MODEL_FILTER_PATH = 'model/best_nano.pt' # 💡 보조 모델 완전히 삭제!
MODEL_MAIN_PATH = 'model/best_small.pt'

CONF_GLOBAL = 0.3
CONF_FILTER = 0.1     
CONF_DENSE = 0.3
CONF_TETRIS = 0.3
CONF_UC = 0.3

DENSE_RATIO_THRESH = 0.30  # 💡 [NEW] 라우팅 임계값: 전체 소형 객체의 50% 이상 밀집 시 DAHI 구역 인정
# IOU_FILTER_MATCH = 0.97 # 💡 사용 안함 (중복검사 삭제됨)
NMS_CONF_THRESH = 0.3
NMS_IOU_THRESH = 0.4    

DENSE_WINDOW_SIZE = 512
DENSE_STEP = 320      

MERGE_PAD = 16
CROP_PAD_LARGE = 80     
CROP_PAD_SMALL = 16
CROP_PAD_THRESH = 200

CANVAS_SIZE = 960
CANVAS_MARGIN = 2
CANVAS_BG_COLOR = 114

UPSCALE_RATIO = 1.5        
UPSCALE_MAX_THRESH = 200    

NUM_TEST_IMAGES = 5000
HR_THRESHOLD = 1920 * 1080 

# =========================================================
dataset_root = 'data/valid'
img_dir, lbl_dir = os.path.join(dataset_root, 'images'), os.path.join(dataset_root, 'labels')
img_list = sorted(os.listdir(img_dir))[:NUM_TEST_IMAGES]

print(f"🚀 [Ablation Study] Single-Model Pipeline! 완벽한 재현 시작! (Total {len(img_list)} images)")

# m1 = YOLO(MODEL_FILTER_PATH) # 💡 보조 모델 삭제
m2 = YOLO(MODEL_MAIN_PATH)

def calculate_iou(box1, box2):
    xi1, yi1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    xi2, yi2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])

def get_size_category(w, h):
    area = w * h
    if area < 32 ** 2: return 'small'
    elif area < 96 ** 2: return 'medium'
    else: return 'large'

def merge_clusters_dynamic(boxes, img_w, img_h, merge_pad=MERGE_PAD):
    if not len(boxes): return []
    def get_padded(b, pad): return [max(0, b[0]-pad), max(0, b[1]-pad), min(img_w, b[2]+pad), min(img_h, b[3]+pad)]
    def is_overlap(b1, b2):
        p1, p2 = get_padded(b1, merge_pad), get_padded(b2, merge_pad)
        return (min(p1[2], p2[2]) > max(p1[0], p2[0])) and (min(p1[3], p2[3]) > max(p1[1], p2[1]))
    curr = boxes.copy()
    while True:
        merged, flags = [], [False]*len(curr)
        for i in range(len(curr)):
            if flags[i]: continue
            b = curr[i]
            for j in range(i+1, len(curr)):
                if not flags[j] and is_overlap(b, curr[j]):
                    b = [min(b[0], curr[j][0]), min(b[1], curr[j][1]), max(b[2], curr[j][2]), max(b[3], curr[j][3])]
                    flags[j] = True
            merged.append(b)
        if len(merged) == len(curr): break
        curr = merged
    final_boxes = []
    for b in curr:
        bw, bh = b[2] - b[0], b[3] - b[1]
        crop_pad = CROP_PAD_LARGE if max(bw, bh) < CROP_PAD_THRESH else CROP_PAD_SMALL 
        final_boxes.append(get_padded(b, crop_pad))
    return final_boxes

def run_official_ablation_benchmark(method_name):
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        
    all_gts = {}; all_preds = []
    img_infos = {} 
    
    stats = {
        'ALL': {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'HR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'LR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()}
    }

    pbar = tqdm.tqdm(img_list, desc=f"⏳ {method_name}", bar_format='{l_bar}{bar:30}{r_bar}')
    for img_idx, img_name in enumerate(pbar):
        img_path, lbl_path = os.path.join(img_dir, img_name), os.path.join(lbl_dir, img_name.replace('.jpg', '.txt'))
        img = cv2.imread(img_path); h, w, _ = img.shape
        
        img_infos[img_idx] = {'w': w, 'h': h, 'name': img_name}
        
        gts = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    c, xc, yc, bw, bh = map(float, line.split())
                    gts.append([int(c), (xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h]) 
        all_gts[img_idx] = gts

        t_pipe_start = time.time()
        img_inf_time, img_inf_cnt = 0, 0
        
        # =====================================================================
        # 1. Baseline: UC (2x2 Uniform Crop)
        # =====================================================================
        if method_name == "UC (2x2 Uniform Crop)":
            # UC 평가가 너무 오래 걸려 비활성화
            ch, cw = h // 2, w // 2
            # crops, offsets = [img], [(0, 0)]
            # for y in [0, ch]:
            #     for x in [0, cw]:
            #         crops.append(img[y:y+ch, x:x+cw])
            #         offsets.append((x, y))
            
            # t_inf_start = time.time()
            # results2 = m2.predict(crops, conf=CONF_UC, verbose=False, batch=5)
            # img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 5 
            
            # temp_boxes, temp_scores, temp_classes = [], [], []
            # for i, res in enumerate(results2):
            #     ox, oy = offsets[i]
            #     for b in res.boxes:
            #         # 💡 [FIX] .item() 을 사용하여 텐서를 완벽하게 파이썬 float으로 변환!
            #         bx1 = b.xyxy[0][0].item() + ox
            #         by1 = b.xyxy[0][1].item() + oy
            #         bx2 = b.xyxy[0][2].item() + ox
            #         by2 = b.xyxy[0][3].item() + oy
                    
            #         temp_boxes.append([bx1, by1, bx2, by2])
            #         temp_scores.append(float(b.conf[0]))
            #         temp_classes.append(int(b.cls[0]))
                    
            # for c in set(temp_classes):
            #     c_boxes = [b for j, b in enumerate(temp_boxes) if temp_classes[j] == c]
            #     c_scores = [s for j, s in enumerate(temp_scores) if temp_classes[j] == c]
            #     cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
            #     indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
            #     if len(indices) > 0:
            #         for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 2. 제안 1: Ours (DAHI Only)
        # =====================================================================
        elif method_name == "Ours (DAHI Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            # local_boxes, local_scores, local_classes = [], [], []
            # roi_boxes = []
            
            # # 💡 [핵심 최적화] 메인 모델(m2)로 CONF_FILTER(0.1) 기준 단 1번만 스캔
            # t_inf_start = time.time()
            # res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            # img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            # for b in res_global_all[0].boxes:
            #     bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
            #     conf = float(b.conf[0])
                
            #     # 1. 글로벌 확정 박스 (0.3 이상)
            #     if conf >= CONF_GLOBAL:
            #         boosted_conf = min(1.0, conf * 1.10)
            #         global_final_boxes.append([bx1, by1, bx2, by2])
            #         global_final_scores.append(boosted_conf)
            #         global_final_classes.append(int(b.cls[0]))
                
            #     # 2. ROI 박스 등록 (모든 탐지 객체 재검사)
            #     roi_boxes.append([bx1, by1, bx2, by2])
            
            # remaining_boxes = roi_boxes.copy()
            # dense_regions = []
            
            # while len(remaining_boxes) > 0:
            #     best_count, best_region = -1, None
            #     for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
            #         for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
            #             count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
            #             if count > best_count: 
            #                 best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
            #     if best_region and best_count >= 1:
            #         dense_regions.append(best_region)
            #         dx1, dy1, dx2, dy2 = best_region
            #         remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
            #     else: break
            
            # unified_infer_list = [img[dy1:dy2, dx1:dx2] for dx1, dy1, dx2, dy2 in dense_regions]
            # if len(unified_infer_list) > 0:
            #     t_inf_start = time.time()
            #     res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
            #     img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
            #     for idx, (dx1, dy1, dx2, dy2) in enumerate(dense_regions):
            #         cw_dense, ch_dense = dx2 - dx1, dy2 - dy1
            #         for b in res_all[idx].boxes:
            #             bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
            #             if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
            #             local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
            
            # final_local_preds = []
            # for c in set(local_classes):
            #     c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
            #     cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
            #     indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
            #     if len(indices) > 0:
            #         for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            # combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            # for c in set(combined_classes):
            #     c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
            #     cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
            #     indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
            #     if len(indices) > 0:
            #         for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 3. 제안 2: Ours (Tetris Only)
        # =====================================================================
        elif method_name == "Ours (Tetris Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            # local_boxes, local_scores, local_classes = [], [], []
            # roi_boxes = []
            
            # t_inf_start = time.time()
            # res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            # img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            # for b in res_global_all[0].boxes:
            #     bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
            #     conf = float(b.conf[0])
            #     if conf >= CONF_GLOBAL:
            #         global_final_boxes.append([bx1, by1, bx2, by2])
            #         global_final_scores.append(min(1.0, conf * 1.10))
            #         global_final_classes.append(int(b.cls[0]))
            #     roi_boxes.append([bx1, by1, bx2, by2])
            
            # canvases, canvas_infos = [], []
            # if len(roi_boxes) > 0:
            #     clustered_boxes = merge_clusters_dynamic(roi_boxes, w, h, merge_pad=MERGE_PAD)
            #     crops_to_pack = []
            #     for cb in clustered_boxes:
            #         cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
            #         scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
            #         cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
            #         if cw_crop > 0 and ch_crop > 0:
            #             crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
            #             if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
            #             else: crop_img = crop_img[:ch_crop, :cw_crop]
            #             crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
            #     crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
            #     current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
            #     cx, cy, max_h = 0, 0, 0
            #     for item in crops_to_pack:
            #         if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
            #         if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
            #         current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
            #         canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
            #         cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
            #     if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
            #     for c_idx, canvas in enumerate(canvases):
            #         c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
            #         if not c_infos: continue
            #         unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
            #         for i, cy1 in enumerate(unique_cy1s):
            #             row_items = [info for info in c_infos if info['cy1'] == cy1]
            #             row_items.sort(key=lambda x: x['cx1'])
            #             next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
            #             for j, info in enumerate(row_items):
            #                 item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
            #                 ox, oy, s = info['ox'], info['oy'], info['scale']
            #                 org_w, org_h = int(item_w / s), int(item_h / s) 
            #                 next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
            #                 gap_w = next_cx1 - info['cx2']
            #                 if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
            #                 if gap_w > 0:
            #                     ext_w_org = min(int(gap_w / s), w - (ox + org_w))
            #                     if ext_w_org > 0:
            #                         ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
            #                         if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
            #                         canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
            #                         info['cx2'] += ext_crop.shape[1]
            #                 gap_h = next_cy1 - info['cy2']
            #                 if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
            #                 if gap_h > 0:
            #                     ext_h_org = min(int(gap_h / s), h - (oy + org_h))
            #                     if ext_h_org > 0:
            #                         ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
            #                         if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
            #                         canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
            #                         info['cy2'] += ext_crop.shape[0]

            # if len(canvases) > 0:
            #     t_inf_start = time.time()
            #     res_pack = m2.predict(canvases, conf=CONF_TETRIS, verbose=False, batch=16)
            #     img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(canvases)
                
            #     for c_idx, res in enumerate(res_pack):
            #         for b in res.boxes:
            #             bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
            #             bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
            #             for info in canvas_infos:
            #                 if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
            #                     if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
            #                     s = info['scale']
            #                     orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
            #                     orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
            #                     local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
            #                     break
                                        
            # final_local_preds = []
            # for c in set(local_classes):
            #     c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
            #     cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
            #     indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
            #     if len(indices) > 0:
            #         for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            # combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            # for c in set(combined_classes):
            #     c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
            #     cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
            #     indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
            #     if len(indices) > 0:
            #         for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 4. 제안 3: Ours (DAHI + Tetris)
        # =====================================================================
        elif method_name == "Ours (DAHI + Tetris)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            roi_boxes = []
            
            t_inf_start = time.time()
            res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global_all[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = float(b.conf[0])
                if conf >= CONF_GLOBAL:
                    global_final_boxes.append([bx1, by1, bx2, by2])
                    global_final_scores.append(min(1.0, conf * 1.10))
                    global_final_classes.append(int(b.cls[0]))
                roi_boxes.append([bx1, by1, bx2, by2])
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            # 💡 [NEW] CAD-Router (Class-Aware Density Router) 로직 시작
            # 1. 라우팅 판단을 위한 "소형 객체" 중심점만 추출 (중/대형 객체 배제)
            small_centroids = []
            for b in remaining_boxes:
                bx1, by1, bx2, by2 = b
                bw, bh = bx2 - bx1, by2 - by1
                if get_size_category(bw, bh) == 'small':
                    cx, cy = (bx1 + bx2) / 2, (by1 + by2) / 2
                    small_centroids.append((cx, cy))
            
            total_small_objs = len(small_centroids)
            # 기준점: 소형 객체 총합의 DENSE_RATIO_THRESH (50%) 이상 (단, 최소 1개 보장)
            route_threshold = max(1, int(total_small_objs * DENSE_RATIO_THRESH)) 
            
            if total_small_objs > 0:
                current_small_centroids = small_centroids.copy()
                
                # 밀집 구역이 기준을 충족하는 한 계속해서 DAHI 영역을 추출 (Spatial NMS)
                while len(current_small_centroids) > 0:
                    best_count, best_region = -1, None
                    
                    # 2. 슬라이딩 탐색으로 최적의 밀집 구역 1개 찾기
                    for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                        for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                            # 해당 창 내부에 있는 '소형 객체' 점의 개수만 카운트
                            count = sum(1 for cx, cy in current_small_centroids if x <= cx <= x + DENSE_WINDOW_SIZE and y <= cy <= y + DENSE_WINDOW_SIZE)
                            if count > best_count: 
                                best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                    
                    # 3. 라우팅 조건 판별
                    if best_region and best_count >= route_threshold:
                        dx1, dy1, dx2, dy2 = best_region
                        
                        # ---------------------------------------------------------
                        # 💡 [NEW] Gravity Snapping (무게중심 유도 기법) 적용
                        # ---------------------------------------------------------
                        # 1. 현재 대략적으로 잡힌 창 내부의 소형 객체들만 추출
                        pts_in_window = [[cx, cy] for cx, cy in current_small_centroids if dx1 <= cx <= dx2 and dy1 <= cy <= dy2]
                        
                        if len(pts_in_window) > 0:
                            pts_arr = np.array(pts_in_window)
                            # 2. 내부 객체들의 실제 X, Y 평균(무게중심) 계산
                            mean_x = np.mean(pts_arr[:, 0])
                            mean_y = np.mean(pts_arr[:, 1])
                            
                            # 3. 창의 정중앙이 무게중심에 오도록 좌표 재조정
                            new_dx1 = int(mean_x - (DENSE_WINDOW_SIZE / 2))
                            new_dy1 = int(mean_y - (DENSE_WINDOW_SIZE / 2))
                            
                            # 4. 이미지 밖으로 벗어나지 않도록 경계 처리(Clamping)
                            new_dx1 = max(0, min(new_dx1, w - DENSE_WINDOW_SIZE))
                            new_dy1 = max(0, min(new_dy1, h - DENSE_WINDOW_SIZE))
                            new_dx2 = new_dx1 + DENSE_WINDOW_SIZE
                            new_dy2 = new_dy1 + DENSE_WINDOW_SIZE
                            
                            # 5. 스냅된 새로운 좌표로 갱신
                            dx1, dy1, dx2, dy2 = new_dx1, new_dy1, new_dx2, new_dy2
                            best_region = (dx1, dy1, dx2, dy2)
                        # ---------------------------------------------------------

                        # 기준을 넘었으므로 DAHI (밀집 구역) 리스트에 추가
                        dense_regions.append(best_region)
                        
                        # Spatial NMS: 방금 채택된 영역에 속한 중심점들은 계산에서 제외
                        current_small_centroids = [(cx, cy) for cx, cy in current_small_centroids if not (dx1 <= cx <= dx2 and dy1 <= cy <= dy2)]
                        
                        # 실제 추론 큐(remaining_boxes)에서도 해당 영역에 속한 '모든 객체(대/중/소)'를 DAHI로 넘김 (제거)
                        remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                    else:
                        # 더 이상 임계값(n%)을 넘는 밀집 구역이 없다면 탐색 즉시 중단
                        # 남은 remaining_boxes는 아래쪽 코드에 의해 자연스럽게 Tetris(Spatial Packing)로 라우팅 됨
                        break
            
            # (이후 코드는 기존과 동일하게 unified_infer_list 생성 및 Tetris Canvas 생성 로직으로 이어짐)
            unified_infer_list = []
            dense_idx_list = []
            
            for dx1, dy1, dx2, dy2 in dense_regions:
                unified_infer_list.append(img[dy1:dy2, dx1:dx2])
                dense_idx_list.append((len(unified_infer_list) - 1, dx1, dy1, dx2, dy2))

            canvases, canvas_infos = [], []
            canvas_start_idx = -1
            if len(remaining_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(remaining_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

                if len(canvases) > 0:
                    canvas_start_idx = len(unified_infer_list)
                    unified_infer_list.extend(canvases)

            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for d_idx, dx1, dy1, dx2, dy2 in dense_idx_list:
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1; res_dense = res_all[d_idx]
                    for b in res_dense.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                
                if canvas_start_idx != -1:
                    res_pack = res_all[canvas_start_idx:]
                    for c_idx, res in enumerate(res_pack):
                        for b in res.boxes:
                            bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                            bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                            for info in canvas_infos:
                                if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                    if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                    s = info['scale']
                                    orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                    orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                    local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                    break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # 통계 저장
        img_total_time = time.time() - t_pipe_start
        is_hr = (w * h >= HR_THRESHOLD)
        target_keys = ['ALL', 'HR'] if is_hr else ['ALL', 'LR']
        for k in target_keys:
            stats[k]['count'] += 1
            stats[k]['inf_time'] += img_inf_time
            stats[k]['total_time'] += img_total_time
            stats[k]['inf_cnt'] += img_inf_cnt
            stats[k]['indices'].add(img_idx)

    # ---------------------------------------------------------
    # 💡 공식 COCO API 연산 엔진
    # ---------------------------------------------------------
    def calc_official_coco_metrics(subset_indices):
        if not subset_indices: return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}
        
        gt_dict = {"images": [], "annotations": [], "categories": []}
        for i in range(10): gt_dict["categories"].append({"id": i, "name": f"class_{i}"})
            
        ann_id = 1
        for img_idx in subset_indices:
            info = img_infos[img_idx]
            gt_dict["images"].append({"id": img_idx, "width": info['w'], "height": info['h'], "file_name": info['name']})
            for gt in all_gts[img_idx]:
                c, x1, y1, x2, y2 = gt
                bw, bh = x2 - x1, y2 - y1
                gt_dict["annotations"].append({"id": ann_id, "image_id": img_idx, "category_id": int(c), "bbox": [x1, y1, bw, bh], "area": bw * bh, "iscrowd": 0})
                ann_id += 1

        cocoGt = COCO()
        cocoGt.dataset = gt_dict
        cocoGt.createIndex()

        sub_preds = [p for p in all_preds if p[0] in subset_indices]
        pred_list = []
        for pred in sub_preds:
            img_idx, c, score, x1, y1, x2, y2 = pred
            bw, bh = x2 - x1, y2 - y1
            pred_list.append({"image_id": img_idx, "category_id": int(c), "bbox": [x1, y1, bw, bh], "score": float(score)})

        if not pred_list: return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}

        cocoDt = cocoGt.loadRes(pred_list)

        cocoEval = COCOeval(cocoGt, cocoDt, 'bbox')
        cocoEval.params.maxDets = [100, 300, 500] 
        
        cocoEval.evaluate()
        cocoEval.accumulate()
        
        with contextlib.redirect_stdout(io.StringIO()):
            cocoEval.summarize()

        if len(cocoEval.stats) < 12:
            return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0, "precision": None} # None 추가

        return {
            "AP50:95": cocoEval.stats[0],
            "AP50": cocoEval.stats[1],
            "AP_small": cocoEval.stats[3],  
            "AP_medium": cocoEval.stats[4], 
            "AP_large": cocoEval.stats[5],
            "precision": cocoEval.eval['precision'] # 💡 [NEW] PR 커브용 Raw 데이터 추가
        }

    result_dict = {}
    for group in ['ALL', 'HR', 'LR']:
        c = stats[group]['count']
        res = calc_official_coco_metrics(stats[group]['indices'])
        res['Img_Cnt'] = c
        res['Avg_Inf_Cnt'] = stats[group]['inf_cnt'] / c if c else 0
        res['Avg_Inf_Time'] = (stats[group]['inf_time'] / c) * 1000 if c else 0
        res['Avg_Tot_Time'] = (stats[group]['total_time'] / c) * 1000 if c else 0
        result_dict[group] = res
        
    result_dict['Peak_VRAM'] = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0
    return result_dict

# =========================================================
# 실행 및 다중 표 그리기 (Official COCO Protocol)
# =========================================================
methods = [
    "UC (2x2 Uniform Crop)", 
    "Ours (DAHI Only)",
    "Ours (Tetris Only)",
    "Ours (DAHI + Tetris)"
]

final_stats = {}
for m in methods: 
    final_stats[m] = run_official_ablation_benchmark(m)

print("\n" + "="*145)
print(f"🏆 [Ablation Study] Single Model & Official COCO Protocol Evaluation (Total {NUM_TEST_IMAGES} Images) 🏆")
print("="*145)
print(f"{'Method':<32} | {'Type':<4} | {'Img':<4} | {'mAP':<6} | {'AP50':<6} | {'APs':<6} | {'APm':<6} | {'APl':<6} | {'Inf Cnt':<7} | {'Inf Time':<9} | {'Tot Time':<9}")
print("-" * 145)

for m, groups in final_stats.items():
    for g in ['ALL', 'HR', 'LR']:
        s = groups[g]
        if s['Img_Cnt'] == 0: continue
        
        mAP  = s.get('AP50:95', 0.0)
        ap50 = s.get('AP50', 0.0)
        aps  = s.get('AP_small', 0.0)
        apm  = s.get('AP_medium', 0.0)
        apl  = s.get('AP_large', 0.0)
        
        print(f"{m if g == 'ALL' else '':<32} | {g:<4} | {s['Img_Cnt']:<4} | {mAP:.4f} | {ap50:.4f} | {aps:.4f} | {apm:.4f} | {apl:.4f} | {s['Avg_Inf_Cnt']:4.1f} /i | {s['Avg_Inf_Time']:5.1f} ms | {s['Avg_Tot_Time']:5.1f} ms")
    
    print(f"{'':<32} > Peak VRAM: {groups.get('Peak_VRAM', 0.0):.1f} MB")
    print("-" * 145)


import matplotlib.pyplot as plt
import numpy as np

# 논문용 영문 폰트 및 스타일 세팅 (한글 깨짐 문제 원천 차단)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-whitegrid')

# 색상 및 마커 지정
colors = ['#7f8c8d', '#e74c3c', '#3498db', '#2ecc71']
markers = ['s', 'o', '^', 'D']
methods = list(final_stats.keys())

# 💡 [NEW] 논문용 세련된 네이밍 매핑
display_names = {
    "UC (2x2 Uniform Crop)": "UC",
    "Ours (DAHI Only)": "DAHI Only",
    "Ours (Tetris Only)": "Tetris Only",
    "Ours (DAHI + Tetris)": "HAHI"
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Ablation Study of Inference Optimization Framework', fontsize=22, fontweight='bold')

# ==========================================
# 1. mAP vs Latency (Total Time) Trade-off
# ==========================================
ax = axes[0, 0]
for idx, m in enumerate(methods):
    if final_stats[m]['ALL']['Img_Cnt'] == 0: continue
    latency = final_stats[m]['ALL']['Avg_Tot_Time']
    mAP = final_stats[m]['ALL']['AP50:95']
    name = display_names[m]
    
    ax.scatter(latency, mAP, color=colors[idx], marker=markers[idx], s=200, label=name, edgecolor='black', zorder=5)
    ax.annotate(name, (latency, mAP), xytext=(10, -5), textcoords='offset points', fontsize=12, fontweight='medium')

ax.set_title('Overall mAP vs Total Latency', fontsize=15, pad=10)
ax.set_xlabel('Average Total Time (ms/img)', fontsize=13)
ax.set_ylabel('COCO mAP (AP50:95)', fontsize=13)
ax.grid(True, linestyle='--', alpha=0.7)

# ==========================================
# 2. AP_small vs Latency Trade-off (핵심 지표)
# ==========================================
ax = axes[0, 1]
for idx, m in enumerate(methods):
    if final_stats[m]['ALL']['Img_Cnt'] == 0: continue
    latency = final_stats[m]['ALL']['Avg_Tot_Time']
    aps = final_stats[m]['ALL']['AP_small']
    name = display_names[m]
    
    ax.scatter(latency, aps, color=colors[idx], marker=markers[idx], s=200, label=name, edgecolor='black', zorder=5)
    ax.annotate(name, (latency, aps), xytext=(10, -5), textcoords='offset points', fontsize=12, fontweight='medium')
    
ax.set_title('Small Object Accuracy (AP_small) vs Total Latency', fontsize=15, pad=10)
ax.set_xlabel('Average Total Time (ms/img)', fontsize=13)
ax.set_ylabel('AP_small', fontsize=13)
ax.grid(True, linestyle='--', alpha=0.7)

# ==========================================
# 3. Peak VRAM Usage Comparison
# ==========================================
ax = axes[1, 0]
vram_usages = [final_stats[m].get('Peak_VRAM', 0.0) for m in methods]
x_labels = [display_names[m] for m in methods]

bars = ax.bar(x_labels, vram_usages, color=colors, edgecolor='black', width=0.6)

ax.set_title('Peak VRAM Usage', fontsize=15, pad=10)
ax.set_ylabel('VRAM (MB)', fontsize=13)
ax.tick_params(axis='x', labelsize=12)

for bar in bars:
    yval = bar.get_height()
    if yval > 0:
        ax.text(bar.get_x() + bar.get_width()/2, yval + 5, f'{yval:.1f} MB', ha='center', va='bottom', fontsize=12, fontweight='bold')

# ==========================================
# 4. Precision-Recall Curve (for ALL area)
# ==========================================
ax = axes[1, 1]
for idx, m in enumerate(methods):
    precision_array = final_stats[m]['ALL'].get('precision', None)
    if precision_array is not None:
        pr_iou50 = precision_array[0, :, :, 0, -1] 
        pr_mean = np.mean([p[p > -1] for p in pr_iou50.T if len(p[p > -1]) > 0], axis=0) if len(pr_iou50) > 0 else []
        
        if len(pr_mean) == 101:
            recalls = np.linspace(0.0, 1.0, 101)
            # 💡 [NEW] drawstyle='steps-post'를 사용하여 학술적인 형태의 조밀한 계단식 커브 생성
            ax.plot(recalls, pr_mean, color=colors[idx], label=display_names[m], linewidth=2.5, drawstyle='steps-post', alpha=0.9)

ax.set_title('Precision-Recall Curve (IoU=0.50)', fontsize=15, pad=10)
ax.set_xlabel('Recall', fontsize=13)
ax.set_ylabel('Precision', fontsize=13)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.5, 1.05]) # 💡 [NEW] Y축을 0.5부터 시작하여 곡선 간의 차이를 부각
ax.legend(loc='lower left', fontsize=11, frameon=True, shadow=True)
ax.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.subplots_adjust(top=0.90)

# 저장 및 출력
plt.savefig('ablation_results_hahi.png', dpi=300, bbox_inches='tight')
print("\n✅ 영문 폰트 및 HAHI 네이밍이 적용된 그래프가 'ablation_results_hahi.png'로 저장되었습니다.")
plt.show()

🚀 [Ablation Study] Single-Model Pipeline! 완벽한 재현 시작! (Total 1294 images)


⏳ UC (2x2 Uniform Crop): 100%|██████████████████████████████| 1294/1294 [00:16<00:00, 77.19it/s]


creating index...
index created!
creating index...
index created!
creating index...
index created!


⏳ Ours (DAHI Only): 100%|██████████████████████████████| 1294/1294 [00:16<00:00, 80.40it/s]


creating index...
index created!
creating index...
index created!
creating index...
index created!


⏳ Ours (Tetris Only): 100%|██████████████████████████████| 1294/1294 [00:16<00:00, 79.79it/s]


creating index...
index created!
creating index...
index created!
creating index...
index created!


⏳ Ours (DAHI + Tetris): 100%|██████████████████████████████| 1294/1294 [01:31<00:00, 14.09it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.06s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=33.46s).
Accumulating evaluation results...
DONE (t=1.49s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=7.23s).
Accumulating evaluation results...
DONE (t=0.28s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.04s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=25.89s).
Accumulating evaluation results...
DONE (t=1.25s).

🏆 [Ablation Study] Single Model & Official COCO Protocol Evaluation (Total 5000 Images) 🏆
Method                           | Type | Img  | mAP    | AP50   | APs    | APm    | APl    | Inf Cnt | Inf Time  | Tot Time 
----------------------------------------------

In [ ]:
import cv2
import os
import time
import numpy as np
import tqdm
import torch
from ultralytics import YOLO

import contextlib
import io

# 💡 [NEW] 공식 COCO API 임포트
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# =========================================================
# ⚙️ 하이퍼파라미터 (Hyperparameters) - Single Model Edition
# =========================================================
# MODEL_FILTER_PATH = 'model/best_nano.pt' # 💡 보조 모델 완전히 삭제!
MODEL_MAIN_PATH = 'model/best_small.pt'

CONF_GLOBAL = 0.3
CONF_FILTER = 0.1     
CONF_DENSE = 0.3
CONF_TETRIS = 0.3
CONF_UC = 0.3

DENSE_RATIO_THRESH = 0.30  # 💡 [NEW] 라우팅 임계값: 전체 소형 객체의 50% 이상 밀집 시 DAHI 구역 인정
# IOU_FILTER_MATCH = 0.97 # 💡 사용 안함 (중복검사 삭제됨)
NMS_CONF_THRESH = 0.3
NMS_IOU_THRESH = 0.4    

DENSE_WINDOW_SIZE = 512
DENSE_STEP = 320      

MERGE_PAD = 16
CROP_PAD_LARGE = 80     
CROP_PAD_SMALL = 16
CROP_PAD_THRESH = 200

CANVAS_SIZE = 960
CANVAS_MARGIN = 2
CANVAS_BG_COLOR = 114

UPSCALE_RATIO = 1.5        
UPSCALE_MAX_THRESH = 200    

NUM_TEST_IMAGES = 5000
HR_THRESHOLD = 1920 * 1080 

# =========================================================
dataset_root = 'data/valid'
img_dir, lbl_dir = os.path.join(dataset_root, 'images'), os.path.join(dataset_root, 'labels')
img_list = sorted(os.listdir(img_dir))[:NUM_TEST_IMAGES]

print(f"🚀 [Ablation Study] Single-Model Pipeline! 완벽한 재현 시작! (Total {len(img_list)} images)")

# m1 = YOLO(MODEL_FILTER_PATH) # 💡 보조 모델 삭제
m2 = YOLO(MODEL_MAIN_PATH)

def calculate_iou(box1, box2):
    xi1, yi1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    xi2, yi2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])

def get_size_category(w, h):
    area = w * h
    if area < 32 ** 2: return 'small'
    elif area < 96 ** 2: return 'medium'
    else: return 'large'

def merge_clusters_dynamic(boxes, img_w, img_h, merge_pad=MERGE_PAD):
    if not len(boxes): return []
    def get_padded(b, pad): return [max(0, b[0]-pad), max(0, b[1]-pad), min(img_w, b[2]+pad), min(img_h, b[3]+pad)]
    def is_overlap(b1, b2):
        p1, p2 = get_padded(b1, merge_pad), get_padded(b2, merge_pad)
        return (min(p1[2], p2[2]) > max(p1[0], p2[0])) and (min(p1[3], p2[3]) > max(p1[1], p2[1]))
    curr = boxes.copy()
    while True:
        merged, flags = [], [False]*len(curr)
        for i in range(len(curr)):
            if flags[i]: continue
            b = curr[i]
            for j in range(i+1, len(curr)):
                if not flags[j] and is_overlap(b, curr[j]):
                    b = [min(b[0], curr[j][0]), min(b[1], curr[j][1]), max(b[2], curr[j][2]), max(b[3], curr[j][3])]
                    flags[j] = True
            merged.append(b)
        if len(merged) == len(curr): break
        curr = merged
    final_boxes = []
    for b in curr:
        bw, bh = b[2] - b[0], b[3] - b[1]
        crop_pad = CROP_PAD_LARGE if max(bw, bh) < CROP_PAD_THRESH else CROP_PAD_SMALL 
        final_boxes.append(get_padded(b, crop_pad))
    return final_boxes

def run_official_ablation_benchmark(method_name):
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        
    all_gts = {}; all_preds = []
    img_infos = {} 
    
    stats = {
        'ALL': {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'HR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'LR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()}
    }

    pbar = tqdm.tqdm(img_list, desc=f"⏳ {method_name}", bar_format='{l_bar}{bar:30}{r_bar}')
    for img_idx, img_name in enumerate(pbar):
        img_path, lbl_path = os.path.join(img_dir, img_name), os.path.join(lbl_dir, img_name.replace('.jpg', '.txt'))
        img = cv2.imread(img_path); h, w, _ = img.shape
        
        img_infos[img_idx] = {'w': w, 'h': h, 'name': img_name}
        
        gts = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    c, xc, yc, bw, bh = map(float, line.split())
                    gts.append([int(c), (xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h]) 
        all_gts[img_idx] = gts

        t_pipe_start = time.time()
        img_inf_time, img_inf_cnt = 0, 0
        
        # =====================================================================
        # 1. Baseline: UC (2x2 Uniform Crop)
        # =====================================================================
        if method_name == "UC (2x2 Uniform Crop)":
            # UC 평가가 너무 오래 걸려 비활성화
            ch, cw = h // 2, w // 2
            crops, offsets = [img], [(0, 0)]
            for y in [0, ch]:
                for x in [0, cw]:
                    crops.append(img[y:y+ch, x:x+cw])
                    offsets.append((x, y))
            
            t_inf_start = time.time()
            results2 = m2.predict(crops, conf=CONF_UC, verbose=False, batch=5)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 5 
            
            temp_boxes, temp_scores, temp_classes = [], [], []
            for i, res in enumerate(results2):
                ox, oy = offsets[i]
                for b in res.boxes:
                    # 💡 [FIX] .item() 을 사용하여 텐서를 완벽하게 파이썬 float으로 변환!
                    bx1 = b.xyxy[0][0].item() + ox
                    by1 = b.xyxy[0][1].item() + oy
                    bx2 = b.xyxy[0][2].item() + ox
                    by2 = b.xyxy[0][3].item() + oy
                    
                    temp_boxes.append([bx1, by1, bx2, by2])
                    temp_scores.append(float(b.conf[0]))
                    temp_classes.append(int(b.cls[0]))
                    
            for c in set(temp_classes):
                c_boxes = [b for j, b in enumerate(temp_boxes) if temp_classes[j] == c]
                c_scores = [s for j, s in enumerate(temp_scores) if temp_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 2. 제안 1: Ours (DAHI Only)
        # =====================================================================
        elif method_name == "Ours (DAHI Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            roi_boxes = []
            
            # 💡 [핵심 최적화] 메인 모델(m2)로 CONF_FILTER(0.1) 기준 단 1번만 스캔
            t_inf_start = time.time()
            res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global_all[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = float(b.conf[0])
                
                # 1. 글로벌 확정 박스 (0.3 이상)
                if conf >= CONF_GLOBAL:
                    boosted_conf = min(1.0, conf * 1.10)
                    global_final_boxes.append([bx1, by1, bx2, by2])
                    global_final_scores.append(boosted_conf)
                    global_final_classes.append(int(b.cls[0]))
                
                # 2. ROI 박스 등록 (모든 탐지 객체 재검사)
                roi_boxes.append([bx1, by1, bx2, by2])
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            while len(remaining_boxes) > 0:
                best_count, best_region = -1, None
                for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                        count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                        if count > best_count: 
                            best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                if best_region and best_count >= 1:
                    dense_regions.append(best_region)
                    dx1, dy1, dx2, dy2 = best_region
                    remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                else: break
            
            unified_infer_list = [img[dy1:dy2, dx1:dx2] for dx1, dy1, dx2, dy2 in dense_regions]
            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for idx, (dx1, dy1, dx2, dy2) in enumerate(dense_regions):
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1
                    for b in res_all[idx].boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
            
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 3. 제안 2: Ours (Tetris Only)
        # =====================================================================
        elif method_name == "Ours (Tetris Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            roi_boxes = []
            
            t_inf_start = time.time()
            res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global_all[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = float(b.conf[0])
                if conf >= CONF_GLOBAL:
                    global_final_boxes.append([bx1, by1, bx2, by2])
                    global_final_scores.append(min(1.0, conf * 1.10))
                    global_final_classes.append(int(b.cls[0]))
                roi_boxes.append([bx1, by1, bx2, by2])
            
            canvases, canvas_infos = [], []
            if len(roi_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(roi_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

            if len(canvases) > 0:
                t_inf_start = time.time()
                res_pack = m2.predict(canvases, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(canvases)
                
                for c_idx, res in enumerate(res_pack):
                    for b in res.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                        for info in canvas_infos:
                            if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                s = info['scale']
                                orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 4. 제안 3: Ours (DAHI + Tetris)
        # =====================================================================
        elif method_name == "Ours (DAHI + Tetris)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            roi_boxes = []
            
            t_inf_start = time.time()
            res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global_all[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = float(b.conf[0])
                if conf >= CONF_GLOBAL:
                    global_final_boxes.append([bx1, by1, bx2, by2])
                    global_final_scores.append(min(1.0, conf * 1.10))
                    global_final_classes.append(int(b.cls[0]))
                roi_boxes.append([bx1, by1, bx2, by2])
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            # 💡 [NEW] CAD-Router (Class-Aware Density Router) 로직 시작
            # SBSI (Small object Based Slicing Inference)
            # 1. "소형 객체" 중심점만 추출 (중/대형 객체 배제)
            small_centroids = []
            for b in remaining_boxes:
                bx1, by1, bx2, by2 = b
                if get_size_category(bx2 - bx1, by2 - by1) == 'small':
                    small_centroids.append([(bx1 + bx2) / 2, (by1 + by2) / 2])
            
            total_small_objs = len(small_centroids)
            route_threshold = max(1, int(total_small_objs * DENSE_RATIO_THRESH)) 
            
            # -----------------------------------------------------------------
            # 💡 [NEW] 앵커 기반 무게중심 유도 슬라이싱 (Anchored Mean-Shift)
            # -----------------------------------------------------------------
            if total_small_objs > 0:
                pts = np.array(small_centroids) # Shape: (N, 2)
                
                while len(pts) > 0:
                    # 1. 벡터화된 앵커 탐색: 남은 점들을 모두 윈도우 중심으로 가정하고 내부 객체 수 동시 계산
                    # pts[:, None, :] - pts[None, :, :] 는 N x N 거리 행렬을 순식간에 계산합니다.
                    diffs = np.abs(pts[:, None, :] - pts[None, :, :]) 
                    in_window = (diffs[:, :, 0] <= DENSE_WINDOW_SIZE / 2) & (diffs[:, :, 1] <= DENSE_WINDOW_SIZE / 2)
                    counts = np.sum(in_window, axis=1) # 각 점을 중심으로 했을 때 포함되는 객체 수
                    
                    best_idx = np.argmax(counts)
                    best_count = counts[best_idx]
                    
                    if best_count >= route_threshold:
                        # 2. 가장 밀도가 높은 앵커 좌표
                        anchor_x, anchor_y = pts[best_idx]
                        
                        # 3. 무게 중심 스내핑 (Center of Mass Snapping)
                        # 해당 앵커 윈도우 안에 들어온 점들의 실제 평균 좌표로 중심을 한 번 더 미세 조정
                        inside_pts = pts[in_window[best_idx]]
                        if len(inside_pts) > 0:
                            cx, cy = np.mean(inside_pts[:, 0]), np.mean(inside_pts[:, 1])
                        else:
                            cx, cy = anchor_x, anchor_y
                            
                        # 4. 이미지 경계를 벗어나지 않도록 512x512 고정 창 확정
                        dx1 = int(max(0, cx - DENSE_WINDOW_SIZE / 2))
                        dy1 = int(max(0, cy - DENSE_WINDOW_SIZE / 2))
                        dx2 = int(min(w, dx1 + DENSE_WINDOW_SIZE))
                        dy2 = int(min(h, dy1 + DENSE_WINDOW_SIZE))
                        
                        # 화면 끝에 걸려 512 미만으로 찌그러진 경우 강제로 512로 크기 복원
                        if dx2 - dx1 < DENSE_WINDOW_SIZE: dx1 = max(0, dx2 - DENSE_WINDOW_SIZE)
                        if dy2 - dy1 < DENSE_WINDOW_SIZE: dy1 = max(0, dy2 - DENSE_WINDOW_SIZE)
                        
                        dense_regions.append((dx1, dy1, dx2, dy2))
                        
                        # 5. Spatial NMS: 방금 확정된 512x512 영역에 포함된 점들은 다음 계산에서 제외
                        mask = ~((pts[:, 0] >= dx1) & (pts[:, 0] <= dx2) & (pts[:, 1] >= dy1) & (pts[:, 1] <= dy2))
                        pts = pts[mask]
                        
                        # 실제 추론 큐(remaining_boxes)에서도 덜어냄
                        remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                    else:
                        break # 임계값을 넘는 군집이 더 이상 없으면 탐색 종료
            
            # (이후 코드는 기존과 동일하게 unified_infer_list 생성 및 Tetris Canvas 생성 로직으로 이어짐)
            # 동적 문맥 재배치 파이프라인 (Dynamic Context Rearrangement Pipeline)
            unified_infer_list = []
            dense_idx_list = []
            
            for dx1, dy1, dx2, dy2 in dense_regions:
                unified_infer_list.append(img[dy1:dy2, dx1:dx2])
                dense_idx_list.append((len(unified_infer_list) - 1, dx1, dy1, dx2, dy2))

            canvases, canvas_infos = [], []
            canvas_start_idx = -1
            if len(remaining_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(remaining_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

                if len(canvases) > 0:
                    canvas_start_idx = len(unified_infer_list)
                    unified_infer_list.extend(canvases)

            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for d_idx, dx1, dy1, dx2, dy2 in dense_idx_list:
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1; res_dense = res_all[d_idx]
                    for b in res_dense.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                
                if canvas_start_idx != -1:
                    res_pack = res_all[canvas_start_idx:]
                    for c_idx, res in enumerate(res_pack):
                        for b in res.boxes:
                            bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                            bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                            for info in canvas_infos:
                                if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                    if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                    s = info['scale']
                                    orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                    orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                    local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                    break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # 통계 저장
        img_total_time = time.time() - t_pipe_start
        is_hr = (w * h >= HR_THRESHOLD)
        target_keys = ['ALL', 'HR'] if is_hr else ['ALL', 'LR']
        for k in target_keys:
            stats[k]['count'] += 1
            stats[k]['inf_time'] += img_inf_time
            stats[k]['total_time'] += img_total_time
            stats[k]['inf_cnt'] += img_inf_cnt
            stats[k]['indices'].add(img_idx)

    # ---------------------------------------------------------
    # 💡 공식 COCO API 연산 엔진
    # ---------------------------------------------------------
    def calc_official_coco_metrics(subset_indices):
        if not subset_indices: return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}
        
        gt_dict = {"images": [], "annotations": [], "categories": []}
        for i in range(10): gt_dict["categories"].append({"id": i, "name": f"class_{i}"})
            
        ann_id = 1
        for img_idx in subset_indices:
            info = img_infos[img_idx]
            gt_dict["images"].append({"id": img_idx, "width": info['w'], "height": info['h'], "file_name": info['name']})
            for gt in all_gts[img_idx]:
                c, x1, y1, x2, y2 = gt
                bw, bh = x2 - x1, y2 - y1
                gt_dict["annotations"].append({"id": ann_id, "image_id": img_idx, "category_id": int(c), "bbox": [x1, y1, bw, bh], "area": bw * bh, "iscrowd": 0})
                ann_id += 1

        cocoGt = COCO()
        cocoGt.dataset = gt_dict
        cocoGt.createIndex()

        sub_preds = [p for p in all_preds if p[0] in subset_indices]
        pred_list = []
        for pred in sub_preds:
            img_idx, c, score, x1, y1, x2, y2 = pred
            bw, bh = x2 - x1, y2 - y1
            pred_list.append({"image_id": img_idx, "category_id": int(c), "bbox": [x1, y1, bw, bh], "score": float(score)})

        if not pred_list: return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}

        cocoDt = cocoGt.loadRes(pred_list)

        cocoEval = COCOeval(cocoGt, cocoDt, 'bbox')
        cocoEval.params.maxDets = [100, 300, 500] 
        
        cocoEval.evaluate()
        cocoEval.accumulate()
        
        with contextlib.redirect_stdout(io.StringIO()):
            cocoEval.summarize()

        if len(cocoEval.stats) < 12:
            return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}

        return {
            "AP50:95": cocoEval.stats[0],
            "AP50": cocoEval.stats[1],
            "AP_small": cocoEval.stats[3],  
            "AP_medium": cocoEval.stats[4], 
            "AP_large": cocoEval.stats[5],
        }

    result_dict = {}
    for group in ['ALL', 'HR', 'LR']:
        c = stats[group]['count']
        res = calc_official_coco_metrics(stats[group]['indices'])
        res['Img_Cnt'] = c
        res['Avg_Inf_Cnt'] = stats[group]['inf_cnt'] / c if c else 0
        res['Avg_Inf_Time'] = (stats[group]['inf_time'] / c) * 1000 if c else 0
        res['Avg_Tot_Time'] = (stats[group]['total_time'] / c) * 1000 if c else 0
        result_dict[group] = res
        
    result_dict['Peak_VRAM'] = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0
    return result_dict

# =========================================================
# 실행 및 다중 표 그리기 (Official COCO Protocol)
# =========================================================
methods = [
    "UC (2x2 Uniform Crop)", 
    "Ours (DAHI Only)",
    "Ours (Tetris Only)",
    "Ours (DAHI + Tetris)"
]

final_stats = {}
for m in methods: 
    final_stats[m] = run_official_ablation_benchmark(m)

print("\n" + "="*145)
print(f"🏆 [Ablation Study] Single Model & Official COCO Protocol Evaluation (Total {NUM_TEST_IMAGES} Images) 🏆")
print("="*145)
print(f"{'Method':<32} | {'Type':<4} | {'Img':<4} | {'mAP':<6} | {'AP50':<6} | {'APs':<6} | {'APm':<6} | {'APl':<6} | {'Inf Cnt':<7} | {'Inf Time':<9} | {'Tot Time':<9}")
print("-" * 145)

for m, groups in final_stats.items():
    for g in ['ALL', 'HR', 'LR']:
        s = groups[g]
        if s['Img_Cnt'] == 0: continue
        
        mAP  = s.get('AP50:95', 0.0)
        ap50 = s.get('AP50', 0.0)
        aps  = s.get('AP_small', 0.0)
        apm  = s.get('AP_medium', 0.0)
        apl  = s.get('AP_large', 0.0)
        
        print(f"{m if g == 'ALL' else '':<32} | {g:<4} | {s['Img_Cnt']:<4} | {mAP:.4f} | {ap50:.4f} | {aps:.4f} | {apm:.4f} | {apl:.4f} | {s['Avg_Inf_Cnt']:4.1f} /i | {s['Avg_Inf_Time']:5.1f} ms | {s['Avg_Tot_Time']:5.1f} ms")
    
    print(f"{'':<32} > Peak VRAM: {groups.get('Peak_VRAM', 0.0):.1f} MB")
    print("-" * 145)

🚀 [Ablation Study] Single-Model Pipeline! 완벽한 재현 시작! (Total 1294 images)


⏳ UC (2x2 Uniform Crop): 100%|██████████████████████████████| 1294/1294 [01:33<00:00, 13.77it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.20s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=31.62s).
Accumulating evaluation results...
DONE (t=1.38s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=6.94s).
Accumulating evaluation results...
DONE (t=0.26s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.18s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=25.07s).
Accumulating evaluation results...
DONE (t=1.19s).


⏳ Ours (DAHI Only): 100%|██████████████████████████████| 1294/1294 [01:38<00:00, 13.19it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.24s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=33.88s).
Accumulating evaluation results...
DONE (t=1.57s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=7.64s).
Accumulating evaluation results...
DONE (t=0.31s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.21s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=25.97s).
Accumulating evaluation results...
DONE (t=1.17s).


⏳ Ours (Tetris Only): 100%|██████████████████████████████| 1294/1294 [01:28<00:00, 14.69it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.31s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=30.47s).
Accumulating evaluation results...
DONE (t=1.46s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=6.41s).
Accumulating evaluation results...
DONE (t=0.25s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=23.80s).
Accumulating evaluation results...
DONE (t=1.15s).


⏳ Ours (DAHI + Tetris): 100%|██████████████████████████████| 1294/1294 [01:36<00:00, 13.43it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.23s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=34.17s).
Accumulating evaluation results...
DONE (t=1.57s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=7.24s).
Accumulating evaluation results...
DONE (t=0.28s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=26.83s).
Accumulating evaluation results...
DONE (t=1.30s).

🏆 [Ablation Study] Single Model & Official COCO Protocol Evaluation (Total 5000 Images) 🏆
Method                           | Type | Img  | mAP    | AP50   | APs    | APm    | APl    | Inf Cnt | Inf Time  | Tot Time 
----------------------------------------------

In [ ]:
import cv2
import os
import time
import numpy as np
import tqdm
import torch
from ultralytics import YOLO

import contextlib
import io

# 💡 [NEW] 공식 COCO API 임포트
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# =========================================================
# ⚙️ 하이퍼파라미터 (Hyperparameters) - Single Model Edition
# =========================================================
# MODEL_FILTER_PATH = 'model/best_nano.pt' # 💡 보조 모델 완전히 삭제!
MODEL_MAIN_PATH = 'model/best_small.pt'

CONF_GLOBAL = 0.3
CONF_FILTER = 0.1     
CONF_DENSE = 0.3
CONF_TETRIS = 0.3
CONF_UC = 0.3

DENSE_RATIO_THRESH = 0.30  # 💡 [NEW] 라우팅 임계값: 전체 소형 객체의 50% 이상 밀집 시 DAHI 구역 인정
# IOU_FILTER_MATCH = 0.97 # 💡 사용 안함 (중복검사 삭제됨)
NMS_CONF_THRESH = 0.3
NMS_IOU_THRESH = 0.4    

DENSE_WINDOW_SIZE = 512
DENSE_STEP = 320      

MERGE_PAD = 16
CROP_PAD_LARGE = 80     
CROP_PAD_SMALL = 16
CROP_PAD_THRESH = 200

CANVAS_SIZE = 960
CANVAS_MARGIN = 2
CANVAS_BG_COLOR = 114

UPSCALE_RATIO = 1.5        
UPSCALE_MAX_THRESH = 200    

NUM_TEST_IMAGES = 5000
HR_THRESHOLD = 1920 * 1080 

# =========================================================
dataset_root = 'data/valid'
img_dir, lbl_dir = os.path.join(dataset_root, 'images'), os.path.join(dataset_root, 'labels')
img_list = sorted(os.listdir(img_dir))[:NUM_TEST_IMAGES]

print(f"🚀 [Ablation Study] Single-Model Pipeline! 완벽한 재현 시작! (Total {len(img_list)} images)")

# m1 = YOLO(MODEL_FILTER_PATH) # 💡 보조 모델 삭제
m2 = YOLO(MODEL_MAIN_PATH)

def calculate_iou(box1, box2):
    xi1, yi1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    xi2, yi2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])

def get_size_category(w, h):
    area = w * h
    if area < 32 ** 2: return 'small'
    elif area < 96 ** 2: return 'medium'
    else: return 'large'

def merge_clusters_dynamic(boxes, img_w, img_h, merge_pad=MERGE_PAD):
    if not len(boxes): return []
    def get_padded(b, pad): return [max(0, b[0]-pad), max(0, b[1]-pad), min(img_w, b[2]+pad), min(img_h, b[3]+pad)]
    def is_overlap(b1, b2):
        p1, p2 = get_padded(b1, merge_pad), get_padded(b2, merge_pad)
        return (min(p1[2], p2[2]) > max(p1[0], p2[0])) and (min(p1[3], p2[3]) > max(p1[1], p2[1]))
    curr = boxes.copy()
    while True:
        merged, flags = [], [False]*len(curr)
        for i in range(len(curr)):
            if flags[i]: continue
            b = curr[i]
            for j in range(i+1, len(curr)):
                if not flags[j] and is_overlap(b, curr[j]):
                    b = [min(b[0], curr[j][0]), min(b[1], curr[j][1]), max(b[2], curr[j][2]), max(b[3], curr[j][3])]
                    flags[j] = True
            merged.append(b)
        if len(merged) == len(curr): break
        curr = merged
    final_boxes = []
    for b in curr:
        bw, bh = b[2] - b[0], b[3] - b[1]
        crop_pad = CROP_PAD_LARGE if max(bw, bh) < CROP_PAD_THRESH else CROP_PAD_SMALL 
        final_boxes.append(get_padded(b, crop_pad))
    return final_boxes

def run_official_ablation_benchmark(method_name):
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        
    all_gts = {}; all_preds = []
    img_infos = {} 
    
    stats = {
        'ALL': {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'HR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'LR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()}
    }

    pbar = tqdm.tqdm(img_list, desc=f"⏳ {method_name}", bar_format='{l_bar}{bar:30}{r_bar}')
    for img_idx, img_name in enumerate(pbar):
        img_path, lbl_path = os.path.join(img_dir, img_name), os.path.join(lbl_dir, img_name.replace('.jpg', '.txt'))
        img = cv2.imread(img_path); h, w, _ = img.shape
        
        img_infos[img_idx] = {'w': w, 'h': h, 'name': img_name}
        
        gts = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    c, xc, yc, bw, bh = map(float, line.split())
                    gts.append([int(c), (xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h]) 
        all_gts[img_idx] = gts

        t_pipe_start = time.time()
        img_inf_time, img_inf_cnt = 0, 0
        
        # =====================================================================
        # 1. Baseline: UC (2x2 Uniform Crop)
        # =====================================================================
        if method_name == "UC (2x2 Uniform Crop)":
            # UC 평가가 너무 오래 걸려 비활성화
            ch, cw = h // 2, w // 2
            # crops, offsets = [img], [(0, 0)]
            # for y in [0, ch]:
            #     for x in [0, cw]:
            #         crops.append(img[y:y+ch, x:x+cw])
            #         offsets.append((x, y))
            
            # t_inf_start = time.time()
            # results2 = m2.predict(crops, conf=CONF_UC, verbose=False, batch=5)
            # img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 5 
            
            # temp_boxes, temp_scores, temp_classes = [], [], []
            # for i, res in enumerate(results2):
            #     ox, oy = offsets[i]
            #     for b in res.boxes:
            #         # 💡 [FIX] .item() 을 사용하여 텐서를 완벽하게 파이썬 float으로 변환!
            #         bx1 = b.xyxy[0][0].item() + ox
            #         by1 = b.xyxy[0][1].item() + oy
            #         bx2 = b.xyxy[0][2].item() + ox
            #         by2 = b.xyxy[0][3].item() + oy
                    
            #         temp_boxes.append([bx1, by1, bx2, by2])
            #         temp_scores.append(float(b.conf[0]))
            #         temp_classes.append(int(b.cls[0]))
                    
            # for c in set(temp_classes):
            #     c_boxes = [b for j, b in enumerate(temp_boxes) if temp_classes[j] == c]
            #     c_scores = [s for j, s in enumerate(temp_scores) if temp_classes[j] == c]
            #     cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
            #     indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
            #     if len(indices) > 0:
            #         for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 2. 제안 1: Ours (DAHI Only)
        # =====================================================================
        elif method_name == "Ours (DAHI Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            # local_boxes, local_scores, local_classes = [], [], []
            # roi_boxes = []
            
            # # 💡 [핵심 최적화] 메인 모델(m2)로 CONF_FILTER(0.1) 기준 단 1번만 스캔
            # t_inf_start = time.time()
            # res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            # img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            # for b in res_global_all[0].boxes:
            #     bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
            #     conf = float(b.conf[0])
                
            #     # 1. 글로벌 확정 박스 (0.3 이상)
            #     if conf >= CONF_GLOBAL:
            #         boosted_conf = min(1.0, conf * 1.10)
            #         global_final_boxes.append([bx1, by1, bx2, by2])
            #         global_final_scores.append(boosted_conf)
            #         global_final_classes.append(int(b.cls[0]))
                
            #     # 2. ROI 박스 등록 (모든 탐지 객체 재검사)
            #     roi_boxes.append([bx1, by1, bx2, by2])
            
            # remaining_boxes = roi_boxes.copy()
            # dense_regions = []
            
            # while len(remaining_boxes) > 0:
            #     best_count, best_region = -1, None
            #     for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
            #         for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
            #             count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
            #             if count > best_count: 
            #                 best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
            #     if best_region and best_count >= 1:
            #         dense_regions.append(best_region)
            #         dx1, dy1, dx2, dy2 = best_region
            #         remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
            #     else: break
            
            # unified_infer_list = [img[dy1:dy2, dx1:dx2] for dx1, dy1, dx2, dy2 in dense_regions]
            # if len(unified_infer_list) > 0:
            #     t_inf_start = time.time()
            #     res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
            #     img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
            #     for idx, (dx1, dy1, dx2, dy2) in enumerate(dense_regions):
            #         cw_dense, ch_dense = dx2 - dx1, dy2 - dy1
            #         for b in res_all[idx].boxes:
            #             bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
            #             if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
            #             local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
            
            # final_local_preds = []
            # for c in set(local_classes):
            #     c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
            #     cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
            #     indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
            #     if len(indices) > 0:
            #         for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            # combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            # for c in set(combined_classes):
            #     c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
            #     cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
            #     indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
            #     if len(indices) > 0:
            #         for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 3. 제안 2: Ours (Tetris Only)
        # =====================================================================
        elif method_name == "Ours (Tetris Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            # local_boxes, local_scores, local_classes = [], [], []
            # roi_boxes = []
            
            # t_inf_start = time.time()
            # res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            # img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            # for b in res_global_all[0].boxes:
            #     bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
            #     conf = float(b.conf[0])
            #     if conf >= CONF_GLOBAL:
            #         global_final_boxes.append([bx1, by1, bx2, by2])
            #         global_final_scores.append(min(1.0, conf * 1.10))
            #         global_final_classes.append(int(b.cls[0]))
            #     roi_boxes.append([bx1, by1, bx2, by2])
            
            # canvases, canvas_infos = [], []
            # if len(roi_boxes) > 0:
            #     clustered_boxes = merge_clusters_dynamic(roi_boxes, w, h, merge_pad=MERGE_PAD)
            #     crops_to_pack = []
            #     for cb in clustered_boxes:
            #         cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
            #         scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
            #         cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
            #         if cw_crop > 0 and ch_crop > 0:
            #             crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
            #             if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
            #             else: crop_img = crop_img[:ch_crop, :cw_crop]
            #             crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
            #     crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
            #     current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
            #     cx, cy, max_h = 0, 0, 0
            #     for item in crops_to_pack:
            #         if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
            #         if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
            #         current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
            #         canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
            #         cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
            #     if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
            #     for c_idx, canvas in enumerate(canvases):
            #         c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
            #         if not c_infos: continue
            #         unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
            #         for i, cy1 in enumerate(unique_cy1s):
            #             row_items = [info for info in c_infos if info['cy1'] == cy1]
            #             row_items.sort(key=lambda x: x['cx1'])
            #             next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
            #             for j, info in enumerate(row_items):
            #                 item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
            #                 ox, oy, s = info['ox'], info['oy'], info['scale']
            #                 org_w, org_h = int(item_w / s), int(item_h / s) 
            #                 next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
            #                 gap_w = next_cx1 - info['cx2']
            #                 if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
            #                 if gap_w > 0:
            #                     ext_w_org = min(int(gap_w / s), w - (ox + org_w))
            #                     if ext_w_org > 0:
            #                         ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
            #                         if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
            #                         canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
            #                         info['cx2'] += ext_crop.shape[1]
            #                 gap_h = next_cy1 - info['cy2']
            #                 if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
            #                 if gap_h > 0:
            #                     ext_h_org = min(int(gap_h / s), h - (oy + org_h))
            #                     if ext_h_org > 0:
            #                         ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
            #                         if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
            #                         canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
            #                         info['cy2'] += ext_crop.shape[0]

            # if len(canvases) > 0:
            #     t_inf_start = time.time()
            #     res_pack = m2.predict(canvases, conf=CONF_TETRIS, verbose=False, batch=16)
            #     img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(canvases)
                
            #     for c_idx, res in enumerate(res_pack):
            #         for b in res.boxes:
            #             bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
            #             bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
            #             for info in canvas_infos:
            #                 if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
            #                     if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
            #                     s = info['scale']
            #                     orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
            #                     orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
            #                     local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
            #                     break
                                        
            # final_local_preds = []
            # for c in set(local_classes):
            #     c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
            #     cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
            #     indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
            #     if len(indices) > 0:
            #         for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            # combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            # for c in set(combined_classes):
            #     c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
            #     cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
            #     indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
            #     if len(indices) > 0:
            #         for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 4. 제안 3: Ours (DAHI + Tetris)
        # =====================================================================
        elif method_name == "Ours (DAHI + Tetris)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            roi_boxes = []
            
            t_inf_start = time.time()
            res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global_all[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = float(b.conf[0])
                if conf >= CONF_GLOBAL:
                    global_final_boxes.append([bx1, by1, bx2, by2])
                    global_final_scores.append(min(1.0, conf * 1.10))
                    global_final_classes.append(int(b.cls[0]))
                roi_boxes.append([bx1, by1, bx2, by2, conf]) # 💡 [수정] conf를 리스트 마지막에 추가
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            # SBSI (Small object Based Slicing Inference)
            # -----------------------------------------------------------------
            # 💡 [NEW V4] 크기 비례 가중치 (Secure the Larger-Small)
            # -----------------------------------------------------------------
            small_info = []
            for b in remaining_boxes:
                bx1, by1, bx2, by2, conf = b 
                w_box, h_box = bx2 - bx1, by2 - by1
                if get_size_category(w_box, h_box) == 'small':
                    cx, cy = (bx1 + bx2) / 2, (by1 + by2) / 2
                    area = w_box * h_box
                    
                    # 💡 [수정된 가중치 수식]
                    # 신뢰도가 낮고(1-conf), 크기가 클수록(sqrt(area)) 높은 가중치
                    weight = (1.0 - conf) * np.sqrt(area)
                    
                    small_info.append([cx, cy, weight])
            
            total_small_objs = len(small_info)
            # 임계값은 원시 객체 수(Discrete Count) 기준으로 확정
            route_threshold = max(1, int(total_small_objs * DENSE_RATIO_THRESH)) 
            
            if total_small_objs > 0:
                pts = np.array([[info[0], info[1]] for info in small_info]) # Shape: (N, 2)
                weights = np.array([info[2] for info in small_info])
                
                # 가중치 정규화
                norm_weights = weights / (np.mean(weights) + 1e-6)
                sigma = DENSE_WINDOW_SIZE / 4.0 
                
                while len(pts) > 0:
                    # 2. KDE 연산: 완벽한 무게중심 좌표를 찾기 위한 '유도(Guidance)' 역할
                    dist_sq = np.sum((pts[:, None, :] - pts[None, :, :]) ** 2, axis=-1) 
                    kernel = np.exp(-dist_sq / (2 * sigma ** 2)) 
                    densities = kernel @ norm_weights 
                    
                    best_idx = np.argmax(densities)
                    
                    # 가중 무게중심 스내핑
                    influence = norm_weights * kernel[best_idx]
                    sum_influence = np.sum(influence)
                    
                    if sum_influence > 0:
                        cx = np.sum(influence * pts[:, 0]) / sum_influence
                        cy = np.sum(influence * pts[:, 1]) / sum_influence
                    else:
                        cx, cy = pts[best_idx]
                        
                    # 윈도우 경계 확정
                    dx1 = int(max(0, cx - DENSE_WINDOW_SIZE / 2))
                    dy1 = int(max(0, cy - DENSE_WINDOW_SIZE / 2))
                    dx2 = int(min(w, dx1 + DENSE_WINDOW_SIZE))
                    dy2 = int(min(h, dy1 + DENSE_WINDOW_SIZE))
                    
                    if dx2 - dx1 < DENSE_WINDOW_SIZE: dx1 = max(0, dx2 - DENSE_WINDOW_SIZE)
                    if dy2 - dy1 < DENSE_WINDOW_SIZE: dy1 = max(0, dy2 - DENSE_WINDOW_SIZE)
                    
                    # 3. 이산 검증 (Discrete Confirmation): 확정된 윈도우 안에 '실제' 점이 몇 개인지 카운트
                    actual_in_window = (pts[:, 0] >= dx1) & (pts[:, 0] <= dx2) & (pts[:, 1] >= dy1) & (pts[:, 1] <= dy2)
                    actual_count = np.sum(actual_in_window)
                    
                    # 임계값 누수 차단: 실제 카운트 값으로 통과 여부 결정
                    if actual_count >= route_threshold:
                        dense_regions.append((dx1, dy1, dx2, dy2))
                        
                        # 처리된 영역 배제
                        mask = ~actual_in_window
                        pts = pts[mask]
                        norm_weights = norm_weights[mask]
                        
                        remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                    else:
                        # 최고 밀도 앵커조차 임계값을 넘지 못하면 라우팅 완전 종료
                        break
            
            
            # (이후 코드는 기존과 동일하게 unified_infer_list 생성 및 Tetris Canvas 생성 로직으로 이어짐)
            # 동적 문맥 재배치 파이프라인 (Dynamic Context Rearrangement Pipeline)
            unified_infer_list = []
            dense_idx_list = []
            
            for dx1, dy1, dx2, dy2 in dense_regions:
                unified_infer_list.append(img[dy1:dy2, dx1:dx2])
                dense_idx_list.append((len(unified_infer_list) - 1, dx1, dy1, dx2, dy2))

            canvases, canvas_infos = [], []
            canvas_start_idx = -1
            if len(remaining_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(remaining_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

                if len(canvases) > 0:
                    canvas_start_idx = len(unified_infer_list)
                    unified_infer_list.extend(canvases)

            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for d_idx, dx1, dy1, dx2, dy2 in dense_idx_list:
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1; res_dense = res_all[d_idx]
                    for b in res_dense.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                
                if canvas_start_idx != -1:
                    res_pack = res_all[canvas_start_idx:]
                    for c_idx, res in enumerate(res_pack):
                        for b in res.boxes:
                            bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                            bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                            for info in canvas_infos:
                                if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                    if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                    s = info['scale']
                                    orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                    orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                    local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                    break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # 통계 저장
        img_total_time = time.time() - t_pipe_start
        is_hr = (w * h >= HR_THRESHOLD)
        target_keys = ['ALL', 'HR'] if is_hr else ['ALL', 'LR']
        for k in target_keys:
            stats[k]['count'] += 1
            stats[k]['inf_time'] += img_inf_time
            stats[k]['total_time'] += img_total_time
            stats[k]['inf_cnt'] += img_inf_cnt
            stats[k]['indices'].add(img_idx)

    # ---------------------------------------------------------
    # 💡 공식 COCO API 연산 엔진
    # ---------------------------------------------------------
    def calc_official_coco_metrics(subset_indices):
        if not subset_indices: return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}
        
        gt_dict = {"images": [], "annotations": [], "categories": []}
        for i in range(10): gt_dict["categories"].append({"id": i, "name": f"class_{i}"})
            
        ann_id = 1
        for img_idx in subset_indices:
            info = img_infos[img_idx]
            gt_dict["images"].append({"id": img_idx, "width": info['w'], "height": info['h'], "file_name": info['name']})
            for gt in all_gts[img_idx]:
                c, x1, y1, x2, y2 = gt
                bw, bh = x2 - x1, y2 - y1
                gt_dict["annotations"].append({"id": ann_id, "image_id": img_idx, "category_id": int(c), "bbox": [x1, y1, bw, bh], "area": bw * bh, "iscrowd": 0})
                ann_id += 1

        cocoGt = COCO()
        cocoGt.dataset = gt_dict
        cocoGt.createIndex()

        sub_preds = [p for p in all_preds if p[0] in subset_indices]
        pred_list = []
        for pred in sub_preds:
            img_idx, c, score, x1, y1, x2, y2 = pred
            bw, bh = x2 - x1, y2 - y1
            pred_list.append({"image_id": img_idx, "category_id": int(c), "bbox": [x1, y1, bw, bh], "score": float(score)})

        if not pred_list: return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}

        cocoDt = cocoGt.loadRes(pred_list)

        cocoEval = COCOeval(cocoGt, cocoDt, 'bbox')
        cocoEval.params.maxDets = [100, 300, 500] 
        
        cocoEval.evaluate()
        cocoEval.accumulate()
        
        with contextlib.redirect_stdout(io.StringIO()):
            cocoEval.summarize()

        if len(cocoEval.stats) < 12:
            return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}

        return {
            "AP50:95": cocoEval.stats[0],
            "AP50": cocoEval.stats[1],
            "AP_small": cocoEval.stats[3],  
            "AP_medium": cocoEval.stats[4], 
            "AP_large": cocoEval.stats[5],
        }

    result_dict = {}
    for group in ['ALL', 'HR', 'LR']:
        c = stats[group]['count']
        res = calc_official_coco_metrics(stats[group]['indices'])
        res['Img_Cnt'] = c
        res['Avg_Inf_Cnt'] = stats[group]['inf_cnt'] / c if c else 0
        res['Avg_Inf_Time'] = (stats[group]['inf_time'] / c) * 1000 if c else 0
        res['Avg_Tot_Time'] = (stats[group]['total_time'] / c) * 1000 if c else 0
        result_dict[group] = res
        
    result_dict['Peak_VRAM'] = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0
    return result_dict

# =========================================================
# 실행 및 다중 표 그리기 (Official COCO Protocol)
# =========================================================
methods = [
    "UC (2x2 Uniform Crop)", 
    "Ours (DAHI Only)",
    "Ours (Tetris Only)",
    "Ours (DAHI + Tetris)"
]

final_stats = {}
for m in methods: 
    final_stats[m] = run_official_ablation_benchmark(m)

print("\n" + "="*145)
print(f"🏆 [Ablation Study] Single Model & Official COCO Protocol Evaluation (Total {NUM_TEST_IMAGES} Images) 🏆")
print("="*145)
print(f"{'Method':<32} | {'Type':<4} | {'Img':<4} | {'mAP':<6} | {'AP50':<6} | {'APs':<6} | {'APm':<6} | {'APl':<6} | {'Inf Cnt':<7} | {'Inf Time':<9} | {'Tot Time':<9}")
print("-" * 145)

for m, groups in final_stats.items():
    for g in ['ALL', 'HR', 'LR']:
        s = groups[g]
        if s['Img_Cnt'] == 0: continue
        
        mAP  = s.get('AP50:95', 0.0)
        ap50 = s.get('AP50', 0.0)
        aps  = s.get('AP_small', 0.0)
        apm  = s.get('AP_medium', 0.0)
        apl  = s.get('AP_large', 0.0)
        
        print(f"{m if g == 'ALL' else '':<32} | {g:<4} | {s['Img_Cnt']:<4} | {mAP:.4f} | {ap50:.4f} | {aps:.4f} | {apm:.4f} | {apl:.4f} | {s['Avg_Inf_Cnt']:4.1f} /i | {s['Avg_Inf_Time']:5.1f} ms | {s['Avg_Tot_Time']:5.1f} ms")
    
    print(f"{'':<32} > Peak VRAM: {groups.get('Peak_VRAM', 0.0):.1f} MB")
    print("-" * 145)

🚀 [Ablation Study] Single-Model Pipeline! 완벽한 재현 시작! (Total 1294 images)


⏳ UC (2x2 Uniform Crop): 100%|██████████████████████████████| 1294/1294 [00:16<00:00, 76.65it/s]


creating index...
index created!
creating index...
index created!
creating index...
index created!


⏳ Ours (DAHI Only): 100%|██████████████████████████████| 1294/1294 [00:16<00:00, 76.17it/s]


creating index...
index created!
creating index...
index created!
creating index...
index created!


⏳ Ours (Tetris Only): 100%|██████████████████████████████| 1294/1294 [00:16<00:00, 77.23it/s]


creating index...
index created!
creating index...
index created!
creating index...
index created!


⏳ Ours (DAHI + Tetris): 100%|██████████████████████████████| 1294/1294 [01:33<00:00, 13.85it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.24s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=34.55s).
Accumulating evaluation results...
DONE (t=1.62s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=7.49s).
Accumulating evaluation results...
DONE (t=0.30s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=27.09s).
Accumulating evaluation results...
DONE (t=1.24s).

🏆 [Ablation Study] Single Model & Official COCO Protocol Evaluation (Total 5000 Images) 🏆
Method                           | Type | Img  | mAP    | AP50   | APs    | APm    | APl    | Inf Cnt | Inf Time  | Tot Time 
----------------------------------------------

In [ ]:
import cv2
import os
import time
import numpy as np
import tqdm
import torch
from ultralytics import YOLO

import contextlib
import io

# 💡 [NEW] 시각화를 위한 라이브러리 임포트
import random
import matplotlib.pyplot as plt

# 💡 [NEW] 공식 COCO API 임포트
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# =========================================================
# ⚙️ 하이퍼파라미터 (Hyperparameters) - Single Model Edition
# =========================================================
# MODEL_FILTER_PATH = 'model/best_nano.pt' # 💡 보조 모델 완전히 삭제!
MODEL_MAIN_PATH = 'model/best_small.pt'

CONF_GLOBAL = 0.3
CONF_FILTER = 0.1     
CONF_DENSE = 0.3
CONF_TETRIS = 0.3
CONF_UC = 0.3

DENSE_RATIO_THRESH = 0.30  # 💡 [NEW] 라우팅 임계값: 전체 소형 객체의 50% 이상 밀집 시 DAHI 구역 인정
# IOU_FILTER_MATCH = 0.97 # 💡 사용 안함 (중복검사 삭제됨)
NMS_CONF_THRESH = 0.3
NMS_IOU_THRESH = 0.4    

DENSE_WINDOW_SIZE = 512
DENSE_STEP = 320      

MERGE_PAD = 16
CROP_PAD_LARGE = 80     
CROP_PAD_SMALL = 16
CROP_PAD_THRESH = 200

CANVAS_SIZE = 960
CANVAS_MARGIN = 2
CANVAS_BG_COLOR = 114

UPSCALE_RATIO = 1.5        
UPSCALE_MAX_THRESH = 200    

NUM_TEST_IMAGES = 5000
HR_THRESHOLD = 1920 * 1080 

# =========================================================
dataset_root = 'data/valid'
img_dir, lbl_dir = os.path.join(dataset_root, 'images'), os.path.join(dataset_root, 'labels')
img_list = sorted(os.listdir(img_dir))[:NUM_TEST_IMAGES]

print(f"🚀 [Ablation Study] Single-Model Pipeline! 완벽한 재현 시작! (Total {len(img_list)} images)")

# m1 = YOLO(MODEL_FILTER_PATH) # 💡 보조 모델 삭제
m2 = YOLO(MODEL_MAIN_PATH)

# 💡 [NEW] 시각화할 캔버스 이미지를 담아둘 전역 리스트
sce_vis_samples = []

def calculate_iou(box1, box2):
    xi1, yi1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    xi2, yi2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])

def get_size_category(w, h):
    area = w * h
    if area < 32 ** 2: return 'small'
    elif area < 96 ** 2: return 'medium'
    else: return 'large'

def merge_clusters_dynamic(boxes, img_w, img_h, merge_pad=MERGE_PAD):
    if not len(boxes): return []
    def get_padded(b, pad): return [max(0, b[0]-pad), max(0, b[1]-pad), min(img_w, b[2]+pad), min(img_h, b[3]+pad)]
    def is_overlap(b1, b2):
        p1, p2 = get_padded(b1, merge_pad), get_padded(b2, merge_pad)
        return (min(p1[2], p2[2]) > max(p1[0], p2[0])) and (min(p1[3], p2[3]) > max(p1[1], p2[1]))
    curr = boxes.copy()
    while True:
        merged, flags = [], [False]*len(curr)
        for i in range(len(curr)):
            if flags[i]: continue
            b = curr[i]
            for j in range(i+1, len(curr)):
                if not flags[j] and is_overlap(b, curr[j]):
                    b = [min(b[0], curr[j][0]), min(b[1], curr[j][1]), max(b[2], curr[j][2]), max(b[3], curr[j][3])]
                    flags[j] = True
            merged.append(b)
        if len(merged) == len(curr): break
        curr = merged
    final_boxes = []
    for b in curr:
        bw, bh = b[2] - b[0], b[3] - b[1]
        crop_pad = CROP_PAD_LARGE if max(bw, bh) < CROP_PAD_THRESH else CROP_PAD_SMALL 
        final_boxes.append(get_padded(b, crop_pad))
    return final_boxes

def run_official_ablation_benchmark(method_name):
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        
    all_gts = {}; all_preds = []
    img_infos = {} 
    
    stats = {
        'ALL': {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'HR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'LR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()}
    }

    pbar = tqdm.tqdm(img_list, desc=f"⏳ {method_name}", bar_format='{l_bar}{bar:30}{r_bar}')
    for img_idx, img_name in enumerate(pbar):
        img_path, lbl_path = os.path.join(img_dir, img_name), os.path.join(lbl_dir, img_name.replace('.jpg', '.txt'))
        img = cv2.imread(img_path); h, w, _ = img.shape
        
        img_infos[img_idx] = {'w': w, 'h': h, 'name': img_name}
        
        gts = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    c, xc, yc, bw, bh = map(float, line.split())
                    gts.append([int(c), (xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h]) 
        all_gts[img_idx] = gts

        t_pipe_start = time.time()
        img_inf_time, img_inf_cnt = 0, 0
        
        # =====================================================================
        # 1. Baseline: UC (2x2 Uniform Crop)
        # =====================================================================
        if method_name == "UC (2x2 Uniform Crop)":
            # UC 평가가 너무 오래 걸려 비활성화
            ch, cw = h // 2, w // 2
            crops, offsets = [img], [(0, 0)]
            for y in [0, ch]:
                for x in [0, cw]:
                    crops.append(img[y:y+ch, x:x+cw])
                    offsets.append((x, y))
            
            t_inf_start = time.time()
            results2 = m2.predict(crops, conf=CONF_UC, verbose=False, batch=5)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 5 
            
            temp_boxes, temp_scores, temp_classes = [], [], []
            for i, res in enumerate(results2):
                ox, oy = offsets[i]
                for b in res.boxes:
                    # 💡 [FIX] .item() 을 사용하여 텐서를 완벽하게 파이썬 float으로 변환!
                    bx1 = b.xyxy[0][0].item() + ox
                    by1 = b.xyxy[0][1].item() + oy
                    bx2 = b.xyxy[0][2].item() + ox
                    by2 = b.xyxy[0][3].item() + oy
                    
                    temp_boxes.append([bx1, by1, bx2, by2])
                    temp_scores.append(float(b.conf[0]))
                    temp_classes.append(int(b.cls[0]))
                    
            for c in set(temp_classes):
                c_boxes = [b for j, b in enumerate(temp_boxes) if temp_classes[j] == c]
                c_scores = [s for j, s in enumerate(temp_scores) if temp_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 2. 제안 1: Ours (DAHI Only)
        # =====================================================================
        elif method_name == "Ours (DAHI Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            roi_boxes = []
            
            # 💡 [핵심 최적화] 메인 모델(m2)로 CONF_FILTER(0.1) 기준 단 1번만 스캔
            t_inf_start = time.time()
            res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global_all[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = float(b.conf[0])
                
                # 1. 글로벌 확정 박스 (0.3 이상)
                if conf >= CONF_GLOBAL:
                    boosted_conf = min(1.0, conf * 1.10)
                    global_final_boxes.append([bx1, by1, bx2, by2])
                    global_final_scores.append(boosted_conf)
                    global_final_classes.append(int(b.cls[0]))
                
                # 2. ROI 박스 등록 (모든 탐지 객체 재검사)
                roi_boxes.append([bx1, by1, bx2, by2])
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            while len(remaining_boxes) > 0:
                best_count, best_region = -1, None
                for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                    for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                        count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                        if count > best_count: 
                            best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                if best_region and best_count >= 1:
                    dense_regions.append(best_region)
                    dx1, dy1, dx2, dy2 = best_region
                    remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                else: break
            
            unified_infer_list = [img[dy1:dy2, dx1:dx2] for dx1, dy1, dx2, dy2 in dense_regions]
            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for idx, (dx1, dy1, dx2, dy2) in enumerate(dense_regions):
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1
                    for b in res_all[idx].boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
            
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 3. 제안 2: Ours (Tetris Only)
        # =====================================================================
        elif method_name == "Ours (Tetris Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            roi_boxes = []
            
            t_inf_start = time.time()
            res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global_all[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = float(b.conf[0])
                if conf >= CONF_GLOBAL:
                    global_final_boxes.append([bx1, by1, bx2, by2])
                    global_final_scores.append(min(1.0, conf * 1.10))
                    global_final_classes.append(int(b.cls[0]))
                roi_boxes.append([bx1, by1, bx2, by2])
            
            canvases, canvas_infos = [], []
            if len(roi_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(roi_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        for j, info in enumerate(row_items):
                            item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            org_w, org_h = int(item_w / s), int(item_h / s) 
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            gap_w = next_cx1 - info['cx2']
                            if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
                            if gap_w > 0:
                                ext_w_org = min(int(gap_w / s), w - (ox + org_w))
                                if ext_w_org > 0:
                                    ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
                                    info['cx2'] += ext_crop.shape[1]
                            gap_h = next_cy1 - info['cy2']
                            if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
                            if gap_h > 0:
                                ext_h_org = min(int(gap_h / s), h - (oy + org_h))
                                if ext_h_org > 0:
                                    ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
                                    if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
                                    canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
                                    info['cy2'] += ext_crop.shape[0]

            if len(canvases) > 0:
                t_inf_start = time.time()
                res_pack = m2.predict(canvases, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(canvases)
                
                for c_idx, res in enumerate(res_pack):
                    for b in res.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                        for info in canvas_infos:
                            if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                s = info['scale']
                                orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 4. 제안 3: Ours (DAHI + Tetris)
        # =====================================================================
        elif method_name == "Ours (DAHI + Tetris)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            roi_boxes = []
            
            t_inf_start = time.time()
            res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global_all[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = float(b.conf[0])
                if conf >= CONF_GLOBAL:
                    global_final_boxes.append([bx1, by1, bx2, by2])
                    global_final_scores.append(min(1.0, conf * 1.10))
                    global_final_classes.append(int(b.cls[0]))
                roi_boxes.append([bx1, by1, bx2, by2, conf]) # 💡 [수정] conf를 리스트 마지막에 추가
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            # SBSI (Small object Based Slicing Inference)
            # -----------------------------------------------------------------
            # 💡 [NEW V4] 크기 비례 가중치 (Secure the Larger-Small)
            # -----------------------------------------------------------------
            small_info = []
            for b in remaining_boxes:
                bx1, by1, bx2, by2, conf = b 
                w_box, h_box = bx2 - bx1, by2 - by1
                if get_size_category(w_box, h_box) == 'small':
                    cx, cy = (bx1 + bx2) / 2, (by1 + by2) / 2
                    area = w_box * h_box
                    
                    # 💡 [수정된 가중치 수식]
                    # 신뢰도가 낮고(1-conf), 크기가 클수록(sqrt(area)) 높은 가중치
                    weight = (1.0 - conf) * np.sqrt(area)
                    
                    small_info.append([cx, cy, weight])
            
            total_small_objs = len(small_info)
            # 임계값은 원시 객체 수(Discrete Count) 기준으로 확정
            route_threshold = max(1, int(total_small_objs * DENSE_RATIO_THRESH)) 
            
            if total_small_objs > 0:
                pts = np.array([[info[0], info[1]] for info in small_info]) # Shape: (N, 2)
                weights = np.array([info[2] for info in small_info])
                
                # 가중치 정규화
                norm_weights = weights / (np.mean(weights) + 1e-6)
                sigma = DENSE_WINDOW_SIZE / 4.0 
                
                while len(pts) > 0:
                    # 2. KDE 연산: 완벽한 무게중심 좌표를 찾기 위한 '유도(Guidance)' 역할
                    dist_sq = np.sum((pts[:, None, :] - pts[None, :, :]) ** 2, axis=-1) 
                    kernel = np.exp(-dist_sq / (2 * sigma ** 2)) 
                    densities = kernel @ norm_weights 
                    
                    best_idx = np.argmax(densities)
                    
                    # 가중 무게중심 스내핑
                    influence = norm_weights * kernel[best_idx]
                    sum_influence = np.sum(influence)
                    
                    if sum_influence > 0:
                        cx = np.sum(influence * pts[:, 0]) / sum_influence
                        cy = np.sum(influence * pts[:, 1]) / sum_influence
                    else:
                        cx, cy = pts[best_idx]
                        
                    # 윈도우 경계 확정
                    dx1 = int(max(0, cx - DENSE_WINDOW_SIZE / 2))
                    dy1 = int(max(0, cy - DENSE_WINDOW_SIZE / 2))
                    dx2 = int(min(w, dx1 + DENSE_WINDOW_SIZE))
                    dy2 = int(min(h, dy1 + DENSE_WINDOW_SIZE))
                    
                    if dx2 - dx1 < DENSE_WINDOW_SIZE: dx1 = max(0, dx2 - DENSE_WINDOW_SIZE)
                    if dy2 - dy1 < DENSE_WINDOW_SIZE: dy1 = max(0, dy2 - DENSE_WINDOW_SIZE)
                    
                    # 3. 이산 검증 (Discrete Confirmation): 확정된 윈도우 안에 '실제' 점이 몇 개인지 카운트
                    actual_in_window = (pts[:, 0] >= dx1) & (pts[:, 0] <= dx2) & (pts[:, 1] >= dy1) & (pts[:, 1] <= dy2)
                    actual_count = np.sum(actual_in_window)
                    
                    # 임계값 누수 차단: 실제 카운트 값으로 통과 여부 결정
                    if actual_count >= route_threshold:
                        dense_regions.append((dx1, dy1, dx2, dy2))
                        
                        # 처리된 영역 배제
                        mask = ~actual_in_window
                        pts = pts[mask]
                        norm_weights = norm_weights[mask]
                        
                        remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                    else:
                        # 최고 밀도 앵커조차 임계값을 넘지 못하면 라우팅 완전 종료
                        break
            
            
            # (이후 코드는 기존과 동일하게 unified_infer_list 생성 및 Tetris Canvas 생성 로직으로 이어짐)
            # 동적 문맥 재배치 파이프라인 (Dynamic Context Rearrangement Pipeline)
            unified_infer_list = []
            dense_idx_list = []
            
            for dx1, dy1, dx2, dy2 in dense_regions:
                unified_infer_list.append(img[dy1:dy2, dx1:dx2])
                dense_idx_list.append((len(unified_infer_list) - 1, dx1, dy1, dx2, dy2))

            canvases, canvas_infos = [], []
            canvas_start_idx = -1
            if len(remaining_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(remaining_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
                # SCE
                # -----------------------------------------------------------------
                # 💡 [NEW V14] 전방위 순수 문맥 확장 (Omnidirectional Contiguous SCE)
                # -----------------------------------------------------------------
                # 💡 여백 분배 함수: 상/하 또는 좌/우로 여백을 균등 분배하되, 가장자리에 막히면 반대편으로 몰아줌
                def get_distribution(gap, avail1, avail2):
                    half = gap // 2
                    if avail1 < half: return avail1, min(gap - avail1, avail2)
                    elif avail2 < half: return min(gap - avail2, avail1), avail2
                    else: return half, gap - half

                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        
                        for j, info in enumerate(row_items):
                            cx1, cy1 = info['cx1'], info['cy1']
                            cx2, cy2 = info['cx2'], info['cy2']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            item_w, item_h = cx2 - cx1, cy2 - cy1
                            org_w, org_h = max(1, int(item_w / s)), max(1, int(item_h / s))
                            
                            # 💡 1. 캔버스에 할당된 셀의 최대 가용 크기(Allocated Space) 계산
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            alloc_w = next_cx1 - cx1
                            if j + 1 < len(row_items): alloc_w -= CANVAS_MARGIN
                            
                            alloc_h = next_cy1 - cy1
                            if i + 1 < len(unique_cy1s): alloc_h -= CANVAS_MARGIN
                            
                            # 현재 크롭이 차지하고 남은 여백 (Gap)
                            gap_w = max(0, alloc_w - item_w)
                            gap_h = max(0, alloc_h - item_h)
                            
                            if gap_w > 0 or gap_h > 0:
                                gap_w_org = int(gap_w / s)
                                gap_h_org = int(gap_h / s)
                                
                                # 💡 2. 4-Way 확장을 위한 픽셀 분배 (상하좌우 가용 공간 체크)
                                ext_left, ext_right = get_distribution(gap_w_org, ox, w - (ox + org_w))
                                ext_top, ext_bottom = get_distribution(gap_h_org, oy, h - (oy + org_h))
                                
                                # 💡 3. 원본 이미지 내에서 확장된 새로운 크롭 좌표 계산
                                new_ox = ox - ext_left
                                new_oy = oy - ext_top
                                new_org_w = org_w + ext_left + ext_right
                                new_org_h = org_h + ext_top + ext_bottom
                                
                                if new_org_w > 0 and new_org_h > 0:
                                    ext_crop = img[new_oy:new_oy+new_org_h, new_ox:new_ox+new_org_w]
                                    actual_w = int(new_org_w * s) if s > 1.0 else new_org_w
                                    actual_h = int(new_org_h * s) if s > 1.0 else new_org_h
                                    
                                    if ext_crop.size > 0:
                                        if s > 1.0: 
                                            ext_crop = cv2.resize(ext_crop, (actual_w, actual_h), interpolation=cv2.INTER_CUBIC)
                                        
                                        # 💡 데이터를 쓰기 직전에 '실제' shape을 구함
                                        h_crop, w_crop = ext_crop.shape[:2]
                                        
                                        # 💡 경계 초과 방지 (캔버스 밖으로 나가는 것만 잘라냄)
                                        y_end = min(cy1 + h_crop, CANVAS_SIZE)
                                        x_end = min(cx1 + w_crop, CANVAS_SIZE)
                                        
                                        # 💡 실제 반영된 크기를 기준으로 모든 좌표 확정
                                        canvas[cy1:y_end, cx1:x_end] = ext_crop[:y_end-cy1, :x_end-cx1]
                                        
                                        # 💡 매핑 정보 단 한 번만 명확하게 갱신
                                        info['ox'] = new_ox
                                        info['oy'] = new_oy
                                        info['cx2'] = cx1 + (x_end - cx1) # 실제 반영된 너비
                                        info['cy2'] = cy1 + (y_end - cy1) # 실제 반영된 높이

                # 텅 비어버린 캔버스 정리
                canvases = [c for i, c in enumerate(canvases) if any(info['c_idx'] == i for info in canvas_infos)]
                for new_idx, old_idx in enumerate(sorted(list(set(info['c_idx'] for info in canvas_infos)))):
                    for info in canvas_infos:
                        if info['c_idx'] == old_idx: info['c_idx'] = new_idx

                # 시각화 수집
                if len(sce_vis_samples) < 10 and len(canvases) > 0:
                    sample_canvas = random.choice(canvases).copy()
                    sample_rgb = cv2.cvtColor(sample_canvas, cv2.COLOR_BGR2RGB)
                    sce_vis_samples.append(sample_rgb)

                if len(canvases) > 0:
                    canvas_start_idx = len(unified_infer_list)
                    unified_infer_list.extend(canvases)

            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for d_idx, dx1, dy1, dx2, dy2 in dense_idx_list:
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1; res_dense = res_all[d_idx]
                    for b in res_dense.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                
                if canvas_start_idx != -1:
                    res_pack = res_all[canvas_start_idx:]
                    for c_idx, res in enumerate(res_pack):
                        for b in res.boxes:
                            bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                            bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                            for info in canvas_infos:
                                if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                    if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                    s = info['scale']
                                    orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                    orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                    local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                    break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # 통계 저장
        img_total_time = time.time() - t_pipe_start
        is_hr = (w * h >= HR_THRESHOLD)
        target_keys = ['ALL', 'HR'] if is_hr else ['ALL', 'LR']
        for k in target_keys:
            stats[k]['count'] += 1
            stats[k]['inf_time'] += img_inf_time
            stats[k]['total_time'] += img_total_time
            stats[k]['inf_cnt'] += img_inf_cnt
            stats[k]['indices'].add(img_idx)

    # ---------------------------------------------------------
    # 💡 공식 COCO API 연산 엔진
    # ---------------------------------------------------------
    def calc_official_coco_metrics(subset_indices):
        if not subset_indices: return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}
        
        gt_dict = {"images": [], "annotations": [], "categories": []}
        for i in range(10): gt_dict["categories"].append({"id": i, "name": f"class_{i}"})
            
        ann_id = 1
        for img_idx in subset_indices:
            info = img_infos[img_idx]
            gt_dict["images"].append({"id": img_idx, "width": info['w'], "height": info['h'], "file_name": info['name']})
            for gt in all_gts[img_idx]:
                c, x1, y1, x2, y2 = gt
                bw, bh = x2 - x1, y2 - y1
                gt_dict["annotations"].append({"id": ann_id, "image_id": img_idx, "category_id": int(c), "bbox": [x1, y1, bw, bh], "area": bw * bh, "iscrowd": 0})
                ann_id += 1

        cocoGt = COCO()
        cocoGt.dataset = gt_dict
        cocoGt.createIndex()

        sub_preds = [p for p in all_preds if p[0] in subset_indices]
        pred_list = []
        for pred in sub_preds:
            img_idx, c, score, x1, y1, x2, y2 = pred
            bw, bh = x2 - x1, y2 - y1
            pred_list.append({"image_id": img_idx, "category_id": int(c), "bbox": [x1, y1, bw, bh], "score": float(score)})

        if not pred_list: return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}

        cocoDt = cocoGt.loadRes(pred_list)

        cocoEval = COCOeval(cocoGt, cocoDt, 'bbox')
        cocoEval.params.maxDets = [100, 300, 500] 
        
        cocoEval.evaluate()
        cocoEval.accumulate()
        
        with contextlib.redirect_stdout(io.StringIO()):
            cocoEval.summarize()

        if len(cocoEval.stats) < 12:
            return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}

        return {
            "AP50:95": cocoEval.stats[0],
            "AP50": cocoEval.stats[1],
            "AP_small": cocoEval.stats[3],  
            "AP_medium": cocoEval.stats[4], 
            "AP_large": cocoEval.stats[5],
        }

    result_dict = {}
    for group in ['ALL', 'HR', 'LR']:
        c = stats[group]['count']
        res = calc_official_coco_metrics(stats[group]['indices'])
        res['Img_Cnt'] = c
        res['Avg_Inf_Cnt'] = stats[group]['inf_cnt'] / c if c else 0
        res['Avg_Inf_Time'] = (stats[group]['inf_time'] / c) * 1000 if c else 0
        res['Avg_Tot_Time'] = (stats[group]['total_time'] / c) * 1000 if c else 0
        result_dict[group] = res
        
    result_dict['Peak_VRAM'] = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0
    return result_dict

# =========================================================
# 실행 및 다중 표 그리기 (Official COCO Protocol)
# =========================================================
methods = [
    "UC (2x2 Uniform Crop)", 
    "Ours (DAHI Only)",
    "Ours (Tetris Only)",
    "Ours (DAHI + Tetris)"
]

final_stats = {}
for m in methods: 
    final_stats[m] = run_official_ablation_benchmark(m)

print("\n" + "="*145)
print(f"🏆 [Ablation Study] Single Model & Official COCO Protocol Evaluation (Total {NUM_TEST_IMAGES} Images) 🏆")
print("="*145)
print(f"{'Method':<32} | {'Type':<4} | {'Img':<4} | {'mAP':<6} | {'AP50':<6} | {'APs':<6} | {'APm':<6} | {'APl':<6} | {'Inf Cnt':<7} | {'Inf Time':<9} | {'Tot Time':<9}")
print("-" * 145)

for m, groups in final_stats.items():
    for g in ['ALL', 'HR', 'LR']:
        s = groups[g]
        if s['Img_Cnt'] == 0: continue
        
        mAP  = s.get('AP50:95', 0.0)
        ap50 = s.get('AP50', 0.0)
        aps  = s.get('AP_small', 0.0)
        apm  = s.get('AP_medium', 0.0)
        apl  = s.get('AP_large', 0.0)
        
        print(f"{m if g == 'ALL' else '':<32} | {g:<4} | {s['Img_Cnt']:<4} | {mAP:.4f} | {ap50:.4f} | {aps:.4f} | {apm:.4f} | {apl:.4f} | {s['Avg_Inf_Cnt']:4.1f} /i | {s['Avg_Inf_Time']:5.1f} ms | {s['Avg_Tot_Time']:5.1f} ms")
    
    print(f"{'':<32} > Peak VRAM: {groups.get('Peak_VRAM', 0.0):.1f} MB")
    print("-" * 145)


# =========================================================
# 💡 [NEW] SCE 결과 캔버스 시각화 (Matplotlib Inline)
# =========================================================
if len(sce_vis_samples) > 0:
    print("\n" + "="*145)
    print("📸 [Visualization] Spatial Context Extension (SCE) 캔버스 결과 샘플")
    print("="*145)
    
    cols = 2
    rows = (len(sce_vis_samples) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 7.5 * rows))
    axes = axes.flatten()
    
    for i, img_rgb in enumerate(sce_vis_samples):
        axes[i].imshow(img_rgb)
        axes[i].set_title(f"SCE Canvas Sample {i+1}", fontsize=14, fontweight='bold')
        axes[i].axis('off')
        
    # 남는 빈 subplot 영역 숨기기
    for i in range(len(sce_vis_samples), len(axes)):
        axes[i].axis('off')
        
    plt.tight_layout()
    plt.show()

🚀 [Ablation Study] Single-Model Pipeline! 완벽한 재현 시작! (Total 1294 images)


⏳ UC (2x2 Uniform Crop): 100%|██████████████████████████████| 1294/1294 [01:31<00:00, 14.18it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.26s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=32.92s).
Accumulating evaluation results...
DONE (t=1.44s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=6.80s).
Accumulating evaluation results...
DONE (t=0.27s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.04s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=26.04s).
Accumulating evaluation results...
DONE (t=1.14s).


⏳ Ours (DAHI Only): 100%|██████████████████████████████| 1294/1294 [01:37<00:00, 13.26it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=34.09s).
Accumulating evaluation results...
DONE (t=1.53s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=7.76s).
Accumulating evaluation results...
DONE (t=0.29s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.24s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=26.24s).
Accumulating evaluation results...
DONE (t=1.14s).


⏳ Ours (Tetris Only): 100%|██████████████████████████████| 1294/1294 [01:21<00:00, 15.83it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.26s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=30.45s).
Accumulating evaluation results...
DONE (t=1.59s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=6.52s).
Accumulating evaluation results...
DONE (t=0.26s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.04s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=24.18s).
Accumulating evaluation results...
DONE (t=1.05s).


⏳ Ours (DAHI + Tetris): 100%|██████████████████████████████| 1294/1294 [01:34<00:00, 13.76it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=35.08s).
Accumulating evaluation results...
DONE (t=1.48s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=7.62s).
Accumulating evaluation results...
DONE (t=0.30s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.25s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=27.41s).
Accumulating evaluation results...
DONE (t=1.20s).

🏆 [Ablation Study] Single Model & Official COCO Protocol Evaluation (Total 5000 Images) 🏆
Method                           | Type | Img  | mAP    | AP50   | APs    | APm    | APl    | Inf Cnt | Inf Time  | Tot Time 
----------------------------------------------

In [ ]:
import torch
print(torch.__version__)

2.7.0+cu128


In [ ]:
import cv2
import os
import time
import numpy as np
import tqdm
import torch
from ultralytics import YOLO

import contextlib
import io

# 💡 [NEW] 시각화를 위한 라이브러리 임포트
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from mpl_toolkits.mplot3d import Axes3D
from scipy.stats import gaussian_kde

# 💡 [NEW] 공식 COCO API 임포트
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# =========================================================
# ⚙️ 하이퍼파라미터 (Hyperparameters) - Single Model Edition
# =========================================================
# MODEL_FILTER_PATH = 'model/best_nano.pt' # 💡 보조 모델 완전히 삭제!
MODEL_MAIN_PATH = 'model/best_small.pt'

CONF_GLOBAL = 0.3
CONF_FILTER = 0.1     
CONF_DENSE = 0.3
CONF_TETRIS = 0.3
CONF_UC = 0.3

DENSE_RATIO_THRESH = 0.30  # 💡 [NEW] 라우팅 임계값: 전체 소형 객체의 50% 이상 밀집 시 DAHI 구역 인정
# IOU_FILTER_MATCH = 0.97 # 💡 사용 안함 (중복검사 삭제됨)
NMS_CONF_THRESH = 0.3
NMS_IOU_THRESH = 0.4    

DENSE_WINDOW_SIZE = 512
DENSE_STEP = 320      

MERGE_PAD = 16
CROP_PAD_LARGE = 80     
CROP_PAD_SMALL = 16
CROP_PAD_THRESH = 200

CANVAS_SIZE = 960
CANVAS_MARGIN = 2
CANVAS_BG_COLOR = 114

UPSCALE_RATIO = 1.5        
UPSCALE_MAX_THRESH = 200    

NUM_TEST_IMAGES = 5000
HR_THRESHOLD = 1920 * 1080 

# =========================================================
dataset_root = 'data/test'
img_dir, lbl_dir = os.path.join(dataset_root, 'images'), os.path.join(dataset_root, 'labels')
img_list = sorted(os.listdir(img_dir))[:NUM_TEST_IMAGES]

print(f"🚀 [Ablation Study] Single-Model Pipeline! 완벽한 재현 시작! (Total {len(img_list)} images)")

# m1 = YOLO(MODEL_FILTER_PATH) # 💡 보조 모델 삭제
m2 = YOLO(MODEL_MAIN_PATH)

# 💡 [NEW] 시각화할 캔버스 이미지를 담아둘 전역 리스트
sce_vis_samples = []

def calculate_iou(box1, box2):
    xi1, yi1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    xi2, yi2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])

def get_size_category(w, h):
    area = w * h
    if area < 32 ** 2: return 'small'
    elif area < 96 ** 2: return 'medium'
    else: return 'large'

def merge_clusters_dynamic(boxes, img_w, img_h, merge_pad=MERGE_PAD):
    if not len(boxes): return []
    def get_padded(b, pad): return [max(0, b[0]-pad), max(0, b[1]-pad), min(img_w, b[2]+pad), min(img_h, b[3]+pad)]
    def is_overlap(b1, b2):
        p1, p2 = get_padded(b1, merge_pad), get_padded(b2, merge_pad)
        return (min(p1[2], p2[2]) > max(p1[0], p2[0])) and (min(p1[3], p2[3]) > max(p1[1], p2[1]))
    curr = boxes.copy()
    while True:
        merged, flags = [], [False]*len(curr)
        for i in range(len(curr)):
            if flags[i]: continue
            b = curr[i]
            for j in range(i+1, len(curr)):
                if not flags[j] and is_overlap(b, curr[j]):
                    b = [min(b[0], curr[j][0]), min(b[1], curr[j][1]), max(b[2], curr[j][2]), max(b[3], curr[j][3])]
                    flags[j] = True
            merged.append(b)
        if len(merged) == len(curr): break
        curr = merged
    final_boxes = []
    for b in curr:
        bw, bh = b[2] - b[0], b[3] - b[1]
        crop_pad = CROP_PAD_LARGE if max(bw, bh) < CROP_PAD_THRESH else CROP_PAD_SMALL 
        final_boxes.append(get_padded(b, crop_pad))
    return final_boxes

def run_official_ablation_benchmark(method_name):
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        
    all_gts = {}; all_preds = []
    img_infos = {} 
    
    stats = {
        'ALL': {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'HR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'LR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()}
    }

    pbar = tqdm.tqdm(img_list, desc=f"⏳ {method_name}", bar_format='{l_bar}{bar:30}{r_bar}')
    for img_idx, img_name in enumerate(pbar):
        img_path, lbl_path = os.path.join(img_dir, img_name), os.path.join(lbl_dir, img_name.replace('.jpg', '.txt'))
        img = cv2.imread(img_path); h, w, _ = img.shape
        
        img_infos[img_idx] = {'w': w, 'h': h, 'name': img_name}
        
        gts = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    c, xc, yc, bw, bh = map(float, line.split())
                    gts.append([int(c), (xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h]) 
        all_gts[img_idx] = gts

        t_pipe_start = time.time()
        img_inf_time, img_inf_cnt = 0, 0
        
        # =====================================================================
        # 1. Baseline: UC (2x2 Uniform Crop)
        # =====================================================================
        if method_name == "UC (2x2 Uniform Crop)":
            # UC 평가가 너무 오래 걸려 비활성화
            ch, cw = h // 2, w // 2
            # crops, offsets = [img], [(0, 0)]
            # for y in [0, ch]:
            #     for x in [0, cw]:
            #         crops.append(img[y:y+ch, x:x+cw])
            #         offsets.append((x, y))
            
            # t_inf_start = time.time()
            # results2 = m2.predict(crops, conf=CONF_UC, verbose=False, batch=5)
            # img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 5 
            
            # temp_boxes, temp_scores, temp_classes = [], [], []
            # for i, res in enumerate(results2):
            #     ox, oy = offsets[i]
            #     for b in res.boxes:
            #         # 💡 [FIX] .item() 을 사용하여 텐서를 완벽하게 파이썬 float으로 변환!
            #         bx1 = b.xyxy[0][0].item() + ox
            #         by1 = b.xyxy[0][1].item() + oy
            #         bx2 = b.xyxy[0][2].item() + ox
            #         by2 = b.xyxy[0][3].item() + oy
                    
            #         temp_boxes.append([bx1, by1, bx2, by2])
            #         temp_scores.append(float(b.conf[0]))
            #         temp_classes.append(int(b.cls[0]))
                    
            # for c in set(temp_classes):
            #     c_boxes = [b for j, b in enumerate(temp_boxes) if temp_classes[j] == c]
            #     c_scores = [s for j, s in enumerate(temp_scores) if temp_classes[j] == c]
            #     cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
            #     indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
            #     if len(indices) > 0:
            #         for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 2. 제안 1: Ours (DAHI Only)
        # =====================================================================
        elif method_name == "Ours (DAHI Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            # local_boxes, local_scores, local_classes = [], [], []
            # roi_boxes = []
            
            # # 💡 [핵심 최적화] 메인 모델(m2)로 CONF_FILTER(0.1) 기준 단 1번만 스캔
            # t_inf_start = time.time()
            # res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            # img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            # for b in res_global_all[0].boxes:
            #     bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
            #     conf = float(b.conf[0])
                
            #     # 1. 글로벌 확정 박스 (0.3 이상)
            #     if conf >= CONF_GLOBAL:
            #         boosted_conf = min(1.0, conf * 1.10)
            #         global_final_boxes.append([bx1, by1, bx2, by2])
            #         global_final_scores.append(boosted_conf)
            #         global_final_classes.append(int(b.cls[0]))
                
            #     # 2. ROI 박스 등록 (모든 탐지 객체 재검사)
            #     roi_boxes.append([bx1, by1, bx2, by2])
            
            # remaining_boxes = roi_boxes.copy()
            # dense_regions = []
            
            # while len(remaining_boxes) > 0:
            #     best_count, best_region = -1, None
            #     for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
            #         for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
            #             count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
            #             if count > best_count: 
            #                 best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
            #     if best_region and best_count >= 1:
            #         dense_regions.append(best_region)
            #         dx1, dy1, dx2, dy2 = best_region
            #         remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
            #     else: break
            
            # unified_infer_list = [img[dy1:dy2, dx1:dx2] for dx1, dy1, dx2, dy2 in dense_regions]
            # if len(unified_infer_list) > 0:
            #     t_inf_start = time.time()
            #     res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
            #     img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
            #     for idx, (dx1, dy1, dx2, dy2) in enumerate(dense_regions):
            #         cw_dense, ch_dense = dx2 - dx1, dy2 - dy1
            #         for b in res_all[idx].boxes:
            #             bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
            #             if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
            #             local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
            
            # final_local_preds = []
            # for c in set(local_classes):
            #     c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
            #     cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
            #     indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
            #     if len(indices) > 0:
            #         for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            # combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            # for c in set(combined_classes):
            #     c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
            #     cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
            #     indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
            #     if len(indices) > 0:
            #         for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 3. 제안 2: Ours (Tetris Only)
        # =====================================================================
        elif method_name == "Ours (Tetris Only)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            # local_boxes, local_scores, local_classes = [], [], []
            # roi_boxes = []
            
            # t_inf_start = time.time()
            # res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            # img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            # for b in res_global_all[0].boxes:
            #     bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
            #     conf = float(b.conf[0])
            #     if conf >= CONF_GLOBAL:
            #         global_final_boxes.append([bx1, by1, bx2, by2])
            #         global_final_scores.append(min(1.0, conf * 1.10))
            #         global_final_classes.append(int(b.cls[0]))
            #     roi_boxes.append([bx1, by1, bx2, by2])
            
            # canvases, canvas_infos = [], []
            # if len(roi_boxes) > 0:
            #     clustered_boxes = merge_clusters_dynamic(roi_boxes, w, h, merge_pad=MERGE_PAD)
            #     crops_to_pack = []
            #     for cb in clustered_boxes:
            #         cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
            #         scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
            #         cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
            #         if cw_crop > 0 and ch_crop > 0:
            #             crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
            #             if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
            #             else: crop_img = crop_img[:ch_crop, :cw_crop]
            #             crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
            #     crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
            #     current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
            #     cx, cy, max_h = 0, 0, 0
            #     for item in crops_to_pack:
            #         if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
            #         if cy + item['ch'] > CANVAS_SIZE: canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
            #         current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
            #         canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
            #         cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
            #     if max_h > 0 or cx > 0: canvases.append(current_canvas)
                
            #     for c_idx, canvas in enumerate(canvases):
            #         c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
            #         if not c_infos: continue
            #         unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
            #         for i, cy1 in enumerate(unique_cy1s):
            #             row_items = [info for info in c_infos if info['cy1'] == cy1]
            #             row_items.sort(key=lambda x: x['cx1'])
            #             next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
            #             for j, info in enumerate(row_items):
            #                 item_w, item_h = info['cx2'] - info['cx1'], info['cy2'] - info['cy1']
            #                 ox, oy, s = info['ox'], info['oy'], info['scale']
            #                 org_w, org_h = int(item_w / s), int(item_h / s) 
            #                 next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
            #                 gap_w = next_cx1 - info['cx2']
            #                 if j + 1 < len(row_items): gap_w -= CANVAS_MARGIN
            #                 if gap_w > 0:
            #                     ext_w_org = min(int(gap_w / s), w - (ox + org_w))
            #                     if ext_w_org > 0:
            #                         ext_crop = img[oy:oy+org_h, ox+org_w:ox+org_w+ext_w_org]
            #                         if s > 1.0: ext_crop = cv2.resize(ext_crop, (gap_w, item_h), interpolation=cv2.INTER_CUBIC)
            #                         canvas[info['cy1']:info['cy2'], info['cx2']:info['cx2']+ext_crop.shape[1]] = ext_crop
            #                         info['cx2'] += ext_crop.shape[1]
            #                 gap_h = next_cy1 - info['cy2']
            #                 if i + 1 < len(unique_cy1s): gap_h -= CANVAS_MARGIN
            #                 if gap_h > 0:
            #                     ext_h_org = min(int(gap_h / s), h - (oy + org_h))
            #                     if ext_h_org > 0:
            #                         ext_crop = img[oy+org_h:oy+org_h+ext_h_org, ox:ox+org_w]
            #                         if s > 1.0: ext_crop = cv2.resize(ext_crop, (item_w, gap_h), interpolation=cv2.INTER_CUBIC)
            #                         canvas[info['cy2']:info['cy2']+ext_crop.shape[0], info['cx1']:info['cx1']+item_w] = ext_crop
            #                         info['cy2'] += ext_crop.shape[0]

            # if len(canvases) > 0:
            #     t_inf_start = time.time()
            #     res_pack = m2.predict(canvases, conf=CONF_TETRIS, verbose=False, batch=16)
            #     img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(canvases)
                
            #     for c_idx, res in enumerate(res_pack):
            #         for b in res.boxes:
            #             bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
            #             bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
            #             for info in canvas_infos:
            #                 if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
            #                     if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
            #                     s = info['scale']
            #                     orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
            #                     orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
            #                     local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
            #                     break
                                        
            # final_local_preds = []
            # for c in set(local_classes):
            #     c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
            #     cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
            #     indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
            #     if len(indices) > 0:
            #         for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            # combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            # for c in set(combined_classes):
            #     c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
            #     cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
            #     indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
            #     if len(indices) > 0:
            #         for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 4. 제안 3: Ours (DAHI + Tetris)
        # =====================================================================
        elif method_name == "Ours (DAHI + Tetris)":
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            roi_boxes = []
            
            t_inf_start = time.time()
            res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global_all[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = float(b.conf[0])
                if conf >= CONF_GLOBAL:
                    global_final_boxes.append([bx1, by1, bx2, by2])
                    global_final_scores.append(min(1.0, conf * 1.10))
                    global_final_classes.append(int(b.cls[0]))
                roi_boxes.append([bx1, by1, bx2, by2, conf]) # 💡 [수정] conf를 리스트 마지막에 추가
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            # SBSI (Small object Based Slicing Inference)
            # -----------------------------------------------------------------
            # 💡 [NEW V4] 크기 비례 가중치 (Secure the Larger-Small)
            # -----------------------------------------------------------------
            small_info = []
            for b in remaining_boxes:
                bx1, by1, bx2, by2, conf = b 
                w_box, h_box = bx2 - bx1, by2 - by1
                if get_size_category(w_box, h_box) == 'small':
                    cx, cy = (bx1 + bx2) / 2, (by1 + by2) / 2
                    area = w_box * h_box
                    
                    # 💡 [수정된 가중치 수식]
                    # 신뢰도가 낮고(1-conf), 크기가 클수록(sqrt(area)) 높은 가중치
                    weight = (1.0 - conf) * np.sqrt(area)
                    
                    small_info.append([cx, cy, weight])
            
            total_small_objs = len(small_info)
            # 임계값은 원시 객체 수(Discrete Count) 기준으로 확정
            route_threshold = max(1, int(total_small_objs * DENSE_RATIO_THRESH)) 
            
            if total_small_objs > 0:
                pts = np.array([[info[0], info[1]] for info in small_info]) # Shape: (N, 2)
                weights = np.array([info[2] for info in small_info])
                
                # 가중치 정규화
                norm_weights = weights / (np.mean(weights) + 1e-6)
                sigma = DENSE_WINDOW_SIZE / 4.0 
                
                while len(pts) > 0:
                    # 2. KDE 연산: 완벽한 무게중심 좌표를 찾기 위한 '유도(Guidance)' 역할
                    dist_sq = np.sum((pts[:, None, :] - pts[None, :, :]) ** 2, axis=-1) 
                    kernel = np.exp(-dist_sq / (2 * sigma ** 2)) 
                    densities = kernel @ norm_weights 
                    
                    best_idx = np.argmax(densities)
                    
                    # 가중 무게중심 스내핑
                    influence = norm_weights * kernel[best_idx]
                    sum_influence = np.sum(influence)
                    
                    if sum_influence > 0:
                        cx = np.sum(influence * pts[:, 0]) / sum_influence
                        cy = np.sum(influence * pts[:, 1]) / sum_influence
                    else:
                        cx, cy = pts[best_idx]
                        
                    # 윈도우 경계 확정
                    dx1 = int(max(0, cx - DENSE_WINDOW_SIZE / 2))
                    dy1 = int(max(0, cy - DENSE_WINDOW_SIZE / 2))
                    dx2 = int(min(w, dx1 + DENSE_WINDOW_SIZE))
                    dy2 = int(min(h, dy1 + DENSE_WINDOW_SIZE))
                    
                    if dx2 - dx1 < DENSE_WINDOW_SIZE: dx1 = max(0, dx2 - DENSE_WINDOW_SIZE)
                    if dy2 - dy1 < DENSE_WINDOW_SIZE: dy1 = max(0, dy2 - DENSE_WINDOW_SIZE)
                    
                    # 3. 이산 검증 (Discrete Confirmation): 확정된 윈도우 안에 '실제' 점이 몇 개인지 카운트
                    actual_in_window = (pts[:, 0] >= dx1) & (pts[:, 0] <= dx2) & (pts[:, 1] >= dy1) & (pts[:, 1] <= dy2)
                    actual_count = np.sum(actual_in_window)
                    
                    # 임계값 누수 차단: 실제 카운트 값으로 통과 여부 결정
                    if actual_count >= route_threshold:
                        dense_regions.append((dx1, dy1, dx2, dy2))
                        
                        # 처리된 영역 배제
                        mask = ~actual_in_window
                        pts = pts[mask]
                        norm_weights = norm_weights[mask]
                        
                        remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                    else:
                        # 최고 밀도 앵커조차 임계값을 넘지 못하면 라우팅 완전 종료
                        break
            
            
            # (이후 코드는 기존과 동일하게 unified_infer_list 생성 및 Tetris Canvas 생성 로직으로 이어짐)
            # 동적 문맥 재배치 파이프라인 (Dynamic Context Rearrangement Pipeline)
            unified_infer_list = []
            dense_idx_list = []
            
            for dx1, dy1, dx2, dy2 in dense_regions:
                unified_infer_list.append(img[dy1:dy2, dx1:dx2])
                dense_idx_list.append((len(unified_infer_list) - 1, dx1, dy1, dx2, dy2))

            canvases, canvas_infos = [], []
            canvas_start_idx = -1
            if len(remaining_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(remaining_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)# (생략: 기존 코드의 DAHI 라우팅 및 crops_to_pack 정렬까지는 동일함)
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                
                # 💡 [NEW V13/14 Baseline] 4방향 확장 전, "순수 테트리스 패킹" 결과 수집
                # 캔버스를 하나 따로 만들어서 SCE 없는 상태를 시각화용으로 저장합니다.
                pre_sce_vis_list = [] # 여기에 SCE 없는 순수 패킹 캔버스를 담습니다.

                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: 
                        # 시각화 수집 (최대 3장만)
                        if len(pre_sce_vis_list) < 3: pre_sce_vis_list.append(cv2.cvtColor(current_canvas, cv2.COLOR_BGR2RGB))
                        canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    
                    # 패킹 실행
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: 
                    # 시각화 수집
                    if len(pre_sce_vis_list) < 3: pre_sce_vis_list.append(cv2.cvtColor(current_canvas, cv2.COLOR_BGR2RGB))
                    canvases.append(current_canvas)
                
                # SCE
                # -----------------------------------------------------------------
                # 💡 [NEW V14] 전방위 순수 문맥 확장 (Omnidirectional Contiguous SCE)
                # -----------------------------------------------------------------
                # 💡 여백 분배 함수 (상하좌우 가용 공간 체크)
                def get_distribution(gap, avail1, avail2):
                    half = gap // 2
                    if avail1 < half: return avail1, min(gap - avail1, avail2)
                    elif avail2 < half: return min(gap - avail2, avail1), avail2
                    else: return half, gap - half

                # 💡 데이터를 쓰기 직전에 '실제' shape을 구해서 canvas 영역을 조절하는 L자 공백 L-Shape 방어 로직 V14
                for c_idx, canvas in enumerate(canvases):
                    c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                    if not c_infos: continue
                    unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                    
                    for i, cy1 in enumerate(unique_cy1s):
                        row_items = [info for info in c_infos if info['cy1'] == cy1]
                        row_items.sort(key=lambda x: x['cx1'])
                        next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                        
                        for j, info in enumerate(row_items):
                            cx1, cy1 = info['cx1'], info['cy1']
                            cx2, cy2 = info['cx2'], info['cy2']
                            ox, oy, s = info['ox'], info['oy'], info['scale']
                            item_w, item_h = cx2 - cx1, cy2 - cy1
                            org_w, org_h = max(1, int(item_w / s)), max(1, int(item_h / s))
                            
                            # 💡 1. 캔버스에 할당된 셀의 최대 가용 크기 계산
                            next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                            alloc_w = next_cx1 - cx1
                            if j + 1 < len(row_items): alloc_w -= CANVAS_MARGIN
                            
                            alloc_h = next_cy1 - cy1
                            if i + 1 < len(unique_cy1s): alloc_h -= CANVAS_MARGIN
                            
                            # 현재 크롭이 차지하고 남은 여백 (Gap)
                            gap_w = max(0, alloc_w - item_w)
                            gap_h = max(0, alloc_h - item_h)
                            
                            if gap_w > 0 or gap_h > 0:
                                gap_w_org = int(gap_w / s)
                                gap_h_org = int(gap_h / s)
                                
                                # 💡 2. 4-Way 확장을 위한 픽셀 분배
                                ext_left, ext_right = get_distribution(gap_w_org, ox, w - (ox + org_w))
                                ext_top, ext_bottom = get_distribution(gap_h_org, oy, h - (oy + org_h))
                                
                                # 💡 3. 원본 이미지 내에서 확장된 새로운 크롭 좌표 계산
                                new_ox = ox - ext_left
                                new_oy = oy - ext_top
                                new_org_w = org_w + ext_left + ext_right
                                new_org_h = org_h + ext_top + ext_bottom
                                
                                if new_org_w > 0 and new_org_h > 0:
                                    ext_crop = img[new_oy:new_oy+new_org_h, new_ox:new_ox+new_org_w]
                                    actual_w = int(new_org_w * s) if s > 1.0 else new_org_w
                                    actual_h = int(new_org_h * s) if s > 1.0 else new_org_h
                                    
                                    if ext_crop.size > 0:
                                        if s > 1.0: 
                                            ext_crop = cv2.resize(ext_crop, (actual_w, actual_h), interpolation=cv2.INTER_CUBIC)
                                        
                                        # 💡 데이터를 쓰기 직전에 '실제' shape을 구함
                                        h_crop, w_crop = ext_crop.shape[:2]
                                        
                                        # 💡 경계 초과 방지
                                        y_end = min(cy1 + h_crop, CANVAS_SIZE)
                                        x_end = min(cx1 + w_crop, CANVAS_SIZE)
                                        
                                        # 💡 실제 반영된 크기를 기준으로 모든 좌표 확정
                                        canvas[cy1:y_end, cx1:x_end] = ext_crop[:y_end-cy1, :x_end-cx1]
                                        
                                        # 💡 매핑 정보 갱신
                                        info['ox'] = new_ox
                                        info['oy'] = new_oy
                                        info['cx2'] = cx1 + (x_end - cx1) # 실제 반영된 너비
                                        info['cy2'] = cy1 + (y_end - cy1) # 실제 반영된 높이

                # 텅 비어버린 캔버스 정리
                canvases = [c for i, c in enumerate(canvases) if any(info['c_idx'] == i for info in canvas_infos)]
                for new_idx, old_idx in enumerate(sorted(list(set(info['c_idx'] for info in canvas_infos)))):
                    for info in canvas_infos:
                        if info['c_idx'] == old_idx: info['c_idx'] = new_idx

                # 💡 [Visual Hook] 시각화 비교 수집 (V14 Baseline vs Ours with SCE)
                if len(sce_vis_samples) < 5 and len(canvases) > 0:
                    random_idx = random.randrange(len(canvases))
                    if random_idx < len(pre_sce_vis_list):
                        pre_canvas = pre_sce_vis_list[random_idx]
                        post_canvas_rgb = cv2.cvtColor(canvases[random_idx], cv2.COLOR_BGR2RGB)
                        
                        # 💡 [수정] vis_img_name을 img_name으로 변경
                        sce_vis_samples.append((img_name, pre_canvas, post_canvas_rgb))

                if len(canvases) > 0:
                    canvas_start_idx = len(unified_infer_list)
                    unified_infer_list.extend(canvases)

            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for d_idx, dx1, dy1, dx2, dy2 in dense_idx_list:
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1; res_dense = res_all[d_idx]
                    for b in res_dense.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                
                if canvas_start_idx != -1:
                    res_pack = res_all[canvas_start_idx:]
                    for c_idx, res in enumerate(res_pack):
                        for b in res.boxes:
                            bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                            bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                            for info in canvas_infos:
                                if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                    if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                    s = info['scale']
                                    orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                    orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                    local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                    break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # 통계 저장
        img_total_time = time.time() - t_pipe_start
        is_hr = (w * h >= HR_THRESHOLD)
        target_keys = ['ALL', 'HR'] if is_hr else ['ALL', 'LR']
        for k in target_keys:
            stats[k]['count'] += 1
            stats[k]['inf_time'] += img_inf_time
            stats[k]['total_time'] += img_total_time
            stats[k]['inf_cnt'] += img_inf_cnt
            stats[k]['indices'].add(img_idx)

    # ---------------------------------------------------------
    # 💡 공식 COCO API 연산 엔진
    # ---------------------------------------------------------
    def calc_official_coco_metrics(subset_indices):
        if not subset_indices: return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}
        
        gt_dict = {"images": [], "annotations": [], "categories": []}
        for i in range(10): gt_dict["categories"].append({"id": i, "name": f"class_{i}"})
            
        ann_id = 1
        for img_idx in subset_indices:
            info = img_infos[img_idx]
            gt_dict["images"].append({"id": img_idx, "width": info['w'], "height": info['h'], "file_name": info['name']})
            for gt in all_gts[img_idx]:
                c, x1, y1, x2, y2 = gt
                bw, bh = x2 - x1, y2 - y1
                gt_dict["annotations"].append({"id": ann_id, "image_id": img_idx, "category_id": int(c), "bbox": [x1, y1, bw, bh], "area": bw * bh, "iscrowd": 0})
                ann_id += 1

        cocoGt = COCO()
        cocoGt.dataset = gt_dict
        cocoGt.createIndex()

        sub_preds = [p for p in all_preds if p[0] in subset_indices]
        pred_list = []
        for pred in sub_preds:
            img_idx, c, score, x1, y1, x2, y2 = pred
            bw, bh = x2 - x1, y2 - y1
            pred_list.append({"image_id": img_idx, "category_id": int(c), "bbox": [x1, y1, bw, bh], "score": float(score)})

        if not pred_list: return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}

        cocoDt = cocoGt.loadRes(pred_list)

        cocoEval = COCOeval(cocoGt, cocoDt, 'bbox')
        cocoEval.params.maxDets = [100, 300, 500] 
        
        cocoEval.evaluate()
        cocoEval.accumulate()
        
        with contextlib.redirect_stdout(io.StringIO()):
            cocoEval.summarize()

        if len(cocoEval.stats) < 12:
            return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}

        return {
            "AP50:95": cocoEval.stats[0],
            "AP50": cocoEval.stats[1],
            "AP_small": cocoEval.stats[3],  
            "AP_medium": cocoEval.stats[4], 
            "AP_large": cocoEval.stats[5],
        }

    result_dict = {}
    for group in ['ALL', 'HR', 'LR']:
        c = stats[group]['count']
        res = calc_official_coco_metrics(stats[group]['indices'])
        res['Img_Cnt'] = c
        res['Avg_Inf_Cnt'] = stats[group]['inf_cnt'] / c if c else 0
        res['Avg_Inf_Time'] = (stats[group]['inf_time'] / c) * 1000 if c else 0
        res['Avg_Tot_Time'] = (stats[group]['total_time'] / c) * 1000 if c else 0
        result_dict[group] = res
        
    result_dict['Peak_VRAM'] = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0
    return result_dict

# =========================================================
# 실행 및 다중 표 그리기 (Official COCO Protocol)
# =========================================================
methods = [
    "UC (2x2 Uniform Crop)", 
    "Ours (DAHI Only)",
    "Ours (Tetris Only)",
    "Ours (DAHI + Tetris)"
]

final_stats = {}
for m in methods: 
    final_stats[m] = run_official_ablation_benchmark(m)

print("\n" + "="*145)
print(f"🏆 [Ablation Study] Single Model & Official COCO Protocol Evaluation (Total {NUM_TEST_IMAGES} Images) 🏆")
print("="*145)
print(f"{'Method':<32} | {'Type':<4} | {'Img':<4} | {'mAP':<6} | {'AP50':<6} | {'APs':<6} | {'APm':<6} | {'APl':<6} | {'Inf Cnt':<7} | {'Inf Time':<9} | {'Tot Time':<9}")
print("-" * 145)

for m, groups in final_stats.items():
    for g in ['ALL', 'HR', 'LR']:
        s = groups[g]
        if s['Img_Cnt'] == 0: continue
        
        mAP  = s.get('AP50:95', 0.0)
        ap50 = s.get('AP50', 0.0)
        aps  = s.get('AP_small', 0.0)
        apm  = s.get('AP_medium', 0.0)
        apl  = s.get('AP_large', 0.0)
        
        print(f"{m if g == 'ALL' else '':<32} | {g:<4} | {s['Img_Cnt']:<4} | {mAP:.4f} | {ap50:.4f} | {aps:.4f} | {apm:.4f} | {apl:.4f} | {s['Avg_Inf_Cnt']:4.1f} /i | {s['Avg_Inf_Time']:5.1f} ms | {s['Avg_Tot_Time']:5.1f} ms")
    
    print(f"{'':<32} > Peak VRAM: {groups.get('Peak_VRAM', 0.0):.1f} MB")
    print("-" * 145)


# ---------------------------------------------------------
# 💡 [논문용 Figure 생성기] 라우팅 메커니즘 시각화
# ---------------------------------------------------------
def generate_thesis_figures(img_list, img_dir, model, window_size=DENSE_WINDOW_SIZE):
    print("\n" + "="*145)
    print("🎨 [Thesis Figure] 학위 논문용 라우팅 메커니즘 시각화 렌더링 시작...")
    
    # 1. 시각화하기 좋은 HR 이미지 무작위 탐색
    vis_img_name = random.choice(img_list)
    img_path = os.path.join(img_dir, vis_img_name)
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape
    
    # 모델 1차 스캔
    res = model.predict(img, conf=CONF_FILTER, verbose=False)[0]
    
    small_info = []
    roi_boxes = []
    
    for b in res.boxes:
        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
        conf = float(b.conf[0])
        w_box, h_box = bx2 - bx1, by2 - by1
        roi_boxes.append([bx1, by1, bx2, by2])
        
        if get_size_category(w_box, h_box) == 'small':
            cx, cy = (bx1 + bx2) / 2, (by1 + by2) / 2
            area = w_box * h_box
            # [수식 1] 스케일-신뢰도 비례 가중치
            weight = (1.0 - conf) * np.sqrt(area)
            small_info.append([cx, cy, weight, bx1, by1, bx2, by2])
            
    if len(small_info) == 0:
        print("⚠️ 소형 객체가 없는 이미지입니다. 다른 이미지로 재시도합니다.")
        return generate_thesis_figures(img_list, img_dir, model, window_size)

    pts = np.array([[info[0], info[1]] for info in small_info])
    weights = np.array([info[2] for info in small_info])
    norm_weights = weights / (np.mean(weights) + 1e-6)
    sigma = window_size / 4.0
    
    # ---------------------------------------------------------
    # 📊 데이터 연산 1: 공간 확률 밀도(KDE) 2D 그리드 생성
    # ---------------------------------------------------------
    grid_size = 100 # 고해상도 히트맵 해상도
    x_grid = np.linspace(0, w, grid_size)
    y_grid = np.linspace(0, h, grid_size)
    X, Y = np.meshgrid(x_grid, y_grid)
    grid_pts = np.vstack([X.ravel(), Y.ravel()]).T
    
    # 벡터화된 밀도 계산
    Z = np.zeros(grid_pts.shape[0])
    for i, pt in enumerate(pts):
        dist_sq = np.sum((grid_pts - pt) ** 2, axis=1)
        Z += norm_weights[i] * np.exp(-dist_sq / (2 * sigma ** 2))
    Z = Z.reshape(X.shape)
    
    # ---------------------------------------------------------
    # 📊 데이터 연산 2: 최적 앵커 및 무게중심 스내핑
    # ---------------------------------------------------------
    dist_sq_pts = np.sum((pts[:, None, :] - pts[None, :, :]) ** 2, axis=-1)
    kernel_pts = np.exp(-dist_sq_pts / (2 * sigma ** 2))
    densities_pts = kernel_pts @ norm_weights
    
    best_idx = np.argmax(densities_pts)
    anchor_pt = pts[best_idx]
    
    influence = norm_weights * kernel_pts[best_idx]
    sum_influence = np.sum(influence)
    com_x = np.sum(influence * pts[:, 0]) / sum_influence
    com_y = np.sum(influence * pts[:, 1]) / sum_influence
    com_pt = np.array([com_x, com_y])
    
    # 창 좌표 확정
    def get_window(cx, cy):
        x1 = max(0, cx - window_size / 2)
        y1 = max(0, cy - window_size / 2)
        x2 = min(w, x1 + window_size)
        y2 = min(h, y1 + window_size)
        if x2 - x1 < window_size: x1 = max(0, x2 - window_size)
        if y2 - y1 < window_size: y1 = max(0, y2 - window_size)
        return x1, y1, x2, y2

    init_window = get_window(anchor_pt[0], anchor_pt[1])
    snap_window = get_window(com_x, com_y)

    # =========================================================
    # 🖼️ Figure 1: 가중치 기반 가우시안 앵커 탐색
    # =========================================================
    plt.rcParams['font.family'] = 'sans-serif' # 논문용 폰트 설정 가능
    fig1 = plt.figure(figsize=(24, 7))
    fig1.suptitle('Fig 1. Weighted Gaussian Anchor Search in Dense Regions', fontsize=20, fontweight='bold', y=0.98)
    
    # [1-1] 원본 ROI
    ax1 = fig1.add_subplot(1, 3, 1)
    ax1.imshow(img_rgb)
    for b in roi_boxes:
        rect = patches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1], linewidth=1, edgecolor='cyan', facecolor='none')
        ax1.add_patch(rect)
    ax1.set_title('(a) Original Image with Detected ROIs', fontsize=16)
    ax1.axis('off')
    
    # [1-2] 열화상 히트맵 (Heatmap)
    ax2 = fig1.add_subplot(1, 3, 2)
    ax2.imshow(img_rgb)
    heatmap = ax2.imshow(Z, extent=[0, w, h, 0], cmap='magma', alpha=0.6, interpolation='bicubic')
    ax2.scatter(pts[:, 0], pts[:, 1], c='white', s=10, alpha=0.5, label='Small Objects')
    ax2.set_title('(b) Continuous Information Density Heatmap', fontsize=16)
    ax2.axis('off')
    
    # [1-3] 3D 공간 밀도 지형 (Surface Plot)
    ax3 = fig1.add_subplot(1, 3, 3, projection='3d')
    surf = ax3.plot_surface(X, Y, Z, cmap='magma', edgecolor='none', alpha=0.9)
    ax3.set_title('(c) 3D Surface of Spatial Information Density', fontsize=16)
    ax3.set_xlabel('Image Width (X)')
    ax3.set_ylabel('Image Height (Y)')
    ax3.set_zlabel('Density $D(p_k)$')
    ax3.view_init(elev=35, azim=45) # 관측 각도
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

    # =========================================================
    # 🖼️ Figure 2: 가중 무게 중심 스내핑 (Center of Mass Snapping)
    # =========================================================
    fig2 = plt.figure(figsize=(24, 7))
    fig2.suptitle('Fig 2. Weighted Center of Mass Snapping Mechanism', fontsize=20, fontweight='bold', y=0.98)
    
    # [2-1] 초기 앵커 및 슬라이싱 영역
    ax4 = fig2.add_subplot(1, 3, 1)
    ax4.imshow(img_rgb)
    ax4.scatter(anchor_pt[0], anchor_pt[1], c='red', s=150, marker='*', zorder=5, label='Optimal Anchor $p_{k^*}$')
    rect_init = patches.Rectangle((init_window[0], init_window[1]), window_size, window_size, 
                                  linewidth=3, edgecolor='red', linestyle='--', facecolor='none', label='Initial Window')
    ax4.add_patch(rect_init)
    ax4.set_xlim(init_window[0] - 200, init_window[2] + 200)
    ax4.set_ylim(init_window[3] + 200, init_window[1] - 200) # Local Crop View
    ax4.set_title('(a) Initial Sliding Window at Optimal Anchor', fontsize=16)
    ax4.legend(loc='upper right')
    ax4.axis('off')
    
    # [2-2] 커널 영향력 및 무게 중심
    ax5 = fig2.add_subplot(1, 3, 2)
    ax5.imshow(img_rgb)
    
    # Local Heatmap for influence
    dist_sq_local = np.sum((grid_pts - anchor_pt) ** 2, axis=1)
    Z_local = np.exp(-dist_sq_local / (2 * sigma ** 2)).reshape(X.shape)
    ax5.imshow(Z_local, extent=[0, w, h, 0], cmap='viridis', alpha=0.5, interpolation='bicubic')
    
    # Point sizes based on influence
    scatter_sizes = (influence / np.max(influence)) * 200
    ax5.scatter(pts[:, 0], pts[:, 1], c='white', s=scatter_sizes, edgecolors='black')
    ax5.scatter(anchor_pt[0], anchor_pt[1], c='red', s=100, marker='*')
    ax5.scatter(com_x, com_y, c='lime', s=200, marker='P', zorder=5, label='Center of Mass $C$')
    
    ax5.set_xlim(init_window[0] - 200, init_window[2] + 200)
    ax5.set_ylim(init_window[3] + 200, init_window[1] - 200)
    ax5.set_title('(b) Local Kernel Influence & Center of Mass Calculation', fontsize=16)
    ax5.legend(loc='upper right')
    ax5.axis('off')
    
    # [2-3] 최종 스내핑된 슬라이싱 영역
    ax6 = fig2.add_subplot(1, 3, 3)
    ax6.imshow(img_rgb)
    
    # Initial (Faded)
    rect_init_fade = patches.Rectangle((init_window[0], init_window[1]), window_size, window_size, 
                                       linewidth=2, edgecolor='red', linestyle=':', alpha=0.5)
    ax6.add_patch(rect_init_fade)
    
    # Final Snapped
    rect_snap = patches.Rectangle((snap_window[0], snap_window[1]), window_size, window_size, 
                                  linewidth=3, edgecolor='lime', facecolor='none', label='Snapped Window')
    ax6.add_patch(rect_snap)
    ax6.scatter(com_x, com_y, c='lime', s=150, marker='P', zorder=5)
    
    # Shift Arrow
    ax6.annotate("", xy=(com_x, com_y), xytext=(anchor_pt[0], anchor_pt[1]),
                 arrowprops=dict(arrowstyle="->", color="white", lw=3))
                 
    ax6.set_xlim(init_window[0] - 200, init_window[2] + 200)
    ax6.set_ylim(init_window[3] + 200, init_window[1] - 200)
    ax6.set_title('(c) Slicing Window Shift (Self-alignment)', fontsize=16)
    ax6.legend(loc='upper right')
    ax6.axis('off')
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

# 💡 실행 구문 (전체 평가 루프가 끝난 후 독립적으로 실행)
generate_thesis_figures(img_list, img_dir, m2, DENSE_WINDOW_SIZE)

# =========================================================
# 💡 [NEW Thesis Visual] SCE 적용 전/후 캔버스 비교 시각화 (Before vs After)
# =========================================================
if len(sce_vis_samples) > 0:
    print("\n" + "="*145)
    print("📸 [Thesis Visual] Spatial Context Extension (SCE) 효과 증명 (Before vs After) 샘플")
    print("="*145)
    
    sample_count = len(sce_vis_samples)
    fig, axes = plt.subplots(sample_count, 2, figsize=(20, 10 * sample_count))
    
    # 학술 스타일 폰트
    plt.rcParams['font.family'] = 'sans-serif'
    
    if sample_count == 1: axes = [axes] # subplot이 하나일 때 예외 방어

    for i, (img_name, canvas_pre, canvas_post) in enumerate(sce_vis_samples):
        # 💡 [좌측] 테트리스 패킹 (V14 Baseline - NO SCE)
        axes[i][0].imshow(canvas_pre)
        axes[i][0].set_title(f"Sample {i+1} (a): Pure Tetris Packing (Baseline - No SCE)\nSource: {img_name}", fontsize=14)
        axes[i][0].axis('off')
        
        # 💡 [우측] Ours (DAHI + Tetris + Omnidirectional SCE)
        axes[i][1].imshow(canvas_post)
        axes[i][1].set_title(f"Sample {i+1} (b): Our Omnidirectional SCE Canvas\n(Seamless Real-pixel Context Restored)", fontsize=14, fontweight='bold')
        axes[i][1].axis('off')
        
        # 여백 강조를 위한 선 (Optional)
        for ax in axes[i]:
             for spine in ax.spines.values(): spine.set_visible(True)

    plt.tight_layout(pad=3.0)
    plt.show()

🚀 [Ablation Study] Single-Model Pipeline! 완벽한 재현 시작! (Total 430 images)


⏳ UC (2x2 Uniform Crop): 100%|██████████████████████████████| 430/430 [00:05<00:00, 75.68it/s] 


creating index...
index created!
creating index...
index created!
creating index...
index created!


⏳ Ours (DAHI Only): 100%|██████████████████████████████| 430/430 [00:05<00:00, 73.87it/s] 


creating index...
index created!
creating index...
index created!
creating index...
index created!


⏳ Ours (Tetris Only): 100%|██████████████████████████████| 430/430 [00:05<00:00, 80.34it/s] 


creating index...
index created!
creating index...
index created!
creating index...
index created!


⏳ Ours (DAHI + Tetris): 100%|██████████████████████████████| 430/430 [00:31<00:00, 13.77it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.18s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=10.22s).
Accumulating evaluation results...
DONE (t=0.47s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=3.25s).
Accumulating evaluation results...
DONE (t=0.13s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.02s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=7.08s).
Accumulating evaluation results...
DONE (t=0.40s).

🏆 [Ablation Study] Single Model & Official COCO Protocol Evaluation (Total 5000 Images) 🏆
Method                           | Type | Img  | mAP    | AP50   | APs    | APm    | APl    | Inf Cnt | Inf Time  | Tot Time 
-----------------------------------------------


📸 [Thesis Visual] Spatial Context Extension (SCE) 효과 증명 (Before vs After) 샘플


In [ ]:
import cv2
import os
import time
import numpy as np
import tqdm
import torch
from ultralytics import YOLO

import contextlib
import io
import random
import matplotlib.pyplot as plt

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# =========================================================
# ⚙️ 하이퍼파라미터 (Hyperparameters)
# =========================================================
MODEL_MAIN_PATH = 'model/best_small.pt'

CONF_GLOBAL = 0.3
CONF_FILTER = 0.1     
CONF_DENSE = 0.3
CONF_TETRIS = 0.3
CONF_UC = 0.3

DENSE_RATIO_THRESH = 0.30  
NMS_CONF_THRESH = 0.3
NMS_IOU_THRESH = 0.4    

DENSE_WINDOW_SIZE = 512
DENSE_STEP = 320      

MERGE_PAD = 16
CROP_PAD_LARGE = 80     
CROP_PAD_SMALL = 16
CROP_PAD_THRESH = 200

CANVAS_SIZE = 960
CANVAS_MARGIN = 2
CANVAS_BG_COLOR = 114

UPSCALE_RATIO = 1.5        
UPSCALE_MAX_THRESH = 200    

NUM_TEST_IMAGES = 5000
HR_THRESHOLD = 1920 * 1080 

# =========================================================
dataset_root = 'data/valid'
img_dir, lbl_dir = os.path.join(dataset_root, 'images'), os.path.join(dataset_root, 'labels')
img_list = sorted(os.listdir(img_dir))[:NUM_TEST_IMAGES]

print(f"🚀 [Ablation Study] 단계적 성능 검증 파이프라인 시작! (Total {len(img_list)} images)")

m2 = YOLO(MODEL_MAIN_PATH)
sce_vis_samples = []

def get_size_category(w, h):
    area = w * h
    if area < 32 ** 2: return 'small'
    elif area < 96 ** 2: return 'medium'
    else: return 'large'

def merge_clusters_dynamic(boxes, img_w, img_h, merge_pad=MERGE_PAD):
    if not len(boxes): return []
    def get_padded(b, pad): return [max(0, b[0]-pad), max(0, b[1]-pad), min(img_w, b[2]+pad), min(img_h, b[3]+pad)]
    def is_overlap(b1, b2):
        p1, p2 = get_padded(b1, merge_pad), get_padded(b2, merge_pad)
        return (min(p1[2], p2[2]) > max(p1[0], p2[0])) and (min(p1[3], p2[3]) > max(p1[1], p2[1]))
    curr = boxes.copy()
    while True:
        merged, flags = [], [False]*len(curr)
        for i in range(len(curr)):
            if flags[i]: continue
            b = curr[i]
            for j in range(i+1, len(curr)):
                if not flags[j] and is_overlap(b, curr[j]):
                    b = [min(b[0], curr[j][0]), min(b[1], curr[j][1]), max(b[2], curr[j][2]), max(b[3], curr[j][3])]
                    flags[j] = True
            merged.append(b)
        if len(merged) == len(curr): break
        curr = merged
    final_boxes = []
    for b in curr:
        bw, bh = b[2] - b[0], b[3] - b[1]
        crop_pad = CROP_PAD_LARGE if max(bw, bh) < CROP_PAD_THRESH else CROP_PAD_SMALL 
        final_boxes.append(get_padded(b, crop_pad))
    return final_boxes

def run_official_ablation_benchmark(method_name):
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        
    all_gts = {}; all_preds = []
    img_infos = {} 
    
    stats = {
        'ALL': {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'HR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'LR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()}
    }

    pbar = tqdm.tqdm(img_list, desc=f"⏳ {method_name}", bar_format='{l_bar}{bar:30}{r_bar}')
    for img_idx, img_name in enumerate(pbar):
        img_path, lbl_path = os.path.join(img_dir, img_name), os.path.join(lbl_dir, img_name.replace('.jpg', '.txt'))
        img = cv2.imread(img_path); h, w, _ = img.shape
        
        img_infos[img_idx] = {'w': w, 'h': h, 'name': img_name}
        
        gts = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    c, xc, yc, bw, bh = map(float, line.split())
                    gts.append([int(c), (xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h]) 
        all_gts[img_idx] = gts

        t_pipe_start = time.time()
        img_inf_time, img_inf_cnt = 0, 0
        
        # =====================================================================
        # 1. Baseline: UC (비활성화) / Ours DAHI Only (비활성화) / Ours Tetris Only (비활성화)
        # =====================================================================
        if method_name in ["UC (2x2 Uniform Crop)", "Ours (DAHI Only)", "Ours (Tetris Only)"]:
            pass # (사용자 기존 코드대로 빠른 실험을 위해 주석/패스 처리)

        # =====================================================================
        # 💡 [논문 성능 증명용 3단계 Ablation Study 모듈]
        # =====================================================================
        elif method_name in ["Ours (DAHI + Tetris: Naive)", "Ours (SBSI + Tetris)", "Ours (SBSI + Tetris + SCE)"]:
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            roi_boxes = []
            
            t_inf_start = time.time()
            res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global_all[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = float(b.conf[0])
                if conf >= CONF_GLOBAL:
                    global_final_boxes.append([bx1, by1, bx2, by2])
                    global_final_scores.append(min(1.0, conf * 1.10))
                    global_final_classes.append(int(b.cls[0]))
                roi_boxes.append([bx1, by1, bx2, by2, conf]) 
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            # -----------------------------------------------------------------
            # 💡 단계 1 & 2: 라우팅 (Routing) 모듈 분기
            # -----------------------------------------------------------------
            if method_name == "Ours (DAHI + Tetris: Naive)":
                # [Phase 1] 기존 단순 격자 스캔 + 카운트 기반 (Naive DAHI)
                while len(remaining_boxes) > 0:
                    best_count, best_region = -1, None
                    for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                        for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                            count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                            if count > best_count: 
                                best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                    if best_region and best_count >= 1:
                        dense_regions.append(best_region)
                        dx1, dy1, dx2, dy2 = best_region
                        remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                    else: break
            else:
                # [Phase 2 & 3] SBSI (KDE 기반 가중치 탐색 + 무게중심 스내핑)
                small_info = []
                for b in remaining_boxes:
                    bx1, by1, bx2, by2, conf = b 
                    w_box, h_box = bx2 - bx1, by2 - by1
                    if get_size_category(w_box, h_box) == 'small':
                        cx, cy = (bx1 + bx2) / 2, (by1 + by2) / 2
                        area = w_box * h_box
                        weight = (1.0 - conf) * np.sqrt(area)
                        small_info.append([cx, cy, weight])
                
                total_small_objs = len(small_info)
                route_threshold = max(1, int(total_small_objs * DENSE_RATIO_THRESH)) 
                
                if total_small_objs > 0:
                    pts = np.array([[info[0], info[1]] for info in small_info]) 
                    weights = np.array([info[2] for info in small_info])
                    norm_weights = weights / (np.mean(weights) + 1e-6)
                    sigma = DENSE_WINDOW_SIZE / 4.0 
                    
                    while len(pts) > 0:
                        dist_sq = np.sum((pts[:, None, :] - pts[None, :, :]) ** 2, axis=-1) 
                        kernel = np.exp(-dist_sq / (2 * sigma ** 2)) 
                        densities = kernel @ norm_weights 
                        
                        best_idx = np.argmax(densities)
                        influence = norm_weights * kernel[best_idx]
                        sum_influence = np.sum(influence)
                        
                        if sum_influence > 0:
                            cx = np.sum(influence * pts[:, 0]) / sum_influence
                            cy = np.sum(influence * pts[:, 1]) / sum_influence
                        else:
                            cx, cy = pts[best_idx]
                            
                        dx1 = int(max(0, cx - DENSE_WINDOW_SIZE / 2))
                        dy1 = int(max(0, cy - DENSE_WINDOW_SIZE / 2))
                        dx2 = int(min(w, dx1 + DENSE_WINDOW_SIZE))
                        dy2 = int(min(h, dy1 + DENSE_WINDOW_SIZE))
                        
                        if dx2 - dx1 < DENSE_WINDOW_SIZE: dx1 = max(0, dx2 - DENSE_WINDOW_SIZE)
                        if dy2 - dy1 < DENSE_WINDOW_SIZE: dy1 = max(0, dy2 - DENSE_WINDOW_SIZE)
                        
                        actual_in_window = (pts[:, 0] >= dx1) & (pts[:, 0] <= dx2) & (pts[:, 1] >= dy1) & (pts[:, 1] <= dy2)
                        actual_count = np.sum(actual_in_window)
                        
                        if actual_count >= route_threshold:
                            dense_regions.append((dx1, dy1, dx2, dy2))
                            mask = ~actual_in_window
                            pts = pts[mask]
                            norm_weights = norm_weights[mask]
                            remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                        else:
                            break
            
            # 라우팅된 영역 이미지 수집
            unified_infer_list = []
            dense_idx_list = []
            for dx1, dy1, dx2, dy2 in dense_regions:
                unified_infer_list.append(img[dy1:dy2, dx1:dx2])
                dense_idx_list.append((len(unified_infer_list) - 1, dx1, dy1, dx2, dy2))

            # -----------------------------------------------------------------
            # 💡 단계 3: 패킹 및 공간 문맥 확장 (SCE) 모듈 분기
            # -----------------------------------------------------------------
            canvases, canvas_infos = [], []
            canvas_start_idx = -1
            if len(remaining_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(remaining_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                
                pre_sce_vis_list = [] 
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: 
                        if len(pre_sce_vis_list) < 3: pre_sce_vis_list.append(cv2.cvtColor(current_canvas, cv2.COLOR_BGR2RGB))
                        canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: 
                    if len(pre_sce_vis_list) < 3: pre_sce_vis_list.append(cv2.cvtColor(current_canvas, cv2.COLOR_BGR2RGB))
                    canvases.append(current_canvas)
                
                # 오직 SCE가 포함된 모델에서만 확장 수행
                if method_name == "Ours (SBSI + Tetris + SCE)":
                    def get_distribution(gap, avail1, avail2):
                        half = gap // 2
                        if avail1 < half: return avail1, min(gap - avail1, avail2)
                        elif avail2 < half: return min(gap - avail2, avail1), avail2
                        else: return half, gap - half

                    for c_idx, canvas in enumerate(canvases):
                        c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                        if not c_infos: continue
                        unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                        
                        for i, cy1 in enumerate(unique_cy1s):
                            row_items = [info for info in c_infos if info['cy1'] == cy1]
                            row_items.sort(key=lambda x: x['cx1'])
                            next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                            
                            for j, info in enumerate(row_items):
                                cx1, cy1 = info['cx1'], info['cy1']
                                cx2, cy2 = info['cx2'], info['cy2']
                                ox, oy, s = info['ox'], info['oy'], info['scale']
                                item_w, item_h = cx2 - cx1, cy2 - cy1
                                org_w, org_h = max(1, int(item_w / s)), max(1, int(item_h / s))
                                
                                next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                                alloc_w = next_cx1 - cx1
                                if j + 1 < len(row_items): alloc_w -= CANVAS_MARGIN
                                
                                alloc_h = next_cy1 - cy1
                                if i + 1 < len(unique_cy1s): alloc_h -= CANVAS_MARGIN
                                
                                gap_w = max(0, alloc_w - item_w)
                                gap_h = max(0, alloc_h - item_h)
                                
                                if gap_w > 0 or gap_h > 0:
                                    gap_w_org = int(gap_w / s)
                                    gap_h_org = int(gap_h / s)
                                    
                                    ext_left, ext_right = get_distribution(gap_w_org, ox, w - (ox + org_w))
                                    ext_top, ext_bottom = get_distribution(gap_h_org, oy, h - (oy + org_h))
                                    
                                    new_ox = ox - ext_left
                                    new_oy = oy - ext_top
                                    new_org_w = org_w + ext_left + ext_right
                                    new_org_h = org_h + ext_top + ext_bottom
                                    
                                    if new_org_w > 0 and new_org_h > 0:
                                        ext_crop = img[new_oy:new_oy+new_org_h, new_ox:new_ox+new_org_w]
                                        actual_w = int(new_org_w * s) if s > 1.0 else new_org_w
                                        actual_h = int(new_org_h * s) if s > 1.0 else new_org_h
                                        
                                        if ext_crop.size > 0:
                                            if s > 1.0: 
                                                ext_crop = cv2.resize(ext_crop, (actual_w, actual_h), interpolation=cv2.INTER_CUBIC)
                                            h_crop, w_crop = ext_crop.shape[:2]
                                            y_end = min(cy1 + h_crop, CANVAS_SIZE)
                                            x_end = min(cx1 + w_crop, CANVAS_SIZE)
                                            
                                            canvas[cy1:y_end, cx1:x_end] = ext_crop[:y_end-cy1, :x_end-cx1]
                                            info['ox'] = new_ox
                                            info['oy'] = new_oy
                                            info['cx2'] = cx1 + (x_end - cx1) 
                                            info['cy2'] = cy1 + (y_end - cy1) 

                    canvases = [c for i, c in enumerate(canvases) if any(info['c_idx'] == i for info in canvas_infos)]
                    for new_idx, old_idx in enumerate(sorted(list(set(info['c_idx'] for info in canvas_infos)))):
                        for info in canvas_infos:
                            if info['c_idx'] == old_idx: info['c_idx'] = new_idx

                    # 시각화 데이터 수집
                    if len(sce_vis_samples) < 5 and len(canvases) > 0:
                        random_idx = random.randrange(len(canvases))
                        if random_idx < len(pre_sce_vis_list):
                            pre_canvas = pre_sce_vis_list[random_idx]
                            post_canvas_rgb = cv2.cvtColor(canvases[random_idx], cv2.COLOR_BGR2RGB)
                            sce_vis_samples.append((img_name, pre_canvas, post_canvas_rgb))

                if len(canvases) > 0:
                    canvas_start_idx = len(unified_infer_list)
                    unified_infer_list.extend(canvases)

            # -----------------------------------------------------------------
            # 💡 공통: 추론 및 NMS
            # -----------------------------------------------------------------
            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for d_idx, dx1, dy1, dx2, dy2 in dense_idx_list:
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1; res_dense = res_all[d_idx]
                    for b in res_dense.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                
                if canvas_start_idx != -1:
                    res_pack = res_all[canvas_start_idx:]
                    for c_idx, res in enumerate(res_pack):
                        for b in res.boxes:
                            bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                            bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                            for info in canvas_infos:
                                if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                    if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                    s = info['scale']
                                    orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                    orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                    local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                    break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # 통계 저장
        img_total_time = time.time() - t_pipe_start
        is_hr = (w * h >= HR_THRESHOLD)
        target_keys = ['ALL', 'HR'] if is_hr else ['ALL', 'LR']
        for k in target_keys:
            stats[k]['count'] += 1
            stats[k]['inf_time'] += img_inf_time
            stats[k]['total_time'] += img_total_time
            stats[k]['inf_cnt'] += img_inf_cnt
            stats[k]['indices'].add(img_idx)

    # ---------------------------------------------------------
    # 💡 공식 COCO API 연산 엔진
    # ---------------------------------------------------------
    def calc_official_coco_metrics(subset_indices):
        if not subset_indices: return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}
        
        gt_dict = {"images": [], "annotations": [], "categories": []}
        for i in range(10): gt_dict["categories"].append({"id": i, "name": f"class_{i}"})
            
        ann_id = 1
        for img_idx in subset_indices:
            info = img_infos[img_idx]
            gt_dict["images"].append({"id": img_idx, "width": info['w'], "height": info['h'], "file_name": info['name']})
            for gt in all_gts[img_idx]:
                c, x1, y1, x2, y2 = gt
                bw, bh = x2 - x1, y2 - y1
                gt_dict["annotations"].append({"id": ann_id, "image_id": img_idx, "category_id": int(c), "bbox": [x1, y1, bw, bh], "area": bw * bh, "iscrowd": 0})
                ann_id += 1

        cocoGt = COCO()
        cocoGt.dataset = gt_dict
        cocoGt.createIndex()

        sub_preds = [p for p in all_preds if p[0] in subset_indices]
        pred_list = []
        for pred in sub_preds:
            img_idx, c, score, x1, y1, x2, y2 = pred
            bw, bh = x2 - x1, y2 - y1
            pred_list.append({"image_id": img_idx, "category_id": int(c), "bbox": [x1, y1, bw, bh], "score": float(score)})

        if not pred_list: return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}

        cocoDt = cocoGt.loadRes(pred_list)

        cocoEval = COCOeval(cocoGt, cocoDt, 'bbox')
        cocoEval.params.maxDets = [100, 300, 500] 
        
        cocoEval.evaluate()
        cocoEval.accumulate()
        
        with contextlib.redirect_stdout(io.StringIO()):
            cocoEval.summarize()

        if len(cocoEval.stats) < 12:
            return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0}

        return {
            "AP50:95": cocoEval.stats[0],
            "AP50": cocoEval.stats[1],
            "AP_small": cocoEval.stats[3],  
            "AP_medium": cocoEval.stats[4], 
            "AP_large": cocoEval.stats[5],
        }

    result_dict = {}
    for group in ['ALL', 'HR', 'LR']:
        c = stats[group]['count']
        res = calc_official_coco_metrics(stats[group]['indices'])
        res['Img_Cnt'] = c
        res['Avg_Inf_Cnt'] = stats[group]['inf_cnt'] / c if c else 0
        res['Avg_Inf_Time'] = (stats[group]['inf_time'] / c) * 1000 if c else 0
        res['Avg_Tot_Time'] = (stats[group]['total_time'] / c) * 1000 if c else 0
        result_dict[group] = res
        
    result_dict['Peak_VRAM'] = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0
    return result_dict

# =========================================================
# 실행 및 다중 표 그리기 (Official COCO Protocol)
# =========================================================
# 💡 [핵심] Ablation Study를 위한 3단계 세분화 모델 평가 목록
methods = [
    # "UC (2x2 Uniform Crop)", 
    # "Ours (DAHI Only)",
    # "Ours (Tetris Only)",
    "Ours (DAHI + Tetris: Naive)", # Step 1: 베이스라인 조합
    "Ours (SBSI + Tetris)",        # Step 2: 라우팅 고도화
    "Ours (SBSI + Tetris + SCE)"   # Step 3: 패킹 고도화 (최종 SOTA)
]

final_stats = {}
for m in methods: 
    final_stats[m] = run_official_ablation_benchmark(m)

print("\n" + "="*145)
print(f"🏆 [Ablation Study] 단계적 성능 검증 (Total {NUM_TEST_IMAGES} Images) 🏆")
print("="*145)
print(f"{'Method':<32} | {'Type':<4} | {'Img':<4} | {'mAP':<6} | {'AP50':<6} | {'APs':<6} | {'APm':<6} | {'APl':<6} | {'Inf Cnt':<7} | {'Inf Time':<9} | {'Tot Time':<9}")
print("-" * 145)

for m, groups in final_stats.items():
    for g in ['ALL', 'HR', 'LR']:
        s = groups[g]
        if s['Img_Cnt'] == 0: continue
        
        mAP  = s.get('AP50:95', 0.0)
        ap50 = s.get('AP50', 0.0)
        aps  = s.get('AP_small', 0.0)
        apm  = s.get('AP_medium', 0.0)
        apl  = s.get('AP_large', 0.0)
        
        print(f"{m if g == 'ALL' else '':<32} | {g:<4} | {s['Img_Cnt']:<4} | {mAP:.4f} | {ap50:.4f} | {aps:.4f} | {apm:.4f} | {apl:.4f} | {s['Avg_Inf_Cnt']:4.1f} /i | {s['Avg_Inf_Time']:5.1f} ms | {s['Avg_Tot_Time']:5.1f} ms")
    
    print(f"{'':<32} > Peak VRAM: {groups.get('Peak_VRAM', 0.0):.1f} MB")
    print("-" * 145)


# =========================================================
# 💡 [NEW Thesis Visual] SCE 적용 전/후 캔버스 비교 시각화
# =========================================================
if len(sce_vis_samples) > 0:
    print("\n" + "="*145)
    print("📸 [Thesis Visual] Spatial Context Extension (SCE) 효과 증명 (Before vs After) 샘플")
    print("="*145)
    
    sample_count = len(sce_vis_samples)
    fig, axes = plt.subplots(sample_count, 2, figsize=(20, 10 * sample_count))
    plt.rcParams['font.family'] = 'sans-serif'
    
    if sample_count == 1: axes = [axes]

    for i, (img_name, canvas_pre, canvas_post) in enumerate(sce_vis_samples):
        axes[i][0].imshow(canvas_pre)
        axes[i][0].set_title(f"Sample {i+1} (a): Pure Tetris Packing (Baseline)\nSource: {img_name}", fontsize=14)
        axes[i][0].axis('off')
        
        axes[i][1].imshow(canvas_post)
        axes[i][1].set_title(f"Sample {i+1} (b): Our Omnidirectional SCE Canvas\n(Seamless Real-pixel Context Restored)", fontsize=14, fontweight='bold')
        axes[i][1].axis('off')
        
        for ax in axes[i]:
             for spine in ax.spines.values(): spine.set_visible(True)

    plt.tight_layout(pad=3.0)
    plt.show()

🚀 [Ablation Study] 단계적 성능 검증 파이프라인 시작! (Total 1294 images)


⏳ Ours (DAHI + Tetris: Naive): 100%|██████████████████████████████| 1294/1294 [02:23<00:00,  9.01it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.24s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=35.12s).
Accumulating evaluation results...
DONE (t=1.59s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.14s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=7.73s).
Accumulating evaluation results...
DONE (t=0.31s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.20s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=26.91s).
Accumulating evaluation results...
DONE (t=1.20s).


⏳ Ours (SBSI + Tetris): 100%|██████████████████████████████| 1294/1294 [01:27<00:00, 14.84it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=34.00s).
Accumulating evaluation results...
DONE (t=1.44s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=7.15s).
Accumulating evaluation results...
DONE (t=0.28s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.04s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=26.53s).
Accumulating evaluation results...
DONE (t=1.23s).


⏳ Ours (SBSI + Tetris + SCE): 100%|██████████████████████████████| 1294/1294 [01:38<00:00, 13.18it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.06s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=34.03s).
Accumulating evaluation results...
DONE (t=1.45s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=7.50s).
Accumulating evaluation results...
DONE (t=0.29s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.23s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=26.18s).
Accumulating evaluation results...
DONE (t=1.17s).

🏆 [Ablation Study] 단계적 성능 검증 (Total 5000 Images) 🏆
Method                           | Type | Img  | mAP    | AP50   | APs    | APm    | APl    | Inf Cnt | Inf Time  | Tot Time 
-------------------------------------------------------------------------------------

In [ ]:
import cv2
import os
import time
import numpy as np
import tqdm
import torch
from ultralytics import YOLO

import contextlib
import io
import random
import matplotlib.pyplot as plt

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# =========================================================
# ⚙️ 하이퍼파라미터 (Hyperparameters)
# =========================================================
MODEL_MAIN_PATH = 'model/best_small.pt'

CONF_GLOBAL = 0.3
CONF_FILTER = 0.1     
CONF_DENSE = 0.3
CONF_TETRIS = 0.3
CONF_UC = 0.3

DENSE_RATIO_THRESH = 0.30  
NMS_CONF_THRESH = 0.3
NMS_IOU_THRESH = 0.4    

DENSE_WINDOW_SIZE = 512
DENSE_STEP = 320      

MERGE_PAD = 16
CROP_PAD_LARGE = 80     
CROP_PAD_SMALL = 16
CROP_PAD_THRESH = 200

CANVAS_SIZE = 960
CANVAS_MARGIN = 2
CANVAS_BG_COLOR = 114

UPSCALE_RATIO = 1.5        
UPSCALE_MAX_THRESH = 200    

NUM_TEST_IMAGES = 5000
HR_THRESHOLD = 1920 * 1080 

# =========================================================
dataset_root = 'data/valid'
img_dir, lbl_dir = os.path.join(dataset_root, 'images'), os.path.join(dataset_root, 'labels')
img_list = sorted(os.listdir(img_dir))[:NUM_TEST_IMAGES]

print(f"🚀 [Ablation Study] 6단계 종합 성능 검증 파이프라인 시작! (Total {len(img_list)} images)")

m2 = YOLO(MODEL_MAIN_PATH)
sce_vis_samples = []

def get_size_category(w, h):
    area = w * h
    if area < 32 ** 2: return 'small'
    elif area < 96 ** 2: return 'medium'
    else: return 'large'

def merge_clusters_dynamic(boxes, img_w, img_h, merge_pad=MERGE_PAD):
    if not len(boxes): return []
    def get_padded(b, pad): return [max(0, b[0]-pad), max(0, b[1]-pad), min(img_w, b[2]+pad), min(img_h, b[3]+pad)]
    def is_overlap(b1, b2):
        p1, p2 = get_padded(b1, merge_pad), get_padded(b2, merge_pad)
        return (min(p1[2], p2[2]) > max(p1[0], p2[0])) and (min(p1[3], p2[3]) > max(p1[1], p2[1]))
    curr = boxes.copy()
    while True:
        merged, flags = [], [False]*len(curr)
        for i in range(len(curr)):
            if flags[i]: continue
            b = curr[i]
            for j in range(i+1, len(curr)):
                if not flags[j] and is_overlap(b, curr[j]):
                    b = [min(b[0], curr[j][0]), min(b[1], curr[j][1]), max(b[2], curr[j][2]), max(b[3], curr[j][3])]
                    flags[j] = True
            merged.append(b)
        if len(merged) == len(curr): break
        curr = merged
    final_boxes = []
    for b in curr:
        bw, bh = b[2] - b[0], b[3] - b[1]
        crop_pad = CROP_PAD_LARGE if max(bw, bh) < CROP_PAD_THRESH else CROP_PAD_SMALL 
        final_boxes.append(get_padded(b, crop_pad))
    return final_boxes

def run_official_ablation_benchmark(method_name):
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        
    all_gts = {}; all_preds = []
    img_infos = {} 
    
    stats = {
        'ALL': {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'HR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()},
        'LR':  {'count': 0, 'inf_time': 0, 'total_time': 0, 'inf_cnt': 0, 'indices': set()}
    }

    pbar = tqdm.tqdm(img_list, desc=f"⏳ {method_name}", bar_format='{l_bar}{bar:30}{r_bar}')
    for img_idx, img_name in enumerate(pbar):
        img_path, lbl_path = os.path.join(img_dir, img_name), os.path.join(lbl_dir, img_name.replace('.jpg', '.txt'))
        img = cv2.imread(img_path); h, w, _ = img.shape
        
        img_infos[img_idx] = {'w': w, 'h': h, 'name': img_name}
        
        gts = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    c, xc, yc, bw, bh = map(float, line.split())
                    gts.append([int(c), (xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h]) 
        all_gts[img_idx] = gts

        t_pipe_start = time.time()
        img_inf_time, img_inf_cnt = 0, 0
        
        # =====================================================================
        # 1. Baseline: UC (2x2 Uniform Crop)
        # =====================================================================
        if method_name == "UC (2x2 Uniform Crop)":
            ch, cw = h // 2, w // 2
            crops, offsets = [img], [(0, 0)]
            for y in [0, ch]:
                for x in [0, cw]:
                    crops.append(img[y:y+ch, x:x+cw])
                    offsets.append((x, y))
            
            t_inf_start = time.time()
            results2 = m2.predict(crops, conf=CONF_UC, verbose=False, batch=5)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 5 
            
            temp_boxes, temp_scores, temp_classes = [], [], []
            for i, res in enumerate(results2):
                ox, oy = offsets[i]
                for b in res.boxes:
                    bx1 = b.xyxy[0][0].item() + ox
                    by1 = b.xyxy[0][1].item() + oy
                    bx2 = b.xyxy[0][2].item() + ox
                    by2 = b.xyxy[0][3].item() + oy
                    temp_boxes.append([bx1, by1, bx2, by2])
                    temp_scores.append(float(b.conf[0]))
                    temp_classes.append(int(b.cls[0]))
                    
            for c in set(temp_classes):
                c_boxes = [b for j, b in enumerate(temp_boxes) if temp_classes[j] == c]
                c_scores = [s for j, s in enumerate(temp_scores) if temp_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # =====================================================================
        # 2 ~ 6. Ours 제안 기법 통합 분기
        # =====================================================================
        else:
            global_final_boxes, global_final_scores, global_final_classes = [], [], []
            local_boxes, local_scores, local_classes = [], [], []
            roi_boxes = []
            
            t_inf_start = time.time()
            res_global_all = m2.predict(img, conf=CONF_FILTER, verbose=False)
            img_inf_time += (time.time() - t_inf_start); img_inf_cnt += 1
            
            for b in res_global_all[0].boxes:
                bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist())
                conf = float(b.conf[0])
                if conf >= CONF_GLOBAL:
                    global_final_boxes.append([bx1, by1, bx2, by2])
                    global_final_scores.append(min(1.0, conf * 1.10))
                    global_final_classes.append(int(b.cls[0]))
                roi_boxes.append([bx1, by1, bx2, by2, conf])
            
            remaining_boxes = roi_boxes.copy()
            dense_regions = []
            
            # --- [라우팅(Routing) 모듈] ---
            if method_name == "Packing Only":
                pass # 라우팅 생략
                
            elif method_name in ["DAHI Only", "DAHI + Packing"]:
                while len(remaining_boxes) > 0:
                    best_count, best_region = -1, None
                    for y in range(0, h - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                        for x in range(0, w - DENSE_WINDOW_SIZE + 1, DENSE_STEP):
                            count = sum(1 for rb in remaining_boxes if rb[0] >= x and rb[1] >= y and rb[2] <= x + DENSE_WINDOW_SIZE and rb[3] <= y + DENSE_WINDOW_SIZE)
                            if count > best_count: 
                                best_count, best_region = count, (x, y, x + DENSE_WINDOW_SIZE, y + DENSE_WINDOW_SIZE)
                    if best_region and best_count >= 1:
                        dense_regions.append(best_region)
                        dx1, dy1, dx2, dy2 = best_region
                        remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                    else: break
                    
                    # 💡 [DAHI 딱 1회 스캔 조건] DAHI + Packing 조합일 경우 1개만 찾고 루프 종료
                    if method_name == "DAHI + Packing":
                        break
                        
            elif method_name in ["SBSI + Packing", "SBSI + DCRP"]:
                small_info = []
                for b in remaining_boxes:
                    bx1, by1, bx2, by2, conf = b 
                    w_box, h_box = bx2 - bx1, by2 - by1
                    if get_size_category(w_box, h_box) == 'small':
                        cx, cy = (bx1 + bx2) / 2, (by1 + by2) / 2
                        weight = (1.0 - conf) * np.sqrt(w_box * h_box)
                        small_info.append([cx, cy, weight])
                
                total_small_objs = len(small_info)
                route_threshold = max(1, int(total_small_objs * DENSE_RATIO_THRESH)) 
                
                if total_small_objs > 0:
                    pts = np.array([[info[0], info[1]] for info in small_info]) 
                    weights = np.array([info[2] for info in small_info])
                    norm_weights = weights / (np.mean(weights) + 1e-6)
                    sigma = DENSE_WINDOW_SIZE / 4.0 
                    
                    while len(pts) > 0:
                        dist_sq = np.sum((pts[:, None, :] - pts[None, :, :]) ** 2, axis=-1) 
                        kernel = np.exp(-dist_sq / (2 * sigma ** 2)) 
                        densities = kernel @ norm_weights 
                        
                        best_idx = np.argmax(densities)
                        influence = norm_weights * kernel[best_idx]
                        sum_influence = np.sum(influence)
                        
                        if sum_influence > 0:
                            cx = np.sum(influence * pts[:, 0]) / sum_influence
                            cy = np.sum(influence * pts[:, 1]) / sum_influence
                        else:
                            cx, cy = pts[best_idx]
                            
                        dx1 = int(max(0, cx - DENSE_WINDOW_SIZE / 2))
                        dy1 = int(max(0, cy - DENSE_WINDOW_SIZE / 2))
                        dx2 = int(min(w, dx1 + DENSE_WINDOW_SIZE))
                        dy2 = int(min(h, dy1 + DENSE_WINDOW_SIZE))
                        
                        if dx2 - dx1 < DENSE_WINDOW_SIZE: dx1 = max(0, dx2 - DENSE_WINDOW_SIZE)
                        if dy2 - dy1 < DENSE_WINDOW_SIZE: dy1 = max(0, dy2 - DENSE_WINDOW_SIZE)
                        
                        actual_in_window = (pts[:, 0] >= dx1) & (pts[:, 0] <= dx2) & (pts[:, 1] >= dy1) & (pts[:, 1] <= dy2)
                        actual_count = np.sum(actual_in_window)
                        
                        if actual_count >= route_threshold:
                            dense_regions.append((dx1, dy1, dx2, dy2))
                            mask = ~actual_in_window
                            pts = pts[mask]
                            norm_weights = norm_weights[mask]
                            remaining_boxes = [rb for rb in remaining_boxes if not (rb[0] >= dx1 and rb[1] >= dy1 and rb[2] <= dx2 and rb[3] <= dy2)]
                        else: break

            # --- [추론 및 패킹(Packing) 모듈] ---
            unified_infer_list = []
            dense_idx_list = []
            for dx1, dy1, dx2, dy2 in dense_regions:
                unified_infer_list.append(img[dy1:dy2, dx1:dx2])
                dense_idx_list.append((len(unified_infer_list) - 1, dx1, dy1, dx2, dy2))

            canvases, canvas_infos = [], []
            canvas_start_idx = -1
            
            # DAHI Only는 패킹하지 않음
            if method_name != "DAHI Only" and len(remaining_boxes) > 0:
                clustered_boxes = merge_clusters_dynamic(remaining_boxes, w, h, merge_pad=MERGE_PAD)
                crops_to_pack = []
                for cb in clustered_boxes:
                    cx1, cy1, cx2, cy2 = map(int, cb[:4]); cw_org, ch_org = cx2 - cx1, cy2 - cy1
                    scale_ratio = UPSCALE_RATIO if max(cw_org, ch_org) <= UPSCALE_MAX_THRESH else 1.0
                    cw_crop, ch_crop = min(int(cw_org * scale_ratio), CANVAS_SIZE), min(int(ch_org * scale_ratio), CANVAS_SIZE)
                    if cw_crop > 0 and ch_crop > 0:
                        crop_img = img[cy1:cy1+ch_org, cx1:cx1+cw_org]
                        if scale_ratio > 1.0: crop_img = cv2.resize(crop_img, (cw_crop, ch_crop), interpolation=cv2.INTER_CUBIC)
                        else: crop_img = crop_img[:ch_crop, :cw_crop]
                        crops_to_pack.append({'crop': crop_img, 'ox': cx1, 'oy': cy1, 'cw': cw_crop, 'ch': ch_crop, 'scale': scale_ratio})
                
                crops_to_pack.sort(key=lambda x: x['ch'], reverse=True)
                
                current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8)
                cx, cy, max_h = 0, 0, 0
                for item in crops_to_pack:
                    if cx + item['cw'] > CANVAS_SIZE: cx = 0; cy += max_h + CANVAS_MARGIN; max_h = 0
                    if cy + item['ch'] > CANVAS_SIZE: 
                        canvases.append(current_canvas); current_canvas = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), CANVAS_BG_COLOR, dtype=np.uint8); cx, cy, max_h = 0, 0, 0
                    current_canvas[cy:cy+item['ch'], cx:cx+item['cw']] = item['crop']
                    canvas_infos.append({'c_idx': len(canvases), 'cx1': cx, 'cy1': cy, 'cx2': cx+item['cw'], 'cy2': cy+item['ch'], 'ox': item['ox'], 'oy': item['oy'], 'scale': item['scale']})
                    cx += item['cw'] + CANVAS_MARGIN; max_h = max(max_h, item['ch'])
                if max_h > 0 or cx > 0: 
                    canvases.append(current_canvas)
                
                # 💡 오직 SCE 파이프라인에서만 전방위 여백 확장 수행
                if method_name == "SBSI + DCRP":
                    def get_distribution(gap, avail1, avail2):
                        half = gap // 2
                        if avail1 < half: return avail1, min(gap - avail1, avail2)
                        elif avail2 < half: return min(gap - avail2, avail1), avail2
                        else: return half, gap - half

                    for c_idx, canvas in enumerate(canvases):
                        c_infos = [info for info in canvas_infos if info['c_idx'] == c_idx]
                        if not c_infos: continue
                        unique_cy1s = sorted(list(set([info['cy1'] for info in c_infos])))
                        for i, cy1 in enumerate(unique_cy1s):
                            row_items = [info for info in c_infos if info['cy1'] == cy1]
                            row_items.sort(key=lambda x: x['cx1'])
                            next_cy1 = unique_cy1s[i+1] if i + 1 < len(unique_cy1s) else CANVAS_SIZE
                            for j, info in enumerate(row_items):
                                cx1, cy1 = info['cx1'], info['cy1']
                                cx2, cy2 = info['cx2'], info['cy2']
                                ox, oy, s = info['ox'], info['oy'], info['scale']
                                item_w, item_h = cx2 - cx1, cy2 - cy1
                                org_w, org_h = max(1, int(item_w / s)), max(1, int(item_h / s))
                                
                                next_cx1 = row_items[j+1]['cx1'] if j + 1 < len(row_items) else CANVAS_SIZE
                                alloc_w = next_cx1 - cx1
                                if j + 1 < len(row_items): alloc_w -= CANVAS_MARGIN
                                alloc_h = next_cy1 - cy1
                                if i + 1 < len(unique_cy1s): alloc_h -= CANVAS_MARGIN
                                
                                gap_w, gap_h = max(0, alloc_w - item_w), max(0, alloc_h - item_h)
                                
                                if gap_w > 0 or gap_h > 0:
                                    ext_left, ext_right = get_distribution(int(gap_w/s), ox, w - (ox + org_w))
                                    ext_top, ext_bottom = get_distribution(int(gap_h/s), oy, h - (oy + org_h))
                                    
                                    new_ox, new_oy = ox - ext_left, oy - ext_top
                                    new_org_w, new_org_h = org_w + ext_left + ext_right, org_h + ext_top + ext_bottom
                                    
                                    if new_org_w > 0 and new_org_h > 0:
                                        ext_crop = img[new_oy:new_oy+new_org_h, new_ox:new_ox+new_org_w]
                                        actual_w = int(new_org_w * s) if s > 1.0 else new_org_w
                                        actual_h = int(new_org_h * s) if s > 1.0 else new_org_h
                                        
                                        if ext_crop.size > 0:
                                            if s > 1.0: ext_crop = cv2.resize(ext_crop, (actual_w, actual_h), interpolation=cv2.INTER_CUBIC)
                                            h_crop, w_crop = ext_crop.shape[:2]
                                            y_end = min(cy1 + h_crop, CANVAS_SIZE)
                                            x_end = min(cx1 + w_crop, CANVAS_SIZE)
                                            canvas[cy1:y_end, cx1:x_end] = ext_crop[:y_end-cy1, :x_end-cx1]
                                            
                                            info['ox'], info['oy'] = new_ox, new_oy
                                            info['cx2'], info['cy2'] = cx1 + (x_end - cx1), cy1 + (y_end - cy1)

                    canvases = [c for i, c in enumerate(canvases) if any(info['c_idx'] == i for info in canvas_infos)]
                    for new_idx, old_idx in enumerate(sorted(list(set(info['c_idx'] for info in canvas_infos)))):
                        for info in canvas_infos:
                            if info['c_idx'] == old_idx: info['c_idx'] = new_idx

                if len(canvases) > 0:
                    canvas_start_idx = len(unified_infer_list)
                    unified_infer_list.extend(canvases)

            if len(unified_infer_list) > 0:
                t_inf_start = time.time()
                res_all = m2.predict(unified_infer_list, conf=CONF_TETRIS, verbose=False, batch=16)
                img_inf_time += (time.time() - t_inf_start); img_inf_cnt += len(unified_infer_list)
                
                for d_idx, dx1, dy1, dx2, dy2 in dense_idx_list:
                    cw_dense, ch_dense = dx2 - dx1, dy2 - dy1; res_dense = res_all[d_idx]
                    for b in res_dense.boxes:
                        bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                        if bx1 <= 5 or by1 <= 5 or bx2 >= cw_dense - 5 or by2 >= ch_dense - 5: conf *= 0.8 
                        local_boxes.append([bx1+dx1, by1+dy1, bx2+dx1, by2+dy1]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                
                if canvas_start_idx != -1:
                    res_pack = res_all[canvas_start_idx:]
                    for c_idx, res in enumerate(res_pack):
                        for b in res.boxes:
                            bx1, by1, bx2, by2 = map(float, b.xyxy[0].tolist()); conf = float(b.conf[0])
                            bcx, bcy = (bx1+bx2)/2, (by1+by2)/2 
                            for info in canvas_infos:
                                if info['c_idx'] == c_idx and info['cx1'] <= bcx <= info['cx2'] and info['cy1'] <= bcy <= info['cy2']:
                                    if bx1 <= info['cx1'] + 3 or by1 <= info['cy1'] + 3 or bx2 >= info['cx2'] - 3 or by2 >= info['cy2'] - 3: conf *= 0.8
                                    s = info['scale']
                                    orig_x1 = ((bx1 - info['cx1']) / s) + info['ox']; orig_y1 = ((by1 - info['cy1']) / s) + info['oy']
                                    orig_x2 = ((bx2 - info['cx1']) / s) + info['ox']; orig_y2 = ((by2 - info['cy1']) / s) + info['oy']
                                    local_boxes.append([orig_x1, orig_y1, orig_x2, orig_y2]); local_scores.append(conf); local_classes.append(int(b.cls[0]))
                                    break
                                        
            final_local_preds = []
            for c in set(local_classes):
                c_boxes = [b for j, b in enumerate(local_boxes) if local_classes[j] == c]; c_scores = [s for j, s in enumerate(local_scores) if local_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, NMS_IOU_THRESH)
                if len(indices) > 0:
                    for idx in indices.flatten(): final_local_preds.append([c, c_scores[idx]] + c_boxes[idx])

            combined_boxes = global_final_boxes + [p[2:6] for p in final_local_preds]; combined_scores = global_final_scores + [p[1] for p in final_local_preds]; combined_classes = global_final_classes + [p[0] for p in final_local_preds]
            for c in set(combined_classes):
                c_boxes = [b for j, b in enumerate(combined_boxes) if combined_classes[j] == c]; c_scores = [s for j, s in enumerate(combined_scores) if combined_classes[j] == c]
                cv_boxes = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in c_boxes]
                indices = cv2.dnn.NMSBoxes(cv_boxes, c_scores, NMS_CONF_THRESH, 0.45) 
                if len(indices) > 0:
                    for idx in indices.flatten(): all_preds.append([img_idx, c, c_scores[idx]] + c_boxes[idx])

        # 💡 [버그 수정] 누락된 전체 시간 측정 및 HR/LR 그룹 분류 로직 복구
        img_total_time = time.time() - t_pipe_start
        is_hr = (w * h >= HR_THRESHOLD)
        target_keys = ['ALL', 'HR'] if is_hr else ['ALL', 'LR']
        
        for k in target_keys:
            stats[k]['count'] += 1
            stats[k]['inf_time'] += img_inf_time
            stats[k]['total_time'] += img_total_time
            stats[k]['inf_cnt'] += img_inf_cnt
            stats[k]['indices'].add(img_idx)

    # ---------------------------------------------------------
    # 💡 공식 COCO API 연산 엔진 (PR 커브 추출 지원)
    # ---------------------------------------------------------
    def calc_official_coco_metrics(subset_indices):
        if not subset_indices: return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0, "pr_curve": None}
        
        gt_dict = {"images": [], "annotations": [], "categories": []}
        for i in range(10): gt_dict["categories"].append({"id": i, "name": f"class_{i}"})
            
        ann_id = 1
        for img_idx in subset_indices:
            info = img_infos[img_idx]
            gt_dict["images"].append({"id": img_idx, "width": info['w'], "height": info['h'], "file_name": info['name']})
            for gt in all_gts[img_idx]:
                c, x1, y1, x2, y2 = gt
                bw, bh = x2 - x1, y2 - y1
                gt_dict["annotations"].append({"id": ann_id, "image_id": img_idx, "category_id": int(c), "bbox": [x1, y1, bw, bh], "area": bw * bh, "iscrowd": 0})
                ann_id += 1

        cocoGt = COCO()
        cocoGt.dataset = gt_dict
        cocoGt.createIndex()

        sub_preds = [p for p in all_preds if p[0] in subset_indices]
        pred_list = []
        for pred in sub_preds:
            img_idx, c, score, x1, y1, x2, y2 = pred
            bw, bh = x2 - x1, y2 - y1
            pred_list.append({"image_id": img_idx, "category_id": int(c), "bbox": [x1, y1, bw, bh], "score": float(score)})

        if not pred_list: return {"AP50:95": 0, "AP50": 0, "AP_small": 0, "AP_medium": 0, "AP_large": 0, "pr_curve": None}

        cocoDt = cocoGt.loadRes(pred_list)
        cocoEval = COCOeval(cocoGt, cocoDt, 'bbox')
        cocoEval.params.maxDets = [100, 300, 500] 
        cocoEval.evaluate()
        cocoEval.accumulate()
        
        with contextlib.redirect_stdout(io.StringIO()):
            cocoEval.summarize()
            
        pr_curve = None
        if cocoEval.eval is not None:
            # 💡 [핵심] Precision 배열 추출 (IoU=0.5, Area=All, maxDets=500)
            precision = cocoEval.eval['precision'][0, :, :, 0, 2] 
            valid_mask = precision > -1
            pr_curve = np.zeros(101)
            for r_idx in range(101):
                valid_cats = precision[r_idx, valid_mask[r_idx]]
                if len(valid_cats) > 0: pr_curve[r_idx] = np.mean(valid_cats)

        return {
            "AP50:95": cocoEval.stats[0] if len(cocoEval.stats) >= 1 else 0,
            "AP50": cocoEval.stats[1] if len(cocoEval.stats) >= 2 else 0,
            "AP_small": cocoEval.stats[3] if len(cocoEval.stats) >= 4 else 0,  
            "AP_medium": cocoEval.stats[4] if len(cocoEval.stats) >= 5 else 0, 
            "AP_large": cocoEval.stats[5] if len(cocoEval.stats) >= 6 else 0,
            "pr_curve": pr_curve
        }

    # 💡 [버그 수정] HR/LR 딕셔너리 구조 복구
    result_dict = {}
    for group in ['ALL', 'HR', 'LR']:
        c = stats[group]['count']
        res = calc_official_coco_metrics(stats[group]['indices'])
        res['Img_Cnt'] = c
        res['Avg_Inf_Cnt'] = stats[group]['inf_cnt'] / c if c else 0
        res['Avg_Inf_Time'] = (stats[group]['inf_time'] / c) * 1000 if c else 0
        res['Avg_Tot_Time'] = (stats[group]['total_time'] / c) * 1000 if c else 0
        result_dict[group] = res
        
    result_dict['Peak_VRAM'] = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0
    return result_dict

# =========================================================
# 실행 및 다중 표 그리기 (Official COCO Protocol)
# =========================================================
methods = [
    "UC (2x2 Uniform Crop)", 
    "DAHI Only",
    "Packing Only",
    "DAHI + Packing", 
    "SBSI + Packing",        
    "SBSI + DCRP"   
]

final_stats = {}
for m in methods: 
    final_stats[m] = run_official_ablation_benchmark(m)

print("\n" + "="*145)
print(f"🏆 [Ablation Study] 6단계 종합 성능 검증 결과 (Total {NUM_TEST_IMAGES} Images) 🏆")
print("="*145)
print(f"{'Method':<32} | {'Type':<4} | {'Img':<4} | {'mAP':<6} | {'AP50':<6} | {'APs':<6} | {'APm':<6} | {'APl':<6} | {'Inf Cnt':<7} | {'Inf Time':<9} | {'Tot Time':<9}")
print("-" * 145)

for m in methods:
    groups = final_stats[m]
    for g in ['ALL', 'HR', 'LR']:
        s = groups[g]
        if s['Img_Cnt'] == 0: continue
        
        mAP  = s.get('AP50:95', 0.0)
        ap50 = s.get('AP50', 0.0)
        aps  = s.get('AP_small', 0.0)
        apm  = s.get('AP_medium', 0.0)
        apl  = s.get('AP_large', 0.0)
        
        print(f"{m if g == 'ALL' else '':<32} | {g:<4} | {s['Img_Cnt']:<4} | {mAP:.4f} | {ap50:.4f} | {aps:.4f} | {apm:.4f} | {apl:.4f} | {s['Avg_Inf_Cnt']:4.1f} /i | {s['Avg_Inf_Time']:5.1f} ms | {s['Avg_Tot_Time']:5.1f} ms")
    
    print(f"{'':<32} > Peak VRAM: {groups.get('Peak_VRAM', 0.0):.1f} MB")
    print("-" * 145)


# =========================================================
# 📊 [논문용 시각화] 6단계 성능 증명 종합 그래프 렌더링
# =========================================================
print("\n🎨 [Thesis Visual] 6단계 성능 증명 그래프 렌더링 시작...")
plt.rcParams['font.family'] = 'sans-serif'
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(24, 7))

# 논문 스타일 마커 및 색상
colors = ['#7f7f7f', '#17becf', '#e377c2', '#ff7f0e', '#2ca02c', '#d62728']
markers = ['o', 'v', '^', 's', 'D', '*']

# 1. Precision-Recall Curve (IoU=0.5)
recall_vals = np.linspace(0.0, 1.0, 101)
for idx, m in enumerate(methods):
    # 💡 딕셔너리 구조 복구에 따른 'ALL' 키 접근
    pr = final_stats[m]['ALL'].get('pr_curve') 
    if pr is not None:
        ax1.plot(recall_vals, pr, label=m, color=colors[idx], linewidth=2.5)

ax1.set_title('(a) Precision-Recall Curve @ IoU=0.50', fontsize=16, fontweight='bold')
ax1.set_xlabel('Recall', fontsize=14)
ax1.set_ylabel('Precision', fontsize=14)
ax1.grid(True, linestyle='--', alpha=0.7)
ax1.legend(loc='lower left', fontsize=11)
ax1.set_xlim([0.0, 1.0])
ax1.set_ylim([0.0, 1.05])

# 2. 핵심 지표 (AP50:95, APs, APl) 막대 그래프 비교
metrics_to_plot = ['AP50:95', 'AP_small', 'AP_large']
x = np.arange(len(metrics_to_plot))
width = 0.12

for idx, m in enumerate(methods):
    vals = [final_stats[m]['ALL'][metric] for metric in metrics_to_plot]
    bars = ax2.bar(x + idx*width - (width*5/2), vals, width, label=m, color=colors[idx], alpha=0.9)

ax2.set_title('(b) Average Precision (AP) Comparison', fontsize=16, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(['mAP (50:95)', 'AP (Small)', 'AP (Large)'], fontsize=14)
ax2.set_ylabel('Score', fontsize=14)
ax2.grid(axis='y', linestyle='--', alpha=0.7)
ax2.legend(loc='upper left', fontsize=10)

# 3. 효율성 vs 성능 (Inf Time vs mAP) 분산 그래프
for idx, m in enumerate(methods):
    inf_time = final_stats[m]['ALL']['Avg_Inf_Time']
    map_score = final_stats[m]['ALL']['AP50:95']
    ax3.scatter(inf_time, map_score, color=colors[idx], s=300, marker=markers[idx], label=m, edgecolor='black', zorder=5)
    
ax3.set_title('(c) Efficiency vs. Accuracy Trade-off', fontsize=16, fontweight='bold')
ax3.set_xlabel('Average Inference Time (ms)', fontsize=14)
ax3.set_ylabel('mAP (50:95)', fontsize=14)
ax3.grid(True, linestyle='--', alpha=0.7)
ax3.legend(loc='lower right', fontsize=11)

plt.tight_layout()
plt.show()

# =========================================================
# 💡 [NEW Thesis Visual] SCE 적용 전/후 캔버스 비교 시각화
# =========================================================
if len(sce_vis_samples) > 0:
    print("\n" + "="*145)
    print("📸 [Thesis Visual] Spatial Context Extension (SCE) 효과 증명 (Before vs After) 샘플")
    print("="*145)
    
    sample_count = len(sce_vis_samples)
    fig, axes = plt.subplots(sample_count, 2, figsize=(20, 10 * sample_count))
    plt.rcParams['font.family'] = 'sans-serif'
    
    if sample_count == 1: axes = [axes]

    for i, (img_name, canvas_pre, canvas_post) in enumerate(sce_vis_samples):
        axes[i][0].imshow(canvas_pre)
        axes[i][0].set_title(f"Sample {i+1} (a): Pure Tetris Packing (Baseline)\nSource: {img_name}", fontsize=14)
        axes[i][0].axis('off')
        
        axes[i][1].imshow(canvas_post)
        axes[i][1].set_title(f"Sample {i+1} (b): Our Omnidirectional SCE Canvas\n(Seamless Real-pixel Context Restored)", fontsize=14, fontweight='bold')
        axes[i][1].axis('off')
        
        for ax in axes[i]:
             for spine in ax.spines.values(): spine.set_visible(True)

    plt.tight_layout(pad=3.0)
    plt.show()

# =========================================================
# 📊 [NEW Thesis Visual] 연산 효율성 (속도 & VRAM) 비교 막대그래프
# =========================================================
print("\n🎨 [Thesis Visual] 연산 효율성(VRAM & 속도) 막대그래프 렌더링 시작...")

fig4, (ax_time, ax_vram) = plt.subplots(1, 2, figsize=(18, 7))
plt.rcParams['font.family'] = 'sans-serif'

x = np.arange(len(methods))
width = 0.35

# 긴 Method 이름을 그래프 X축에 맞게 축약
short_names = [m.replace("Ours (", "").replace(")", "").replace("2x2 Uniform Crop", "UC 2x2") for m in methods]

# ---------------------------------------------------------
# 1. 속도 비교 막대그래프 (Inf Time vs Total Time)
# ---------------------------------------------------------
inf_times = [final_stats[m]['ALL']['Avg_Inf_Time'] for m in methods]
tot_times = [final_stats[m]['ALL']['Avg_Tot_Time'] for m in methods]

# 막대 그리기
bars_inf = ax_time.bar(x - width/2, inf_times, width, label='Inference Time', color='#1f77b4', edgecolor='black', alpha=0.85)
bars_tot = ax_time.bar(x + width/2, tot_times, width, label='Total Pipeline Time', color='#aec7e8', edgecolor='black', alpha=0.85)

ax_time.set_title('(a) Processing Time Overhead Analysis', fontsize=16, fontweight='bold')
ax_time.set_xticks(x)
ax_time.set_xticklabels(short_names, rotation=20, ha='right', fontsize=13)
ax_time.set_ylabel('Time (ms)', fontsize=14)
ax_time.grid(axis='y', linestyle='--', alpha=0.7)
ax_time.legend(loc='upper left', fontsize=12)

# 막대 위에 값(ms) 표기
for bar in bars_tot:
    yval = bar.get_height()
    if yval > 0:
        ax_time.text(bar.get_x() + bar.get_width()/2, yval + (max(tot_times)*0.02), f'{yval:.1f}', 
                     ha='center', va='bottom', fontsize=11, fontweight='bold', color='#333333')

# ---------------------------------------------------------
# 2. Peak VRAM 비교 막대그래프
# ---------------------------------------------------------
vrams = [final_stats[m].get('Peak_VRAM', 0) for m in methods]

# 막대 그리기 (주황색 테마)
bars_vram = ax_vram.bar(x, vrams, width*1.2, color='#ff7f0e', edgecolor='black', alpha=0.85)

ax_vram.set_title('(b) Peak VRAM Consumption', fontsize=16, fontweight='bold')
ax_vram.set_xticks(x)
ax_vram.set_xticklabels(short_names, rotation=20, ha='right', fontsize=13)
ax_vram.set_ylabel('Memory (MB)', fontsize=14)
ax_vram.grid(axis='y', linestyle='--', alpha=0.7)

# 막대 위에 값(MB) 표기
for bar in bars_vram:
    yval = bar.get_height()
    if yval > 0:
        ax_vram.text(bar.get_x() + bar.get_width()/2, yval + (max(vrams)*0.02), f'{yval:.1f}', 
                     ha='center', va='bottom', fontsize=11, fontweight='bold', color='#333333')

# 그래프 간격 조절 및 출력
plt.tight_layout(pad=3.0)
plt.show()

🚀 [Ablation Study] 6단계 종합 성능 검증 파이프라인 시작! (Total 1294 images)


⏳ UC (2x2 Uniform Crop): 100%|██████████████████████████████| 1294/1294 [01:40<00:00, 12.93it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.25s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=32.49s).
Accumulating evaluation results...
DONE (t=1.39s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=6.70s).
Accumulating evaluation results...
DONE (t=0.26s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.22s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=25.44s).
Accumulating evaluation results...
DONE (t=1.15s).


⏳ DAHI Only: 100%|██████████████████████████████| 1294/1294 [01:37<00:00, 13.26it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=35.11s).
Accumulating evaluation results...
DONE (t=1.56s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=8.14s).
Accumulating evaluation results...
DONE (t=0.31s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=28.01s).
Accumulating evaluation results...
DONE (t=1.28s).


...

⏳ SBSI + Packing: 100%|██████████████████████████████| 1294/1294 [01:40<00:00, 12.91it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.26s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=37.62s).
Accumulating evaluation results...
DONE (t=1.71s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=8.17s).
Accumulating evaluation results...
DONE (t=0.35s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.25s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=29.26s).
Accumulating evaluation results...
DONE (t=1.32s).


⏳ SBSI + DCRP: 100%|██████████████████████████████| 1294/1294 [01:41<00:00, 12.81it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=34.04s).
Accumulating evaluation results...
DONE (t=1.54s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=7.49s).
Accumulating evaluation results...
DONE (t=0.30s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.04s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=26.46s).
Accumulating evaluation results...
DONE (t=1.23s).

🏆 [Ablation Study] 6단계 종합 성능 검증 결과 (Total 5000 Images) 🏆
Method                           | Type | Img  | mAP    | AP50   | APs    | APm    | APl    | Inf Cnt | Inf Time  | Tot Time 
-------------------------------------------------------------------------------


🎨 [Thesis Visual] 연산 효율성(VRAM & 속도) 막대그래프 렌더링 시작...
